# Sanskrit Cross-Domain POS Tagging — Complete Ordered Notebook

**Consolidated source:** original experiment notebook + optimized mapping workbench + latest JQ review cells.

## Recommended execution order
1. **Setup and dataset preparation** (first section; creates the data structures needed by mapping).
2. **Occurrence-level source-tag mapping** (PPR, JQ and the other family reviews; then the safe mapping workbench and JQ review queue).
3. **Model experiments** (BiLSTM-CRF and evaluation cells) only after mapping is finalized and the approved occurrence-level labels have been integrated into the source training sequences.

**Important:** Do not use *Runtime → Run all*. Run the notebook in order and inspect each section's outputs. The mapping workbench's optional commit cell requires explicit approval. Unresolved cases should remain unresolved rather than receiving guessed labels.

**Important research safeguard:** The existing data-preparation cell builds `source_train` before the occurrence-level review workflow. Changes in `manual_review` do not automatically update `source_train` or its token labels. Before running model training, rebuild/integrate the source sequences from the approved occurrence-level mappings and verify that the model inputs actually contain those labels. The locked 30% target test split must remain untouched.

**Gemini note:** The three supplied files do not contain Gemini API configuration or Gemini-driven mapping code. This notebook consolidates the supplied research cells; it does not add a Gemini integration that was not present in the source files.


# Part A — Environment, datasets, splits, and training examples

Run this part first. It establishes imports, Drive paths, corpus data, tag inventories, leakage-resistant target splits, and source/target sequence structures required by the mapping workflow.

# Sanskrit Cross-Domain POS Tagging
## Dataset preparation → BiLSTM-CRF → XLM-RoBERTa → IndicBERT v2
### Runtime: Google Colab, GPU runtime, approximately 8 GB system RAM.

This notebook implements a reproducible, conservative partial-mapping experiment:
- Literature corpus keeps its original tag strings; only explicitly mapped source tags are used as source supervision.
- News corpus uses the 24-tag target inventory.
- Unmapped source tags are excluded from the 24-tag source loss, not guessed.
- A document/sentence-level news split is made once and reused by all models.
- Each model is trained on source data first, evaluated zero-shot on news, then adapted with nested few-shot news subsets.

# 0. Colab setup
1. In Colab choose Runtime → Change runtime type → T4 GPU (or another available GPU).
2. Upload both CSV files, or mount Google Drive and set paths below.
3. Run cells from top to bottom. Do not run model cells before completing dataset checks.

In [5]:
import torch
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Check your Colab runtime configuration.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [6]:
!pip -q install "transformers>=4.44,<5" datasets accelerate seqeval pytorch-crf sentencepiece scikit-learn pandas numpy matplotlib seaborn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
import torch
import transformers
import datasets
import sklearn
import pandas
import numpy
import seqeval
import torchcrf

print("All required packages imported successfully!")
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

All required packages imported successfully!
Transformers: 4.57.6
PyTorch: 2.11.0+cu130


In [8]:
import os
import re
import json
import random
import gc
import warnings
import unicodedata

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup
)

from torchcrf import CRF

# --------------------------------------------------
# 1. Experiment configuration
# --------------------------------------------------

SEED = 42

def seed_everything(seed=SEED):
    """Set random seeds for reproducible experiments."""
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Improve reproducibility on CUDA where supported
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# --------------------------------------------------
# 2. Device configuration
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("EXPERIMENT ENVIRONMENT")
print("=" * 50)

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    )

    print(f"GPU memory: {gpu_memory:.2f} GB")

    torch.cuda.empty_cache()
else:
    print("GPU unavailable. Running on CPU.")

print("=" * 50)
print("Environment initialization completed.")
print("=" * 50)


EXPERIMENT ENVIRONMENT
PyTorch version: 2.11.0+cu130
NumPy version: 2.1.3
Pandas version: 2.3.3
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB
Environment initialization completed.


In [2]:
import os
import shutil
import zipfile
from pathlib import Path

PROJECT_NAME = "Cross_Domain"

# --------------------------------------------------
# 1. Environment Detection (Mutually Exclusive)
# --------------------------------------------------
is_colab = Path("/content").is_dir()
is_kaggle = not is_colab and Path("/kaggle/working").is_dir()

cwd = Path.cwd().resolve()

if is_colab:
    PROJECT_DIR = Path("/content/Cross_Domain")
    print("Google Colab environment detected. Project root:", PROJECT_DIR)
elif is_kaggle:
    PROJECT_DIR = Path("/kaggle/working/Cross_Domain")
    print("Kaggle environment detected. Project root:", PROJECT_DIR)
elif cwd.name == PROJECT_NAME:
    PROJECT_DIR = cwd
elif cwd.parent.name == PROJECT_NAME:
    PROJECT_DIR = cwd.parent
else:
    candidates = [
        cwd,
        cwd.parent if cwd.name in ["notebooks", "src", "data", "outputs", "configs"] else cwd,
        Path.home() / "Desktop" / "Cross_Domain",
        Path("/home/badshah/Desktop/Cross_Domain").resolve(),
        Path("/root/Desktop/Cross_Domain"),
    ]
    try:
        for u in Path("/home").glob("*"):
            candidates.append(u / "Desktop" / "Cross_Domain")
    except Exception:
        pass
    valid = next((c for c in candidates if (c / "data" / "raw" / "Literature_Corpus.csv").is_file()), None)
    if valid is not None:
        PROJECT_DIR = valid
    else:
        PROJECT_DIR = Path.home() / "Desktop" / PROJECT_NAME
    print("Local environment detected. Project root:", PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATASET_OUTPUT_DIR = OUTPUT_DIR / "datasets"
MAPPING_REVIEW_DIR = OUTPUT_DIR / "mapping_reviews"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "models"
BILSTM_OUTPUT_DIR = MODEL_OUTPUT_DIR / "bilstm_crf"
XLMR_OUTPUT_DIR = MODEL_OUTPUT_DIR / "xlm_roberta"
INDICBERT_OUTPUT_DIR = MODEL_OUTPUT_DIR / "indicbert"
EVALUATION_DIR = OUTPUT_DIR / "evaluations"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
EXPERIMENT_DIR = OUTPUT_DIR / "experiments"

for directory in (
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUT_DIR,
    DATASET_OUTPUT_DIR,
    MAPPING_REVIEW_DIR,
    MODEL_OUTPUT_DIR,
    BILSTM_OUTPUT_DIR,
    XLMR_OUTPUT_DIR,
    INDICBERT_OUTPUT_DIR,
    EVALUATION_DIR,
    FIGURE_DIR,
    LOG_DIR,
    EXPERIMENT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = MAPPING_REVIEW_DIR
OUT_DIR = MAPPING_REVIEW_DIR
SOURCE_FILE = RAW_DATA_DIR / "Literature_Corpus.csv"
TARGET_FILE = RAW_DATA_DIR / "News_Annotated.csv"
UNLABELED_NEWS_FILE = RAW_DATA_DIR / "News_Unannotated.csv"

# --------------------------------------------------
# 2. Check local directories first if on local machine
# --------------------------------------------------
if not SOURCE_FILE.is_file():
    search_roots = [
        Path("/home/badshah/Desktop/Cross_Domain"),
        *Path("/home").glob("*/Desktop/Cross_Domain"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for sroot in search_roots:
        s_raw = sroot / "data" / "raw"
        if (s_raw / "Literature_Corpus.csv").is_file():
            print(f"Copying missing datasets from {s_raw} -> {RAW_DATA_DIR}...")
            for f in s_raw.glob("*.csv"):
                shutil.copy2(f, RAW_DATA_DIR / f.name)
            for sub in ["outputs/datasets", "configs", "src"]:
                if (sroot / sub).is_dir():
                    shutil.copytree(sroot / sub, PROJECT_DIR / sub, dirs_exist_ok=True)
            break

# --------------------------------------------------
# 3. If running on Kaggle and attached under /kaggle/input, copy them
# --------------------------------------------------
if is_kaggle and not SOURCE_FILE.is_file():
    for f in Path("/kaggle/input").rglob("*"):
        if f.is_file() and f.name in ["Literature_Corpus.csv", "News_Annotated.csv", "News_Unannotated.csv"]:
            shutil.copy2(f, RAW_DATA_DIR / f.name)

# --------------------------------------------------
# 4. Universal Cloud Auto-Download: Fetch dataset directly into PROJECT_DIR
# --------------------------------------------------
if not SOURCE_FILE.is_file():
    print(f"Cloud environment detected! Fetching project datasets directly into {PROJECT_DIR}...")
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_c595dea7dbc740b795edcc95f32ed945"
    k_dir = Path.home() / ".kaggle"
    k_dir.mkdir(parents=True, exist_ok=True)
    (k_dir / "access_token").write_text("KGAT_c595dea7dbc740b795edcc95f32ed945", encoding="utf-8")
    try:
        (k_dir / "access_token").chmod(0o600)
    except Exception:
        pass
    os.system(f"pip install -q kaggle && kaggle datasets download -d badshahkumar/sanskrit-pos-crossdomain --unzip -p {PROJECT_DIR}")
    for b_src, b_dst in [
        (PROJECT_DIR / "kaggle_bundle/outputs/datasets", DATASET_OUTPUT_DIR),
        (PROJECT_DIR / "kaggle_bundle/configs", PROJECT_DIR / "configs"),
        (PROJECT_DIR / "kaggle_bundle/src", PROJECT_DIR / "src"),
        (PROJECT_DIR / "kaggle_bundle/data/raw", RAW_DATA_DIR),
        (PROJECT_DIR / "outputs/datasets", DATASET_OUTPUT_DIR),
    ]:
        if b_src.is_dir() and b_src != b_dst:
            shutil.copytree(b_src, b_dst, dirs_exist_ok=True)

print("Local project:", PROJECT_DIR)
print("Source dataset:", SOURCE_FILE, SOURCE_FILE.is_file())
print("Target dataset:", TARGET_FILE, TARGET_FILE.is_file())
print("Unannotated news:", UNLABELED_NEWS_FILE, UNLABELED_NEWS_FILE.is_file())



RuntimeError: Could not locate the Cross_Domain project. Set PROJECT_DIR explicitly to the directory containing data/raw, outputs, src, and configs.

# 1. Configure dataset paths and column names
Expected source (literature) CSV columns from the corpus discussed earlier:
sentence_id, word_number, word, lemma, tag, sentence

Expected target (news) CSV: one token per row, with a sentence/document identifier, token, and gold POS tag. Edit the column names below to match your annotated file. If your target file uses a different structure, adapt this section before proceeding.

In [ ]:

# --------------------------------------------------
# 1. Dataset paths
# --------------------------------------------------

SOURCE_CSV = str(RAW_DATA_DIR / "Literature_Corpus.csv")
TARGET_CSV = str(RAW_DATA_DIR / "News_Annotated.csv")

# --------------------------------------------------
# 2. Source corpus configuration
# --------------------------------------------------

SRC_SENT_ID = "sentence_id"
SRC_TOKEN = "word"
SRC_TAG = "tag"
SRC_DOC_ID = None

# --------------------------------------------------
# 3. Target corpus configuration
# --------------------------------------------------

TGT_SENT_ID = "sentence_id"
TGT_TOKEN = "token"
TGT_TAG = "gold_sanskrit_tag"
TGT_DOC_ID = "doc_id"

TARGET_IS_SENTENCE_TEXT = False
TGT_SENTENCE_TEXT = "sentence_text"
TGT_TAG_SEQUENCE = None

# --------------------------------------------------
# 4. Check files
# --------------------------------------------------

assert os.path.exists(SOURCE_CSV), (
    f"Source CSV not found: {SOURCE_CSV}"
)

assert os.path.exists(TARGET_CSV), (
    f"Target CSV not found: {TARGET_CSV}"
)

# --------------------------------------------------
# 5. Load datasets
# --------------------------------------------------

source_raw = pd.read_csv(SOURCE_CSV)
target_raw = pd.read_csv(TARGET_CSV)

# --------------------------------------------------
# 6. Validate required columns
# --------------------------------------------------

source_required = [
    SRC_SENT_ID,
    SRC_TOKEN,
    SRC_TAG
]

target_required = [
    TGT_SENT_ID,
    TGT_TOKEN,
    TGT_TAG,
    TGT_DOC_ID,
    "token_id",
    "annotation_status"
]

missing_source = [
    col for col in source_required
    if col not in source_raw.columns
]

missing_target = [
    col for col in target_required
    if col not in target_raw.columns
]

assert not missing_source, (
    f"Missing source columns: {missing_source}"
)

assert not missing_target, (
    f"Missing target columns: {missing_target}"
)

# --------------------------------------------------
# 7. Dataset overview
# --------------------------------------------------

print("=" * 60)
print("DATASET LOADING SUMMARY")
print("=" * 60)

print("\nSource corpus:")
print("Shape:", source_raw.shape)
print("Columns:", source_raw.columns.tolist())

print("\nTarget corpus:")
print("Shape:", target_raw.shape)
print("Columns:", target_raw.columns.tolist())

print("\nTarget annotation status:")
print(target_raw["annotation_status"].value_counts(dropna=False))

print("\nUnique target documents:", target_raw["doc_id"].nunique())
print("Unique target sentences:",
      target_raw[["doc_id", "sentence_id"]].drop_duplicates().shape[0])

print("=" * 60)

display(source_raw.head())
display(target_raw.head())

# 2. Define the target tagset and conservative source→target mapping
The target inventory contains 24 labels. The mapping below includes only high-confidence correspondences as an initial working hypothesis. Validate it with the Sanskrit POS guidelines and a domain expert. In particular, do not automatically map ambiguous particle, demonstrative, participle, abbreviation, or compound subtypes.

In [ ]:

# --------------------------------------------------
# 1. Target POS tagset
# --------------------------------------------------

TARGET_TAGS = [
    "N", "NS", "NV", "KNV", "SN", "SNV", "SNS",
    "SAM", "SAMY", "SAMP", "KP", "UP", "AVN",
    "AVP", "AVUP", "SAMB", "AVU", "K", "AKV",
    "AB", "AVC", "DVI", "A", "PU"
]

TAG2ID = {
    tag: idx for idx, tag in enumerate(TARGET_TAGS)
}

ID2TAG = {
    idx: tag for tag, idx in TAG2ID.items()
}

assert len(TARGET_TAGS) == 24
assert len(TAG2ID) == len(TARGET_TAGS)
assert len(ID2TAG) == len(TARGET_TAGS)

# --------------------------------------------------
# 2. Source POS mapping
# --------------------------------------------------

# Prefix mappings
SOURCE_PREFIX_TO_TARGET = {
    "NC": "N",
    "NP": "NS",
    "JJ": "A",
    "PU": "PU",
    "PPR": "SN",
    "PRF": "SN",
    "PRC": "SN",
    "PRL": "SN",
    "PWH": "SN",
    "CIN": "AVU",
    "CCD": "SAMB",
    "CSB": "SAMB",
}

# Exact mappings
SOURCE_EXACT_TO_TARGET = {
    "V": "KP"
}

# --------------------------------------------------
# 3. Mapping validation
# --------------------------------------------------

assert all(
    tag in TARGET_TAGS
    for tag in SOURCE_PREFIX_TO_TARGET.values()
)

assert all(
    tag in TARGET_TAGS
    for tag in SOURCE_EXACT_TO_TARGET.values()
)

print("Target POS tags:", len(TARGET_TAGS))
print("Source prefix mappings:", len(SOURCE_PREFIX_TO_TARGET))
print("Source exact mappings:", len(SOURCE_EXACT_TO_TARGET))

print("\nTarget tag IDs:")
print(TAG2ID)

In [ ]:
print("News dataset columns:")
print(target_raw.columns.tolist())

display(target_raw.head(10))

In [ ]:

# ============================================
# DATASET PREPARATION
# ============================================

# --------------------------------------------
# 1. Text cleaning
# --------------------------------------------

def clean_text(x):
    if pd.isna(x):
        return ""
    return unicodedata.normalize("NFC", str(x)).strip()


# --------------------------------------------
# 2. Source POS tag mapping
# --------------------------------------------

def source_prefix(tag):
    s = clean_text(tag).upper()
    match = re.match(r"^[A-Z]+", s)
    return match.group(0) if match else s


def map_source_tag(tag):
    """
    Map source POS tags to the target Sanskrit tagset.

    Exact mappings are checked first.
    Prefix mappings are checked longest-first.
    V is deliberately handled as an exact mapping only.
    """
    prefix = source_prefix(tag)

    if prefix in SOURCE_EXACT_TO_TARGET:
        return SOURCE_EXACT_TO_TARGET[prefix]

    for source_prefix_key in sorted(
        SOURCE_PREFIX_TO_TARGET,
        key=len,
        reverse=True
    ):
        if prefix.startswith(source_prefix_key):
            return SOURCE_PREFIX_TO_TARGET[source_prefix_key]

    return None


# --------------------------------------------
# 3. Convert token rows into sentence sequences
# --------------------------------------------

def make_sequences(
    df,
    sent_col,
    token_col,
    tag_col,
    doc_col=None,
    order_col=None
):

    required = [sent_col, token_col, tag_col]

    if doc_col is not None:
        required.append(doc_col)

    if order_col is not None:
        required.append(order_col)

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    work = df.copy()

    # Clean sentence and token values
    work[sent_col] = work[sent_col].map(clean_text)
    work[token_col] = work[token_col].map(clean_text)
    work[tag_col] = work[tag_col].map(clean_text)

    # Clean document IDs if available
    if doc_col is not None:
        work[doc_col] = work[doc_col].map(clean_text)

    # Remove rows without sentence IDs or tokens.
    # Do NOT remove rows with empty source tags:
    # their tokens should remain in sentence context.
    valid_rows = (
        (work[sent_col] != "") &
        (work[token_col] != "")
    )

    if doc_col is not None:
        valid_rows &= work[doc_col] != ""

    work = work[valid_rows].copy()

    # Preserve original token order
    if order_col is not None:
        work["__order"] = pd.to_numeric(
            work[order_col],
            errors="coerce"
        )

        if work["__order"].isna().any():
            raise ValueError(
                f"Missing or non-numeric values found in {order_col}."
            )

        group_cols = (
            [doc_col, sent_col]
            if doc_col is not None
            else [sent_col]
        )

        # Stable sort preserves original order for ties
        work = work.sort_values(
            group_cols + ["__order"],
            kind="mergesort"
        )

    # Group tokens into sentences
    group_cols = (
        [doc_col, sent_col]
        if doc_col is not None
        else [sent_col]
    )

    rows = []

    for _, group in work.groupby(
        group_cols,
        sort=False,
        dropna=False
    ):

        tokens = group[token_col].tolist()
        tags = group[tag_col].tolist()

        if not tokens:
            continue

        document_id = (
            group[doc_col].iloc[0]
            if doc_col is not None
            else group[sent_col].iloc[0]
        )

        rows.append({
            "sentence_id": group[sent_col].iloc[0],
            "doc_id": document_id,
            "tokens": tokens,
            "tags": tags
        })

    return pd.DataFrame(
        rows,
        columns=["sentence_id", "doc_id", "tokens", "tags"]
    )


# --------------------------------------------
# 4. Prepare source/literature dataset
# --------------------------------------------

src = make_sequences(
    source_raw,
    SRC_SENT_ID,
    SRC_TOKEN,
    SRC_TAG,
    SRC_DOC_ID,
    order_col=(
        "word_number"
        if "word_number" in source_raw.columns
        else None
    )
)

# Preserve original source tags and derive prefixes
src["source_prefixes"] = src["tags"].apply(
    lambda tags: [source_prefix(tag) for tag in tags]
)

# Map source labels while preserving unmapped positions
src["mapped_tags"] = src["tags"].apply(
    lambda tags: [map_source_tag(tag) for tag in tags]
)

# Mapping statistics
src["mapped_count"] = src["mapped_tags"].apply(
    lambda tags: sum(tag is not None for tag in tags)
)

src["unmapped_count"] = src["mapped_tags"].apply(
    lambda tags: sum(tag is None for tag in tags)
)

src["mapped_coverage"] = (
    src["mapped_count"] /
    src["tags"].apply(lambda tags: max(1, len(tags)))
)


# --------------------------------------------
# 5. Prepare target/news dataset
# --------------------------------------------

# Use expert-verified Sanskrit POS labels.
# Never substitute UPOS or Stanza predictions for gold labels.

target_verified = target_raw[
    (
        target_raw["annotation_status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        == "verified"
    )
    &
    target_raw["gold_sanskrit_tag"].notna()
].copy()

print("Total target token rows:", len(target_raw))
print("Verified target token rows:", len(target_verified))
print(
    "Excluded target rows:",
    len(target_raw) - len(target_verified)
)

# Convert verified target tokens into sentence sequences
tgt = make_sequences(
    target_verified,
    TGT_SENT_ID,
    TGT_TOKEN,
    TGT_TAG,
    TGT_DOC_ID,
    order_col="token_id"
)

# Normalize target labels
tgt["tags"] = tgt["tags"].apply(
    lambda tags: [tag.upper().strip() for tag in tags]
)


# --------------------------------------------
# 6. Dataset validation
# --------------------------------------------

print("\n========== DATASET SUMMARY ==========")

print("Source sentences:", len(src))
print("Target verified sentences:", len(tgt))

source_total_tokens = sum(
    len(sequence) for sequence in src["tags"]
)

source_mapped_tokens = int(src["mapped_count"].sum())

source_mapping_coverage = (
    source_mapped_tokens / max(1, source_total_tokens)
)

print("\nSource tokens:", source_total_tokens)
print("Mapped source tokens:", source_mapped_tokens)
print("Unmapped source tokens:", int(src["unmapped_count"].sum()))

print(
    "Source mapping coverage:",
    round(source_mapping_coverage, 4)
)

print("\nTarget documents:", tgt["doc_id"].nunique())

print(
    "Target sentences:",
    tgt[["doc_id", "sentence_id"]].drop_duplicates().shape[0]
)

# Target label validation
target_observed_tags = {
    tag
    for sequence in tgt["tags"]
    for tag in sequence
}

unknown_tags = sorted(
    target_observed_tags - set(TARGET_TAGS)
)

missing_target_tags = sorted(
    set(TARGET_TAGS) - target_observed_tags
)

print("\nObserved target tags:", sorted(target_observed_tags))
print("Unknown target tags:", unknown_tags)
print("Target tags absent from dataset:", missing_target_tags)

# Token/tag alignment
assert all(
    len(row.tokens) == len(row.tags)
    for row in tgt.itertuples()
), "Target token/tag alignment error!"

assert all(
    len(row.tokens) == len(row.mapped_tags)
    for row in src.itertuples()
), "Source token/mapped-tag alignment error!"

assert len(src) > 0, "No source sentences were prepared."
assert len(tgt) > 0, "No verified target sentences were prepared."

assert not unknown_tags, (
    f"Unknown target tags found: {unknown_tags}. "
    "Check the annotation scheme before training."
)

assert source_total_tokens > 0
assert source_mapped_tokens > 0, (
    "No source tags were mapped. Check source tag mappings."
)

print("\nDataset preparation completed successfully!")

display(src.head())
display(tgt.head())

In [ ]:

# Literature/source dataset
SRC_SENT_ID = "sentence_id"
SRC_TOKEN = "word"
SRC_TAG = "tag"
SRC_DOC_ID = None

# News/target dataset
TGT_SENT_ID = "sentence_id"
TGT_TOKEN = "token"
TGT_TAG = "gold_sanskrit_tag"
TGT_DOC_ID = "doc_id"

In [ ]:

# ============================================
# SOURCE MAPPING AUDIT & TARGET TAG DISTRIBUTION
# ============================================

# --------------------------------------------
# 1. Source tag-prefix distribution
# --------------------------------------------

source_counts = Counter(
    prefix
    for sequence in src["source_prefixes"]
    for prefix in sequence
)

mapping_audit = pd.DataFrame([
    {
        "source_prefix": prefix,
        "tokens": count,
        "mapped_target": map_source_tag(prefix),
        "mapping_status": (
            "Mapped"
            if map_source_tag(prefix) is not None
            else "Unmapped"
        )
    }
    for prefix, count in source_counts.most_common()
])

print("SOURCE TAG MAPPING AUDIT")
display(mapping_audit)

# --------------------------------------------
# 2. Mapping coverage by source prefix
# --------------------------------------------

total_source_tokens = sum(source_counts.values())

mapped_source_tokens = sum(
    count
    for prefix, count in source_counts.items()
    if map_source_tag(prefix) is not None
)

unmapped_source_tokens = total_source_tokens - mapped_source_tokens

print("\nSOURCE MAPPING SUMMARY")
print("Total source tokens:", total_source_tokens)
print("Mapped source tokens:", mapped_source_tokens)
print("Unmapped source tokens:", unmapped_source_tokens)

print(
    "Mapping coverage:",
    round(mapped_source_tokens / max(1, total_source_tokens) * 100, 2),
    "%"
)

# --------------------------------------------
# 3. Target POS tag distribution
# --------------------------------------------

target_counts = Counter(
    tag
    for sequence in tgt["tags"]
    for tag in sequence
)

target_distribution = pd.DataFrame({
    "tag": TARGET_TAGS,
    "count": [
        target_counts.get(tag, 0)
        for tag in TARGET_TAGS
    ]
})

target_distribution["percentage"] = (
    target_distribution["count"] /
    max(1, target_distribution["count"].sum()) * 100
)

target_distribution = target_distribution.sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("\nTARGET SANSKRIT POS TAG DISTRIBUTION")
display(target_distribution)

# --------------------------------------------
# 4. Check missing target classes
# --------------------------------------------

missing_classes = target_distribution[
    target_distribution["count"] == 0
]["tag"].tolist()

print("\nTarget classes with zero occurrences:", missing_classes)

# 3. Validate sequences and create leakage-resistant splits
Splitting unit is doc_id when available; otherwise sentence ID is used as the grouping unit. If many sentences come from the same article/document, set TGT_DOC_ID to that document column before splitting. The target test set is held out once and never used for mapping, training, or hyperparameter selection.

In [ ]:

# ============================================
# NEWS DATASET VALIDATION AND SPLITTING
# ============================================

# --------------------------------------------
# 1. Validate target sequences
# --------------------------------------------

assert len(tgt) > 0, "Target dataset is empty."

bad = tgt[
    tgt.apply(
        lambda r: (
            len(r["tokens"]) != len(r["tags"])
            or len(r["tokens"]) == 0
        ),
        axis=1
    )
]

assert len(bad) == 0, (
    f"{len(bad)} malformed target sequences found."
)

# --------------------------------------------
# 2. Validate target labels
# --------------------------------------------

unknown = sorted(
    set(tag for seq in tgt["tags"] for tag in seq)
    - set(TARGET_TAGS)
)

assert not unknown, (
    f"Unknown target tags: {unknown}. "
    "Resolve these before training."
)

# --------------------------------------------
# 3. Validate document IDs
# --------------------------------------------

tgt = tgt.copy()

tgt["doc_id"] = tgt["doc_id"].astype(str).str.strip()

assert (
    tgt["doc_id"].notna().all()
    and (tgt["doc_id"] != "").all()
), "Missing or empty document IDs found."

groups = np.array(sorted(tgt["doc_id"].unique()))

print("Unique documents:", len(groups))

assert len(groups) >= 5, (
    "At least 5 distinct documents are recommended "
    "for the requested train/validation/test split."
)

# --------------------------------------------
# 4. Split by document
# --------------------------------------------

# First split: 70% train + validation, 30% test

trainval_groups, test_groups = train_test_split(
    groups,
    test_size=0.30,
    random_state=SEED
)

# Second split: 80% train, 20% validation
# Final approximate ratio: 56% / 14% / 30%

train_groups, val_groups = train_test_split(
    trainval_groups,
    test_size=0.20,
    random_state=SEED
)

# --------------------------------------------
# 5. Create datasets
# --------------------------------------------

news_train_pool = (
    tgt[tgt["doc_id"].isin(train_groups)]
    .reset_index(drop=True)
)

news_val = (
    tgt[tgt["doc_id"].isin(val_groups)]
    .reset_index(drop=True)
)

news_test = (
    tgt[tgt["doc_id"].isin(test_groups)]
    .reset_index(drop=True)
)

# --------------------------------------------
# 6. Verify document-level separation
# --------------------------------------------

train_docs = set(news_train_pool["doc_id"])
val_docs = set(news_val["doc_id"])
test_docs = set(news_test["doc_id"])

assert train_docs.isdisjoint(val_docs)
assert train_docs.isdisjoint(test_docs)
assert val_docs.isdisjoint(test_docs)

assert len(news_train_pool) > 0
assert len(news_val) > 0
assert len(news_test) > 0

assert (
    len(news_train_pool) + len(news_val) + len(news_test)
    == len(tgt)
), "Some target sentences were lost during splitting."

# --------------------------------------------
# 7. Save split document IDs
# --------------------------------------------

split_ids = {
    "seed": SEED,
    "train_documents": sorted(train_docs),
    "validation_documents": sorted(val_docs),
    "test_documents": sorted(test_docs)
}

split_file = RESULTS_DIR / "news_document_split.json"

with open(split_file, "w", encoding="utf-8") as f:
    json.dump(split_ids, f, indent=4, ensure_ascii=False)

# --------------------------------------------
# 8. Display split statistics
# --------------------------------------------

total = len(tgt)

print("\n========== DATASET SPLIT SUMMARY ==========")

for name, frame, docs in [
    ("Training pool", news_train_pool, train_docs),
    ("Validation", news_val, val_docs),
    ("Testing", news_test, test_docs)
]:
    print(f"\n{name}:")
    print("Sentences:", len(frame))
    print("Documents:", len(docs))
    print("Percentage:", round(len(frame) / total * 100, 2), "%")
    print("Tokens:", sum(len(x) for x in frame["tokens"]))

print("\nDocument leakage check: PASSED")
print("Target label validation: PASSED")
print("Token/tag alignment: PASSED")
print("All sentences accounted for: PASSED")

print("\nDocument split saved to:", split_file)

# --------------------------------------------
# 9. Inspect POS class distribution per split
# --------------------------------------------

def get_tag_counts(frame):
    return Counter(
        tag
        for sequence in frame["tags"]
        for tag in sequence
    )

class_distribution = pd.DataFrame({
    "tag": TARGET_TAGS,
    "train": [
        get_tag_counts(news_train_pool).get(tag, 0)
        for tag in TARGET_TAGS
    ],
    "validation": [
        get_tag_counts(news_val).get(tag, 0)
        for tag in TARGET_TAGS
    ],
    "test": [
        get_tag_counts(news_test).get(tag, 0)
        for tag in TARGET_TAGS
    ]
})

print("\nPOS CLASS DISTRIBUTION ACROSS SPLITS")
display(class_distribution)

In [ ]:

# ============================================
# SAVE PREPARED DATASETS AND EXPERIMENT CONFIG
# ============================================

import json
from pathlib import Path

# --------------------------------------------
# 1. Verify Google Drive and output directory
# --------------------------------------------

# Reuse RESULTS_DIR defined in Cell 3
OUT_DIR = RESULTS_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)

# --------------------------------------------
# 2. JSON saving utility
# --------------------------------------------

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=4
        )

# --------------------------------------------
# 3. Save prepared datasets
# --------------------------------------------

prepared_datasets = {
    "source_prepared.pkl": src,
    "target_prepared.pkl": tgt,
    "news_train_pool.pkl": news_train_pool,
    "news_validation.pkl": news_val,
    "news_test_locked.pkl": news_test
}

for filename, dataframe in prepared_datasets.items():
    dataframe.to_pickle(OUT_DIR / filename)
    print("Saved:", filename)

# --------------------------------------------
# 4. Calculate source mapping statistics
# --------------------------------------------

total_source_tokens = sum(
    len(sequence) for sequence in src["tags"]
)

mapped_source_tokens = int(src["mapped_count"].sum())

unmapped_source_tokens = (
    total_source_tokens - mapped_source_tokens
)

mapping_coverage = (
    mapped_source_tokens / max(1, total_source_tokens)
)

# --------------------------------------------
# 5. Calculate target split statistics
# --------------------------------------------

def count_tokens(frame):
    return sum(len(sequence) for sequence in frame["tokens"])

# --------------------------------------------
# 6. Save experiment configuration
# --------------------------------------------

experiment_config = {
    "experiment_name": "Cross-Domain Generalization and Few-Shot Adaptation for Sanskrit POS Tagging",

    "seed": int(SEED),

    "target_tags": TARGET_TAGS,
    "number_of_target_tags": len(TARGET_TAGS),

    "source_prefix_to_target": SOURCE_PREFIX_TO_TARGET,
    "source_exact_to_target": SOURCE_EXACT_TO_TARGET,

    "source_csv": SOURCE_CSV,
    "target_csv": TARGET_CSV,

    "source_sentences": len(src),
    "target_verified_sentences": len(tgt),

    "source_total_tokens": total_source_tokens,
    "source_mapped_tokens": mapped_source_tokens,
    "source_unmapped_tokens": unmapped_source_tokens,
    "source_mapping_coverage": mapping_coverage,

    "train_sentences": len(news_train_pool),
    "validation_sentences": len(news_val),
    "test_sentences": len(news_test),

    "train_documents": len(train_docs),
    "validation_documents": len(val_docs),
    "test_documents": len(test_docs),

    "train_tokens": count_tokens(news_train_pool),
    "validation_tokens": count_tokens(news_val),
    "test_tokens": count_tokens(news_test),

    "split_method": "document_level",
    "train_percentage_approx": 56,
    "validation_percentage_approx": 14,
    "test_percentage_approx": 30,

    "test_size": 0.30,
    "validation_size_of_trainval": 0.20,

    "train_document_ids": sorted(train_docs),
    "validation_document_ids": sorted(val_docs),
    "test_document_ids": sorted(test_docs)
}

save_json(
    experiment_config,
    OUT_DIR / "experiment_config.json"
)

# --------------------------------------------
# 7. Save target tag distribution
# --------------------------------------------

target_tag_counts = Counter(
    tag
    for sequence in tgt["tags"]
    for tag in sequence
)

tag_distribution = pd.DataFrame({
    "tag": TARGET_TAGS,
    "count": [
        target_tag_counts.get(tag, 0)
        for tag in TARGET_TAGS
    ]
})

tag_distribution["percentage"] = (
    tag_distribution["count"] /
    max(1, tag_distribution["count"].sum()) * 100
)

tag_distribution.to_csv(
    OUT_DIR / "target_tag_distribution.csv",
    index=False
)

# --------------------------------------------
# 8. Verify saved files
# --------------------------------------------

print("\n========== SAVED FILES ==========")

for file in sorted(OUT_DIR.iterdir()):
    if file.is_file():
        print(
            file.name,
            "-",
            round(file.stat().st_size / 1024, 2),
            "KB"
        )

print("\n========== EXPERIMENT SUMMARY ==========")

print("Source sentences:", len(src))
print("Target sentences:", len(tgt))
print("Source mapping coverage:", round(mapping_coverage * 100, 2), "%")

print("Training sentences:", len(news_train_pool))
print("Validation sentences:", len(news_val))
print("Testing sentences:", len(news_test))

print("\nAll prepared datasets and experiment configuration saved successfully!")

# 4. Build source and target training examples
For source supervision, unmapped source tokens are removed from the sequence so they do not contribute to loss. This can create discontinuous token sequences; the model learns only from mapped source tokens. The original source labels remain preserved in the prepared dataframe for auditability.

Few-shot subsets are nested (50 ⊂ 100 ⊂ 200 ⊂ 500 ⊂ 1,000 sentences) and sampled from the news training pool. The same sampled subsets will be reused for all three models.

In [ ]:

# ============================================
# SOURCE SUPERVISION + FEW-SHOT PREPARATION
# ============================================

# --------------------------------------------
# 1. Prepare source supervised sequences
# --------------------------------------------

def source_supervised_rows(frame):

    rows = []

    for r in frame.itertuples():

        # Preserve every original token.
        # None means the source POS label is unmapped.
        mapped_tags = [
            tag if tag in TAG2ID else None
            for tag in r.mapped_tags
        ]

        if any(tag is not None for tag in mapped_tags):

            rows.append({
                "tokens": list(r.tokens),
                "tags": mapped_tags,
                "sentence_id": r.sentence_id,
                "doc_id": r.doc_id
            })

    return pd.DataFrame(
        rows,
        columns=["tokens", "tags", "sentence_id", "doc_id"]
    )


source_train_all = source_supervised_rows(src)

print("Source supervised sequences:", len(source_train_all))

print(
    "Source total tokens:",
    sum(len(x) for x in source_train_all["tokens"])
)

print(
    "Source mapped tokens:",
    sum(
        sum(tag is not None for tag in sequence)
        for sequence in source_train_all["tags"]
    )
)

print(
    "Source unmapped tokens:",
    sum(
        sum(tag is None for tag in sequence)
        for sequence in source_train_all["tags"]
    )
)

# --------------------------------------------
# 2. Validate source data
# --------------------------------------------

assert len(source_train_all) > 0, (
    "No source sequences contain mapped tags. "
    "Check source POS mappings."
)

assert all(
    len(row.tokens) == len(row.tags)
    for row in source_train_all.itertuples()
), "Source token/tag alignment error."

assert source_train_all["doc_id"].notna().all(), (
    "Missing source document IDs."
)

source_groups = (
    source_train_all["doc_id"]
    .astype(str)
    .unique().tolist()
)

assert len(source_groups) >= 2, (
    "At least two source groups are required."
)

# --------------------------------------------
# 3. Split source data into train/validation
# --------------------------------------------

src_tr_groups, src_va_groups = train_test_split(
    source_groups,
    test_size=0.20,
    random_state=SEED
)

source_train = (
    source_train_all[
        source_train_all["doc_id"].astype(str).isin(src_tr_groups)
    ]
    .reset_index(drop=True)
)

source_val = (
    source_train_all[
        source_train_all["doc_id"].astype(str).isin(src_va_groups)
    ]
    .reset_index(drop=True)
)

# Verify source split separation
source_train_docs = set(source_train["doc_id"].astype(str))
source_val_docs = set(source_val["doc_id"].astype(str))

assert source_train_docs.isdisjoint(source_val_docs)

assert len(source_train) > 0
assert len(source_val) > 0

# --------------------------------------------
# 4. Prepare nested few-shot datasets
# --------------------------------------------

FEWSHOT_BUDGETS = [50, 100, 200, 500, 1000]

assert len(news_train_pool) > 0, (
    "News training pool is empty."
)

# Unique sentence key: document ID + sentence ID
news_train_pool = news_train_pool.copy()

news_train_pool["__sentence_key"] = list(
    zip(
        news_train_pool["doc_id"].astype(str),
        news_train_pool["sentence_id"].astype(str)
    )
)

assert news_train_pool["__sentence_key"].is_unique, (
    "Duplicate document/sentence combinations found."
)

# One shuffled ordering guarantees nested subsets
order = (
    news_train_pool.sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

fewshot_sets = {}

for budget in FEWSHOT_BUDGETS:

    n = min(budget, len(order))

    fewshot_sets[budget] = (
        order.iloc[:n]
        .copy()
        .reset_index(drop=True)
    )

# --------------------------------------------
# 5. Verify few-shot nestedness
# --------------------------------------------

for i in range(len(FEWSHOT_BUDGETS) - 1):

    smaller = FEWSHOT_BUDGETS[i]
    larger = FEWSHOT_BUDGETS[i + 1]

    small_ids = set(
        fewshot_sets[smaller]["__sentence_key"]
    )

    large_ids = set(
        fewshot_sets[larger]["__sentence_key"]
    )

    assert small_ids.issubset(large_ids), (
        f"Few-shot nesting failed: {smaller} vs {larger}"
    )

# --------------------------------------------
# 6. Verify few-shot data does not overlap
#    with validation or test
# --------------------------------------------

train_pool_ids = set(news_train_pool["__sentence_key"])

val_ids = set(
    zip(
        news_val["doc_id"].astype(str),
        news_val["sentence_id"].astype(str)
    )
)

test_ids = set(
    zip(
        news_test["doc_id"].astype(str),
        news_test["sentence_id"].astype(str)
    )
)

assert train_pool_ids.isdisjoint(val_ids)
assert train_pool_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

# --------------------------------------------
# 7. Display source and few-shot summaries
# --------------------------------------------

print("\n========== SOURCE DATA SUMMARY ==========")

print("Source training sequences:", len(source_train))
print("Source validation sequences:", len(source_val))

print("Source training groups:", len(source_train_docs))
print("Source validation groups:", len(source_val_docs))

print("\n========== FEW-SHOT DATASET SUMMARY ==========")

for budget, dataset in fewshot_sets.items():

    print(
        f"{budget}-shot: "
        f"{len(dataset)} sentences, "
        f"{dataset['doc_id'].nunique()} documents, "
        f"{sum(len(x) for x in dataset['tokens'])} tokens"
    )

print("\nSource train/validation separation: PASSED")
print("Few-shot nestedness: PASSED")
print("Train/validation/test separation: PASSED")

print("\nFew-shot preparation completed successfully!")

# Part B — Occurrence-level source mapping and audit

Run these cells after Part A, in their displayed order. They inspect source-tag families, build `manual_review`, preserve exact occurrence keys, and record the existing PPR/JQ and other family mapping decisions.

In [ ]:
from collections import Counter

# Calculate train tag counts from the news training pool
train_tag_counts = Counter(
    tag
    for sequence in news_train_pool["tags"]
    for tag in sequence
)

print("Total training pool tokens counted:", sum(train_tag_counts.values()))
print("train_tag_counts is now defined and ready for the audit!")

In [ ]:
# ============================================================
# CELL 42: SOURCE-TO-TARGET TAG MAPPING AUDIT
# ============================================================

from collections import Counter
import pandas as pd

# Count original source tags
source_original_counts = Counter()

# Count mapped target tags
source_mapped_counts = Counter()

# Count unmapped source tags
source_unmapped_counts = Counter()

for sequence in source_train["tags"]:

    for tag in sequence:

        if tag is None:
            continue

        source_original_counts[tag] += 1

        if tag in TARGET_TAGS:
            mapped_tag = tag

        elif tag in SOURCE_EXACT_TO_TARGET:
            mapped_tag = SOURCE_EXACT_TO_TARGET[tag]

        else:
            mapped_tag = None

            for prefix in sorted(
                SOURCE_PREFIX_TO_TARGET,
                key=len,
                reverse=True
            ):
                if tag.startswith(prefix):
                    mapped_tag = SOURCE_PREFIX_TO_TARGET[prefix]
                    break

        if mapped_tag is not None:
            source_mapped_counts[mapped_tag] += 1
        else:
            source_unmapped_counts[tag] += 1

# Create mapping audit table
mapping_audit_df = pd.DataFrame({
    "Target_Tag": TARGET_TAGS,
    "Source_Mapped_Tokens": [
        source_mapped_counts.get(tag, 0)
        for tag in TARGET_TAGS
    ],
    "News_Training_Tokens": [
        train_tag_counts.get(tag, 0)
        for tag in TARGET_TAGS
    ]
})

print("\n========== SOURCE-TO-TARGET MAPPING AUDIT ==========")

display(mapping_audit_df)

print("\nTotal source mapped tokens:",
      sum(source_mapped_counts.values()))

print("Total unmapped source tokens:",
      sum(source_unmapped_counts.values()))

print("\n========== UNMAPPED SOURCE TAGS ==========")

display(
    pd.DataFrame(
        source_unmapped_counts.most_common(),
        columns=["Source_Tag", "Unmapped_Count"]
    )
)

# Save audit
mapping_audit_df.to_csv(
    RESULTS_DIR / "source_target_mapping_audit.csv",
    index=False
)

print("\nMapping audit saved successfully.")

In [ ]:
# ============================================================
# CELL 43: INSPECT ORIGINAL SOURCE DATA
# ============================================================

import pandas as pd

print("\n========== SOURCE DATAFRAME INSPECTION ==========")

for name, obj in globals().copy().items():

    if isinstance(obj, pd.DataFrame):

        if any(
            key in name.lower()
            for key in ["source", "literature"]
        ):

            print(f"\nDataFrame: {name}")
            print("Shape:", obj.shape)
            print("Columns:", obj.columns.tolist())

            if "tag" in obj.columns:
                print("\nOriginal tag examples:")
                print(obj["tag"].value_counts().head(20))

print("\n========== SOURCE TRAIN STRUCTURE ==========")

print("Shape:", source_train.shape)
print("Columns:", source_train.columns.tolist())

print("\nFirst source training record:")
print(source_train.iloc[0].to_dict())

In [ ]:
# ============================================================
# CELL 44: ORIGINAL SOURCE-TO-TARGET MAPPING AUDIT
# ============================================================

from collections import Counter
import pandas as pd

# Function to reproduce the existing mapping logic
def audit_source_mapping(tag):

    if pd.isna(tag):
        return None

    tag = str(tag).strip()

    # Exact mapping first
    if tag in SOURCE_EXACT_TO_TARGET:
        return SOURCE_EXACT_TO_TARGET[tag]

    # Already-valid target tags
    if tag in TARGET_TAGS:
        return tag

    # Longest-prefix matching
    for prefix in sorted(
        SOURCE_PREFIX_TO_TARGET.keys(),
        key=len,
        reverse=True
    ):

        if tag.startswith(prefix):
            return SOURCE_PREFIX_TO_TARGET[prefix]

    return None


# Count original source tags
original_source_counts = Counter()
mapped_target_counts = Counter()
unmapped_source_counts = Counter()

for tag in source_raw[SRC_TAG]:

    if pd.isna(tag):
        continue

    original_source_counts[str(tag)] += 1

    mapped_tag = audit_source_mapping(tag)

    if mapped_tag is not None:
        mapped_target_counts[mapped_tag] += 1
    else:
        unmapped_source_counts[str(tag)] += 1


# Original source tag distribution
original_tag_df = pd.DataFrame(
    original_source_counts.most_common(),
    columns=["Original_Source_Tag", "Token_Count"]
)

# Target mapping distribution
original_mapping_df = pd.DataFrame({
    "Target_Tag": TARGET_TAGS,
    "Mapped_Source_Tokens": [
        mapped_target_counts.get(tag, 0)
        for tag in TARGET_TAGS
    ]
})

# Mapping statistics
total_source_tokens = sum(original_source_counts.values())
total_mapped_tokens = sum(mapped_target_counts.values())
total_unmapped_tokens = sum(unmapped_source_counts.values())

mapping_coverage = (
    total_mapped_tokens / total_source_tokens * 100
    if total_source_tokens else 0
)

print("\n========== ORIGINAL SOURCE MAPPING SUMMARY ==========")

print("Total original source tokens:", total_source_tokens)
print("Successfully mapped tokens:", total_mapped_tokens)
print("Unmapped tokens:", total_unmapped_tokens)
print("Mapping coverage:", round(mapping_coverage, 2), "%")

print("\n========== TARGET TAG MAPPING DISTRIBUTION ==========")

display(original_mapping_df)

print("\n========== TOP ORIGINAL SOURCE TAGS ==========")

display(original_tag_df.head(30))

print("\n========== UNMAPPED SOURCE TAGS ==========")

unmapped_df = pd.DataFrame(
    unmapped_source_counts.most_common(),
    columns=["Original_Source_Tag", "Unmapped_Count"]
)

display(unmapped_df.head(50))


# Save audit files
original_mapping_df.to_csv(
    RESULTS_DIR / "original_source_mapping_distribution.csv",
    index=False
)

original_tag_df.to_csv(
    RESULTS_DIR / "original_source_tag_distribution.csv",
    index=False
)

unmapped_df.to_csv(
    RESULTS_DIR / "original_unmapped_source_tags.csv",
    index=False
)

print("\nOriginal source mapping audit saved successfully.")

In [ ]:

import pandas as pd

# Count original source tags (not previously mapped tags)
source_tag_counts = (
    source_raw["tag"]
    .dropna()
    .astype(str)
    .value_counts()
    .rename_axis("Source_Tag")
    .reset_index(name="Frequency")
)

# Source tag family = part before the first dot
source_tag_counts["Source_Family"] = (
    source_tag_counts["Source_Tag"].str.split(".").str[0]
)

# Mappings supported by the MSRI paper and our agreed policy
confirmed_mappings = {
    "NC": ("N", "Common noun", "Confirmed"),
    "NP": ("NS", "Proper noun", "Confirmed"),
    "JJ": ("A", "Adjective", "Confirmed"),
    "PU": ("PU", "Punctuation", "Confirmed"),
    "CCD": ("SAMB", "Coordinating conjunction", "Confirmed"),
    "CSB": ("SAMB", "Subordinating conjunction", "Confirmed"),
    "CIN": ("AVU", "Interjection", "Confirmed"),
    "V": ("KP", "Finite verb", "Confirmed"),
    "VBP": ("NV", "Participle form", "Confirmed by agreed policy"),
}

# Categories where the target label depends on grammatical usage
context_dependent = {
    "PPR": "Pronoun: distinguish independent pronoun (SN) from pronominal adjective (SNV).",
    "PRF": "Reflexive pronoun: inspect its grammatical function.",
    "PRC": "Reciprocal pronoun: inspect its grammatical function.",
    "PRL": "Relative pronoun: inspect its grammatical function.",
    "PWH": "Wh-pronoun: inspect its grammatical function.",
}

# Candidate mappings that require further verification
candidate_mappings = {
    "CAD": ("AVP", "Adverb", "Candidate - verify target definition"),
    "CNG": ("AVN", "Negative particle", "Candidate - verify usage"),
    "CEM": ("AVN", "Emphatic particle", "Candidate - verify usage"),
}

def classify_source_tag(row):
    tag = row["Source_Tag"]
    family = row["Source_Family"]

    if family in confirmed_mappings:
        target, meaning, status = confirmed_mappings[family]
        return pd.Series([meaning, target, status, ""])

    if family in context_dependent:
        return pd.Series([
            context_dependent[family],
            None,
            "Context-dependent",
            context_dependent[family]
        ])

    if family in candidate_mappings:
        target, meaning, status = candidate_mappings[family]
        return pd.Series([meaning, target, status, status])

    if family == "VBG":
        return pd.Series([
            "Gerundive/participle candidate; source encoding needs verification",
            None,
            "Needs verification",
            "Check the source corpus convention before mapping to NV."
        ])

    return pd.Series([
        "Not established from the available source documentation",
        None,
        "Unresolved",
        "Do not use as supervised source label until its meaning is verified."
    ])

source_tag_counts[
    ["Grammatical_Meaning", "Target_Tag", "Mapping_Status", "Reason"]
] = source_tag_counts.apply(classify_source_tag, axis=1)

# Organize the audit table
source_tag_mapping_audit = source_tag_counts[
    [
        "Source_Tag",
        "Frequency",
        "Source_Family",
        "Grammatical_Meaning",
        "Target_Tag",
        "Mapping_Status",
        "Reason"
    ]
].copy()

source_tag_mapping_audit = source_tag_mapping_audit.sort_values(
    ["Mapping_Status", "Frequency"],
    ascending=[True, False]
).reset_index(drop=True)

print("Total unique source tags:", len(source_tag_mapping_audit))
print("\nMapping status summary:")
display(
    source_tag_mapping_audit.groupby("Mapping_Status")
    .agg(
        Unique_Tags=("Source_Tag", "count"),
        Total_Tokens=("Frequency", "sum")
    )
    .sort_values("Total_Tokens", ascending=False)
)

print("\nComplete source-tag mapping audit:")
display(source_tag_mapping_audit)

# Save for manual review
mapping_audit_path = RESULTS_DIR / "source_tag_mapping_audit.csv"
source_tag_mapping_audit.to_csv(mapping_audit_path, index=False)

print(f"\nAudit saved to: {mapping_audit_path}")


In [ ]:

# Summarize unresolved and context-dependent tags by source family
review_statuses = ["Unresolved", "Context-dependent", "Needs verification"]

review_df = source_tag_mapping_audit[
    source_tag_mapping_audit["Mapping_Status"].isin(review_statuses)
].copy()

family_summary = (
    review_df.groupby(["Mapping_Status", "Source_Family"])
    .agg(
        Unique_Tags=("Source_Tag", "count"),
        Total_Tokens=("Frequency", "sum")
    )
    .reset_index()
    .sort_values(
        ["Mapping_Status", "Total_Tokens"],
        ascending=[True, False]
    )
)

print("Review summary by source family:")
display(family_summary)

# Show the most frequent tags in each family
print("\nDetailed tags requiring review:")
for (status, family), group in review_df.groupby(
    ["Mapping_Status", "Source_Family"]
):
    print(f"\n{status} | Family: {family}")
    display(
        group[["Source_Tag", "Frequency", "Grammatical_Meaning"]]
        .sort_values("Frequency", ascending=False)
        .head(15)
    )


In [ ]:

# Inspect original source sentences and tokens
families_to_inspect = [
    "JQ", "DAB", "DRL", "DWH",
    "PPR", "PRF", "PRC", "PRL", "PWH",
    "VBG", "ALC", "AMN", "CCL"
]

# Get source tags and their families
context_rows = source_raw.copy()
context_rows["Source_Family"] = (
    context_rows["tag"].astype(str).str.split(".").str[0]
)

# Show up to 8 examples per family, prioritizing frequent tags
for family in families_to_inspect:
    family_data = context_rows[
        context_rows["Source_Family"] == family
    ].copy()

    if family_data.empty:
        continue

    print(f"\n{'='*80}")
    print(f"FAMILY: {family} | Total tokens: {len(family_data)}")
    print(f"{'='*80}")

    # Prioritize the most frequent source tags
    frequent_tags = family_data["tag"].value_counts().head(5).index
    examples = (
        family_data[family_data["tag"].isin(frequent_tags)]
        .drop_duplicates(subset=["sentence_id", "tag", "word"])
        .head(8)
    )

    display(
        examples[["sentence_id", "word", "tag", "sentence"]]
    )


In [ ]:

# Cell 49: Extract each reviewed token with surrounding words

families_for_review = [
    "JQ", "DAB", "DRL", "DWH",
    "PPR", "PRF", "PRC", "PRL", "PWH",
    "ALC", "AMN", "CCL"
]

context_df = source_raw.copy().reset_index(drop=True)

context_df["Source_Family"] = (
    context_df["tag"].astype(str).str.split(".").str[0]
)

# Preserve token order within each sentence
context_df["Token_Position"] = (
    context_df.groupby("sentence_id").cumcount()
)

review_records = []

for sentence_id, group in context_df.groupby("sentence_id", sort=False):
    group = group.reset_index(drop=True)

    for i, row in group.iterrows():
        if row["Source_Family"] not in families_for_review:
            continue

        previous_words = [
            str(word) for word in group.loc[max(0, i-2):i-1, "word"].tolist()
            if pd.notna(word)
        ]
        next_words = [
            str(word) for word in group.loc[i+1:i+2, "word"].tolist()
            if pd.notna(word)
        ]

        review_records.append({
            "sentence_id": sentence_id,
            "Token_Position": i,
            "Word": row["word"],
            "Source_Tag": row["tag"],
            "Source_Family": row["Source_Family"],
            "Previous_Words": " ".join(previous_words),
            "Next_Words": " ".join(next_words),
            "Full_Sentence": row["sentence"]
        })

token_context_review = pd.DataFrame(review_records)

# Save all occurrences
context_path = RESULTS_DIR / "source_token_context_review.csv"
token_context_review.to_csv(context_path, index=False)

print("Total reviewed occurrences:", len(token_context_review))
print("Saved to:", context_path)

# Display a manageable sample from each family
for family in families_for_review:
    sample = token_context_review[
        token_context_review["Source_Family"] == family
    ].head(10)

    if not sample.empty:
        print(f"\n--- {family} ---")
        display(sample[
            [
                "Word",
                "Source_Tag",
                "Previous_Words",
                "Next_Words",
                "Full_Sentence"
            ]
        ])


In [ ]:

# Cell 50: Prepare occurrence-level mapping review

manual_review = token_context_review.copy()

manual_review["Target_Tag"] = None
manual_review["Mapping_Reason"] = None
manual_review["Mapping_Confidence"] = None

# Apply only high-confidence decisions supported by the displayed examples.
# These are examples, not automatic family-wide mappings.

reviewed_examples = {
    ("अयम्", "DAB"): ("SNV", "Demonstrative modifies a noun", "High"),
    ("ईदृशं", "DAB"): ("SNV", "Demonstrative modifies a noun", "High"),
    ("सः", "PPR"): ("SN", "Independent personal pronoun", "High"),
    ("अस्य", "PPR"): ("SNV", "Possessive/pronominal modifier", "High"),
    ("स्वीयस्य", "PRF"): ("SNV", "Reflexive possessive modifier", "High"),
    ("ये", "PRL"): ("SNV", "Relative pronoun modifies a noun", "High"),
    ("कः", "PWH"): ("SNV", "Interrogative pronoun modifies a noun", "High"),
    ("प्रतिदिनम्", "ALC"): ("AVP", "Temporal adverb", "High"),
    ("प्रतिवर्षं", "ALC"): ("AVP", "Temporal adverb", "High"),
    ("तदा", "ALC"): ("AVP", "Temporal adverb", "High"),
    ("तत्र", "ALC"): ("AVP", "Spatial adverb", "High"),
    ("यदा", "ALC"): ("AVP", "Temporal adverb", "High"),
    ("पुरा", "CCL"): ("AVP", "Temporal adverb", "High"),
    ("पश्र्चात्", "CCL"): ("AVP", "Temporal adverb", "High"),
}

for index, row in manual_review.iterrows():
    key = (str(row["Word"]), str(row["Source_Family"]))

    if key in reviewed_examples:
        target, reason, confidence = reviewed_examples[key]

        manual_review.at[index, "Target_Tag"] = target
        manual_review.at[index, "Mapping_Reason"] = reason
        manual_review.at[index, "Mapping_Confidence"] = confidence

# Save the review template
manual_review_path = RESULTS_DIR / "occurrence_level_mapping_review.csv"
manual_review.to_csv(manual_review_path, index=False)

print("Total occurrences:", len(manual_review))
print("Automatically assigned examples:",
      manual_review["Target_Tag"].notna().sum())
print("Pending manual review:",
      manual_review["Target_Tag"].isna().sum())
print("Saved to:", manual_review_path)

display(
    manual_review[
        manual_review["Target_Tag"].notna()
    ][
        ["sentence_id", "Word", "Source_Tag",
         "Target_Tag", "Mapping_Reason", "Mapping_Confidence"]
    ].head(30)
)


In [ ]:
# Cell 51: Audit automatically assigned mappings

auto_review = manual_review[
    manual_review["Target_Tag"].notna()
].copy()

# Extract source family
auto_review["Source_Family"] = (
    auto_review["Source_Tag"]
    .astype(str)
    .str.split(".")
    .str[0]
)

print("Total automatic assignments:", len(auto_review))

print("\n1. Distribution by target tag:")
display(
    auto_review["Target_Tag"]
    .value_counts()
    .rename_axis("Target_Tag")
    .reset_index(name="Occurrences")
)

print("\n2. Distribution by source family:")
display(
    auto_review.groupby("Source_Family")
    .agg(
        Occurrences=("Word", "size"),
        Target_Tags=(
            "Target_Tag",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
    .sort_values("Occurrences", ascending=False)
    .reset_index()
)

print("\n3. Mapping combinations:")
display(
    auto_review.groupby(
        ["Source_Family", "Target_Tag"]
    )
    .size()
    .reset_index(name="Occurrences")
    .sort_values("Occurrences", ascending=False)
)

In [ ]:
# Cell 52: Inspect automatic mapping examples

families_to_check = ["PPR", "PWH", "DAB", "PRL", "PRF", "CCL"]

for family in families_to_check:

    print(f"\n{'='*80}")
    print(f"SOURCE FAMILY: {family}")
    print(f"{'='*80}")

    family_df = auto_review[
        auto_review["Source_Family"] == family
    ]

    display(
        family_df[
            [
                "sentence_id",
                "Word",
                "Source_Tag",
                "Target_Tag",
                "Mapping_Reason",
                "Mapping_Confidence"
            ]
        ].head(30)
    )

In [ ]:
# Cell 53: Inspect automatic mappings with complete sentence context

# Rebuild from the original DataFrame
verify_df = manual_review[
    manual_review["Target_Tag"].notna()
].copy()

# Extract source family
verify_df["Source_Family"] = (
    verify_df["Source_Tag"]
    .astype(str)
    .str.split(".")
    .str[0]
)

# Select families requiring verification
families_to_verify = ["PWH", "DAB", "PRL"]

verify_df = verify_df[
    verify_df["Source_Family"].isin(families_to_verify)
].copy()

# Display complete contexts
display(
    verify_df[
        [
            "Source_Family",
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].sort_values(
        ["Source_Family", "sentence_id", "Token_Position"]
    )
)

print("Total occurrences:", len(verify_df))
print(
    "Missing contexts:",
    verify_df["Full_Sentence"].isna().sum()
)

In [ ]:
# Cell 54: Correct occurrence-level mappings

# Confirmed occurrence-level decisions
corrections = {
    # DAB
    (28, 0): "SNV",
    (69, 4): "SNV",
    (843, 2): "SNV",
    (877, 0): "SNV",
    (972, 0): "SNV",
    (1395, 2): "SN",

    # PRL
    (108, 5): "SN",
    (112, 3): "SNV",
    (370, 0): "SN",
    (599, 1): "SN",

    # PWH
    (202, 0): "SN",
    (243, 1): "SNV",
    (1111, 4): "SN",
    (1232, 1): "SN",
    (1233, 5): "SN",
    (1238, 5): "SN",
    (1239, 0): "SN",
    (1244, 3): "SN",
    (1278, 1): "SN",
    (1286, 1): "SNV",
    (1312, 0): "SNV",
    (1389, 4): "SN",
    (1404, 1): "SNV",
    (1446, 1): "SNV",
}

# Occurrences requiring further review
pending_occurrences = [
    (942, 8),   # DAB: अयम्
    (1403, 9),  # DAB: अयम्
    (934, 7),   # PWH: कः
]

# Apply confirmed corrections
for (sentence_id, token_position), target_tag in corrections.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    )

    if mask.sum() != 1:
        print(
            f"WARNING: Expected one match for "
            f"{sentence_id}, {token_position}; "
            f"found {mask.sum()}"
        )
        continue

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = (
        "Occurrence-level grammatical function verified"
    )
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

# Clear uncertain assignments
for sentence_id, token_position in pending_occurrences:

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    )

    manual_review.loc[mask, "Target_Tag"] = None
    manual_review.loc[mask, "Mapping_Reason"] = (
        "Pending: complete sentence context required"
    )
    manual_review.loc[mask, "Mapping_Confidence"] = "Pending"

print("Confirmed corrections:", len(corrections))
print("Pending occurrences:", len(pending_occurrences))

# Verify updated records
display(
    manual_review[
        manual_review["sentence_id"].isin(
            [x[0] for x in corrections] +
            [x[0] for x in pending_occurrences]
        )
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# Cell 55: Validate the 27 occurrence-level decisions

reviewed_ids = (
    list(corrections.keys())
    + pending_occurrences
)

verification_rows = []

for sentence_id, token_position in reviewed_ids:

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    if len(row) != 1:
        print(
            f"ERROR: Expected one record for "
            f"sentence={sentence_id}, position={token_position}; "
            f"found {len(row)}"
        )
        continue

    verification_rows.append(row)

verified_df = pd.concat(verification_rows)

print("Total reviewed occurrences:", len(verified_df))
print(
    "Confirmed:",
    (verified_df["Mapping_Confidence"] == "High").sum()
)
print(
    "Pending:",
    (verified_df["Mapping_Confidence"] == "Pending").sum()
)

display(
    verified_df[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# Cell 56: Inspect complete contexts of pending occurrences

pending_ids = [
    (934, 7),    # कः
    (942, 8),    # अयम्
    (1403, 9),   # अयम्
]

pending_contexts = []

for sentence_id, token_position in pending_ids:

    row = token_context_review[
        (token_context_review["sentence_id"] == sentence_id)
        & (token_context_review["Token_Position"] == token_position)
    ]

    if len(row) == 1:
        pending_contexts.append(row)
    else:
        print(
            f"Sentence {sentence_id}, position {token_position}: "
            f"{len(row)} matching records"
        )

pending_context_df = pd.concat(pending_contexts, ignore_index=True)

display(
    pending_context_df[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 57: Retrieve original complete sentences

pending_sentence_ids = [934, 942, 1403]

for sid in pending_sentence_ids:

    sentence_data = source_raw[
        source_raw[SRC_SENT_ID] == sid
    ]

    print("\n" + "=" * 80)
    print("SENTENCE ID:", sid)
    print("=" * 80)

    if sentence_data.empty:
        print("Sentence not found in source_raw")
        continue

    # Display original sentence text
    print(sentence_data["sentence"].iloc[0])

In [ ]:
# Cell 58: Finalize remaining pending mappings

final_corrections = {
    (934, 7): (
        "SN",
        "Independent indefinite pronoun"
    ),
    (942, 8): (
        "SN",
        "Independent demonstrative pronoun"
    ),
    (1403, 9): (
        "SN",
        "Independent demonstrative pronoun"
    )
}

for (sentence_id, token_position), (tag, reason) in final_corrections.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one match for {sentence_id}, {token_position}"
    )

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Final mappings applied:", len(final_corrections))

print("\nRemaining pending mappings:",
      manual_review["Mapping_Confidence"].eq("Pending").sum())

display(
    manual_review[
        manual_review["sentence_id"].isin([934, 942, 1403])
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)

In [ ]:
# Cell 59: Save finalized occurrence-level mapping review

from pathlib import Path

OUTPUT_DIR = Path(
    str(RESULTS_DIR)
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR / "occurrence_level_mapping_review.csv"
)

manual_review.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("File saved successfully:", OUTPUT_FILE)
print("Total occurrences:", len(manual_review))
print(
    "Mapped occurrences:",
    manual_review["Target_Tag"].notna().sum()
)
print(
    "Unmapped occurrences:",
    manual_review["Target_Tag"].isna().sum()
)
print(
    "High-confidence mappings:",
    manual_review["Mapping_Confidence"].eq("High").sum()
)
print(
    "Pending mappings:",
    manual_review["Mapping_Confidence"].eq("Pending").sum()
)

In [ ]:
# Cell 60: Analyze remaining unmapped source families

unmapped_review = manual_review[
    manual_review["Target_Tag"].isna()
].copy()

unmapped_review["Source_Family"] = (
    unmapped_review["Source_Tag"]
    .astype(str)
    .str.split(".")
    .str[0]
)

family_summary = (
    unmapped_review
    .groupby("Source_Family")
    .agg(
        Occurrences=("Word", "size"),
        Unique_Words=("Word", "nunique"),
        Unique_Source_Tags=("Source_Tag", "nunique")
    )
    .sort_values("Occurrences", ascending=False)
    .reset_index()
)

print("Total unmapped occurrences:", len(unmapped_review))
print("Unmapped source families:", len(family_summary))

display(family_summary.head(30))

In [ ]:
# Cell 61: Inspect ALC and AMN source-tag distributions

families_to_inspect = ["ALC", "AMN"]

for family in families_to_inspect:

    family_df = unmapped_review[
        unmapped_review["Source_Family"] == family
    ]

    print("\n" + "=" * 80)
    print("SOURCE FAMILY:", family)
    print("Total occurrences:", len(family_df))
    print("Unique words:", family_df["Word"].nunique())
    print("Unique source tags:", family_df["Source_Tag"].nunique())
    print("=" * 80)

    print("\nSource-tag distribution:")
    display(
        family_df["Source_Tag"]
        .value_counts()
        .rename_axis("Source_Tag")
        .reset_index(name="Occurrences")
    )

    print("\nWord distribution:")
    display(
        family_df["Word"]
        .value_counts()
        .head(30)
        .rename_axis("Word")
        .reset_index(name="Occurrences")
    )

In [ ]:
# Cell 62: Contextual review of ALC and AMN

for family in ["ALC", "AMN"]:

    family_df = unmapped_review[
        unmapped_review["Source_Family"] == family
    ].copy()

    print("\n" + "=" * 100)
    print(f"SOURCE FAMILY: {family}")
    print("=" * 100)

    frequent_words = (
        family_df["Word"]
        .value_counts()
        .head(30)
        .index
        .tolist()
    )

    for word in frequent_words:

        examples = family_df[
            family_df["Word"] == word
        ].head(3)

        print(f"\nWord: {word} | Total occurrences: {(family_df['Word'] == word).sum()}")

        display(
            examples[
                [
                    "sentence_id",
                    "Token_Position",
                    "Word",
                    "Source_Tag",
                    "Full_Sentence"
                ]
            ]
        )

In [ ]:
# Cell 63: Complete occurrence-level ALC review

alc_review = unmapped_review[
    unmapped_review["Source_Family"] == "ALC"
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
)

display(
    alc_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

print("Total ALC occurrences:", len(alc_review))
print("Unique ALC words:", alc_review["Word"].nunique())

In [ ]:
# Cell 64: Display ALC occurrences in batches of 50

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Total unmapped ALC occurrences:", len(alc_review))

# Display the first batch
start, end = 0, 50

display(
    alc_review.iloc[start:end][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 65: Apply confirmed occurrence-level ALC mappings

alc_confirmed = {
    (452, 1): ("AVP", "Sudden manner adverb"),
    (867, 3): ("AVP", "Adverb expressing distance"),

    # अत्र: locative adverb
    (164, 1): ("AVP", "Locative adverb meaning here"),
    (169, 0): ("AVP", "Locative adverb meaning here"),
    (206, 1): ("AVP", "Locative adverb meaning here"),
    (379, 0): ("AVP", "Locative adverb meaning here"),
    (429, 3): ("AVP", "Locative adverb meaning here"),
    (435, 0): ("AVP", "Locative adverb meaning here"),
    (440, 1): ("AVP", "Locative adverb meaning here"),
    (453, 3): ("AVP", "Locative adverb meaning here"),
    (456, 1): ("AVP", "Locative adverb meaning here"),
    (491, 1): ("AVP", "Locative adverb meaning here"),
    (551, 8): ("AVP", "Locative adverb meaning here"),
    (936, 3): ("AVP", "Locative adverb meaning here"),
    (1156, 11): ("AVP", "Locative adverb meaning here"),
    (1158, 0): ("AVP", "Locative adverb meaning here"),
    (1195, 0): ("AVP", "Locative adverb meaning here"),
    (1343, 3): ("AVP", "Locative adverb meaning here"),
    (1389, 5): ("AVP", "Locative adverb meaning here"),

    # अत्रैव: emphatic locative adverb
    (1295, 0): ("AVP", "Emphatic locative adverb"),
    (1297, 0): ("AVP", "Emphatic locative adverb"),
    (1301, 1): ("AVP", "Emphatic locative adverb"),

    # अद्य: temporal adverb
    (136, 0): ("AVP", "Temporal adverb meaning today"),
    (145, 4): ("AVP", "Temporal adverb meaning today"),
    (388, 1): ("AVP", "Temporal adverb meaning today"),
    (412, 2): ("AVP", "Temporal adverb meaning today"),
    (614, 8): ("AVP", "Temporal adverb meaning today"),
    (684, 0): ("AVP", "Temporal adverb meaning today"),
    (688, 19): ("AVP", "Temporal adverb meaning today"),

    # अधः: directional adverb
    (712, 2): ("AVP", "Directional adverb meaning downward"),
    (799, 2): ("AVP", "Directional adverb meaning downward"),

    # अधुना: temporal adverb
    (166, 1): ("AVP", "Temporal adverb meaning now"),
    (186, 2): ("AVP", "Temporal adverb meaning now"),
    (316, 3): ("AVP", "Temporal adverb meaning now"),
    (607, 0): ("AVP", "Temporal adverb meaning now"),
    (779, 3): ("AVP", "Temporal adverb meaning now"),
    (963, 0): ("AVP", "Temporal adverb meaning now"),
    (1263, 0): ("AVP", "Temporal adverb meaning now"),
    (1285, 6): ("AVP", "Temporal adverb meaning now"),
    (1301, 0): ("AVP", "Temporal adverb meaning now"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_confirmed.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped occurrences:", updated_count)
print("\nUpdated ALC tag distribution:")

display(
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        manual_review["Target_Tag"].notna()
    ]["Target_Tag"].value_counts()
)

In [ ]:
# Cell 66: Verify अधः in sentence 712

display(
    manual_review[
        (manual_review["sentence_id"] == 712) &
        (manual_review["Word"] == "अधः")
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 67: Correct the verified अधः occurrence

mask = (
    (manual_review["sentence_id"] == 712) &
    (manual_review["Token_Position"] == 3) &
    (manual_review["Word"] == "अधः")
)

manual_review.loc[mask, "Target_Tag"] = "AVP"
manual_review.loc[mask, "Mapping_Reason"] = "Directional adverb meaning downward"
manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Corrected occurrences:", mask.sum())

display(
    manual_review.loc[
        mask,
        ["sentence_id", "Token_Position", "Word", "Target_Tag",
         "Mapping_Reason", "Mapping_Confidence"]
    ]
)

In [ ]:
display(alc_review.iloc[50:100][
    ["sentence_id", "Token_Position", "Word", "Source_Tag", "Full_Sentence"]
])

In [ ]:
# Cell 68: Apply confirmed ALC mappings (occurrences 50–99)

alc_batch2_confirmed = {
    # अनन्तरं
    (577, 0): ("AVP", "Temporal adverb meaning afterward"),
    (821, 10): ("AVP", "Temporal adverb meaning afterward"),
    (1156, 0): ("AVP", "Temporal adverb meaning afterward"),

    # अनुक्षणम्
    (711, 0): ("AVP", "Temporal adverb meaning immediately"),

    # अन्तः
    (397, 0): ("AVP", "Directional adverb meaning inside"),

    # अन्ततः
    (1131, 5): ("AVP", "Temporal adverb meaning finally"),

    # अन्यत्र
    (392, 3): ("AVP", "Locative adverb meaning elsewhere"),
    (482, 5): ("AVP", "Locative adverb meaning elsewhere"),

    # अन्येद्युः
    (725, 0): ("AVP", "Temporal adverb meaning on another day"),

    # इतस्ततः
    (592, 3): ("AVP", "Locative adverb meaning here and there"),
    (926, 3): ("AVP", "Locative adverb meaning here and there"),

    # इदानीं
    (353, 1): ("AVP", "Temporal adverb meaning now"),
    (414, 3): ("AVP", "Temporal adverb meaning now"),
    (501, 0): ("AVP", "Temporal adverb meaning now"),
    (689, 1): ("AVP", "Temporal adverb meaning now"),
    (1197, 1): ("AVP", "Temporal adverb meaning now"),
    (1216, 1): ("AVP", "Temporal adverb meaning now"),
    (1444, 3): ("AVP", "Temporal adverb meaning now"),

    # इदानीम्
    (415, 7): ("AVP", "Temporal adverb meaning now"),
    (431, 2): ("AVP", "Temporal adverb meaning now"),
    (1228, 1): ("AVP", "Temporal adverb meaning now"),
    (1209, 4): ("AVP", "Temporal adverb meaning now"),

    # उपरि
    (584, 0): ("AVP", "Spatial adverb meaning above"),

    # एकदा
    (715, 0): ("AVP", "Temporal adverb meaning once"),
    (821, 2): ("AVP", "Temporal adverb meaning once"),
    (857, 0): ("AVP", "Temporal adverb meaning once"),
    (926, 0): ("AVP", "Temporal adverb meaning once"),
    (940, 2): ("AVP", "Temporal adverb meaning once"),
    (955, 0): ("AVP", "Temporal adverb meaning once"),

    # कदा
    (56, 2): ("AVP", "Interrogative temporal adverb meaning when"),
    (69, 2): ("AVP", "Interrogative temporal adverb meaning when"),
    (429, 0): ("AVP", "Interrogative temporal adverb meaning when"),
    (432, 0): ("AVP", "Interrogative temporal adverb meaning when"),
    (818, 0): ("AVP", "Interrogative temporal adverb meaning when"),

    # कदाचित्
    (481, 0): ("AVP", "Temporal adverb meaning sometimes"),
    (1322, 0): ("AVP", "Temporal adverb meaning sometimes"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch2_confirmed.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC")
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

print("\nALC mapping distribution:")
display(
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        manual_review["Target_Tag"].notna()
    ]["Target_Tag"].value_counts()
)

In [ ]:
# Cell 69: Review the third batch of ALC occurrences

# Refresh the review data to reflect previous mapping updates
alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[100:150][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 70: Apply confirmed ALC mappings (occurrences 100–149)

alc_batch3_confirmed = {
    # परं
    (1139, 1): ("AVP", "Temporal adverb meaning thereafter"),

    # परितः
    (1450, 7): ("AVP", "Spatial adverb meaning around"),

    # परेद्युः
    (741, 1): ("AVP", "Temporal adverb meaning on the following day"),

    # पश्चात् / पश्र्चात्
    (1257, 0): ("AVP", "Temporal adverb meaning afterward"),
    (222, 0): ("AVP", "Temporal adverb meaning afterward"),
    (232, 0): ("AVP", "Temporal adverb meaning afterward"),
    (821, 6): ("AVP", "Temporal adverb meaning afterward"),

    # पूर्वं / पुर्वं
    (620, 1): ("AVP", "Temporal adverb meaning previously"),
    (665, 13): ("AVP", "Temporal adverb meaning previously"),
    (414, 0): ("AVP", "Temporal adverb meaning previously"),
    (436, 1): ("AVP", "Temporal adverb meaning previously"),
    (489, 7): ("AVP", "Temporal adverb meaning previously"),
    (650, 1): ("AVP", "Temporal adverb meaning previously"),
    (1262, 0): ("AVP", "Temporal adverb meaning previously"),

    # प्रतिदिनं
    (56, 1): ("AVP", "Frequency adverb meaning every day"),
    (57, 1): ("AVP", "Frequency adverb meaning every day"),
    (696, 0): ("AVP", "Frequency adverb meaning every day"),
    (739, 1): ("AVP", "Frequency adverb meaning every day"),
    (758, 1): ("AVP", "Frequency adverb meaning every day"),
    (759, 1): ("AVP", "Frequency adverb meaning every day"),
    (1062, 0): ("AVP", "Frequency adverb meaning every day"),

    # प्राक्
    (402, 1): ("AVP", "Temporal adverb meaning earlier"),

    # प्रायः
    (471, 1): ("AVP", "Frequency adverb meaning usually"),
    (733, 0): ("AVP", "Frequency adverb meaning generally"),

    # बहिः
    (396, 2): ("AVP", "Spatial adverb meaning outside"),

    # यत्र
    (318, 5): ("AVP", "Locative relative adverb meaning where"),
    (1306, 13): ("AVP", "Locative relative adverb meaning where"),

    # यावत्: subordinate conjunction
    (748, 0): ("SAMB", "Temporal subordinate conjunction"),
    (929, 0): ("SAMB", "Temporal subordinate conjunction"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch3_confirmed.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC")
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print(
            "Check occurrence:",
            sid, pos,
            "Matches:", mask.sum()
        )

print("Successfully mapped:", updated_count)

print("\nALC mapping distribution:")
display(
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        manual_review["Target_Tag"].notna()
    ]["Target_Tag"].value_counts()
)

print("\nRemaining unmapped ALC occurrences:",
      manual_review[
          (manual_review["Source_Family"] == "ALC") &
          manual_review["Target_Tag"].isna()
      ].shape[0])

In [ ]:
# Cell 71: Review the fourth batch of ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[150:200][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 72: Apply final ALC occurrence-level mappings

alc_final_confirmed = {
    (187, 0): ("N", "Common noun meaning evening"),
    (65, 3): ("AVP", "Temporal adverb meaning in the evening"),
    (224, 0): ("AVP", "Adverb expressing considerable distance"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_final_confirmed.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC")
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

alc_final = manual_review[
    manual_review["Source_Family"] == "ALC"
]

print("\nFinal ALC mapping distribution:")
display(alc_final["Target_Tag"].value_counts(dropna=False))

print("\nTotal ALC occurrences:", len(alc_final))
print("Mapped:", alc_final["Target_Tag"].notna().sum())
print("Unmapped:", alc_final["Target_Tag"].isna().sum())

In [ ]:
# Cell 73: Display the first 50 remaining unmapped ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[0:50][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 74: Apply confirmed ALC mappings (remaining batch)

alc_batch4_confirmed = {
    # अग्रे: directional adverb
    (1088, 6): ("AVP", "Directional adverb meaning forward"),

    # अथ: discourse indeclinable
    (1404, 0): ("AVN", "Discourse particle meaning then"),
    (1426, 2): ("AVN", "Discourse particle meaning then"),

    # अनन्तरं: temporal adverb
    (233, 0): ("AVP", "Temporal adverb meaning afterward"),
    (303, 0): ("AVP", "Temporal adverb meaning afterward"),

    # कदाचित्: temporal adverb
    (1322, 7): ("AVP", "Temporal adverb meaning sometimes"),
    (1322, 11): ("AVP", "Temporal adverb meaning sometimes"),
    (1322, 18): ("AVP", "Temporal adverb meaning sometimes"),
    (1322, 22): ("AVP", "Temporal adverb meaning sometimes"),
    (1428, 3): ("AVP", "Temporal adverb meaning sometimes"),
    (1148, 0): ("AVP", "Temporal adverb meaning sometimes"),

    # कदापि: temporal adverb
    (279, 6): ("AVP", "Temporal adverb meaning ever or at any time"),
    (718, 5): ("AVP", "Temporal adverb meaning ever or at any time"),
    (740, 1): ("AVP", "Temporal adverb meaning ever or at any time"),
    (1145, 3): ("AVP", "Temporal adverb meaning ever or at any time"),

    # कालानन्तरं / कालान्तरम्
    (661, 1): ("AVP", "Temporal adverb meaning after some time"),
    (694, 1): ("AVP", "Temporal adverb meaning after some time"),
    (940, 1): ("AVP", "Temporal adverb meaning after some time"),

    # कुत्र: interrogative/relative locative adverb
    (168, 2): ("AVP", "Interrogative locative adverb meaning where"),
    (418, 1): ("AVP", "Interrogative locative adverb meaning where"),
    (430, 1): ("AVP", "Interrogative locative adverb meaning where"),
    (431, 1): ("AVP", "Interrogative locative adverb meaning where"),
    (460, 0): ("AVP", "Interrogative locative adverb meaning where"),
    (495, 2): ("AVP", "Locative adverb meaning anywhere"),
    (688, 29): ("AVP", "Interrogative locative adverb meaning where"),
    (694, 13): ("AVP", "Interrogative locative adverb meaning where"),
    (1349, 16): ("AVP", "Interrogative locative adverb meaning where"),
    (1351, 7): ("AVP", "Interrogative locative adverb meaning where"),
    (1376, 2): ("AVP", "Interrogative locative adverb meaning where"),
    (1409, 5): ("AVP", "Interrogative locative adverb meaning where"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch4_confirmed.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["Target_Tag"].isna())
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

print("\nALC mapping distribution:")
display(
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        manual_review["Target_Tag"].notna()
    ]["Target_Tag"].value_counts(dropna=False)
)

print("\nRemaining unmapped ALC occurrences:",
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        manual_review["Target_Tag"].isna()
    ].shape[0]
)

In [ ]:
# Cell 75: Review the next batch of unmapped ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[0:50][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)


In [ ]:
# Cell 76: Apply confirmed ALC mappings (next batch)

alc_batch5_confirmed = {
    # कुत्रचित्
    (408, 3): ("AVP", "Locative adverb meaning somewhere"),
    (408, 9): ("AVP", "Locative adverb meaning somewhere"),

    # कुत्रापि
    (1348, 5): ("AVP", "Locative adverb meaning anywhere"),

    # चिरम्
    (1168, 2): ("AVP", "Temporal adverb meaning for a long time"),

    # ततः: sequential/temporal adverb
    (328, 0): ("AVP", "Sequential adverb meaning then"),
    (586, 0): ("AVP", "Sequential adverb meaning then"),
    (690, 0): ("AVP", "Sequential adverb meaning then"),
    (708, 2): ("AVP", "Sequential adverb meaning then"),
    (712, 5): ("AVP", "Sequential adverb meaning then"),
    (731, 0): ("AVP", "Sequential adverb meaning then"),
    (765, 0): ("AVP", "Sequential adverb meaning then"),
    (808, 0): ("AVP", "Sequential adverb meaning then"),
    (843, 0): ("AVP", "Sequential adverb meaning then"),
    (881, 0): ("AVP", "Sequential adverb meaning then"),
    (1052, 0): ("AVP", "Sequential adverb meaning then"),
    (1139, 0): ("AVP", "Sequential adverb meaning then"),
    (1154, 0): ("AVP", "Sequential adverb meaning then"),
    (1155, 0): ("AVP", "Sequential adverb meaning then"),
    (1157, 4): ("AVP", "Sequential adverb meaning then"),
    (1158, 9): ("AVP", "Sequential adverb meaning then"),
    (1173, 1): ("AVP", "Sequential adverb meaning then"),
    (1178, 1): ("AVP", "Sequential adverb meaning then"),
    (1192, 11): ("AVP", "Sequential adverb meaning then"),
    (1204, 1): ("AVP", "Sequential adverb meaning then"),
    (1427, 2): ("AVP", "Sequential adverb meaning then"),
    (1441, 1): ("AVP", "Sequential adverb meaning then"),
    (1444, 0): ("AVP", "Sequential adverb meaning then"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch5_confirmed.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["Target_Tag"].isna())
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print(
            "Check occurrence:",
            sid, pos,
            "Matches:", mask.sum()
        )

print("Successfully mapped:", updated_count)

alc_final = manual_review[
    manual_review["Source_Family"] == "ALC"
]

print("\nALC mapping distribution:")
display(alc_final["Target_Tag"].value_counts(dropna=False))

print("\nTotal ALC occurrences:", len(alc_final))
print("Mapped:", alc_final["Target_Tag"].notna().sum())
print("Unmapped:", alc_final["Target_Tag"].isna().sum())

In [ ]:
# Cell 77: Review the next 50 unmapped ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[0:50][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 78: Apply confirmed ALC mappings

alc_batch6_confirmed = {
    # तदनन्तरं / तदनन्तरम्
    (543, 0): ("AVP", "Temporal adverb meaning thereafter"),
    (600, 0): ("AVP", "Temporal adverb meaning thereafter"),
    (723, 0): ("AVP", "Temporal adverb meaning thereafter"),
    (1050, 0): ("AVP", "Temporal adverb meaning thereafter"),
    (938, 1): ("AVP", "Temporal adverb meaning thereafter"),

    # दूरं
    (859, 7): ("AVP", "Spatial adverb meaning far away"),

    # नीचैः
    (80, 1): ("AVP", "Directional adverb meaning downward"),
    (178, 6): ("AVP", "Directional adverb meaning downward"),
    (181, 3): ("AVP", "Directional adverb meaning downward"),
    (862, 5): ("AVP", "Directional adverb meaning downward"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch6_confirmed.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["Target_Tag"].isna())
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

alc_final = manual_review[
    manual_review["Source_Family"] == "ALC"
]

print("\nALC mapping distribution:")
display(alc_final["Target_Tag"].value_counts(dropna=False))

print("\nTotal ALC occurrences:", len(alc_final))
print("Mapped:", alc_final["Target_Tag"].notna().sum())
print("Unmapped:", alc_final["Target_Tag"].isna().sum())


In [ ]:
# Cell 79: Review remaining unmapped ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 80: Apply confirmed final ALC mappings

alc_final_confirmed = {
    (797, 3): ("N", "Common noun meaning top or front"),
    (794, 0): ("AVP", "Locative adverb meaning everywhere"),
    (1061, 5): ("AVP", "Locative adverb meaning everywhere"),
    (96, 6): ("AVP", "Frequency adverb meaning always"),
    (143, 2): ("AVP", "Frequency adverb meaning always"),
    (415, 1): ("AVP", "Frequency adverb meaning always"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_final_confirmed.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["Target_Tag"].isna())
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

In [ ]:
# Cell 81: Inspect remaining ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC:", len(alc_review))

display(
    alc_review.iloc[0:40][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 82: Apply confirmed ALC mappings from the latest review

alc_batch7_confirmed = {
    # तत्रत्याः: adjective
    (711, 2): ("A", "Adjective meaning situated there"),

    # तावत्: correlative temporal adverb
    (929, 6): ("AVP", "Correlative temporal adverb"),

    # तावत्: discourse particle
    (1198, 3): ("AVN", "Discourse particle"),
    (1225, 4): ("AVN", "Discourse particle"),
    (1385, 8): ("AVN", "Discourse particle"),
    (1391, 1): ("AVN", "Discourse particle"),

    # परीक्षानन्तरम्: temporal adverbial expression
    (440, 0): ("AVP", "Temporal adverbial expression meaning after the examination"),
}

updated_count = 0

for (sid, pos), (tag, reason) in alc_batch7_confirmed.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["Target_Tag"].isna())
    )

    if mask.sum() == 1:
        manual_review.loc[mask, "Target_Tag"] = tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = "High"
        updated_count += 1
    else:
        print("Check occurrence:", sid, pos, "Matches:", mask.sum())

print("Successfully mapped:", updated_count)

alc_final = manual_review[
    manual_review["Source_Family"] == "ALC"
]

print("\nALC mapping distribution:")
display(alc_final["Target_Tag"].value_counts(dropna=False))

print("\nTotal ALC occurrences:", len(alc_final))
print("Mapped:", alc_final["Target_Tag"].notna().sum())
print("Unmapped:", alc_final["Target_Tag"].isna().sum())

In [ ]:
# Cell 83: Review remaining unmapped ALC occurrences

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[0:40][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 84: Map reviewed ALC occurrences (batch 8)

alc_batch8_confirmed = {
    (912, 1): ("AVP", "Spatial adverb: in front"),
    (858, 2): ("AVP", "Spatial adverb: below"),

    (245, 3): ("AVP", "Temporal adverb: afterwards"),
    (375, 16): ("AVP", "Temporal adverb: afterwards"),
    (1291, 2): ("AVP", "Temporal adverb: afterwards"),
    (1317, 1): ("AVP", "Temporal relation: after"),

    (399, 10): ("AVP", "Spatial adverb: inside"),
    (596, 1): ("AVP", "Spatial adverb: inside"),
    (489, 0): ("AVP", "Temporal adverb: at the end"),
    (221, 1): ("AVP", "Temporal adverb: initially"),

    (91, 1): ("AVP", "Spatial adverb: above"),
    (702, 9): ("AVP", "Spatial adverb: above"),
    (779, 6): ("AVP", "Spatial adverb: above"),
    (859, 2): ("AVP", "Spatial adverb: above"),
    (1095, 1): ("AVP", "Relational expression: concerning"),

    (1081, 2): ("AVP", "Duration adverb: for a moment"),
    (11, 2): ("NV", "Gerund/verbal form"),
    (1172, 7): ("AVN", "Indeclinable discourse expression"),
    (231, 1): ("AVP", "Duration expression"),

    (492, 2): ("AVP", "Temporal adverb: afterwards"),
    (969, 1): ("AVP", "Temporal adverb: after"),
    (386, 19): ("AVP", "Spatial adverb: in front"),
    (587, 2): ("AVP", "Spatial adverb: in front"),
    (376, 4): ("AVP", "Temporal adverb: before"),
    (596, 2): ("AVP", "Spatial adverb: behind"),
    (866, 9): ("AVP", "Spatial adverb: behind"),
    (881, 1): ("AVP", "Temporal postposition: since"),
    (952, 5): ("AVP", "Temporal postposition: since"),
}

for (sid, pos), (tag, reason) in alc_batch8_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Successfully mapped:", len(alc_batch8_confirmed))

alc_status = (
    manual_review[manual_review["Source_Family"] == "ALC"]
    ["Target_Tag"].value_counts(dropna=False)
)

display(alc_status)

In [ ]:
# Cell 85: Review remaining ALC occurrences (batch 9)

alc_review = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_review = alc_review.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_review))

display(
    alc_review.iloc[0:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 86: Map confirmed ALC occurrences (batch 10)

alc_batch9_confirmed = {
    (693, 1): ("AVP", "Spatial adverb: outside"),
    (703, 1): ("AVP", "Spatial adverb: outside"),
    (705, 1): ("AVP", "Spatial adverb: outside"),

    (30, 0): ("AVP", "Manner adverb: quickly"),

    (189, 0): ("AVP", "Temporal adverb: tomorrow"),
    (390, 13): ("AVP", "Temporal adverb: tomorrow"),
    (1343, 0): ("AVP", "Temporal adverb: tomorrow"),

    (481, 1): ("AVP", "Temporal expression: after a year"),
    (341, 3): ("AVP", "Temporal expression: after marriage"),

    (232, 2): ("AVP", "Directional adverb: to the left"),

    (733, 1): ("AVP", "Duration expression: for six months"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in alc_batch9_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

alc_status = (
    manual_review[manual_review["Source_Family"] == "ALC"]
    ["Target_Tag"]
    .value_counts(dropna=False)
)

display(alc_status)

print(
    "Total ALC occurrences:",
    (manual_review["Source_Family"] == "ALC").sum()
)

In [ ]:
# Cell 87: Verify missing occurrence and inspect remaining ALC entries

print("Checking sentence 693:")

display(
    manual_review[
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["sentence_id"] == 693)
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Full_Sentence"
        ]
    ]
)

alc_remaining = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_remaining = alc_remaining.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("\nRemaining unmapped ALC occurrences:", len(alc_remaining))

display(
    alc_remaining[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 88: Apply occurrence-level ALC mappings (batch 11)

alc_batch10_confirmed = {
    # Corrected position for बहिः
    (693, 7): ("AVP", "Spatial adverb: outside"),

    # Temporal and spatial adverbs
    (471, 0): ("AVP", "Temporal adverb: always"),
    (98, 4): ("AVP", "Temporal adverb: always"),
    (130, 2): ("AVP", "Temporal adverb: always"),
    (194, 4): ("AVP", "Spatial adverb: everywhere"),
    (272, 3): ("AVP", "Spatial adverb: everywhere"),
    (766, 0): ("AVP", "Spatial adverb: everywhere"),

    # Nominal expressions
    (89, 1): ("N", "Locative nominal form"),
    (57, 2): ("N", "Temporal expression formed from a noun"),
    (382, 10): ("N", "Locative nominal expression"),
    (488, 5): ("N", "Locative noun"),
    (342, 1): ("N", "Locative compound noun"),
    (1435, 2): ("N", "Nominal expression"),
    (871, 7): ("N", "Nominal expression"),
    (1154, 2): ("N", "Directional nominal form"),
    (863, 9): ("N", "Directional nominal form"),
    (387, 0): ("N", "Locative noun"),
    (932, 3): ("N", "Locative noun"),
    (1130, 3): ("N", "Locative noun"),
    (1066, 1): ("N", "Locative noun"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in alc_batch10_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

alc_status = (
    manual_review[
        manual_review["Source_Family"] == "ALC"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(alc_status)

print(
    "Total ALC occurrences:",
    (manual_review["Source_Family"] == "ALC").sum()
)

In [ ]:
# Cell 89: Inspect final unmapped ALC occurrences

alc_remaining = manual_review[
    (manual_review["Source_Family"] == "ALC") &
    (manual_review["Target_Tag"].isna())
].copy()

alc_remaining = alc_remaining.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped ALC occurrences:", len(alc_remaining))

display(
    alc_remaining[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 90: Final ALC occurrence-level mappings

alc_batch11_confirmed = {
    (708, 3): ("N", "Accusative nominal expression indicating destination"),
    (254, 2): ("N", "Instrumental nominal duration expression"),
    (65, 4): ("N", "Locative temporal noun"),
    (1389, 1): ("N", "Accusative noun in a stage direction"),
    (643, 8): ("AVP", "Spatial expression: in front of"),

    # Provisional classifications; retain for final audit
    (476, 1): ("AVP", "Spatial expression: inside; provisional"),
    (1013, 1): ("AVP", "Duration expression; provisional"),
    (1017, 1): ("AVP", "Duration expression; provisional"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in alc_batch11_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "ALC") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = (
        "Medium" if "provisional" in reason.lower() else "High"
    )

    mapped_count += 1

print("Successfully mapped:", mapped_count)

alc_status = (
    manual_review[
        manual_review["Source_Family"] == "ALC"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(alc_status)

print(
    "Total ALC occurrences:",
    (manual_review["Source_Family"] == "ALC").sum()
)

In [ ]:
# Cell 91: Final ALC audit

alc_final = manual_review[
    manual_review["Source_Family"] == "ALC"
].copy()

print("Total ALC occurrences:", len(alc_final))
print("Mapped:", alc_final["Target_Tag"].notna().sum())
print("Unmapped:", alc_final["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
display(alc_final["Target_Tag"].value_counts(dropna=False))

print("\nMapping confidence distribution:")
display(alc_final["Mapping_Confidence"].value_counts(dropna=False))

print("\nRemaining unmapped occurrences:")
display(
    alc_final[alc_final["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

# Validate target labels
invalid_tags = alc_final[
    alc_final["Target_Tag"].notna() &
    ~alc_final["Target_Tag"].isin(TARGET_TAGS)
]

print("\nInvalid target tags:", len(invalid_tags))

# Save updated occurrence-level review
MANUAL_REVIEW_PATH = (
    RESULTS_DIR / "occurrence_level_mapping_review.csv"
)

manual_review.to_csv(MANUAL_REVIEW_PATH, index=False)

print("\nSaved updated review to:", MANUAL_REVIEW_PATH)

In [ ]:
# Cell 92: AMN family overview

amn_review = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

print("Total AMN occurrences:",
      (manual_review["Source_Family"] == "AMN").sum())

print("Already mapped:",
      manual_review[
          (manual_review["Source_Family"] == "AMN") &
          (manual_review["Target_Tag"].notna())
      ].shape[0])

print("Unmapped AMN occurrences:", len(amn_review))

print("\nSource tag distribution:")
display(amn_review["Source_Tag"].value_counts())

print("\nMost frequent words:")
display(amn_review["Word"].value_counts().head(40))

print("\nUnique words:", amn_review["Word"].nunique())

In [ ]:
# Cell 93: AMN occurrence-level contextual review

amn_review = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

# Select frequent words for initial review
priority_words = [
    "एवं", "कथं", "अतः", "पुनः", "कथम्",
    "कदाचित्", "सुखेन", "अकस्मात्",
    "शनैः", "शीघ्रम्", "एवम्", "तथा",
    "त्वरया", "क्रमशः", "सत्यम्",
    "आत्मगतम्", "सत्यं", "कष्टेन",
    "उच्चैः", "सस्मितम्"
]

amn_priority = amn_review[
    amn_review["Word"].isin(priority_words)
].copy()

amn_priority = amn_priority.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Priority AMN occurrences:", len(amn_priority))
print("Unique priority words:", amn_priority["Word"].nunique())

display(
    amn_priority[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 94: Inspect AMN priority occurrences in smaller batches

amn_priority = amn_priority.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

display(
    amn_priority.iloc[0:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 95: AMN occurrence-level mapping (batch 1)

amn_batch1_confirmed = {
    # अकस्मात्: suddenly
    (400, 3): ("AVP", "Adverb of suddenness"),
    (510, 0): ("AVP", "Adverb of suddenness"),
    (685, 4): ("AVP", "Adverb of suddenness"),
    (770, 0): ("AVP", "Adverb of suddenness"),
    (780, 0): ("AVP", "Adverb of suddenness"),
    (1284, 2): ("AVP", "Adverb of suddenness"),

    # अतः: discourse connector
    (234, 0): ("AVN", "Discourse connector: therefore"),
    (411, 0): ("AVN", "Discourse connector: therefore"),
    (564, 0): ("AVN", "Discourse connector: therefore"),
    (568, 0): ("AVN", "Discourse connector: therefore"),
    (659, 0): ("AVN", "Discourse connector: therefore"),
    (665, 0): ("AVN", "Discourse connector: therefore"),
    (764, 0): ("AVN", "Discourse connector: therefore"),
    (827, 0): ("AVN", "Discourse connector: therefore"),
    (835, 0): ("AVN", "Discourse connector: therefore"),
    (852, 0): ("AVN", "Discourse connector: therefore"),
    (1098, 0): ("AVN", "Discourse connector: therefore"),

    # आत्मगतम्: stage-direction marker
    (1406, 3): ("AVN", "Indeclinable stage-direction marker"),
    (1413, 3): ("AVN", "Indeclinable stage-direction marker"),
    (1431, 0): ("AVN", "Indeclinable stage-direction marker"),
    (1447, 3): ("AVN", "Indeclinable stage-direction marker"),

    # उच्चैः: manner/intensity adverb
    (712, 1): ("AVP", "Adverb of manner or intensity"),
    (782, 6): ("AVP", "Adverb of manner or intensity"),
    (1089, 7): ("AVP", "Adverb of manner or intensity"),

    # एवं: thus/in this manner
    (12, 0): ("AVP", "Adverb meaning thus"),
    (14, 0): ("AVP", "Adverb meaning thus"),
    (17, 0): ("AVP", "Adverb meaning thus"),
    (233, 4): ("AVP", "Adverb meaning thus"),
    (310, 15): ("AVP", "Adverb meaning thus"),
    (902, 0): ("AVP", "Adverb meaning thus"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in amn_batch1_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

amn_status = (
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(amn_status)

In [ ]:
# Cell 96: Review the next 30 AMN priority occurrences

amn_priority = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna()) &
    (manual_review["Word"].isin(priority_words))
].copy()

amn_priority = amn_priority.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining priority occurrences:", len(amn_priority))

display(
    amn_priority.iloc[30:60][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 97: AMN occurrence-level mapping (batch 2)

amn_batch2_confirmed = {
    # कथम्: interrogative/manner adverb
    (1431, 2): ("AVP", "Interrogative adverb of manner"),

    # कदाचित्: indefinite temporal adverb
    (613, 9): ("AVP", "Indefinite temporal adverb"),
    (654, 0): ("AVP", "Indefinite temporal adverb"),
    (681, 0): ("AVP", "Indefinite temporal adverb"),
    (734, 7): ("AVP", "Indefinite temporal adverb"),
    (762, 1): ("AVP", "Indefinite temporal adverb"),
    (839, 6): ("AVP", "Indefinite temporal adverb"),

    # कष्टेन: instrumental noun form
    (340, 2): ("N", "Instrumental noun expressing difficulty"),
    (603, 0): ("N", "Instrumental noun expressing difficulty"),
    (609, 3): ("N", "Instrumental noun expressing difficulty"),

    # क्रमशः: adverb meaning gradually/in sequence
    (443, 12): ("AVP", "Adverb of sequence"),
    (469, 3): ("AVP", "Adverb of sequence"),
    (470, 1): ("AVP", "Adverb of sequence"),
    (615, 0): ("AVP", "Adverb of sequence"),

    # तथा: adverb meaning thus/in that manner
    (375, 21): ("AVP", "Adverb meaning thus"),
    (719, 2): ("AVP", "Adverb meaning thus"),
    (854, 8): ("AVP", "Adverb meaning thus"),
    (941, 5): ("AVP", "Adverb meaning thus"),

    # त्वरया: instrumental noun form
    (408, 2): ("N", "Instrumental noun expressing speed"),
    (408, 8): ("N", "Instrumental noun expressing speed"),
    (771, 2): ("N", "Instrumental noun expressing speed"),
    (1260, 2): ("N", "Instrumental noun expressing speed"),

    # पुनः: adverb meaning again
    (246, 1): ("AVP", "Adverb meaning again"),
    (246, 2): ("AVP", "Adverb meaning again"),
    (392, 13): ("AVP", "Adverb meaning again"),
    (740, 0): ("AVP", "Adverb meaning again"),
    (802, 0): ("AVP", "Adverb meaning again"),
    (803, 4): ("AVP", "Adverb meaning again"),
    (1171, 0): ("AVP", "Adverb meaning again"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in amn_batch2_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

amn_status = (
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(amn_status)

In [ ]:
# Cell 98: Review next AMN priority occurrences

amn_priority = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna()) &
    (manual_review["Word"].isin(priority_words))
].copy()

amn_priority = amn_priority.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining priority occurrences:", len(amn_priority))

display(
    amn_priority.iloc[30:60][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 99: AMN occurrence-level mapping (batch 3)

amn_batch3_confirmed = {
    # तथा: discourse expression
    (1384, 2): ("AVN", "Discourse expression meaning thus"),

    # पुनः: again
    (1174, 10): ("AVP", "Adverb meaning again"),

    # शनैः: slowly/gradually
    (171, 1): ("AVP", "Adverb of manner: slowly"),
    (171, 2): ("AVP", "Adverb of manner: slowly"),
    (587, 5): ("AVP", "Adverb of manner: gradually"),
    (587, 6): ("AVP", "Adverb of manner: gradually"),
    (688, 14): ("AVP", "Adverb of manner: slowly"),

    # शीघ्रम्: quickly
    (37, 0): ("AVP", "Adverb of manner: quickly"),
    (75, 0): ("AVP", "Adverb of manner: quickly"),
    (173, 0): ("AVP", "Adverb of manner: quickly"),
    (800, 6): ("AVP", "Adverb of manner: quickly"),
    (867, 2): ("AVP", "Adverb of manner: quickly"),

    # सत्यं: adjectival and nominal uses
    (720, 5): ("A", "Adjective meaning true"),
    (749, 8): ("A", "Adjective meaning true"),
    (1351, 4): ("N", "Noun meaning truth, object of speak"),

    # सत्यम्: context-dependent uses
    (107, 0): ("A", "Predicative adjectival use"),
    (612, 0): ("N", "Noun meaning truth"),
    (1080, 0): ("AVN", "Discourse/intensifying use: really"),
    (1132, 0): ("N", "Noun meaning truth"),

    # सस्मितम्: with a smile
    (1371, 3): ("AVP", "Manner expression: with a smile"),
    (1393, 5): ("AVP", "Manner expression: with a smile"),
    (1440, 3): ("AVP", "Manner expression: with a smile"),

    # सुखेन: instrumental nominal form
    (280, 6): ("N", "Instrumental noun expressing comfort or ease"),
    (282, 7): ("N", "Instrumental noun expressing comfort or ease"),
    (326, 2): ("N", "Instrumental noun expressing well-being"),
    (382, 25): ("N", "Instrumental noun expressing comfort"),
    (586, 4): ("N", "Instrumental noun expressing comfort"),
    (1146, 2): ("N", "Instrumental noun expressing well-being"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in amn_batch3_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = (
        "Medium" if tag in ["A", "AVN"] and
        manual_review.loc[mask, "Word"].iloc[0] in ["सत्यम्", "सत्यं"]
        else "High"
    )

    mapped_count += 1

print("Successfully mapped:", mapped_count)

amn_status = (
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(amn_status)

In [ ]:
# Cell 100: Review remaining AMN priority occurrences

amn_priority_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna()) &
    (manual_review["Word"].isin(priority_words))
].copy()

amn_priority_remaining = amn_priority_remaining.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining priority occurrences:", len(amn_priority_remaining))

display(
    amn_priority_remaining.iloc[0:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 101: AMN occurrence-level mapping (batch 4)

amn_batch4_confirmed = {
    # एवं
    (904, 0): ("AVP", "Adverb meaning thus"),
    (907, 0): ("AVP", "Adverb meaning thus"),
    (1172, 6): ("AVN", "Conditional discourse expression: if so"),
    (1174, 9): ("AVP", "Adverb meaning in this way"),
    (1258, 2): ("AVP", "Adverb meaning in this manner"),
    (1333, 3): ("AVN", "Conditional discourse expression: if so"),
    (1397, 3): ("AVP", "Adverb meaning thus"),

    # एवम्
    (286, 0): ("AVP", "Adverb meaning thus"),
    (739, 2): ("AVP", "Adverb meaning in this way"),
    (992, 0): ("AVP", "Adverb meaning thus"),
    (1170, 1): ("AVP", "Adverb meaning thus"),
    (1345, 1): ("AVN", "Conditional discourse expression: if so"),

    # कथं
    (242, 3): ("AVP", "Interrogative adverb: how"),
    (313, 18): ("AVP", "Interrogative adverb: how"),
    (325, 1): ("AVP", "Interrogative adverb: how"),
    (453, 2): ("AVP", "Interrogative adverb: how"),
    (509, 7): ("AVP", "Interrogative adverb: how"),
    (535, 31): ("AVP", "Interrogative adverb: how"),
    (666, 1): ("AVP", "Interrogative adverb: how"),
    (817, 2): ("AVP", "Interrogative adverb: how"),
    (820, 1): ("AVP", "Interrogative adverb: how"),
    (822, 2): ("AVP", "Interrogative adverb: how"),
    (1248, 0): ("AVP", "Interrogative adverb: how"),
    (1378, 12): ("AVP", "Interrogative adverb: how"),

    # कथम्
    (413, 2): ("AVP", "Interrogative adverb: how"),
    (509, 13): ("AVP", "Interrogative adverb: how"),
    (662, 0): ("AVP", "Interrogative adverb: how"),
    (1358, 0): ("AVP", "Interrogative adverb: how"),
    (1406, 5): ("AVP", "Interrogative adverb: how"),
    (1420, 12): ("AVP", "Interrogative adverb: how"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in amn_batch4_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

amn_status = (
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(amn_status)

In [ ]:
# Cell 102: Review remaining unmapped AMN words

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(amn_remaining["Word"].value_counts().head(50))

print("\nSource tag distribution:")
display(amn_remaining["Source_Tag"].value_counts())

# Display first 30 occurrences with context
amn_remaining = amn_remaining.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

display(
    amn_remaining.iloc[0:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 103: AMN occurrence-level mapping (batch 5)

amn_batch5_confirmed = {
    # Nominal / instrumental forms
    (260, 0): ("N", "Instrumental nominal expression"),
    (633, 0): ("N", "Instrumental nominal form"),
    (280, 7): ("N", "Instrumental nominal expression"),
    (310, 30): ("N", "Instrumental noun expressing pride"),
    (365, 0): ("N", "Instrumental noun expressing pride"),
    (938, 5): ("N", "Nominal expression indicating role or form"),
    (742, 2): ("N", "Nominal expression indicating accordance"),
    (1418, 3): ("N", "Instrumental noun expressing emotion"),
    (299, 5): ("N", "Instrumental noun expressing enthusiasm"),

    # Discourse connector
    (1130, 0): ("AVN", "Discourse connector: therefore"),

    # Indeclinable adverbs
    (1060, 1): ("AVP", "Degree adverb: very"),
    (7, 3): ("AVP", "Degree adverb: excessively"),
    (688, 0): ("AVP", "Temporal adverb: today"),
    (710, 0): ("AVP", "Spatial adverb: inside"),
    (801, 0): ("AVP", "Adverb: otherwise"),
    (500, 0): ("AVP", "Adverb expressing certainty"),
    (511, 3): ("AVP", "Adverb: suddenly"),
    (980, 3): ("AVP", "Spatial adverb: here and there"),
    (958, 0): ("AVP", "Adverb: in this manner"),
    (819, 2): ("AVP", "Frequency adverb: once"),

    # Pronouns
    (903, 0): ("SN", "Independent demonstrative pronoun"),
    (746, 0): ("SN", "Independent demonstrative pronoun"),

    # Adjectives
    (688, 21): ("A", "Adjective modifying स्वरेण"),
    (694, 4): ("A", "Adjective modifying स्वरेण"),
    (1115, 1): ("A", "Demonstrative adjective"),
    (447, 3): ("A", "Demonstrative adjective"),
    (964, 7): ("A", "Demonstrative adjective"),
}

mapped_count = 0

for (sid, pos), (tag, reason) in amn_batch5_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        print(f"Warning: {(sid, pos)} matched {mask.sum()} rows")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)

amn_status = (
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"]
    .value_counts(dropna=False)
)

display(amn_status)

In [ ]:
# Cell 104: Inspect next 30 unmapped AMN occurrences

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_remaining = amn_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

display(
    amn_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 105: AMN occurrence-level mapping (batch 6)

amn_batch6_confirmed = {
    # Instrumental nominal forms
    (12, 1): ("N", "Instrumental nominal form expressing manner"),
    (96, 4): ("N", "Instrumental nominal form"),
    (99, 2): ("N", "Nominal expression indicating form or role"),
    (206, 2): ("N", "Instrumental noun expressing courage"),
    (271, 3): ("N", "Instrumental nominal form expressing speed"),
    (300, 2): ("N", "Instrumental noun expressing pride"),
    (303, 9): ("N", "Nominal expression indicating form"),
    (307, 2): ("N", "Instrumental noun expressing pride"),
    (312, 4): ("N", "Instrumental noun expressing humility"),
    (316, 6): ("N", "Instrumental noun expressing dedication"),
    (316, 8): ("N", "Instrumental noun expressing effort"),
    (364, 1): ("N", "Instrumental nominal form expressing speed"),
    (364, 8): ("N", "Instrumental noun expressing patience"),
    (367, 1): ("N", "Noun functioning as the object of आश्रितवान्"),

    # Adjectives / adjective-like expressions
    (81, 2): ("A", "Adjectival expression: accompanied by puri"),

    # Adverbial indeclinables
    (36, 6): ("AVP", "Adverbial expression: according to procedure"),
    (62, 1): ("AVP", "Adverb of specification"),
    (63, 0): ("AVP", "Adverb of specification"),
    (88, 3): ("AVP", "Adverb expressing similarity"),
    (253, 0): ("AVP", "Adverb expressing manner"),
    (277, 6): ("AVP", "Adverb expressing continuity"),
    (316, 4): ("AVP", "Adverb expressing sincerity"),
    (319, 2): ("AVP", "Adverb expressing absence of charge"),
    (342, 9): ("AVP", "Adverb expressing direct experience"),
    (363, 0): ("AVP", "Adverb expressing manner or detail"),
    (365, 1): ("AVP", "Adverb expressing joy"),

    # Discourse marker
    (132, 1): ("AVN", "Discourse marker: in fact"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in amn_batch6_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated AMN tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 106: Inspect next 30 unmapped AMN occurrences

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_remaining = amn_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

display(
    amn_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 107: AMN occurrence-level mapping (batch 7)

amn_batch7_confirmed = {
    # Nominal forms
    (373, 2): ("N", "Instrumental nominal expression indicating form"),
    (403, 0): ("N", "Instrumental nominal expression: by mere sight"),
    (526, 2): ("N", "Instrumental noun expressing sorrow"),
    (542, 5): ("N", "Nominal expression indicating form or manner"),
    (564, 2): ("N", "Nominal expression indicating role"),
    (576, 4): ("N", "Instrumental noun expressing anger"),
    (637, 8): ("N", "Nominal expression indicating role"),
    (660, 11): ("N", "Instrumental noun expressing devotion"),
    (300, 2): ("N", "Instrumental noun expressing pride"),

    # Adverbs / indeclinables
    (375, 19): ("AVN", "Correlative/subordinating indeclinable"),
    (375, 24): ("AVP", "Indeclinable adverb: somehow"),
    (377, 3): ("AVP", "Adverb: once"),
    (382, 18): ("AVP", "Adverb expressing certainty"),
    (410, 5): ("AVP", "Temporal adverb: recently"),
    (434, 0): ("AVP", "Adverb: simultaneously"),
    (441, 2): ("AVP", "Adverb expressing appropriate timing"),
    (457, 12): ("AVP", "Adverb: in the same way"),
    (468, 1): ("AVP", "Adverb expressing natural disposition"),
    (590, 4): ("AVP", "Adverb: especially"),
    (591, 4): ("AVP", "Adverb: especially"),
    (608, 3): ("N", "Nominal expression indicating form or manner"),

    # Adjectives
    (387, 4): ("A", "Adjective describing a helpless person"),
    (476, 3): ("A", "Adjective: eternal"),

    # Relative and correlative pronouns
    (554, 0): ("SN", "Standalone relative pronoun"),
    (554, 3): ("SN", "Standalone correlative pronoun"),

    # Nominal object
    (575, 5): ("N", "Noun used as the object of तिष्ठतु"),

    # Noun used as an adverbial expression
    (511, 7): ("N", "Nominal expression indicating confusion"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in amn_batch7_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated AMN tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 108: Inspect next 30 unmapped AMN occurrences

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_remaining = amn_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

display(
    amn_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 109: AMN occurrence-level mapping (batch 8)

amn_batch8_confirmed = {
    # AVP: Adverbial indeclinables
    (693, 0): ("AVP", "Temporal expression: by that time"),
    (726, 2): ("AVP", "Adverb expressing secrecy"),
    (736, 2): ("AVP", "Adverb: in the same way"),
    (744, 4): ("AVP", "Adverb: immediately"),
    (773, 5): ("AVP", "Adverb: quickly"),
    (776, 2): ("AVP", "Adverb expressing certainty"),
    (804, 1): ("AVP", "Adverb: directly or personally"),
    (811, 7): ("AVP", "Adverb: in vain"),
    (812, 0): ("AVP", "Adverb: in vain"),
    (838, 0): ("AVP", "Adverb: generally"),
    (859, 9): ("AVP", "Adverb: silently"),
    (890, 4): ("AVP", "Adverb expressing joy"),
    (929, 7): ("AVP", "Adverb expressing completeness"),

    # AVN: General indeclinables / connectives
    (702, 0): ("AVN", "Causal connective: because"),
    (854, 5): ("AVN", "Subordinating connective: as"),

    # N: Nominal forms
    (724, 2): ("N", "Noun: determination, object of कृतवान्"),
    (782, 1): ("N", "Instrumental noun expressing skill"),
    (782, 10): ("N", "Noun: urgency, object of कुरु"),
    (811, 1): ("N", "Instrumental noun expressing anger"),
    (824, 3): ("N", "Instrumental nominal form"),
    (902, 1): ("N", "Instrumental nominal expression"),
    (933, 4): ("N", "Nominal expression indicating role"),

    # SNV: Pronominal adjective modifying रूपेण
    (824, 1): ("SNV", "Interrogative pronoun modifying रूपेण"),

    # A: Adjective
    (931, 1): ("A", "Adjective modifying चण्डरवम्"),

    # AVC: Numerical / frequency expression
    (819, 5): ("AVC", "Numerical frequency expression: three times"),

    # AVP: Comparative adverb
    (706, 2): ("AVP", "Comparative expression: in comparison with that"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in amn_batch8_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated AMN tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 110: Investigate failed occurrence (804, 1)

failed_sid = 804
failed_pos = 1

print("Matching rows in manual_review:")

display(
    manual_review[
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == failed_sid)
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Full_Sentence"
        ]
    ]
)

print("\nAll AMN rows containing साक्षात्:")

display(
    manual_review[
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["Word"].astype(str).str.contains("साक्षात्", na=False))
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 111: Correct साक्षात् mapping

sid = 804
pos = 4

mask = (
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["sentence_id"] == sid) &
    (manual_review["Token_Position"] == pos) &
    (manual_review["Word"] == "साक्षात्")
)

assert mask.sum() == 1, f"Expected 1 matching row, found {mask.sum()}"

manual_review.loc[mask, "Target_Tag"] = "AVP"
manual_review.loc[mask, "Mapping_Reason"] = (
    "Adverb expressing direct or personal presence"
)
manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Corrected occurrence:")
display(
    manual_review.loc[
        mask,
        ["sentence_id", "Token_Position", "Word",
         "Target_Tag", "Mapping_Reason", "Mapping_Confidence"]
    ]
)

print("\nUpdated AMN distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 112: Inspect next 30 unmapped AMN occurrences

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_remaining = amn_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

display(
    amn_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 113: AMN occurrence-level mapping (batch 9)

amn_batch9_confirmed = {
    # AVP: Adverbial indeclinables
    (942, 13): ("AVP", "Adverb expressing distribution: piece by piece"),
    (977, 6): ("AVP", "Adverb expressing manner: sweetly"),
    (1083, 1): ("AVP", "Adverb expressing speed: quickly"),
    (1117, 3): ("AVP", "Adverb expressing joy"),
    (1127, 0): ("AVP", "Adverb: suddenly"),
    (1176, 3): ("AVP", "Adverb expressing comfort or happiness"),
    (1190, 3): ("AVP", "Comparative adverb: more quickly"),
    (1200, 12): ("AVP", "Adverb expressing completeness"),
    (1254, 5): ("AVP", "Adverb expressing naturalness"),
    (1256, 2): ("AVP", "Adverb expressing completeness"),
    (1313, 0): ("AVP", "Adverb: generally"),
    (1313, 21): ("AVP", "Adverb expressing regularity"),
    (1324, 1): ("AVP", "Adverb: properly"),
    (1324, 8): ("AVP", "Indeclinable expressing preference: rather"),
    (1326, 0): ("AVP", "Adverb: generally"),

    # AVN: General connective
    (1172, 5): ("AVN", "Conditional connective: if"),

    # AVU: Interjection
    (1175, 0): ("AVU", "Interjection expressing approval"),

    # N: Nominal instrumental forms
    (1086, 6): ("N", "Instrumental nominal expression indicating form"),
    (1316, 7): ("N", "Instrumental noun expressing sorrow"),

    # SN: Standalone relative pronoun
    (1086, 4): ("SN", "Standalone relative pronoun"),

    # SNV: Pronominal adjectives
    (1241, 5): ("SNV", "Interrogative pronominal adjective"),
    (1260, 1): ("SNV", "Demonstrative pronominal adjective"),

    # A: Adjectives
    (1261, 6): ("A", "Adjective modifying गत्या"),
    (1319, 3): ("A", "Adjective modifying इच्छा"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in amn_batch9_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated AMN tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 114: Investigate नैसर्गिकतया

display(
    manual_review[
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["Word"].astype(str).str.contains(
            "नैसर्गिकतया", na=False
        ))
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 115: Correct नैसर्गिकतया mapping

sid = 1254
pos = 4

mask = (
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["sentence_id"] == sid) &
    (manual_review["Token_Position"] == pos) &
    (manual_review["Word"] == "नैसर्गिकतया")
)

assert mask.sum() == 1, f"Expected 1 row, found {mask.sum()}"

manual_review.loc[mask, "Target_Tag"] = "AVP"
manual_review.loc[mask, "Mapping_Reason"] = (
    "Adverb expressing naturalness"
)
manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Corrected occurrence:")
display(
    manual_review.loc[
        mask,
        ["sentence_id", "Token_Position", "Word",
         "Target_Tag", "Mapping_Reason", "Mapping_Confidence"]
    ]
)

print("\nUpdated AMN distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 116: Inspect final unmapped AMN occurrences

amn_remaining = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_remaining = amn_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped AMN occurrences:", len(amn_remaining))
print("Unique remaining words:", amn_remaining["Word"].nunique())

display(
    amn_remaining[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 117: AMN occurrence-level mapping (final batch)

amn_batch10_confirmed = {
    (1326, 5): ("AVP", "Adverb expressing correctness"),
    (1326, 6): ("N", "Instrumental nominal form"),
    (1327, 3): ("AVP", "Adverb expressing completeness"),
    (1341, 0): ("A", "Adjective modifying गृहम्"),
    (1349, 13): ("AVP", "Adverb expressing falsity"),
    (1409, 3): ("AVP", "Dramatic stage direction"),
    (1417, 0): ("AVP", "Dramatic stage direction"),
    (1430, 3): ("AVP", "Dramatic stage direction"),
    (1448, 1): ("AVP", "Dramatic stage direction"),
    (1460, 0): ("AVP", "Adverb expressing summary"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in amn_batch10_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "AMN") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nFinal AMN tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "AMN"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 118: Final unresolved AMN contextual review

amn_unresolved = manual_review[
    (manual_review["Source_Family"] == "AMN") &
    (manual_review["Target_Tag"].isna())
].copy()

amn_unresolved = amn_unresolved.sort_values(
    ["sentence_id", "Token_Position"]
)

print("Unresolved AMN occurrences:", len(amn_unresolved))

display(
    amn_unresolved[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)


In [ ]:
# Cell 119: Final AMN audit and save

amn_final = manual_review[
    manual_review["Source_Family"] == "AMN"
].copy()

print("========== FINAL AMN AUDIT ==========")

print("Total occurrences:", len(amn_final))
print("Mapped occurrences:", amn_final["Target_Tag"].notna().sum())
print("Unmapped occurrences:", amn_final["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
display(
    amn_final["Target_Tag"].value_counts(dropna=False)
)

print("\nMapping confidence:")
display(
    amn_final["Mapping_Confidence"].value_counts(dropna=False)
)

print("\nUnresolved occurrences:")
display(
    amn_final[
        amn_final["Target_Tag"].isna()
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

# Verify mapping integrity
assert len(amn_final) == 264
assert amn_final["Target_Tag"].notna().sum() == 257
assert amn_final["Target_Tag"].isna().sum() == 7

# Verify target tags
invalid_tags = amn_final[
    amn_final["Target_Tag"].notna() &
    ~amn_final["Target_Tag"].isin(TARGET_TAGS)
]

print("\nInvalid target tags:", len(invalid_tags))

assert len(invalid_tags) == 0

# Save complete occurrence-level review
OUTPUT_FILE = RESULTS_DIR / "occurrence_level_mapping_review.csv"

manual_review.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully to:", OUTPUT_FILE)

In [ ]:
# Cell 120: DAB family overview

dab_data = manual_review[
    manual_review["Source_Family"] == "DAB"
].copy()

print("========== DAB FAMILY OVERVIEW ==========")

print("Total occurrences:", len(dab_data))
print("Unique words:", dab_data["Word"].nunique())
print("Unique source tags:", dab_data["Source_Tag"].nunique())

print("\nSource-tag distribution:")
display(dab_data["Source_Tag"].value_counts())

print("\nMost frequent words:")
display(dab_data["Word"].value_counts().head(40))

print("\nCurrent mapping status:")
display(dab_data["Target_Tag"].value_counts(dropna=False))

In [ ]:
# Cell 121: DAB occurrence-level contextual review

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("========== DAB CONTEXTUAL REVIEW ==========")

print("Total DAB occurrences:",
      len(manual_review[manual_review["Source_Family"] == "DAB"]))

print("Already mapped:",
      manual_review[
          (manual_review["Source_Family"] == "DAB") &
          (manual_review["Target_Tag"].notna())
      ].shape[0])

print("Remaining unmapped:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent unmapped words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nFirst 30 unmapped occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Target_Tag"
        ]
    ]
)

In [ ]:
# Cell 122: DAB occurrence-level mapping (batch 1)

dab_batch1_confirmed = {
    # SNV: Demonstratives modifying nouns
    (10, 5): ("SNV", "Demonstrative modifying तीर्थस्थानेषु"),
    (41, 6): ("SNV", "Demonstrative modifying महर्षिः"),
    (136, 2): ("SNV", "Demonstrative modifying पत्राणि"),
    (145, 0): ("SNV", "Demonstrative modifying कारणेन"),
    (218, 0): ("SNV", "Demonstrative modifying सगरपुत्रैः"),
    (223, 0): ("SNV", "Demonstrative modifying प्रदेशेषु"),
    (223, 1): ("SNV", "Demonstrative modifying प्रदेशेषु"),
    (235, 0): ("SNV", "Demonstrative modifying तीर्थेन"),
    (248, 2): ("SNV", "Demonstrative modifying फलम्"),
    (249, 1): ("SNV", "Demonstrative modifying फलम्"),
    (251, 0): ("SNV", "Demonstrative modifying शय्यायाम्"),
    (251, 3): ("SNV", "Demonstrative modifying फलम्"),
    (253, 1): ("SNV", "Demonstrative modifying स्थले"),
    (259, 2): ("SNV", "Demonstrative modifying फलम्"),
    (259, 5): ("SNV", "Demonstrative modifying फलम्"),
    (264, 3): ("SNV", "Demonstrative modifying फलम्"),
    (268, 0): ("SNV", "Demonstrative modifying फलस्य"),
    (269, 5): ("SNV", "Demonstrative modifying फलम्"),
    (271, 0): ("SNV", "Demonstrative modifying फलस्य"),
    (273, 3): ("SNV", "Demonstrative modifying वृक्षाः"),
    (277, 12): ("SNV", "Demonstrative modifying अवतारविशेषस्य"),
    (278, 3): ("SNV", "Demonstrative modifying फलस्य"),
    (280, 1): ("SNV", "Demonstrative modifying प्रसादफले"),
    (284, 0): ("SNV", "Demonstrative modifying पन्थाः"),
    (285, 0): ("SNV", "Demonstrative modifying मार्गः"),
    (286, 5): ("SNV", "Demonstrative modifying फलम्"),
    (287, 0): ("SNV", "Demonstrative modifying नारिकेलफलम्"),

    # SN: Independent demonstrative pronoun
    (39, 6): ("SN", "Independent pronoun referring to previously mentioned principles"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch1_confirmed.items():

    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    # Avoid overwriting an existing annotation
    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review[
        manual_review["Source_Family"] == "DAB"
    ]["Target_Tag"].value_counts(dropna=False)
)

In [ ]:
# Cell 123: Inspect next 30 unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nNext 30 occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 124: Map confirmed DAB occurrences from batch 2

dab_batch2_confirmed = {
    (300, 7): ("SNV", "Demonstrative modifying वर्षे"),
    (302, 0): ("SNV", "Demonstrative modifying कारणात्"),
    (304, 0): ("SNV", "Demonstrative modifying वर्षे"),
    (306, 0): ("SNV", "Demonstrative modifying दुष्टाः"),
    (316, 0): ("SNV", "Demonstrative modifying कारणात्"),
    (322, 0): ("SNV", "Demonstrative modifying कारणात्"),
    (322, 16): ("SNV", "Demonstrative modifying कारणात्"),
    (324, 1): ("SNV", "Demonstrative modifying राजानम्"),
    (327, 2): ("SNV", "Demonstrative modifying मार्गम्"),
    (328, 4): ("SN", "Independent demonstrative pronoun"),
    (337, 2): ("SNV", "Demonstrative modifying प्रकोष्ठे"),
    (376, 0): ("SNV", "Demonstrative modifying दिने"),
    (410, 3): ("SNV", "Demonstrative modifying नगरे"),
    (412, 0): ("SNV", "Demonstrative modifying दिनात्"),
    (426, 2): ("SNV", "Demonstrative modifying मिष्टान्नम्"),
    (442, 2): ("SNV", "Demonstrative modifying आलोकः"),
    (458, 1): ("SNV", "Demonstrative modifying आलोकम्"),
    (469, 0): ("SNV", "Demonstrative modifying सर्वम्"),
    (488, 11): ("SN", "Independent demonstrative pronoun referring to two people"),
    (499, 4): ("SNV", "Demonstrative modifying नगरम्"),
    (505, 1): ("SN", "Independent pronoun referring to यशोधरा"),
    (506, 0): ("SN", "Independent pronoun referring to वसुधा"),
    (509, 2): ("SNV", "Demonstrative modifying कथा"),
    (526, 6): ("SNV", "Demonstrative modifying बालकः"),
    (528, 3): ("SNV", "Demonstrative modifying विषये"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch2_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 125: Inspect next 30 unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nNext 30 occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 126: Map confirmed DAB occurrences from batch 3

dab_batch3_confirmed = {
    (530, 4): ("SNV", "Demonstrative modifying विषये"),
    (539, 3): ("SNV", "Demonstrative modifying बालकः"),
    (544, 3): ("SNV", "Demonstrative modifying सर्वम्"),
    (550, 4): ("SNV", "Demonstrative modifying समयः"),
    (551, 3): ("SNV", "Demonstrative modifying अनुरोधः"),
    (561, 3): ("SNV", "Demonstrative modifying प्रकारेण"),
    (567, 0): ("SNV", "Demonstrative modifying भावना"),
    (598, 3): ("SNV", "Demonstrative modifying जनाः"),
    (601, 2): ("SNV", "Demonstrative modifying बालकः"),
    (615, 2): ("SNV", "Demonstrative modifying बालकस्य"),
    (618, 1): ("SNV", "Demonstrative modifying द्वारपालकस्य"),
    (643, 1): ("SNV", "Demonstrative modifying धनेन"),
    (643, 9): ("SNV", "Demonstrative modifying अधमस्य"),
    (669, 14): ("SN", "Independent demonstrative pronoun referring to गुरुनानकः"),
    (671, 0): ("SNV", "Demonstrative modifying वचनानि"),
    (683, 7): ("SNV", "Demonstrative modifying वणिजः"),
    (684, 3): ("SNV", "Demonstrative modifying गृहात्"),
    (687, 0): ("SNV", "Demonstrative modifying मनुष्यः"),
    (693, 1): ("SNV", "Demonstrative modifying चोरः"),
    (702, 6): ("SNV", "Demonstrative modifying घटम्"),
    (704, 6): ("SNV", "Demonstrative modifying वणिक्"),
    (707, 0): ("SNV", "Demonstrative modifying implied वस्तु/धनम्"),
    (716, 9): ("SNV", "Demonstrative modifying जगति"),
    (733, 2): ("SNV", "Demonstrative modifying उपक्रमः"),
    (747, 8): ("SNV", "Demonstrative modifying मार्जाराः"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch3_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 127: Inspect next 30 unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nNext 30 occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 128: Map confirmed DAB occurrences from batch 4

dab_batch4_confirmed = {
    (749, 7): ("SNV", "Demonstrative modifying सत्यं"),
    (755, 0): ("SNV", "Demonstrative modifying ग्रामे"),
    (762, 2): ("SNV", "Demonstrative modifying ग्रामे"),
    (771, 3): ("SNV", "Demonstrative modifying याने"),
    (788, 0): ("SNV", "Demonstrative modifying नौकया"),
    (798, 0): ("SNV", "Demonstrative modifying क्षणे"),
    (799, 0): ("SNV", "Demonstrative modifying लघुयातयानम्"),
    (815, 0): ("SNV", "Demonstrative modifying घोरः विश्वासघातः"),
    (832, 2): ("SNV", "Demonstrative modifying अंशम्"),
    (832, 26): ("SNV", "Demonstrative modifying दिनेषु"),
    (835, 1): ("SNV", "Demonstrative modifying प्रसङ्गे"),
    (837, 5): ("SNV", "Demonstrative modifying स्थले"),
    (838, 1): ("SNV", "Demonstrative modifying स्थितिः"),
    (839, 3): ("SNV", "Demonstrative modifying स्थितिः"),
    (840, 2): ("SNV", "Demonstrative modifying महाविद्यालये"),
    (844, 1): ("SNV", "Demonstrative modifying स्थितौ"),
    (846, 5): ("SNV", "Demonstrative modifying विषयाय"),
    (846, 12): ("SNV", "Demonstrative modifying युक्तिवादम्"),
    (860, 0): ("SNV", "Demonstrative modifying काले"),
    (863, 2): ("SNV", "Demonstrative modifying व्याधः"),
    (864, 1): ("SNV", "Demonstrative modifying व्याधम्"),
    (866, 2): ("SNV", "Demonstrative modifying व्याधः"),
    (866, 7): ("SNV", "Demonstrative modifying कपोतानाम्"),
    (870, 1): ("SNV", "Demonstrative modifying जालम्"),
    (872, 1): ("SNV", "Demonstrative modifying जालम्"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch4_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 129: Investigate sentence 832

sentence_832_dab = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["sentence_id"] == 832)
].copy()

print("DAB occurrences in sentence 832:")
display(
    sentence_832_dab[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Full_Sentence"
        ]
    ].sort_values("Token_Position")
)

print("\nAll occurrences in sentence 832:")
display(
    manual_review[
        manual_review["sentence_id"] == 832
    ][
        [
            "Token_Position",
            "Word",
            "Source_Family",
            "Source_Tag",
            "Target_Tag"
        ]
    ].sort_values("Token_Position")
)

In [ ]:
# Cell 130: Correct the missing DAB occurrence in sentence 832

mask = (
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["sentence_id"] == 832) &
    (manual_review["Token_Position"] == 0)
)

assert mask.sum() == 1, "Expected exactly one matching occurrence"
assert manual_review.loc[mask, "Target_Tag"].isna().all(), \
    "Occurrence is already mapped"

manual_review.loc[mask, "Target_Tag"] = "SNV"
manual_review.loc[mask, "Mapping_Reason"] = (
    "Demonstrative modifying अंशम्"
)
manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Corrected occurrence:")
display(
    manual_review.loc[
        mask,
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 131: Inspect next 30 unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nNext 30 occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 132: Map confirmed DAB occurrences from batch 5

dab_batch5_confirmed = {
    (878, 3): ("SNV", "Demonstrative modifying दिवसे"),
    (881, 3): ("SNV", "Demonstrative modifying दिवसे"),
    (881, 6): ("SNV", "Demonstrative modifying उत्सवः"),
    (885, 0): ("SNV", "Demonstrative modifying दिवसे"),
    (890, 1): ("SNV", "Demonstrative modifying उत्सवम्"),
    (900, 5): ("SNV", "Demonstrative modifying तीर्थस्थानेषु"),
    (928, 0): ("SNV", "Demonstrative modifying पात्रम्"),
    (929, 2): ("SNV", "Demonstrative modifying पात्रात्"),
    (930, 0): ("SNV", "Demonstrative modifying अवस्थायाम्"),
    (930, 3): ("SNV", "Demonstrative modifying चण्डरवः"),
    (931, 2): ("SNV", "Demonstrative modifying चण्डरवम्"),
    (933, 2): ("SNV", "Demonstrative modifying वनस्य"),
    (934, 3): ("SNV", "Demonstrative modifying वने"),
    (937, 5): ("SNV", "Demonstrative modifying चण्डरवम्"),
    (938, 2): ("SNV", "Demonstrative modifying शृगालेन"),
    (939, 12): ("SNV", "Demonstrative modifying शृगालाः"),
    (941, 0): ("SNV", "Demonstrative modifying शृगालरवम्"),
    (946, 0): ("SNV", "Demonstrative modifying अवसरे"),
    (948, 0): ("SNV", "Demonstrative modifying नारी"),
    (962, 0): ("SNV", "Demonstrative modifying वर्षे"),
    (968, 16): ("SNV", "Demonstrative modifying ऋतवः"),
    (970, 0): ("SNV", "Demonstrative modifying ऋतौ"),
    (971, 0): ("SNV", "Demonstrative modifying कारणात्"),
    (971, 2): ("SNV", "Demonstrative modifying ऋतौ"),
    (972, 6): ("SNV", "Demonstrative modifying ऋतुः"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch5_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 133: Investigate sentence 934

sentence_934 = manual_review[
    manual_review["sentence_id"] == 934
].copy()

print("All occurrences in sentence 934:")
display(
    sentence_934[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Family",
            "Source_Tag",
            "Target_Tag"
        ]
    ].sort_values("Token_Position")
)

print("\nDAB occurrences in sentence 934:")
display(
    sentence_934[
        sentence_934["Source_Family"] == "DAB"
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag"
        ]
    ].sort_values("Token_Position")
)

In [ ]:
# Cell 134: Correct DAB occurrence in sentence 934

mask = (
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["sentence_id"] == 934) &
    (manual_review["Token_Position"] == 4)
)

assert mask.sum() == 1, "Expected exactly one matching occurrence"
assert manual_review.loc[mask, "Target_Tag"].isna().all(), \
    "Occurrence is already mapped"

manual_review.loc[mask, "Target_Tag"] = "SNV"
manual_review.loc[mask, "Mapping_Reason"] = (
    "Demonstrative modifying वने"
)
manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Corrected occurrence:")
display(
    manual_review.loc[
        mask,
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 135: Inspect next 30 unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nMost frequent remaining words:")
display(dab_remaining["Word"].value_counts().head(30))

print("\nNext 30 occurrences:")
display(
    dab_remaining.iloc[:30][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 136: Map confirmed DAB occurrences from batch 6

dab_batch6_confirmed = {
    (973, 0): ("SNV", "Demonstrative modifying ऋतोः"),
    (974, 0): ("SNV", "Demonstrative modifying ऋतौ"),
    (982, 0): ("SNV", "Demonstrative modifying ऋतौ"),
    (985, 1): ("SNV", "Demonstrative modifying कार्यक्रमेषु"),
    (988, 1): ("SNV", "Demonstrative modifying दिवसे"),
    (992, 1): ("SNV", "Demonstrative modifying ऋतुराजः"),
    (1020, 1): ("SNV", "Demonstrative modifying वारणानां वृन्दम्"),
    (1046, 0): ("SNV", "Demonstrative modifying महापुरुषस्य"),
    (1057, 0): ("SNV", "Demonstrative modifying महात्मा"),
    (1070, 0): ("SNV", "Demonstrative modifying बालः"),
    (1072, 0): ("SNV", "Demonstrative modifying लता"),
    (1074, 0): ("SNV", "Demonstrative modifying बालः"),
    (1076, 0): ("SNV", "Demonstrative modifying गीता"),
    (1085, 7): ("SNV", "Demonstrative modifying शिशोः"),
    (1092, 0): ("SNV", "Demonstrative modifying लता"),
    (1092, 2): ("SNV", "Demonstrative modifying बालम्"),
    (1094, 1): ("SNV", "Demonstrative modifying बालः"),
    (1098, 3): ("SNV", "Demonstrative modifying शिशोः"),
    (1103, 2): ("SNV", "Demonstrative modifying शिशुम्"),
    (1104, 0): ("SNV", "Demonstrative modifying भगिनी"),
    (1116, 0): ("SNV", "Demonstrative modifying उत्तरेण"),
    (1117, 0): ("SN", "Independent demonstrative pronoun referring to चन्द्रशेखरः"),
    (1117, 4): ("SNV", "Demonstrative modifying वेत्राघातम्"),
    (1120, 6): ("SN", "Independent demonstrative pronoun referring to the preceding event"),
    (1121, 4): ("SNV", "Demonstrative modifying सहयोगिभिः"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch6_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 137: Inspect all remaining unmapped DAB occurrences

dab_remaining = manual_review[
    (manual_review["Source_Family"] == "DAB") &
    (manual_review["Target_Tag"].isna())
].copy()

dab_remaining = dab_remaining.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Remaining unmapped DAB occurrences:", len(dab_remaining))
print("Unique remaining words:", dab_remaining["Word"].nunique())

print("\nRemaining word frequency:")
display(dab_remaining["Word"].value_counts())

print("\nAll remaining occurrences:")
display(
    dab_remaining[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

In [ ]:
# Cell 138: Map confirmed DAB occurrences from batch 7

dab_batch7_confirmed = {
    (1122, 4): ("SNV", "Demonstrative modifying क्रान्तिकारिभिः"),
    (1123, 1): ("SNV", "Demonstrative modifying विस्फोटनेन"),
    (1130, 11): ("SNV", "Demonstrative modifying गुलिकया"),
    (1131, 3): ("SNV", "Demonstrative modifying आजादः"),
    (1140, 1): ("SNV", "Demonstrative modifying वानराः"),
    (1156, 7): ("SNV", "Demonstrative modifying पिपासाकुलितेन गजयूथेन"),
    (1166, 6): ("SN", "First-person independent pronoun"),
    (1168, 0): ("SNV", "Demonstrative modifying शशकाः"),
    (1174, 1): ("SNV", "Demonstrative modifying अपराधः"),
    (1180, 0): ("SNV", "Demonstrative modifying दुष्टमहोरगकुलावासः"),
    (1198, 2): ("SNV", "Demonstrative modifying कुम्भपलाशम्"),
    (1209, 5): ("SN", "Independent demonstrative pronoun"),
    (1212, 2): ("SNV", "Demonstrative modifying बलेन"),
    (1217, 0): ("SN", "Independent demonstrative pronoun"),
    (1221, 2): ("SNV", "Demonstrative modifying अभयम्"),
    (1227, 1): ("SNV", "Demonstrative modifying जलम्"),
    (1228, 0): ("SN", "Independent demonstrative pronoun"),
    (1254, 0): ("SNV", "Demonstrative modifying कारणात्"),
    (1261, 2): ("SNV", "Demonstrative modifying चित्राणि"),
    (1270, 0): ("SNV", "Demonstrative modifying चलचित्रे"),
    (1276, 0): ("SNV", "Demonstrative modifying क्रीडाकृत्यस्य"),
    (1304, 0): ("SNV", "Demonstrative modifying चिन्तनम्"),
    (1305, 1): ("SNV", "Demonstrative modifying जगति"),
    (1317, 4): ("SNV", "Demonstrative modifying प्रयोजनम्"),
    (1323, 4): ("SNV", "Demonstrative modifying क्रियासु"),
    (1368, 14): ("SNV", "Demonstrative modifying बाले; verify full sentence"),
    (1370, 5): ("SNV", "Demonstrative modifying केसरिणी"),
    (1374, 0): ("SNV", "Demonstrative modifying बालमृगेन्द्रम्"),
    (1412, 0): ("SNV", "Demonstrative modifying मृत्तिकामयूरस्य"),
    (1419, 2): ("SNV", "Demonstrative associated with सिंहशावकविमर्दात्"),
    (1424, 0): ("SNV", "Demonstrative modifying अपराजिता"),
    (1433, 1): ("SNV", "Demonstrative modifying वृत्तान्तम्"),
    (1440, 5): ("SNV", "Demonstrative modifying विवादः"),
    (1442, 0): ("SN", "Independent demonstrative pronoun referring to शकुन्तला"),
    (1446, 0): ("SNV", "Demonstrative modifying पुरुषः"),
    (1452, 1): ("SNV", "Demonstrative modifying प्रक्रियायाम्"),
}

mapped_count = 0
failed_occurrences = []

for (sid, pos), (tag, reason) in dab_batch7_confirmed.items():
    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        failed_occurrences.append((sid, pos, int(mask.sum())))
        continue

    if manual_review.loc[mask, "Target_Tag"].notna().any():
        print(f"Already mapped: {(sid, pos)}")
        continue

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"
    mapped_count += 1

print("Successfully mapped:", mapped_count)
print("Failed occurrences:", failed_occurrences)

print("\nUpdated DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 139: Investigate failed DAB occurrences

for sid in [1140, 1221]:
    print(f"\n{'='*60}")
    print(f"Sentence ID: {sid}")

    sentence_data = manual_review[
        manual_review["sentence_id"] == sid
    ].copy()

    display(
        sentence_data[
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Source_Family",
                "Source_Tag",
                "Target_Tag"
            ]
        ].sort_values("Token_Position")
    )

In [ ]:
# Cell 140: Correct failed DAB mappings

dab_corrections = {
    (1140, 0): ("SNV", "Demonstrative modifying वानराः"),
    (1221, 3): ("SNV", "Demonstrative modifying अभयम्"),
}

corrected_count = 0

for (sid, pos), (tag, reason) in dab_corrections.items():

    mask = (
        (manual_review["Source_Family"] == "DAB") &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    assert mask.sum() == 1, f"Unexpected match count for {(sid, pos)}"
    assert manual_review.loc[mask, "Target_Tag"].isna().all()

    manual_review.loc[mask, "Target_Tag"] = tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

    corrected_count += 1

print("Successfully corrected:", corrected_count)

print("\nFinal DAB tag distribution:")
display(
    manual_review.loc[
        manual_review["Source_Family"] == "DAB",
        "Target_Tag"
    ].value_counts(dropna=False)
)

In [ ]:
# Cell 141: Final DAB audit

import pandas as pd

dab_final = manual_review[
    manual_review["Source_Family"] == "DAB"
].copy()

# 1. Basic counts
total = len(dab_final)
mapped = dab_final["Target_Tag"].notna().sum()
unmapped = dab_final["Target_Tag"].isna().sum()

print("=" * 60)
print("FINAL DAB MAPPING AUDIT")
print("=" * 60)

print("Total occurrences:", total)
print("Mapped occurrences:", mapped)
print("Unmapped occurrences:", unmapped)
print("Mapping percentage:", round(mapped / total * 100, 2), "%")

# 2. Target tag distribution
print("\nTarget tag distribution:")
display(dab_final["Target_Tag"].value_counts(dropna=False))

# 3. Validate target tags
invalid_tags = dab_final[
    dab_final["Target_Tag"].notna() &
    ~dab_final["Target_Tag"].isin(TARGET_TAGS)
]

print("\nInvalid target tags:", len(invalid_tags))
if len(invalid_tags) > 0:
    display(invalid_tags)

# 4. Confidence distribution
print("\nMapping confidence distribution:")
display(dab_final["Mapping_Confidence"].value_counts(dropna=False))

# 5. Unresolved occurrences
dab_unresolved = dab_final[
    dab_final["Target_Tag"].isna()
].sort_values(["sentence_id", "Token_Position"])

print("\nUnresolved DAB occurrences:", len(dab_unresolved))

display(
    dab_unresolved[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

# 6. Assertions
assert total == 203, "Unexpected DAB occurrence count"
assert mapped == 197, "Unexpected mapped occurrence count"
assert unmapped == 6, "Unexpected unresolved occurrence count"
assert len(invalid_tags) == 0, "Invalid target tags detected"

# 7. Save updated mapping review
OUTPUT_FILE = (
    RESULTS_DIR / "occurrence_level_mapping_review.csv"
)

manual_review.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nAll validation checks passed.")
print("Updated file saved to:", OUTPUT_FILE)

In [ ]:
# Cell 142: Identify next source families for manual review

family_progress = (
    manual_review
    .groupby("Source_Family")
    .agg(
        Total_Occurrences=("Target_Tag", "size"),
        Mapped=("Target_Tag", "count"),
        Unmapped=("Target_Tag", lambda x: x.isna().sum()),
        Unique_Source_Tags=("Source_Tag", "nunique")
    )
    .reset_index()
)

family_progress = family_progress[
    family_progress["Unmapped"] > 0
].sort_values(
    ["Unmapped", "Total_Occurrences"],
    ascending=False
).reset_index(drop=True)

family_progress["Mapping_Percentage"] = (
    family_progress["Mapped"] /
    family_progress["Total_Occurrences"] * 100
).round(2)

print("Remaining source families:")
display(family_progress.head(20))


In [ ]:

# Cell 143: Inspect PPR source-tag distribution

ppr_review = manual_review[
    (manual_review["Source_Family"] == "PPR") &
    (manual_review["Target_Tag"].isna())
].copy()

print("PPR unresolved occurrences:", len(ppr_review))
print("Unique source tags:", ppr_review["Source_Tag"].nunique())

ppr_tag_summary = (
    ppr_review
    .groupby("Source_Tag")
    .agg(
        Occurrences=("Word", "size"),
        Unique_Words=("Word", "nunique")
    )
    .reset_index()
    .sort_values("Occurrences", ascending=False)
)

display(ppr_tag_summary)

print("\nRepresentative contextual examples:")
display(
    ppr_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].head(30)
)


In [ ]:

# Cell 144: Contextual review of frequent PPR source tags

ppr_review = manual_review[
    (manual_review["Source_Family"] == "PPR") &
    (manual_review["Target_Tag"].isna())
].copy()

top_ppr_tags = (
    ppr_review["Source_Tag"]
    .value_counts()
    .head(25)
    .index
    .tolist()
)

for source_tag in top_ppr_tags:
    examples = ppr_review[
        ppr_review["Source_Tag"] == source_tag
    ].head(3)

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("OCCURRENCES:", (ppr_review["Source_Tag"] == source_tag).sum())

    display(
        examples[
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Previous_Words",
                "Next_Words",
                "Full_Sentence"
            ]
        ]
    )


In [ ]:

# Cell 145: Full contextual inspection of first-person genitive PPR

target_source_tag = "PPR.0.sg.1.gen.vi.n.n.prx"

ppr_batch = ppr_review[
    ppr_review["Source_Tag"] == target_source_tag
].copy()

print("Source tag:", target_source_tag)
print("Total unresolved occurrences:", len(ppr_batch))

display(
    ppr_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].sort_values(
        ["sentence_id", "Token_Position"]
    ).reset_index(drop=True)
)


In [ ]:

# Cell 146: Display all PPR genitive occurrences in manageable batches

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

BATCH_SIZE = 20

for start in range(0, len(ppr_batch), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(ppr_batch))

    print(f"\n{'=' * 90}")
    print(f"PPR GENITIVE: OCCURRENCES {start + 1}–{end}")
    print("=" * 90)

    display(
        ppr_batch.iloc[start:end][
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Previous_Words",
                "Next_Words",
                "Full_Sentence"
            ]
        ]
    )


In [ ]:

# Cell 147: Map PPR first-person genitive occurrences

ppr_genitive_mapping = {
    # SNV: possessive pronoun modifying a noun
    (44, 9): "SNV",
    (88, 4): "SNV",
    (89, 0): "SNV",
    (90, 2): "SNV",
    (93, 1): "SNV",
    (94, 0): "SNV",
    (98, 0): "SNV",
    (99, 0): "SNV",
    (100, 1): "SNV",
    (102, 1): "SNV",
    (103, 0): "SNV",
    (106, 3): "SNV",
    (109, 3): "SNV",
    (110, 3): "SNV",
    (114, 0): "SNV",
    (115, 1): "SNV",
    (182, 1): "SNV",
    (185, 0): "SNV",
    (248, 4): "SNV",
    (259, 7): "SNV",
    (264, 5): "SNV",
    (265, 1): "SNV",
    (266, 1): "SNV",
    (266, 10): "SNV",
    (300, 9): "SNV",
    (302, 2): "SNV",
    (303, 4): "SNV",
    (304, 2): "SNV",
    (309, 1): "SNV",
    (310, 20): "SNV",
    (313, 0): "SNV",
    (322, 6): "SNV",
    (335, 0): "SNV",
    (373, 0): "SNV",
    (376, 7): "SNV",
    (398, 4): "SNV",
    (409, 0): "SNV",
    (457, 1): "SNV",
    (464, 0): "SNV",
    (465, 10): "SNV",
    (466, 0): "SNV",
    (469, 2): "SNV",
    (477, 6): "SNV",
    (485, 1): "SNV",
    (499, 14): "SNV",
    (509, 19): "SNV",
    (510, 1): "SNV",
    (511, 1): "SNV",
    (511, 4): "SNV",
    (535, 23): "SNV",
    (551, 2): "SNV",
    (688, 3): "SNV",
    (751, 6): "SNV",
    (759, 6): "SNV",
    (759, 9): "SNV",
    (761, 2): "SNV",
    (775, 10): "SNV",
    (814, 0): "SNV",
    (869, 10): "SNV",
    (1070, 2): "SNV",
    (1091, 2): "SNV",
    (1104, 1): "SNV",
    (1105, 1): "SNV",
    (1169, 3): "SNV",
    (1190, 5): "SNV",
    (1208, 5): "SNV",
    (1219, 0): "SNV",
    (1347, 2): "SNV",
    (1357, 2): "SNV",
    (1368, 19): "SNV",
    (1409, 7): "SNV",
    (1431, 6): "SNV",
    (1438, 2): "SNV",
    (1444, 6): "SNV",

    # SN: independent possessive pronoun
    (1074, 2): "SN",
    (1356, 3): "SN",
}

# Validate that each position identifies exactly one occurrence
for (sid, pos), target_tag in ppr_genitive_mapping.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Family"] == "PPR")
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence: sentence={sid}, position={pos}"
    )
    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Existing annotation found at sentence={sid}, position={pos}"
    )

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = (
        "First-person genitive pronoun used independently"
        if target_tag == "SN"
        else "First-person genitive pronoun qualifying a noun"
    )
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("Mapped occurrences:", len(ppr_genitive_mapping))
print("SNV:", sum(v == "SNV" for v in ppr_genitive_mapping.values()))
print("SN:", sum(v == "SN" for v in ppr_genitive_mapping.values()))
print("Unresolved occurrences:", 80 - len(ppr_genitive_mapping))


In [ ]:

# Cell 148: Audit first-person genitive PPR mapping

ppr_genitive_audit = manual_review[
    (manual_review["Source_Tag"] == "PPR.0.sg.1.gen.vi.n.n.prx")
].copy()

print("Total occurrences:", len(ppr_genitive_audit))
print("Mapped:", ppr_genitive_audit["Target_Tag"].notna().sum())
print("Unmapped:", ppr_genitive_audit["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
display(ppr_genitive_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
display(ppr_genitive_audit["Mapping_Confidence"].value_counts(dropna=False))

print("\nUnresolved occurrences:")
display(
    ppr_genitive_audit[
        ppr_genitive_audit["Target_Tag"].isna()
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ]
)

assert len(ppr_genitive_audit) == 80
assert ppr_genitive_audit["Target_Tag"].notna().sum() == 76
assert ppr_genitive_audit["Target_Tag"].isna().sum() == 4
assert set(ppr_genitive_audit["Target_Tag"].dropna()).issubset(TARGET_TAGS)

print("\nAudit passed.")


In [ ]:

# Cell 149: Inspect first-person nominative PPR occurrences

target_source_tag = "PPR.0.sg.1.nom.i.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Source tag:", target_source_tag)
print("Total unresolved occurrences:", len(ppr_batch))

BATCH_SIZE = 20

for start in range(0, len(ppr_batch), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(ppr_batch))

    print(f"\n{'=' * 80}")
    print(f"OCCURRENCES {start + 1}–{end}")
    print("=" * 80)

    display(
        ppr_batch.iloc[start:end][
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Previous_Words",
                "Next_Words",
                "Full_Sentence"
            ]
        ]
    )


In [ ]:

# Cell 150: Map first-person nominative PPR occurrences

target_source_tag = "PPR.0.sg.1.nom.i.n.n.prx"

ppr_nom_mapping = {}

for _, row in ppr_batch.iterrows():
    key = (row["sentence_id"], row["Token_Position"])
    ppr_nom_mapping[key] = "SN"

print("Occurrences selected:", len(ppr_nom_mapping))
print("Target tag:", "SN")

# Validate all occurrences before updating
for (sid, pos), target_tag in ppr_nom_mapping.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    assert mask.sum() == 1, (
        f"Duplicate or missing occurrence: {(sid, pos)}"
    )
    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Already mapped: {(sid, pos)}"
    )

# Apply mappings
for (sid, pos), target_tag in ppr_nom_mapping.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = (
        "First-person nominative pronoun functioning as sentence subject"
    )
    manual_review.loc[mask, "Mapping_Confidence"] = "High"

print("\nMapping completed.")
print("Mapped:", len(ppr_nom_mapping))
print("SN:", sum(v == "SN" for v in ppr_nom_mapping.values()))


In [ ]:

ppr_nom_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

print("=" * 65)
print("PPR FIRST-PERSON NOMINATIVE AUDIT")
print("=" * 65)

print("Total occurrences:", len(ppr_nom_audit))
print("Mapped occurrences:", ppr_nom_audit["Target_Tag"].notna().sum())
print("Unmapped occurrences:", ppr_nom_audit["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
print(ppr_nom_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(ppr_nom_audit["Mapping_Confidence"].value_counts(dropna=False))

# Validation
assert len(ppr_nom_audit) == 65
assert ppr_nom_audit["Target_Tag"].notna().all()
assert (ppr_nom_audit["Target_Tag"] == "SN").all()
assert ppr_nom_audit["Target_Tag"].dropna().isin(TARGET_TAGS).all()

print("\nAUDIT PASSED")


In [ ]:

target_source_tag = "PPR.mas.sg.3.gen.vi.n.n.dst"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Occurrences found:", len(ppr_batch))
print("Unique sentences:", ppr_batch["sentence_id"].nunique())

print("\nOccurrence-level context:\n")
for _, row in ppr_batch.iterrows():
    print("-" * 80)
    print(f"Sentence ID: {row['sentence_id']}")
    print(f"Token Position: {row['Token_Position']}")
    print(f"Word: {row['Word']}")
    print(f"Previous Words: {row['Previous_Words']}")
    print(f"Next Words: {row['Next_Words']}")
    print(f"Full Sentence: {row['Full_Sentence']}")


In [ ]:

target_source_tag = "PPR.mas.sg.3.gen.vi.n.n.dst"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

# Default: possessive pronoun modifying a noun
ppr_mapping = {}

for _, row in ppr_batch.iterrows():
    key = (row["sentence_id"], row["Token_Position"])
    ppr_mapping[key] = {
        "tag": "SNV",
        "reason": "Third-person genitive demonstrative used possessively with a noun",
        "confidence": "High"
    }

# Independent genitive pronoun
ppr_mapping[(915, 8)] = {
    "tag": "SN",
    "reason": "Independent genitive pronoun in 'तत् तस्य तिष्ठति'",
    "confidence": "Medium"
}

# Unclear adverbial construction: retain as unresolved
ppr_mapping.pop((476, 0), None)

# Validate and apply occurrence-level mappings
mapped_count = 0

for key, info in ppr_mapping.items():
    sid, pos = key

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    assert mask.sum() == 1, f"Occurrence mismatch: {key}"
    assert manual_review.loc[mask, "Target_Tag"].isna().all()

    manual_review.loc[mask, "Target_Tag"] = info["tag"]
    manual_review.loc[mask, "Mapping_Reason"] = info["reason"]
    manual_review.loc[mask, "Mapping_Confidence"] = info["confidence"]

    mapped_count += 1

print("Occurrences selected:", len(ppr_batch))
print("Mapped:", mapped_count)
print("SNV:", sum(v["tag"] == "SNV" for v in ppr_mapping.values()))
print("SN:", sum(v["tag"] == "SN" for v in ppr_mapping.values()))
print("Left unresolved:", len(ppr_batch) - mapped_count)


In [ ]:

ppr_gen_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

print("=" * 65)
print("PPR GENITIVE DEMONSTRATIVE AUDIT")
print("=" * 65)

print("Total occurrences:", len(ppr_gen_audit))
print("Mapped:", ppr_gen_audit["Target_Tag"].notna().sum())
print("Unmapped:", ppr_gen_audit["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
print(ppr_gen_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(ppr_gen_audit["Mapping_Confidence"].value_counts(dropna=False))

print("\nUnresolved occurrence:")
print(ppr_gen_audit.loc[
    ppr_gen_audit["Target_Tag"].isna(),
    ["sentence_id", "Token_Position", "Word", "Full_Sentence"]
].to_string(index=False))

# Validation
assert len(ppr_gen_audit) == 60
assert ppr_gen_audit["Target_Tag"].notna().sum() == 59
assert ppr_gen_audit["Target_Tag"].isna().sum() == 1
assert set(ppr_gen_audit["Target_Tag"].dropna()).issubset(TARGET_TAGS)

print("\nAUDIT PASSED")


In [ ]:

target_source_tag = "PPR.mas.sg.3.nom.i.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Occurrences found:", len(ppr_batch))
print("Unique sentences:", ppr_batch["sentence_id"].nunique())

print("\nOccurrence-level context:\n")

for _, row in ppr_batch.iterrows():
    print("-" * 80)
    print(f"Sentence ID: {row['sentence_id']}")
    print(f"Token Position: {row['Token_Position']}")
    print(f"Word: {row['Word']}")
    print(f"Previous Words: {row['Previous_Words']}")
    print(f"Next Words: {row['Next_Words']}")
    print(f"Full Sentence: {row['Full_Sentence']}")


In [ ]:

batch_keys = set(zip(
    ppr_batch["sentence_id"],
    ppr_batch["Token_Position"]
))

mapping_keys = set(ppr_mapping.keys())

missing_from_batch = mapping_keys - batch_keys
not_in_mapping = batch_keys - mapping_keys

print("Actual batch occurrences:", len(batch_keys))
print("Proposed mapping occurrences:", len(mapping_keys))

print("\nKeys in mapping but missing from batch:")
for key in sorted(missing_from_batch):
    print(key)

print("\nActual batch occurrences not mapped:")
for key in sorted(not_in_mapping):
    rows = ppr_batch[
        (ppr_batch["sentence_id"] == key[0]) &
        (ppr_batch["Token_Position"] == key[1])
    ]
    for _, row in rows.iterrows():
        print(
            key,
            "| Word:", row["Word"],
            "| Sentence:", row["Full_Sentence"]
        )


In [ ]:
# Debug occurrence keys for sentence 688

print("Target source tag:", target_source_tag)

print("\nKeys currently present for sentence 688:")
for key, value in ppr_mapping.items():
    if key[0] == 688:
        print(key, "->", value)

print("\nChecking keys:")
print("(688, 10) exists:", (688, 10) in ppr_mapping)
print("(688, 2) exists:", (688, 2) in ppr_mapping)

print("\nExpected batch keys for sentence 688:")
print([key for key in batch_keys if key[0] == 688])

In [ ]:
# Inspect sentence 688 occurrences in the current batch

for sid, pos in [(688, 2), (688, 26)]:

    result = manual_review[
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    ]

    print("\n" + "=" * 70)
    print(f"Occurrence key: ({sid}, {pos})")
    print("=" * 70)

    if len(result) != 1:
        print("WARNING: Expected exactly one occurrence.")
        display(result)
        continue

    row = result.iloc[0]

    print("Word:", row["Word"])
    print("Source tag:", row["Source_Tag"])
    print("Current target tag:", row["Target_Tag"])
    print("Previous words:", row["Previous_Words"])
    print("Next words:", row["Next_Words"])
    print("Full sentence:", row["Full_Sentence"])

In [ ]:
# Diagnostic: Compare current batch and mapping dictionary

print("=" * 65)
print("BATCH CONSISTENCY DIAGNOSTIC")
print("=" * 65)

print("Target source tag:", target_source_tag)
print("ppr_batch rows:", len(ppr_batch))
print("batch_keys count:", len(batch_keys))
print("ppr_mapping entries:", len(ppr_mapping))

# Reconstruct occurrence keys from the current batch
current_batch_keys = set(
    zip(
        ppr_batch["sentence_id"],
        ppr_batch["Token_Position"]
    )
)

expected_keys = set(batch_keys)
mapping_keys = set(ppr_mapping.keys())

print("\n--- Key comparisons ---")

print("ppr_batch vs batch_keys:",
      current_batch_keys == expected_keys)

print("ppr_batch vs ppr_mapping:",
      current_batch_keys == mapping_keys)

print("batch_keys vs ppr_mapping:",
      expected_keys == mapping_keys)

print("\n--- Differences ---")

print("Missing from mapping:", len(expected_keys - mapping_keys))
print("Extra in mapping:", len(mapping_keys - expected_keys))

print("\n--- Sentence 688 verification ---")

for key in [(688, 2), (688, 26)]:
    print(
        key,
        "| In batch:", key in current_batch_keys,
        "| In batch_keys:", key in expected_keys,
        "| In mapping:", key in mapping_keys
    )

print("\n--- Current batch sample ---")
display(
    ppr_batch[
        ["sentence_id", "Token_Position", "Word", "Source_Tag"]
    ].head(10)
)

print("\n--- Mapping dictionary sample ---")
for key in list(ppr_mapping.keys())[:10]:
    print(key, "->", ppr_mapping[key])

print("\n--- Batch key sample ---")
print(list(expected_keys)[:10])

In [ ]:
# Cell: Inspect current batch without merge

print("=" * 65)
print("CURRENT BATCH MAPPING STATUS")
print("=" * 65)

# Required columns
required_cols = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Target_Tag",
    "Mapping_Reason",
    "Mapping_Confidence"
]

missing_cols = [
    col for col in required_cols
    if col not in manual_review.columns
]

assert not missing_cols, f"Missing columns: {missing_cols}"

batch_status_rows = []

for _, row in ppr_batch.iterrows():

    sid = row["sentence_id"]
    pos = row["Token_Position"]

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    matches = manual_review.loc[mask]

    assert len(matches) == 1, (
        f"Expected exactly one occurrence for ({sid}, {pos}), "
        f"found {len(matches)}"
    )

    result = matches.iloc[0]

    batch_status_rows.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Word": result["Word"],
        "Target_Tag": result["Target_Tag"],
        "Mapping_Reason": result["Mapping_Reason"],
        "Mapping_Confidence": result["Mapping_Confidence"]
    })

batch_status = pd.DataFrame(batch_status_rows)

print("Total batch occurrences:", len(batch_status))

print("\nTarget tag distribution:")
print(batch_status["Target_Tag"].value_counts(dropna=False))

print("\nAlready mapped occurrences:")
display(batch_status[batch_status["Target_Tag"].notna()])

print("\nUnmapped occurrences:")
display(batch_status[batch_status["Target_Tag"].isna()])

print("\nBatch inspection completed. No mappings modified.")

In [ ]:
# Create a fresh mapping dictionary for the current batch
# Preserve the old dictionary for safety.

previous_ppr_mapping = ppr_mapping.copy()

# Rebuild mapping keys from the verified current batch
ppr_mapping = {}

for _, row in batch_status.iterrows():

    sid = row["sentence_id"]
    pos = row["Token_Position"]

    # Only include genuinely unmapped occurrences
    if pd.isna(row["Target_Tag"]):

        key = (sid, pos)

        assert key not in ppr_mapping, (
            f"Duplicate occurrence key detected: {key}"
        )

        ppr_mapping[key] = None

# Validate exact batch coverage
assert set(ppr_mapping.keys()) == batch_keys, (
    "Fresh mapping dictionary does not match batch_keys"
)

print("Fresh mapping dictionary created successfully.")
print("Previous dictionary entries preserved:", len(previous_ppr_mapping))
print("Current batch occurrences:", len(batch_keys))
print("Fresh dictionary entries:", len(ppr_mapping))
print("Exact key validation: PASSED")

In [ ]:
# Display all 41 occurrences with their complete sentence context

for _, row in batch_status.iterrows():

    sid = row["sentence_id"]
    pos = row["Token_Position"]

    result = manual_review[
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    ]

    assert len(result) == 1

    r = result.iloc[0]

    print("\n" + "=" * 90)
    print(f"Sentence ID: {sid} | Token Position: {pos}")
    print(f"Word: {r['Word']}")
    print(f"Previous Words: {r['Previous_Words']}")
    print(f"Next Words: {r['Next_Words']}")
    print(f"Full Sentence: {r['Full_Sentence']}")

In [ ]:
# Populate verified occurrence-level mappings
# Source: PPR.mas.sg.3.nom.i.n.n.prx

# SNV: Pronouns modifying or identifying nouns
snv_keys = {
    (391, 6),
    (472, 4)
}

# SN: Independent pronouns
sn_keys = batch_keys - snv_keys

# Validate complete and disjoint coverage
assert len(sn_keys) == 39
assert len(snv_keys) == 2
assert sn_keys.isdisjoint(snv_keys)
assert sn_keys | snv_keys == batch_keys

# Medium-confidence cases
medium_confidence = {
    (352, 1),
    (382, 17),
    (472, 4)
}

# Populate the mapping dictionary
for key in batch_keys:

    sid, pos = key

    row = batch_status[
        (batch_status["sentence_id"] == sid) &
        (batch_status["Token_Position"] == pos)
    ]

    assert len(row) == 1
    word = row.iloc[0]["Word"]

    if key in snv_keys:
        tag = "SNV"

        if key == (391, 6):
            reason = "Demonstrative pronoun अयम् directly identifies and modifies अर्थः."

        else:
            reason = "Post-nominal demonstrative अयम् identifies संसारः."

    else:
        tag = "SN"

        if word == "भवान्":
            reason = "Independent honorific personal pronoun, not modifying a noun."

        else:
            reason = "Independent demonstrative pronoun functioning without a modified noun."

    confidence = (
        "Medium" if key in medium_confidence else "High"
    )

    ppr_mapping[key] = {
        "tag": tag,
        "reason": reason,
        "confidence": confidence
    }

# Final validation
assert set(ppr_mapping.keys()) == batch_keys
assert all(
    info["tag"] in ["SN", "SNV"]
    for info in ppr_mapping.values()
)

print("Mapping dictionary populated successfully.")
print("Total occurrences:", len(ppr_mapping))
print("SN:", sum(v["tag"] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v["tag"] == "SNV" for v in ppr_mapping.values()))
print("High confidence:", sum(v["confidence"] == "High" for v in ppr_mapping.values()))
print("Medium confidence:", sum(v["confidence"] == "Medium" for v in ppr_mapping.values()))
print("Low confidence:", sum(v["confidence"] == "Low" for v in ppr_mapping.values()))
print("Exact occurrence-key validation: PASSED")
print("manual_review modified: NO")

In [ ]:
# Apply validated occurrence-level mappings

# Step 1: Validate complete dictionary coverage
assert set(ppr_mapping.keys()) == batch_keys
assert len(ppr_mapping) == len(batch_keys)

# Step 2: Pre-validate every occurrence before updating
validated_masks = {}

for (sid, pos), info in ppr_mapping.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence for ({sid}, {pos}); "
        f"found {mask.sum()}"
    )

    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Occurrence already mapped: ({sid}, {pos})"
    )

    assert info["tag"] in ["SN", "SNV"]
    assert info["confidence"] in ["High", "Medium", "Low"]

    validated_masks[(sid, pos)] = mask

# Step 3: Apply mappings after all validations pass
for key, info in ppr_mapping.items():

    mask = validated_masks[key]

    manual_review.loc[mask, "Target_Tag"] = info["tag"]
    manual_review.loc[mask, "Mapping_Reason"] = info["reason"]
    manual_review.loc[mask, "Mapping_Confidence"] = info["confidence"]

# Step 4: Verify results
updated_group = manual_review[
    manual_review["Source_Tag"] == target_source_tag
]

print("=" * 65)
print("MAPPING APPLICATION SUMMARY")
print("=" * 65)

print("Occurrences selected:", len(ppr_mapping))
print("Mapped:", len(ppr_mapping))
print("SN:", (updated_group["Target_Tag"] == "SN").sum())
print("SNV:", (updated_group["Target_Tag"] == "SNV").sum())
print("Unresolved:", updated_group["Target_Tag"].isna().sum())

print("\nMapping confidence:")
print(updated_group["Mapping_Confidence"].value_counts())

print("\nMapping completed successfully.")

In [ ]:
# Complete audit: PPR.mas.sg.3.nom.i.n.n.prx

SOURCE_TAG = "PPR.mas.sg.3.nom.i.n.n.prx"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

batch_key_set = set(batch_keys)

# Build occurrence keys
group["Occurrence_Key"] = list(zip(
    group["sentence_id"],
    group["Token_Position"]
))

# Identify occurrences outside current batch
outside_batch = group[
    ~group["Occurrence_Key"].isin(batch_key_set)
].copy()

inside_batch = group[
    group["Occurrence_Key"].isin(batch_key_set)
].copy()

print("=" * 70)
print("COMPLETE SOURCE-TAG AUDIT")
print("=" * 70)

print("Source tag:", SOURCE_TAG)
print("Total occurrences:", len(group))
print("Unique occurrence keys:", group["Occurrence_Key"].nunique())
print("Current batch occurrences:", len(inside_batch))
print("Occurrences outside batch:", len(outside_batch))

print("\n--- Complete target-tag distribution ---")
print(group["Target_Tag"].value_counts(dropna=False))

print("\n--- Complete confidence distribution ---")
print(group["Mapping_Confidence"].value_counts(dropna=False))

print("\n--- Current batch distribution ---")
print(inside_batch["Target_Tag"].value_counts(dropna=False))

print("\n--- Occurrences outside current batch ---")

display(
    outside_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\n--- Unresolved occurrences in complete group ---")

display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence"
        ]
    ]
)

print("\n--- Duplicate occurrence keys ---")

duplicates = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

print("Duplicate rows:", len(duplicates))
display(duplicates)

print("\nAudit inspection completed. No mappings modified.")

In [ ]:
# Find remaining unresolved PPR source tags

ppr_unresolved = manual_review[
    manual_review["Source_Tag"].str.startswith(
        "PPR.", na=False
    )
    & manual_review["Target_Tag"].isna()
].copy()

summary = (
    ppr_unresolved
    .groupby("Source_Tag")
    .agg(
        Unresolved_Occurrences=("Word", "size"),
        Unique_Sentences=("sentence_id", "nunique")
    )
    .sort_values(
        "Unresolved_Occurrences",
        ascending=False
    )
)

print("=" * 65)
print("REMAINING UNRESOLVED PPR SOURCE TAGS")
print("=" * 65)

print("Total unresolved PPR occurrences:", len(ppr_unresolved))
print("Remaining source tags:", len(summary))

display(summary)

if not summary.empty:
    print("\nNext source tag:")
    print(summary.index[0])
else:
    print("\nAll PPR occurrences have been mapped or reviewed.")

In [ ]:
# Investigate previously completed PPR source tag

SOURCE_TAG = "PPR.mas.sg.3.gen.vi.n.n.prx"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

print("=" * 65)
print("PREVIOUSLY COMPLETED SOURCE TAG VERIFICATION")
print("=" * 65)

print("Source tag:", SOURCE_TAG)
print("Total occurrences:", len(group))

print("\nTarget tag distribution:")
print(group["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(group["Mapping_Confidence"].value_counts(dropna=False))

print("\nUnresolved occurrences:",
      group["Target_Tag"].isna().sum())

print("\nDuplicate occurrence keys:",
      group.duplicated(
          subset=["sentence_id", "Token_Position"],
          keep=False
      ).sum())

print("\nUnresolved occurrence details:")

display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].sort_values(
        ["sentence_id", "Token_Position"]
    )
)

print("\nPreviously mapped occurrence count:",
      group["Target_Tag"].notna().sum())

print("\nInspection completed. No mappings modified.")

In [ ]:

# Inspect all 42 occurrences of this source tag
ppr_full_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

print("Total occurrences:", len(ppr_full_audit))

print("\nAll occurrences:")
print(
    ppr_full_audit[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(
        ["sentence_id", "Token_Position"]
    ).to_string(index=False)
)

print("\nOccurrences outside the original 41-row batch:")
original_keys = set(zip(
    ppr_batch["sentence_id"],
    ppr_batch["Token_Position"]
))

for _, row in ppr_full_audit.iterrows():
    key = (row["sentence_id"], row["Token_Position"])
    if key not in original_keys:
        print(
            key,
            "| Word:", row["Word"],
            "| Tag:", row["Target_Tag"],
            "| Context:", row["Full_Sentence"]
        )


In [ ]:

ppr_nom_demo_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

print("=" * 65)
print("PPR NOMINATIVE DEMONSTRATIVE FINAL AUDIT")
print("=" * 65)

print("Total occurrences:", len(ppr_nom_demo_audit))
print("Mapped:", ppr_nom_demo_audit["Target_Tag"].notna().sum())
print("Unmapped:", ppr_nom_demo_audit["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
print(ppr_nom_demo_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(ppr_nom_demo_audit["Mapping_Confidence"].value_counts(dropna=False))

assert len(ppr_nom_demo_audit) == 42
assert ppr_nom_demo_audit["Target_Tag"].notna().all()
assert ppr_nom_demo_audit["Target_Tag"].isin(TARGET_TAGS).all()

print("\nAUDIT PASSED")


In [ ]:

target_source_tag = "PPR.mas.sg.3.gen.vi.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Occurrences found:", len(ppr_batch))
print("Unique sentences:", ppr_batch["sentence_id"].nunique())

print("\nOccurrence-level context:\n")

for _, row in ppr_batch.iterrows():
    print("-" * 80)
    print(f"Sentence ID: {row['sentence_id']}")
    print(f"Token Position: {row['Token_Position']}")
    print(f"Word: {row['Word']}")
    print(f"Previous Words: {row['Previous_Words']}")
    print(f"Next Words: {row['Next_Words']}")
    print(f"Full Sentence: {row['Full_Sentence']}")


In [ ]:

target_source_tag = "PPR.mas.sg.3.gen.vi.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_mapping = {}

# SNV: possessive/genitive forms modifying another expression
snv_positions = [
    (124, 1), (231, 7), (261, 4), (277, 11),
    (281, 5), (346, 0), (362, 1),
    (528, 2), (534, 2), (535, 3), (545, 6),
    (548, 0), (559, 2), (562, 2), (563, 2),
    (565, 0), (568, 5), (569, 5), (627, 2),
    (643, 7), (645, 0), (658, 5), (686, 0),
    (1048, 0), (1095, 0)
]

# SN: independent genitive pronoun
sn_positions = [
    (369, 4)
]

for key in snv_positions:
    ppr_mapping[key] = {
        "tag": "SNV",
        "reason": "Genitive pronoun used possessively or modifying another expression",
        "confidence": "High"
    }

for key in sn_positions:
    ppr_mapping[key] = {
        "tag": "SN",
        "reason": "Independent genitive pronoun",
        "confidence": "Medium"
    }

# Validate exact occurrence keys
batch_keys = set(zip(
    ppr_batch["sentence_id"],
    ppr_batch["Token_Position"]
))

assert len(ppr_batch) == 26
assert set(ppr_mapping.keys()) == batch_keys

# Apply mappings
for (sid, pos), info in ppr_mapping.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    assert mask.sum() == 1
    assert manual_review.loc[mask, "Target_Tag"].isna().all()

    manual_review.loc[mask, "Target_Tag"] = info["tag"]
    manual_review.loc[mask, "Mapping_Reason"] = info["reason"]
    manual_review.loc[mask, "Mapping_Confidence"] = info["confidence"]

print("Occurrences selected:", len(ppr_batch))
print("Mapped:", len(ppr_mapping))
print("SN:", len(sn_positions))
print("SNV:", len(snv_positions))
print("Unmapped:", len(ppr_batch) - len(ppr_mapping))
print("\nMapping completed.")


In [ ]:

# Get all occurrences of this source tag
ppr_full_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

# Compare against the 26 occurrences extracted for this batch
original_keys = set(zip(
    ppr_batch["sentence_id"],
    ppr_batch["Token_Position"]
))

print("Total occurrences:", len(ppr_full_audit))
print("Original batch:", len(original_keys))
print("Additional occurrences:", len(ppr_full_audit) - len(original_keys))

print("\nAdditional occurrences:\n")

for _, row in ppr_full_audit.iterrows():
    key = (row["sentence_id"], row["Token_Position"])

    if key not in original_keys:
        print("-" * 80)
        print("Sentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Existing Target Tag:", row["Target_Tag"])
        print("Confidence:", row["Mapping_Confidence"])
        print("Full Sentence:", row["Full_Sentence"])


In [ ]:

additional_keys = [
    (20, 0), (22, 0), (33, 0), (122, 0),
    (207, 4), (978, 0), (1036, 6), (1047, 0),
    (1056, 5), (1071, 2), (1075, 2), (1086, 5),
    (1093, 2), (1200, 13), (1286, 0), (1383, 1),
    (1396, 5), (1401, 2), (1404, 2), (1413, 9),
    (1417, 4), (1424, 4)
]

additional_audit = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in additional_keys
        and r["Source_Tag"] == target_source_tag,
        axis=1
    )
].copy()

print("Additional occurrences:", len(additional_audit))
print("\nTarget tag distribution:")
print(additional_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(additional_audit["Mapping_Confidence"].value_counts(dropna=False))

assert len(additional_audit) == 22
assert additional_audit["Target_Tag"].eq("SNV").all()

print("\nADDITIONAL OCCURRENCES VERIFIED")


In [ ]:

ppr_gen_audit = manual_review[
    manual_review["Source_Tag"] == target_source_tag
].copy()

print("=" * 65)
print("PPR GENITIVE PRONOUN FINAL AUDIT")
print("=" * 65)

print("Total occurrences:", len(ppr_gen_audit))
print("Mapped:", ppr_gen_audit["Target_Tag"].notna().sum())
print("Unmapped:", ppr_gen_audit["Target_Tag"].isna().sum())

print("\nTarget tag distribution:")
print(ppr_gen_audit["Target_Tag"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(ppr_gen_audit["Mapping_Confidence"].value_counts(dropna=False))

assert len(ppr_gen_audit) == 48
assert ppr_gen_audit["Target_Tag"].notna().all()
assert ppr_gen_audit["Target_Tag"].isin(TARGET_TAGS).all()

print("\nAUDIT PASSED")


In [ ]:

target_source_tag = "PPR.0.sg.1.ins.iii.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_batch = ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).reset_index(drop=True)

print("Occurrences found:", len(ppr_batch))
print("Unique sentences:", ppr_batch["sentence_id"].nunique())

print("\nOccurrence-level context:\n")

for _, row in ppr_batch.iterrows():
    print("-" * 80)
    print(f"Sentence ID: {row['sentence_id']}")
    print(f"Token Position: {row['Token_Position']}")
    print(f"Word: {row['Word']}")
    print(f"Previous Words: {row['Previous_Words']}")
    print(f"Next Words: {row['Next_Words']}")
    print(f"Full Sentence: {row['Full_Sentence']}")


In [ ]:

target_source_tag = "PPR.0.sg.1.ins.iii.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == target_source_tag) &
    (manual_review["Target_Tag"].isna())
].copy()

ppr_mapping = {}

for _, row in ppr_batch.iterrows():
    key = (row["sentence_id"], row["Token_Position"])

    ppr_mapping[key] = {
        "tag": "SN",
        "reason": "First-person instrumental pronoun functioning independently as agent or complement",
        "confidence": "High"
    }

# Validate exact occurrence keys
batch_keys = set(zip(
    ppr_batch["sentence_id"],
    ppr_batch["Token_Position"]
))

assert len(ppr_batch) == 24
assert set(ppr_mapping.keys()) == batch_keys

# Apply mappings
for (sid, pos), info in ppr_mapping.items():
    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == target_source_tag)
    )

    assert mask.sum() == 1
    assert manual_review.loc[mask, "Target_Tag"].isna().all()

    manual_review.loc[mask, "Target_Tag"] = info["tag"]
    manual_review.loc[mask, "Mapping_Reason"] = info["reason"]
    manual_review.loc[mask, "Mapping_Confidence"] = info["confidence"]

print("Occurrences selected:", len(ppr_batch))
print("Mapped:", len(ppr_mapping))
print("SN:", len(ppr_mapping))
print("Unmapped:", len(ppr_batch) - len(ppr_mapping))
print("\nMapping completed.")


In [ ]:
# Cell 167: Complete audit of PPR.0.sg.1.ins.iii.n.n.prx

import pandas as pd

SOURCE_TAG = "PPR.0.sg.1.ins.iii.n.n.prx"

# Extract the complete source-tag group
group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

if group.empty:
    raise AssertionError(f"No occurrences found for {SOURCE_TAG}")

# Normalize target labels for auditing only
target_labels = group["Target_Tag"].apply(
    lambda x: str(x).strip() if pd.notna(x) else None
)

# Count mappings
total = len(group)
mapped = target_labels.notna().sum()
unmapped = target_labels.isna().sum()

sn_count = (target_labels == "SN").sum()
snv_count = (target_labels == "SNV").sum()

other_mapped = (
    target_labels.notna()
    & ~target_labels.isin(["SN", "SNV"])
).sum()

# Check exact occurrence-key uniqueness
duplicate_keys = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

print("=" * 65)
print(f"SOURCE TAG AUDIT: {SOURCE_TAG}")
print("=" * 65)

print(f"Total occurrences       : {total}")
print(f"Mapped occurrences      : {mapped}")
print(f"SN                      : {sn_count}")
print(f"SNV                     : {snv_count}")
print(f"Other mapped tags       : {other_mapped}")
print(f"Unresolved              : {unmapped}")
print(f"Duplicate occurrence keys: {len(duplicate_keys)}")

print("\nMapping distribution:")
print(target_labels.value_counts(dropna=False))

print("\nComplete occurrence-level mapping:")
display(
    group[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(
        ["sentence_id", "Token_Position"]
    )
)

if len(duplicate_keys) > 0:
    print("\nWARNING: Duplicate occurrence keys found.")
    display(
        duplicate_keys[
            ["sentence_id", "Token_Position", "Word", "Target_Tag"]
        ]
    )
else:
    print("\nOccurrence-key uniqueness check: PASSED")

print("\nAudit summary:")
print(f"Additional occurrences beyond the 24 extracted: {total - 24}")
print(f"All occurrences mapped: {mapped == total}")

In [ ]:
# Cell 168: Extract next unresolved PPR source tag

SOURCE_TAG = "PPR.0.sg.2.gen.vi.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == SOURCE_TAG) &
    (manual_review["Target_Tag"].isna())
].copy()

# Validate occurrence-level uniqueness
batch_keys = set(
    zip(
        ppr_batch["sentence_id"],
        ppr_batch["Token_Position"]
    )
)

assert len(ppr_batch) == len(batch_keys), \
    "Duplicate occurrence keys detected."

print("=" * 70)
print("NEXT PPR SOURCE TAG")
print("=" * 70)

print("Source tag:", SOURCE_TAG)
print("Unresolved occurrences:", len(ppr_batch))
print("Unique occurrence keys:", len(batch_keys))

print("\nOccurrence-level context:")

for _, row in ppr_batch.iterrows():

    print("\n" + "-" * 80)
    print("Sentence ID:", row["sentence_id"])
    print("Token Position:", row["Token_Position"])
    print("Word:", row["Word"])
    print("Previous Words:", row["Previous_Words"])
    print("Next Words:", row["Next_Words"])
    print("Full Sentence:", row["Full_Sentence"])

print("\nExtraction completed. No mappings modified.")

In [ ]:
# Cell 169: Populate PPR.0.sg.2.gen.vi.n.n.prx mappings

SOURCE_TAG = "PPR.0.sg.2.gen.vi.n.n.prx"

# Verify the current batch
assert len(ppr_batch) == 23
assert len(batch_keys) == 23

assert set(
    zip(ppr_batch["sentence_id"], ppr_batch["Token_Position"])
) == batch_keys

# SNV: Possessive pronouns modifying nouns
snv_keys = {
    (78, 0),
    (108, 0),
    (110, 0),
    (112, 15),
    (181, 0),
    (184, 0),
    (416, 1),
    (420, 2),
    (429, 4),
    (442, 0),
    (611, 2),
    (806, 2),
    (816, 11),
    (823, 4),
    (1109, 1),
    (1111, 2),
    (1210, 8),
    (1220, 1),
    (1363, 1),
    (1367, 1),
    (1401, 4)
}

# SN: Independent possessive pronoun
sn_keys = {
    (916, 0)
}

# Uncertain occurrence
unresolved_keys = {
    (1402, 2)
}

# Validate complete and non-overlapping coverage
assert snv_keys.isdisjoint(sn_keys)
assert snv_keys.isdisjoint(unresolved_keys)
assert sn_keys.isdisjoint(unresolved_keys)

assert snv_keys | sn_keys | unresolved_keys == batch_keys

assert len(snv_keys) == 21
assert len(sn_keys) == 1
assert len(unresolved_keys) == 1

# Create a fresh dictionary
ppr_mapping = {}

for key in batch_keys:

    sid, pos = key

    row = ppr_batch[
        (ppr_batch["sentence_id"] == sid) &
        (ppr_batch["Token_Position"] == pos)
    ]

    assert len(row) == 1

    word = row.iloc[0]["Word"]

    if key in snv_keys:

        tag = "SNV"

        reason = (
            "Second-person genitive pronoun functioning "
            "as a possessive modifier of a noun."
        )

        confidence = (
            "Medium" if key in {(611, 2), (1401, 4)}
            else "High"
        )

    elif key in sn_keys:

        tag = "SN"

        reason = (
            "Independent substantival possessive pronoun "
            "in the construction तव अपि अस्ति मम अपि अस्ति."
        )

        confidence = "Medium"

    else:

        tag = None

        reason = (
            "Uncertain genitive/dative function; "
            "insufficient context to establish independent "
            "or noun-modifying usage."
        )

        confidence = "Low"

    ppr_mapping[key] = {
        "tag": tag,
        "reason": reason,
        "confidence": confidence
    }

# Final validation
assert set(ppr_mapping.keys()) == batch_keys

print("Mapping dictionary populated successfully.")
print("Total occurrences:", len(ppr_mapping))
print("SN:", sum(v["tag"] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v["tag"] == "SNV" for v in ppr_mapping.values()))
print("Unresolved:", sum(v["tag"] is None for v in ppr_mapping.values()))

print("\nConfidence distribution:")
print(pd.Series(
    [v["confidence"] for v in ppr_mapping.values()]
).value_counts())

print("\nUnresolved occurrence:")
for key, info in ppr_mapping.items():
    if info["tag"] is None:
        print(key, info)

print("\nExact occurrence-key validation: PASSED")
print("manual_review modified: NO")

In [ ]:
# Debug Cell: Verify source-tag variables and occurrence (442, 0)

print("=" * 65)
print("SOURCE TAG VARIABLE DIAGNOSTIC")
print("=" * 65)

print("SOURCE_TAG:", SOURCE_TAG)
print("target_source_tag:", target_source_tag)

print("\nCurrent batch source tags:")
print(ppr_batch["Source_Tag"].value_counts(dropna=False))

print("\nChecking occurrence (442, 0):")

check = manual_review[
    (manual_review["sentence_id"] == 442) &
    (manual_review["Token_Position"] == 0)
]

display(
    check[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Target_Tag"
        ]
    ]
)

print("\nOccurrence in current batch:")

display(
    ppr_batch[
        (ppr_batch["sentence_id"] == 442) &
        (ppr_batch["Token_Position"] == 0)
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag"
        ]
    ]
)

print("\nDictionary entry:")
print(ppr_mapping.get((442, 0)))

In [ ]:
# Cell 170: Apply validated PPR mappings
# Correct source-tag variable: SOURCE_TAG

assert set(ppr_mapping.keys()) == batch_keys

confirmed_mappings = {
    key: info
    for key, info in ppr_mapping.items()
    if info["tag"] is not None
}

unresolved_mappings = {
    key: info
    for key, info in ppr_mapping.items()
    if info["tag"] is None
}

validated_masks = {}

# Step 1: Validate all confirmed occurrences
for (sid, pos), info in confirmed_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == SOURCE_TAG)
    )

    assert mask.sum() == 1, (
        f"Invalid occurrence key: ({sid}, {pos})"
    )

    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Occurrence already mapped: ({sid}, {pos})"
    )

    validated_masks[(sid, pos)] = mask

# Step 2: Validate unresolved occurrences
for (sid, pos), info in unresolved_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos) &
        (manual_review["Source_Tag"] == SOURCE_TAG)
    )

    assert mask.sum() == 1
    assert manual_review.loc[mask, "Target_Tag"].isna().all()

# Step 3: Apply confirmed mappings
for key, info in confirmed_mappings.items():

    mask = validated_masks[key]

    manual_review.loc[mask, "Target_Tag"] = info["tag"]
    manual_review.loc[mask, "Mapping_Reason"] = info["reason"]
    manual_review.loc[mask, "Mapping_Confidence"] = info["confidence"]

# Step 4: Verify current batch
updated_group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
]

updated_keys = set(
    zip(updated_group["sentence_id"], updated_group["Token_Position"])
)

assert batch_keys.issubset(updated_keys)

batch_result = updated_group[
    updated_group.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in batch_keys,
        axis=1
    )
]

print("=" * 65)
print("MAPPING APPLICATION SUMMARY")
print("=" * 65)

print("Source tag:", SOURCE_TAG)
print("Total occurrences selected:", len(ppr_mapping))
print("Mapped:", len(confirmed_mappings))
print("SN:", (batch_result["Target_Tag"] == "SN").sum())
print("SNV:", (batch_result["Target_Tag"] == "SNV").sum())
print("Unresolved:", batch_result["Target_Tag"].isna().sum())

print("\nMapping completed successfully.")

In [ ]:
# Cell 171: Complete audit of PPR.0.sg.2.gen.vi.n.n.prx

SOURCE_TAG = "PPR.0.sg.2.gen.vi.n.n.prx"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

assert not group.empty, f"No occurrences found for {SOURCE_TAG}"

target_labels = group["Target_Tag"].apply(
    lambda x: str(x).strip() if pd.notna(x) else None
)

total = len(group)
mapped = target_labels.notna().sum()
unmapped = target_labels.isna().sum()

sn_count = (target_labels == "SN").sum()
snv_count = (target_labels == "SNV").sum()

other_mapped = (
    target_labels.notna()
    & ~target_labels.isin(["SN", "SNV"])
).sum()

duplicate_keys = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

print("=" * 65)
print(f"SOURCE TAG AUDIT: {SOURCE_TAG}")
print("=" * 65)

print("Total occurrences:", total)
print("Mapped occurrences:", mapped)
print("SN:", sn_count)
print("SNV:", snv_count)
print("Other mapped tags:", other_mapped)
print("Unresolved:", unmapped)
print("Duplicate occurrence keys:", len(duplicate_keys))

print("\nMapping distribution:")
print(target_labels.value_counts(dropna=False))

print("\nOccurrences outside current batch:")

current_keys = batch_keys

group_keys = set(
    zip(group["sentence_id"], group["Token_Position"])
)

outside_keys = group_keys - current_keys

outside_batch = group[
    [
        (sid, pos) in outside_keys
        for sid, pos in zip(
            group["sentence_id"],
            group["Token_Position"]
        )
    ]
]

display(
    outside_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\nUnresolved occurrences:")

display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence"
        ]
    ]
)

print("\nAudit summary:")
print("All occurrences mapped:", mapped == total)
print("Unique occurrence keys:", group_keys == set(
    zip(group["sentence_id"], group["Token_Position"])
))

In [ ]:
# Cell 172: Find next unresolved PPR source tag

ppr_unresolved = manual_review[
    manual_review["Source_Tag"].str.startswith(
        "PPR.", na=False
    )
    & manual_review["Target_Tag"].isna()
].copy()

summary = (
    ppr_unresolved
    .groupby("Source_Tag")
    .agg(
        Unresolved_Occurrences=("Word", "size"),
        Unique_Sentences=("sentence_id", "nunique")
    )
    .sort_values(
        "Unresolved_Occurrences",
        ascending=False
    )
)

print("=" * 65)
print("REMAINING UNRESOLVED PPR SOURCE TAGS")
print("=" * 65)

print("Total unresolved PPR occurrences:", len(ppr_unresolved))
print("Remaining source tags:", len(summary))

display(summary)

if not summary.empty:
    print("\nNext source tag:")
    print(summary.index[0])
else:
    print("\nAll PPR occurrences have been reviewed.")

In [ ]:
# Cell 173: Inspect PPR.mas.sg.3.acc.ii.n.n.dst

SOURCE_TAG = "PPR.mas.sg.3.acc.ii.n.n.dst"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == SOURCE_TAG) &
    (manual_review["Target_Tag"].isna())
].copy()

batch_keys = set(
    zip(ppr_batch["sentence_id"], ppr_batch["Token_Position"])
)

print("=" * 80)
print("SOURCE TAG:", SOURCE_TAG)
print("=" * 80)

print("Total unresolved occurrences:", len(ppr_batch))
print("Unique occurrence keys:", len(batch_keys))

for _, row in ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).iterrows():

    print("\n" + "-" * 80)
    print("Sentence ID:", row["sentence_id"])
    print("Token Position:", row["Token_Position"])
    print("Word:", row["Word"])
    print("Previous Words:", row["Previous_Words"])
    print("Next Words:", row["Next_Words"])
    print("Full Sentence:", row["Full_Sentence"])

In [ ]:
# Cell 174: Mapping for PPR.mas.sg.3.acc.ii.n.n.dst

SOURCE_TAG = "PPR.mas.sg.3.acc.ii.n.n.dst"

ppr_mapping = {
    (150, 5): ("SN", "Correlative pronoun referring to स्वप्नम्", "High"),
    (169, 1): ("SN", "Independent demonstrative object pronoun", "High"),
    (270, 2): ("SN", "Independent demonstrative object pronoun", "High"),
    (344, 0): ("SNV", "Demonstrative modifier of पितरम्", "High"),
    (540, 4): ("SN", "Independent demonstrative object pronoun", "High"),
    (541, 2): ("SN", "Independent demonstrative object pronoun", "High"),
    (579, 0): ("SN", "Independent demonstrative object pronoun", "High"),
    (599, 3): ("SN", "Independent demonstrative object pronoun", "High"),
    (600, 3): ("SNV", "Demonstrative modifier of बालकम्", "High"),
    (605, 1): ("SNV", "Demonstrative modifier of बालम्", "High"),
    (632, 7): ("SN", "Independent demonstrative object pronoun", "High"),
    (635, 2): ("SN", "Independent demonstrative object pronoun", "High"),
    (639, 9): ("SN", "Independent demonstrative object pronoun", "High"),
    (694, 6): ("SN", "Independent demonstrative object pronoun", "Medium"),
    (711, 4): ("SN", "Independent demonstrative object pronoun", "High"),
    (828, 6): ("SN", "Correlative demonstrative pronoun", "High"),
    (942, 12): ("SN", "Independent demonstrative object pronoun", "High"),
    (1085, 1): ("SNV", "Demonstrative modifier of आरक्षकम्", "High"),
    (1172, 14): ("SN", "Independent demonstrative object pronoun", "High"),
    (1278, 3): ("SN", "Independent demonstrative object pronoun", "High"),
    (1279, 11): ("SN", "Independent demonstrative object pronoun", "High"),
    (1383, 0): ("SN", "Independent demonstrative object pronoun", "High"),
    (1427, 3): ("SN", "Independent demonstrative object pronoun", "High"),
}

# Validate exact occurrence keys
assert len(ppr_mapping) == len(batch_keys), (
    f"Mapping count mismatch: {len(ppr_mapping)} vs {len(batch_keys)}"
)

assert set(ppr_mapping.keys()) == batch_keys, (
    "Occurrence-key mismatch. Do not apply the mapping."
)

print("Mapping dictionary populated successfully.")
print("Total occurrences:", len(ppr_mapping))
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Unresolved:", sum(v[0] is None for v in ppr_mapping.values()))
print("Exact occurrence-key validation: PASSED")
print("manual_review modified: NO")

In [ ]:
# Cell 175: Apply PPR.mas.sg.3.acc.ii.n.n.dst mappings

SOURCE_TAG = "PPR.mas.sg.3.acc.ii.n.n.dst"

applied = 0

for key, (target_tag, reason, confidence) in ppr_mapping.items():

    sid, pos = key

    mask = (
        (manual_review["Source_Tag"] == SOURCE_TAG) &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence for {key}, found {mask.sum()}"
    )

    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Occurrence {key} is already mapped. Stopping."
    )

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

    applied += 1

print("Source tag:", SOURCE_TAG)
print("Total mappings applied:", applied)
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Mapping completed successfully.")

In [ ]:
# Cell 176: Complete source-tag audit

SOURCE_TAG = "PPR.mas.sg.3.acc.ii.n.n.dst"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

assert not group.empty, f"No occurrences found for {SOURCE_TAG}"

target_labels = group["Target_Tag"].apply(
    lambda x: str(x).strip() if pd.notna(x) else None
)

total = len(group)
mapped = target_labels.notna().sum()
unmapped = target_labels.isna().sum()

sn_count = (target_labels == "SN").sum()
snv_count = (target_labels == "SNV").sum()

other_mapped = (
    target_labels.notna()
    & ~target_labels.isin(["SN", "SNV"])
).sum()

duplicate_keys = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

group_keys = set(
    zip(group["sentence_id"], group["Token_Position"])
)

outside_keys = group_keys - batch_keys

outside_batch = group[
    [
        (sid, pos) in outside_keys
        for sid, pos in zip(
            group["sentence_id"],
            group["Token_Position"]
        )
    ]
]

print("=" * 65)
print("SOURCE TAG AUDIT:", SOURCE_TAG)
print("=" * 65)

print("Total occurrences:", total)
print("Mapped occurrences:", mapped)
print("SN:", sn_count)
print("SNV:", snv_count)
print("Other mapped tags:", other_mapped)
print("Unresolved:", unmapped)
print("Duplicate occurrence keys:", len(duplicate_keys))

print("\nMapping distribution:")
print(target_labels.value_counts(dropna=False))

print("\nOccurrences outside current batch:")
display(
    outside_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\nUnresolved occurrences:")
display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence"
        ]
    ]
)

print("\nAudit summary:")
print("All occurrences mapped:", mapped == total)
print("Duplicate keys absent:", len(duplicate_keys) == 0)

In [ ]:
# Cell 177: Inspect next unresolved PPR source tag

SOURCE_TAG = "PPR.0.sg.2.nom.i.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == SOURCE_TAG) &
    (manual_review["Target_Tag"].isna())
].copy()

batch_keys = set(
    zip(ppr_batch["sentence_id"], ppr_batch["Token_Position"])
)

print("=" * 80)
print("SOURCE TAG:", SOURCE_TAG)
print("=" * 80)

print("Total unresolved occurrences:", len(ppr_batch))
print("Unique occurrence keys:", len(batch_keys))

for _, row in ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).iterrows():

    print("\n" + "-" * 80)
    print("Sentence ID:", row["sentence_id"])
    print("Token Position:", row["Token_Position"])
    print("Word:", row["Word"])
    print("Previous Words:", row["Previous_Words"])
    print("Next Words:", row["Next_Words"])
    print("Full Sentence:", row["Full_Sentence"])

In [ ]:
# Cell 178: Mapping for PPR.0.sg.2.nom.i.n.n.prx

SOURCE_TAG = "PPR.0.sg.2.nom.i.n.n.prx"

ppr_mapping = {
    (43, 0): ("SN", "Independent second-person personal pronoun", "High"),
    (54, 0): ("SN", "Independent second-person personal pronoun", "High"),
    (56, 0): ("SN", "Independent second-person personal pronoun", "High"),
    (58, 0): ("SN", "Independent second-person personal pronoun", "High"),
    (68, 1): ("SN", "Independent second-person personal pronoun", "High"),
    (413, 4): ("SN", "Independent second-person personal pronoun", "High"),
    (428, 3): ("SN", "Independent second-person personal pronoun", "High"),
    (449, 3): ("SN", "Independent second-person personal pronoun", "High"),
    (464, 6): ("SN", "Independent second-person personal pronoun", "High"),
    (493, 2): ("SN", "Independent second-person personal pronoun", "High"),
    (510, 6): ("SN", "Independent second-person personal pronoun", "High"),
    (935, 2): ("SN", "Independent second-person personal pronoun", "High"),
    (946, 13): ("SN", "Independent second-person personal pronoun", "High"),
    (1164, 8): ("SN", "Independent second-person personal pronoun", "High"),
    (1332, 4): ("SN", "Independent second-person personal pronoun", "High"),
    (1334, 3): ("SN", "Independent second-person personal pronoun", "High"),
    (1337, 1): ("SN", "Independent second-person personal pronoun", "High"),
    (1339, 0): ("SN", "Independent second-person personal pronoun", "High"),
    (1342, 8): ("SN", "Independent second-person personal pronoun", "High"),
    (1349, 12): ("SN", "Independent second-person personal pronoun", "High"),
    (1351, 3): ("SN", "Independent second-person personal pronoun", "High"),
    (1381, 1): ("SN", "Independent second-person personal pronoun", "High"),
}

assert len(ppr_mapping) == len(batch_keys), (
    f"Mapping count mismatch: {len(ppr_mapping)} vs {len(batch_keys)}"
)

assert set(ppr_mapping.keys()) == batch_keys, (
    "Occurrence-key mismatch. Do not apply the mapping."
)

print("Mapping dictionary populated successfully.")
print("Total occurrences:", len(ppr_mapping))
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Unresolved:", sum(v[0] is None for v in ppr_mapping.values()))
print("Exact occurrence-key validation: PASSED")
print("manual_review modified: NO")

In [ ]:
# Cell 179: Apply PPR.0.sg.2.nom.i.n.n.prx mappings

SOURCE_TAG = "PPR.0.sg.2.nom.i.n.n.prx"

applied = 0

for key, (target_tag, reason, confidence) in ppr_mapping.items():

    sid, pos = key

    mask = (
        (manual_review["Source_Tag"] == SOURCE_TAG) &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence for {key}, found {mask.sum()}"
    )

    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Occurrence {key} is already mapped. Stopping."
    )

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

    applied += 1

print("Source tag:", SOURCE_TAG)
print("Total mappings applied:", applied)
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Mapping completed successfully.")

In [ ]:
# Cell 180: Complete source-tag audit

SOURCE_TAG = "PPR.0.sg.2.nom.i.n.n.prx"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

assert not group.empty, f"No occurrences found for {SOURCE_TAG}"

target_labels = group["Target_Tag"].apply(
    lambda x: str(x).strip() if pd.notna(x) else None
)

total = len(group)
mapped = target_labels.notna().sum()
unmapped = target_labels.isna().sum()

sn_count = (target_labels == "SN").sum()
snv_count = (target_labels == "SNV").sum()

other_mapped = (
    target_labels.notna()
    & ~target_labels.isin(["SN", "SNV"])
).sum()

duplicate_keys = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

group_keys = set(
    zip(group["sentence_id"], group["Token_Position"])
)

outside_keys = group_keys - batch_keys

outside_batch = group[
    [
        (sid, pos) in outside_keys
        for sid, pos in zip(
            group["sentence_id"],
            group["Token_Position"]
        )
    ]
]

print("=" * 65)
print("SOURCE TAG AUDIT:", SOURCE_TAG)
print("=" * 65)

print("Total occurrences:", total)
print("Mapped occurrences:", mapped)
print("SN:", sn_count)
print("SNV:", snv_count)
print("Other mapped tags:", other_mapped)
print("Unresolved:", unmapped)
print("Duplicate occurrence keys:", len(duplicate_keys))

print("\nMapping distribution:")
print(target_labels.value_counts(dropna=False))

print("\nOccurrences outside current batch:")
display(
    outside_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\nUnresolved occurrences:")
display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence"
        ]
    ]
)

print("\nAudit summary:")
print("All occurrences mapped:", mapped == total)
print("Duplicate keys absent:", len(duplicate_keys) == 0)

In [ ]:
# Cell 181: Inspect next unresolved PPR source tag

SOURCE_TAG = "PPR.0.sg.1.acc.ii.n.n.prx"

ppr_batch = manual_review[
    (manual_review["Source_Tag"] == SOURCE_TAG) &
    (manual_review["Target_Tag"].isna())
].copy()

batch_keys = set(
    zip(ppr_batch["sentence_id"], ppr_batch["Token_Position"])
)

print("=" * 80)
print("SOURCE TAG:", SOURCE_TAG)
print("=" * 80)

print("Total unresolved occurrences:", len(ppr_batch))
print("Unique occurrence keys:", len(batch_keys))

for _, row in ppr_batch.sort_values(
    ["sentence_id", "Token_Position"]
).iterrows():

    print("\n" + "-" * 80)
    print("Sentence ID:", row["sentence_id"])
    print("Token Position:", row["Token_Position"])
    print("Word:", row["Word"])
    print("Previous Words:", row["Previous_Words"])
    print("Next Words:", row["Next_Words"])
    print("Full Sentence:", row["Full_Sentence"])

In [ ]:
# Cell 182: Mapping for PPR.0.sg.1.acc.ii.n.n.prx

SOURCE_TAG = "PPR.0.sg.1.acc.ii.n.n.prx"

ppr_mapping = {
    (389, 2): ("SN", "Independent first-person personal pronoun", "High"),
    (399, 1): ("SN", "Independent first-person personal pronoun", "High"),
    (407, 7): ("SN", "Independent first-person personal pronoun", "High"),
    (458, 3): ("SN", "Independent first-person personal pronoun", "High"),
    (488, 1): ("SN", "Independent first-person personal pronoun", "High"),
    (495, 0): ("SN", "Independent first-person personal pronoun", "High"),
    (558, 2): ("SN", "Independent first-person personal pronoun", "High"),
    (571, 1): ("SN", "Independent first-person personal pronoun in elliptical construction", "High"),
    (629, 1): ("SN", "Independent first-person personal pronoun in elliptical construction", "High"),
    (688, 23): ("SN", "Independent first-person personal pronoun; surrounding syntax is irregular", "Medium"),
    (722, 5): ("SN", "Enclitic first-person pronoun in मे भाति", "Medium"),
    (769, 5): ("SN", "Independent first-person personal pronoun", "High"),
    (776, 1): ("SN", "Independent first-person personal pronoun", "High"),
    (787, 4): ("SN", "Independent first-person personal pronoun", "High"),
    (805, 1): ("SN", "Independent first-person personal pronoun", "High"),
    (932, 10): ("SN", "Independent first-person personal pronoun", "High"),
    (934, 1): ("SN", "Independent first-person personal pronoun", "High"),
    (1369, 2): ("SN", "Independent first-person personal pronoun", "High"),
    (1388, 0): ("SN", "Independent first-person personal pronoun", "High"),
    (1434, 7): ("SN", "Independent first-person personal pronoun", "High"),
    (1446, 4): ("SN", "Independent first-person personal pronoun", "High"),
}

# Validate exact occurrence keys
assert len(ppr_mapping) == len(batch_keys), (
    f"Mapping count mismatch: {len(ppr_mapping)} vs {len(batch_keys)}"
)

assert set(ppr_mapping.keys()) == batch_keys, (
    "Occurrence-key mismatch. Do not apply the mapping."
)

print("Mapping dictionary populated successfully.")
print("Total occurrences:", len(ppr_mapping))
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Unresolved:", sum(v[0] is None for v in ppr_mapping.values()))

print("\nConfidence distribution:")
print(pd.Series(
    [v[2] for v in ppr_mapping.values()]
).value_counts())

print("Exact occurrence-key validation: PASSED")
print("manual_review modified: NO")

In [ ]:
# Cell 183: Apply PPR.0.sg.1.acc.ii.n.n.prx mappings

SOURCE_TAG = "PPR.0.sg.1.acc.ii.n.n.prx"

applied = 0

for key, (target_tag, reason, confidence) in ppr_mapping.items():

    sid, pos = key

    mask = (
        (manual_review["Source_Tag"] == SOURCE_TAG) &
        (manual_review["sentence_id"] == sid) &
        (manual_review["Token_Position"] == pos)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence for {key}, found {mask.sum()}"
    )

    assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
        f"Occurrence {key} is already mapped. Stopping."
    )

    manual_review.loc[mask, "Target_Tag"] = target_tag
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

    applied += 1

print("Source tag:", SOURCE_TAG)
print("Total mappings applied:", applied)
print("SN:", sum(v[0] == "SN" for v in ppr_mapping.values()))
print("SNV:", sum(v[0] == "SNV" for v in ppr_mapping.values()))
print("Mapping completed successfully.")

In [ ]:
# Cell 184: Complete source-tag audit

SOURCE_TAG = "PPR.0.sg.1.acc.ii.n.n.prx"

group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

assert not group.empty, f"No occurrences found for {SOURCE_TAG}"

target_labels = group["Target_Tag"].apply(
    lambda x: str(x).strip() if pd.notna(x) else None
)

total = len(group)
mapped = target_labels.notna().sum()
unmapped = target_labels.isna().sum()

sn_count = (target_labels == "SN").sum()
snv_count = (target_labels == "SNV").sum()

other_mapped = (
    target_labels.notna()
    & ~target_labels.isin(["SN", "SNV"])
).sum()

duplicate_keys = group[
    group.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
]

group_keys = set(
    zip(group["sentence_id"], group["Token_Position"])
)

outside_keys = group_keys - batch_keys

outside_batch = group[
    [
        (sid, pos) in outside_keys
        for sid, pos in zip(
            group["sentence_id"],
            group["Token_Position"]
        )
    ]
]

print("=" * 65)
print("SOURCE TAG AUDIT:", SOURCE_TAG)
print("=" * 65)

print("Total occurrences:", total)
print("Mapped occurrences:", mapped)
print("SN:", sn_count)
print("SNV:", snv_count)
print("Other mapped tags:", other_mapped)
print("Unresolved:", unmapped)
print("Duplicate occurrence keys:", len(duplicate_keys))

print("\nMapping distribution:")
print(target_labels.value_counts(dropna=False))

print("\nOccurrences outside current batch:")
display(
    outside_batch[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\nUnresolved occurrences:")
display(
    group[group["Target_Tag"].isna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence"
        ]
    ]
)

print("\nAudit summary:")
print("All occurrences mapped:", mapped == total)
print("Duplicate keys absent:", len(duplicate_keys) == 0)

In [ ]:
# Cell 185: Verify current tag and identify next 5 batches

SOURCE_TAG = "PPR.0.sg.1.acc.ii.n.n.prx"

# 1. Audit the recently mapped source tag
group = manual_review[
    manual_review["Source_Tag"] == SOURCE_TAG
].copy()

assert not group.empty, f"No occurrences found for {SOURCE_TAG}"

unresolved = group["Target_Tag"].isna().sum()

duplicates = group.duplicated(
    subset=["sentence_id", "Token_Position"],
    keep=False
).sum()

assert unresolved == 0, (
    f"Current tag still has {unresolved} unresolved occurrences."
)

assert duplicates == 0, (
    f"Current tag has {duplicates} duplicate occurrence rows."
)

print("CURRENT SOURCE TAG AUDIT: PASSED")
print("Source tag:", SOURCE_TAG)
print("Total occurrences:", len(group))
print("SN:", (group["Target_Tag"] == "SN").sum())
print("SNV:", (group["Target_Tag"] == "SNV").sum())
print("Unresolved:", unresolved)
print("Duplicate keys:", duplicates)

# 2. Identify remaining unresolved PPR source tags
remaining = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

next_batches = (
    remaining
    .groupby("Source_Tag")
    .agg(
        Unresolved_Occurrences=("Word", "size"),
        Unique_Sentences=("sentence_id", "nunique")
    )
    .sort_values(
        ["Unresolved_Occurrences", "Unique_Sentences"],
        ascending=False
    )
    .head(5)
)

print("\n" + "=" * 75)
print("NEXT 5 SOURCE-TAG BATCHES")
print("=" * 75)

print("Remaining unresolved occurrences:", len(remaining))
print("Remaining source tags:", remaining["Source_Tag"].nunique())

display(next_batches)

In [ ]:
# Cell 186: Extract next 5 PPR source-tag batches

NEXT_SOURCE_TAGS = [
    "PPR.fem.sg.3.nom.i.n.n.dst",
    "PPR.fem.sg.3.gen.vi.n.n.prx",
    "PPR.mas.sg.3.ins.iii.n.n.dst",
    "PPR.0.pl.1.gen.vi.n.n.prx",
    "PPR.fem.sg.3.nom.i.n.n.prx"
]

ppr_batches = {}

for source_tag in NEXT_SOURCE_TAGS:

    batch = manual_review[
        (manual_review["Source_Tag"] == source_tag) &
        (manual_review["Target_Tag"].isna())
    ].copy()

    keys = set(
        zip(batch["sentence_id"], batch["Token_Position"])
    )

    assert len(keys) == len(batch), (
        f"Duplicate occurrence keys found in {source_tag}"
    )

    ppr_batches[source_tag] = batch

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("Total occurrences:", len(batch))
    print("Unique occurrence keys:", len(keys))
    print("=" * 100)

    for _, row in batch.sort_values(
        ["sentence_id", "Token_Position"]
    ).iterrows():

        print("\n" + "-" * 80)
        print("Sentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Previous Words:", row["Previous_Words"])
        print("Next Words:", row["Next_Words"])
        print("Full Sentence:", row["Full_Sentence"])

# Final extraction validation
print("\n" + "=" * 100)
print("BATCH EXTRACTION SUMMARY")
print("=" * 100)

print("Total source tags:", len(ppr_batches))
print(
    "Total extracted occurrences:",
    sum(len(batch) for batch in ppr_batches.values())
)

for source_tag, batch in ppr_batches.items():
    print(source_tag, ":", len(batch))

In [ ]:
# Cell 187: Create mappings for all 5 PPR source tags

# Helper: create occurrence-level mappings
def make_ppr_mapping(sn_keys, snv_keys, sn_reason, snv_reason):

    mapping = {}

    for key in sn_keys:
        mapping[key] = (
            "SN",
            sn_reason,
            "High"
        )

    for key in snv_keys:
        confidence = "Medium" if key == (1365, 1) else "High"

        mapping[key] = (
            "SNV",
            snv_reason,
            confidence
        )

    return mapping


# Batch 1: Feminine third-person nominative demonstrative
TAG_1 = "PPR.fem.sg.3.nom.i.n.n.dst"

SN_1 = [
    (30, 2), (223, 4), (232, 1), (245, 4), (401, 13),
    (404, 0), (407, 8), (433, 5), (457, 5), (934, 0),
    (951, 7), (955, 1), (959, 4), (961, 2), (962, 5),
    (963, 1), (1090, 0), (1093, 0), (1097, 12), (1099, 0)
]

SNV_1 = []


# Batch 2: Feminine third-person genitive demonstrative
TAG_2 = "PPR.fem.sg.3.gen.vi.n.n.prx"

SN_2 = []

SNV_2 = [
    (7, 1), (9, 0), (10, 8), (11, 1), (15, 0),
    (444, 4), (897, 1), (899, 0), (901, 1), (905, 0),
    (949, 0), (950, 2), (953, 2), (954, 1), (954, 6),
    (957, 2), (968, 0), (1370, 11), (1428, 4)
]


# Batch 3: Masculine third-person instrumental demonstrative
TAG_3 = "PPR.mas.sg.3.ins.iii.n.n.dst"

SN_3 = [
    (227, 2), (334, 2), (338, 4), (340, 5), (585, 3),
    (624, 9), (633, 1), (726, 4), (733, 5), (939, 14),
    (957, 0), (1120, 10), (1159, 1), (1173, 14),
    (1297, 4), (1458, 5)
]

SNV_3 = [
    (639, 13), (854, 14)
]


# Batch 4: First-person plural genitive pronoun
TAG_4 = "PPR.0.pl.1.gen.vi.n.n.prx"

SN_4 = [
    (547, 3), (1057, 4)
]

SNV_4 = [
    (305, 4), (315, 3), (326, 9), (350, 2), (545, 25),
    (546, 0), (565, 9), (700, 0), (1045, 2), (1054, 2),
    (1271, 6), (1365, 1), (1449, 0), (1450, 12), (1451, 15)
]


# Batch 5: Feminine third-person nominative pronoun
TAG_5 = "PPR.fem.sg.3.nom.i.n.n.prx"

SN_5 = [
    (3, 0), (16, 0), (448, 2), (509, 6), (665, 22),
    (688, 18), (893, 0), (906, 0), (951, 1), (952, 6),
    (964, 6), (1077, 2), (1101, 0), (1359, 0),
    (1425, 0), (1442, 1)
]

SNV_5 = []


# Build independent dictionaries
ppr_mappings = {
    TAG_1: make_ppr_mapping(
        SN_1, SNV_1,
        "Independent feminine demonstrative pronoun",
        "Feminine demonstrative pronoun modifying a noun"
    ),

    TAG_2: make_ppr_mapping(
        SN_2, SNV_2,
        "Independent feminine genitive pronoun",
        "Genitive demonstrative pronoun expressing possession or relation to a noun"
    ),

    TAG_3: make_ppr_mapping(
        SN_3, SNV_3,
        "Independent instrumental pronoun",
        "Instrumental demonstrative modifying a noun"
    ),

    TAG_4: make_ppr_mapping(
        SN_4, SNV_4,
        "Independent first-person plural pronoun governed by कृते",
        "First-person plural genitive pronoun modifying a noun"
    ),

    TAG_5: make_ppr_mapping(
        SN_5, SNV_5,
        "Independent feminine demonstrative or personal pronoun",
        "Feminine demonstrative pronoun modifying a noun"
    )
}


# Validate each dictionary independently
print("=" * 80)
print("FIVE-BATCH MAPPING VALIDATION")
print("=" * 80)

for source_tag, mapping in ppr_mappings.items():

    expected_batch = ppr_batches[source_tag]

    expected_keys = set(
        zip(
            expected_batch["sentence_id"],
            expected_batch["Token_Position"]
        )
    )

    actual_keys = set(mapping.keys())

    assert len(mapping) == len(expected_batch), (
        f"Mapping count mismatch: {source_tag}"
    )

    assert actual_keys == expected_keys, (
        f"Occurrence-key mismatch: {source_tag}"
    )

    sn = sum(v[0] == "SN" for v in mapping.values())
    snv = sum(v[0] == "SNV" for v in mapping.values())
    unresolved = sum(v[0] is None for v in mapping.values())

    print("\nSource tag:", source_tag)
    print("Total:", len(mapping))
    print("SN:", sn)
    print("SNV:", snv)
    print("Unresolved:", unresolved)
    print("Exact key validation: PASSED")


# Overall validation
total_mappings = sum(
    len(mapping) for mapping in ppr_mappings.values()
)

assert total_mappings == 90

print("\n" + "=" * 80)
print("ALL FIVE MAPPING DICTIONARIES VALIDATED")
print("Total mappings:", total_mappings)
print("manual_review modified: NO")
print("=" * 80)

In [ ]:
# Cell 188: Apply all 5 validated PPR mappings

total_applied = 0

for source_tag, mapping in ppr_mappings.items():

    print("\nProcessing:", source_tag)

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag) &
            (manual_review["sentence_id"] == sid) &
            (manual_review["Token_Position"] == pos)
        )

        # Verify exactly one occurrence
        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        # Never overwrite an existing annotation
        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already mapped: {source_tag}, {key}"
        )

        # Apply annotation
        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        total_applied += 1

    print("Mappings applied:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))

assert total_applied == 90

print("\n" + "=" * 75)
print("ALL FIVE BATCHES APPLIED SUCCESSFULLY")
print("Total mappings applied:", total_applied)
print("=" * 75)

In [ ]:
# Cell 190: Rebuild audit and identify next PPR batches

import pandas as pd

# Rebuild audit summary for the five completed source tags
completed_tags = [
    "PPR.fem.sg.3.nom.i.n.n.dst",
    "PPR.fem.sg.3.gen.vi.n.n.prx",
    "PPR.mas.sg.3.ins.iii.n.n.dst",
    "PPR.0.pl.1.gen.vi.n.n.prx",
    "PPR.fem.sg.3.nom.i.n.n.prx"
]

audit_results = []

for source_tag in completed_tags:

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ].copy()

    labels = group["Target_Tag"].apply(
        lambda x: str(x).strip() if pd.notna(x) else None
    )

    audit_results.append({
        "Source_Tag": source_tag,
        "Total": len(group),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Unresolved": labels.isna().sum(),
        "Other": (
            labels.notna() &
            ~labels.isin(["SN", "SNV"])
        ).sum(),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"],
            keep=False
        ).sum()
    })

audit_df = pd.DataFrame(audit_results)

# Validate completed batches
all_passed = (
    audit_df["Total"].sum() == 90
    and audit_df["Unresolved"].sum() == 0
    and audit_df["Other"].sum() == 0
    and audit_df["Duplicates"].sum() == 0
)

print("Five-batch audit passed:", all_passed)
display(audit_df)

assert all_passed, "Audit failed. Inspect the table."

# Calculate remaining unresolved PPR occurrences
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\n" + "=" * 75)
print("REMAINING PPR PROGRESS")
print("=" * 75)

print("Unresolved occurrences:", len(remaining_ppr))
print("Unresolved source tags:", len(remaining_summary))

print("\nNext five source tags:")
display(
    remaining_summary.head(5)
    .rename("Unresolved_Count")
    .to_frame()
)

In [ ]:
# Cell 191: Extract next five unresolved PPR source-tag batches

next_ppr_tags = remaining_summary.head(5).index.tolist()

ppr_batches = {}

for source_tag in next_ppr_tags:

    batch = remaining_ppr[
        remaining_ppr["Source_Tag"] == source_tag
    ].copy()

    batch = batch.sort_values(
        ["sentence_id", "Token_Position"]
    )

    ppr_batches[source_tag] = batch

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("TOTAL OCCURRENCES:", len(batch))
    print("=" * 100)

    for _, row in batch.iterrows():

        print("\nSentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Previous Words:", row["Previous_Words"])
        print("Next Words:", row["Next_Words"])
        print("Full Sentence:", row["Full_Sentence"])

print("\n" + "=" * 100)
print("TOTAL BATCHES:", len(ppr_batches))
print("TOTAL OCCURRENCES:", sum(len(x) for x in ppr_batches.values()))
print("=" * 100)

In [ ]:
# Cell 192: Prepare and validate next five PPR mappings

def make_mapping(keys, target_tag, reason, confidence="High"):
    return {
        key: (target_tag, reason, confidence)
        for key in keys
    }


new_ppr_mappings = {

    # 1. Masculine plural nominative demonstrative
    "PPR.mas.pl.3.nom.i.n.n.dst": make_mapping(
        [
            (7, 0), (112, 12), (147, 3), (316, 2),
            (350, 8), (432, 2), (542, 1), (592, 2),
            (599, 6), (880, 1), (883, 0), (897, 0),
            (1136, 1), (1169, 0)
        ],
        "SN",
        "Independent masculine plural demonstrative pronoun used substantively."
    ),

    # 2. Masculine plural genitive demonstrative
    "PPR.mas.pl.3.gen.vi.n.n.dst": {
        **make_mapping(
            [
                (272, 1), (386, 9), (590, 0), (591, 0),
                (745, 0), (847, 1), (853, 6), (854, 0),
                (932, 0), (1260, 3), (1260, 7), (1335, 6)
            ],
            "SNV",
            "Genitive demonstrative pronoun dependent on a noun."
        ),
        (868, 2): (
            "SNV",
            "Genitive pronoun dependent on the verbal noun अनुसरणे; objective relationship.",
            "Medium"
        )
    },

    # 3. Neuter singular nominative demonstrative
    "PPR.neu.sg.3.nom.i.n.n.prx": {
        **make_mapping(
            [
                (70, 0), (203, 0), (653, 0), (659, 12),
                (662, 1), (666, 0), (871, 1), (878, 0),
                (947, 1), (1253, 0), (1419, 1)
            ],
            "SN",
            "Independent neuter demonstrative pronoun used substantively."
        ),
        (118, 2): (
            "SNV",
            "Demonstrative pronoun एतत् modifies the noun चित्रम्.",
            "High"
        )
    },

    # 4. Masculine singular nominative demonstrative
    "PPR.mas.sg.3.nom.i.n.n.dst": make_mapping(
        [
            (236, 6), (237, 10), (324, 0), (613, 1),
            (615, 1), (617, 5), (688, 11), (759, 0),
            (1256, 1), (1275, 5), (1285, 2), (1294, 8)
        ],
        "SN",
        "Independent masculine singular demonstrative pronoun used substantively."
    ),

    # 5. Masculine singular instrumental pronoun
    "PPR.mas.sg.3.ins.iii.n.n.prx": make_mapping(
        [
            (530, 6), (535, 11), (535, 32), (536, 2),
            (556, 4), (659, 14), (665, 14), (825, 0),
            (1378, 15), (1385, 6), (1421, 1)
        ],
        "SN",
        "Independent instrumental pronoun functioning as an agent or instrument."
    )
}


# Validate exact occurrence keys and annotation status
total_validated = 0

for source_tag, mapping in new_ppr_mappings.items():

    batch = ppr_batches[source_tag]

    batch_keys = set(
        zip(batch["sentence_id"], batch["Token_Position"])
    )

    mapping_keys = set(mapping.keys())

    assert batch_keys == mapping_keys, (
        f"Key mismatch in {source_tag}. "
        f"Missing: {batch_keys - mapping_keys}; "
        f"Extra: {mapping_keys - batch_keys}"
    )

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already annotated: {source_tag}, {key}"
        )

        assert target_tag in ["SN", "SNV"]
        assert confidence in ["High", "Medium", "Low"]

        total_validated += 1

    print("\nSource tag:", source_tag)
    print("Total:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))
    print("Exact keys validated:", True)

assert total_validated == 62

print("\n" + "=" * 75)
print("ALL FIVE BATCHES VALIDATED SUCCESSFULLY")
print("Total mappings:", total_validated)
print("manual_review modified: NO")
print("=" * 75)

In [ ]:
# Cell 193: Apply five validated PPR mappings

total_applied = 0

for source_tag, mapping in new_ppr_mappings.items():

    print("\nProcessing:", source_tag)

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        # Verify exactly one occurrence
        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        # Never overwrite existing annotations
        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already mapped: {source_tag}, {key}"
        )

        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        total_applied += 1

    print("Mappings applied:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))

assert total_applied == 62

print("\n" + "=" * 75)
print("ALL FIVE BATCHES APPLIED SUCCESSFULLY")
print("Total mappings applied:", total_applied)
print("=" * 75)

In [ ]:
# Cell 194: Exact-key audit of the five newly processed batches

audit_results = []

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ].copy()

    batch_rows = []

    # Retrieve only the exact occurrences processed in Cell 193
    for key, expected in mapping.items():

        sid, pos = key
        expected_tag = expected[0]

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == expected_tag, (
            f"Incorrect annotation: {source_tag}, {key}. "
            f"Expected {expected_tag}, found {row['Target_Tag']}"
        )

        batch_rows.append(row)

    batch_df = pd.DataFrame(batch_rows)

    labels = batch_df["Target_Tag"]

    duplicate_count = group.duplicated(
        ["sentence_id", "Token_Position"],
        keep=False
    ).sum()

    batch_keys = set(mapping.keys())

    group_keys = set(
        zip(group["sentence_id"], group["Token_Position"])
    )

    outside_count = len(group_keys - batch_keys)

    audit_results.append({
        "Source_Tag": source_tag,
        "Batch_Total": len(batch_df),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Unresolved": labels.isna().sum(),
        "Other": (
            labels.notna()
            & ~labels.isin(["SN", "SNV"])
        ).sum(),
        "Duplicates_In_Group": duplicate_count,
        "Previously_Outside_Batch": outside_count
    })

audit_df = pd.DataFrame(audit_results)

print("=" * 90)
print("EXACT-KEY FIVE-BATCH AUDIT")
print("=" * 90)

display(audit_df)

print("\nTotal batch occurrences:", audit_df["Batch_Total"].sum())
print("Total SN:", audit_df["SN"].sum())
print("Total SNV:", audit_df["SNV"].sum())
print("Total unresolved:", audit_df["Unresolved"].sum())
print("Total other:", audit_df["Other"].sum())
print("Total duplicate rows:", audit_df["Duplicates_In_Group"].sum())
print("Previously outside batch:", audit_df["Previously_Outside_Batch"].sum())

all_passed = (
    audit_df["Batch_Total"].sum() == 62
    and audit_df["Unresolved"].sum() == 0
    and audit_df["Other"].sum() == 0
    and audit_df["Duplicates_In_Group"].sum() == 0
)

print("\nALL FIVE BATCH AUDITS PASSED:", all_passed)

assert all_passed, "Exact-key audit failed. Inspect the results."

In [ ]:
# Cell 195: Calculate remaining PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("=" * 75)
print("UPDATED PPR ANNOTATION PROGRESS")
print("=" * 75)

print("Unresolved PPR occurrences:", len(remaining_ppr))
print("Unresolved source tags:", len(remaining_summary))

print("\nNext five source tags:")

display(
    remaining_summary.head(5)
    .rename("Unresolved_Count")
    .to_frame()
)

print("\nTotal completed in latest batch:", audit_df["Batch_Total"].sum())
print("Latest batch audit:", all_passed)

In [ ]:
# Cell 196: Extract next five unresolved PPR batches

next_ppr_tags = remaining_summary.head(5).index.tolist()

ppr_batches = {}

for source_tag in next_ppr_tags:

    batch = remaining_ppr[
        remaining_ppr["Source_Tag"] == source_tag
    ].copy()

    batch = batch.sort_values(
        ["sentence_id", "Token_Position"]
    )

    ppr_batches[source_tag] = batch

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("TOTAL OCCURRENCES:", len(batch))
    print("=" * 100)

    for _, row in batch.iterrows():

        print("\nSentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Previous Words:", row["Previous_Words"])
        print("Next Words:", row["Next_Words"])
        print("Full Sentence:", row["Full_Sentence"])

print("\n" + "=" * 100)
print("TOTAL BATCHES:", len(ppr_batches))
print(
    "TOTAL OCCURRENCES:",
    sum(len(batch) for batch in ppr_batches.values())
)
print("=" * 100)

In [ ]:
# Cell 197: Prepare and validate next five PPR mappings

def make_mapping(keys, target_tag, reason, confidence="High"):
    return {
        key: (target_tag, reason, confidence)
        for key in keys
    }


new_ppr_mappings = {

    # 1. Neuter singular demonstrative
    "PPR.neu.sg.3.0.i.n.n.prx": {

        **make_mapping(
            [
                (27, 0),
                (34, 0),
                (44, 8),
                (93, 0),
                (323, 0),
                (1170, 10),
                (1460, 1)
            ],
            "SN",
            "Independent neuter demonstrative pronoun used substantively."
        ),

        (41, 0): (
            "SNV",
            "एतावत् qualifies the noun phrase महत् कार्यम्.",
            "High"
        ),

        (1453, 0): (
            "SNV",
            "एतद् is associated with अतिरिक्तम् in the expression एतद् अतिरिक्तम्.",
            "Medium"
        )
    },

    # 2. Neuter singular accusative demonstrative
    "PPR.neu.sg.3.acc.ii.n.n.prx": {

        **make_mapping(
            [
                (826, 1),
                (937, 1),
                (1021, 1),
                (1158, 6),
                (1170, 6),
                (1377, 1),
                (1420, 9)
            ],
            "SN",
            "Independent neuter demonstrative pronoun functioning substantively."
        ),

        (1324, 4): (
            "SNV",
            "एतत् qualifies सर्वम् in the expression एतत् सर्वम्.",
            "High"
        )
    },

    # 3. Neuter singular demonstrative
    "PPR.neu.sg.3.0.i.n.n.dst": make_mapping(
        [
            (313, 17),
            (351, 1),
            (573, 0),
            (588, 3),
            (915, 7),
            (1167, 13)
        ],
        "SN",
        "Independent demonstrative pronoun used substantively."
    ),

    # 4. Feminine singular genitive demonstrative
    "PPR.fem.sg.3.gen.vi.n.n.dst": make_mapping(
        [
            (128, 3),
            (361, 0),
            (407, 2),
            (507, 8),
            (841, 5),
            (956, 1),
            (1300, 1)
        ],
        "SNV",
        "Feminine genitive demonstrative pronoun modifying or possessing a noun."
    ),

    # 5. Second-person singular nominative pronoun
    "PPR.0.sg.2.0.i.n.n.prx": make_mapping(
        [
            (43, 5),
            (43, 7),
            (43, 12),
            (44, 0),
            (44, 4),
            (44, 6),
            (1439, 1)
        ],
        "SN",
        "Independent second-person pronoun used substantively."
    )
}


# Explicitly preserve the uncertain occurrence
ppr_unresolved_keys = {
    "PPR.neu.sg.3.0.i.n.n.dst": {
        (656, 12)
    }
}


# Exact-key validation
total_validated = 0
total_unresolved = 0

for source_tag, mapping in new_ppr_mappings.items():

    batch = ppr_batches[source_tag]

    batch_keys = set(
        zip(batch["sentence_id"], batch["Token_Position"])
    )

    mapping_keys = set(mapping.keys())

    unresolved_keys = ppr_unresolved_keys.get(source_tag, set())

    # Every batch occurrence must be mapped or explicitly unresolved
    assert batch_keys == mapping_keys | unresolved_keys, (
        f"Key mismatch in {source_tag}. "
        f"Missing: {batch_keys - mapping_keys - unresolved_keys}; "
        f"Extra: {(mapping_keys | unresolved_keys) - batch_keys}"
    )

    # Validate mapped occurrences
    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already annotated: {source_tag}, {key}"
        )

        assert target_tag in ["SN", "SNV"]
        assert confidence in ["High", "Medium", "Low"]

        total_validated += 1

    # Validate unresolved occurrences without assigning a tag
    for sid, pos in unresolved_keys:

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1
        assert manual_review.loc[mask, "Target_Tag"].isna().all()

        total_unresolved += 1

    print("\nSource tag:", source_tag)
    print("Total occurrences:", len(batch))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))
    print("Explicitly unresolved:", len(unresolved_keys))
    print("Exact keys validated: True")


assert total_validated == 37
assert total_unresolved == 1

print("\n" + "=" * 75)
print("ALL FIVE BATCHES VALIDATED")
print("Mappings ready:", total_validated)
print("Preserved unresolved:", total_unresolved)
print("Total occurrences checked:", total_validated + total_unresolved)
print("manual_review modified: NO")
print("=" * 75)

In [ ]:
# Cell 198: Apply the 37 validated mappings

total_applied = 0

for source_tag, mapping in new_ppr_mappings.items():

    print("\nProcessing:", source_tag)

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, f"Occurrence mismatch: {source_tag}, {key}"

        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already mapped: {source_tag}, {key}"
        )

        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        total_applied += 1

    print("Mappings applied:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))

assert total_applied == 37

print("\n" + "=" * 75)
print("MAPPINGS APPLIED SUCCESSFULLY")
print("Total mappings applied:", total_applied)
print("Preserved unresolved occurrence:", (656, 12))
print("=" * 75)

In [ ]:
# Cell 199: Exact-key audit of the five newly processed PPR batches

audit_results = []

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ].copy()

    batch_rows = []

    for key, (expected_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == expected_tag, (
            f"Incorrect target tag: {source_tag}, {key}"
        )

        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        batch_rows.append(row)

    batch_df = pd.DataFrame(batch_rows)

    labels = batch_df["Target_Tag"]

    unresolved_keys = ppr_unresolved_keys.get(source_tag, set())

    # Confirm explicitly unresolved occurrences remain untouched
    for sid, pos in unresolved_keys:

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1
        assert manual_review.loc[mask, "Target_Tag"].isna().all()

    batch_keys = set(mapping.keys()) | unresolved_keys

    group_keys = set(
        zip(group["sentence_id"], group["Token_Position"])
    )

    duplicate_count = group.duplicated(
        ["sentence_id", "Token_Position"],
        keep=False
    ).sum()

    audit_results.append({
        "Source_Tag": source_tag,
        "Batch_Total": len(batch_df) + len(unresolved_keys),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Unresolved": len(unresolved_keys),
        "Other": (
            labels.notna()
            & ~labels.isin(["SN", "SNV"])
        ).sum(),
        "Duplicates": duplicate_count,
        "Outside_Batch": len(group_keys - batch_keys)
    })

audit_df = pd.DataFrame(audit_results)

print("=" * 90)
print("EXACT-KEY FIVE-BATCH AUDIT")
print("=" * 90)

display(audit_df)

print("\nTotal occurrences:", audit_df["Batch_Total"].sum())
print("Total SN:", audit_df["SN"].sum())
print("Total SNV:", audit_df["SNV"].sum())
print("Total unresolved:", audit_df["Unresolved"].sum())
print("Total other:", audit_df["Other"].sum())
print("Duplicate rows:", audit_df["Duplicates"].sum())
print("Previously outside batch:", audit_df["Outside_Batch"].sum())

all_passed = (
    audit_df["Batch_Total"].sum() == 38
    and audit_df["Unresolved"].sum() == 1
    and audit_df["Other"].sum() == 0
    and audit_df["Duplicates"].sum() == 0
    and audit_df["Outside_Batch"].sum() == 0
)

print("\nALL FIVE BATCH AUDITS PASSED:", all_passed)

assert all_passed, "Audit failed. Inspect the results."

In [ ]:
# Cell 200: Calculate remaining PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("=" * 75)
print("UPDATED PPR ANNOTATION PROGRESS")
print("=" * 75)

print("Unresolved PPR occurrences:", len(remaining_ppr))
print("Unresolved source tags:", len(remaining_summary))

print("\nNext five source tags:")

display(
    remaining_summary.head(5)
    .rename("Unresolved_Count")
    .to_frame()
)

print("\nLatest batch completed mappings:", 37)
print("Latest batch preserved unresolved:", 1)
print("Latest batch audit:", all_passed)

In [ ]:
# Cell 201: Extract next five unresolved PPR batches

next_ppr_tags = remaining_summary.head(5).index.tolist()

ppr_batches = {}

for source_tag in next_ppr_tags:

    batch = remaining_ppr[
        remaining_ppr["Source_Tag"] == source_tag
    ].copy()

    batch = batch.sort_values(
        ["sentence_id", "Token_Position"]
    )

    ppr_batches[source_tag] = batch

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("TOTAL OCCURRENCES:", len(batch))
    print("=" * 100)

    for _, row in batch.iterrows():

        print("\nSentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Previous Words:", row["Previous_Words"])
        print("Next Words:", row["Next_Words"])
        print("Full Sentence:", row["Full_Sentence"])

print("\n" + "=" * 100)
print("TOTAL BATCHES:", len(ppr_batches))
print(
    "TOTAL OCCURRENCES:",
    sum(len(batch) for batch in ppr_batches.values())
)
print("=" * 100)

In [ ]:
# Cell 202: Prepare and validate five PPR mapping batches

def make_mapping(keys, target_tag, reason, confidence="High"):
    return {
        key: (target_tag, reason, confidence)
        for key in keys
    }


new_ppr_mappings = {

    # 1. First-person plural nominative pronoun
    "PPR.0.pl.1.nom.i.n.n.prx": make_mapping(
        [
            (235, 2), (562, 0), (965, 0), (1151, 0),
            (1252, 2), (1254, 2), (1303, 6)
        ],
        "SN",
        "Independent first-person plural pronoun used as sentence subject."
    ),

    # 2. Neuter singular nominative demonstrative
    "PPR.neu.sg.3.nom.i.n.n.dst": {
        **make_mapping(
            [
                (382, 1), (423, 3), (566, 0),
                (1105, 5), (1321, 7)
            ],
            "SN",
            "Independent neuter demonstrative pronoun used substantively."
        ),

        (1321, 8): (
            "SNV",
            "Second तत् qualifies सर्वम् in the expression तत् सर्वम्.",
            "High"
        ),

        (1450, 9): (
            "SNV",
            "तत् qualifies सर्वम् in the expression तत् सर्वम्.",
            "High"
        )
    },

    # 3. Masculine plural instrumental demonstrative
    "PPR.mas.pl.3.ins.iii.n.n.dst": {

        **make_mapping(
            [
                (434, 2), (717, 15), (744, 7),
                (944, 1), (1461, 3)
            ],
            "SN",
            "Independent instrumental pronoun functioning as an agent or instrument."
        ),

        (1451, 13): (
            "SNV",
            "तैः qualifies सर्वैः in the expression तैः सर्वैः.",
            "High"
        )
    },

    # 4. First-person singular genitive pronoun
    "PPR.mas.sg.1.gen.vi.n.n.prx": make_mapping(
        [
            (346, 5), (464, 3), (488, 4),
            (488, 9), (1095, 2), (1406, 8)
        ],
        "SNV",
        "First-person genitive pronoun in a possessive or dependent construction."
    ),

    # 5. Masculine plural nominative quantifier
    "PPR.mas.pl.3.nom.i.n.n.0": {

        **make_mapping(
            [
                (51, 0), (51, 3), (52, 0)
            ],
            "SN",
            "Independent quantifier pronoun used substantively."
        ),

        **make_mapping(
            [
                (39, 7), (235, 3), (327, 1)
            ],
            "SNV",
            "Quantifier सर्वे qualifies an accompanying pronoun."
        )
    }
}


# Exact occurrence-level validation
total_validated = 0

for source_tag, mapping in new_ppr_mappings.items():

    batch = ppr_batches[source_tag]

    batch_keys = set(
        zip(batch["sentence_id"], batch["Token_Position"])
    )

    mapping_keys = set(mapping.keys())

    assert batch_keys == mapping_keys, (
        f"Key mismatch in {source_tag}. "
        f"Missing: {batch_keys - mapping_keys}; "
        f"Extra: {mapping_keys - batch_keys}"
    )

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already annotated: {source_tag}, {key}"
        )

        assert target_tag in ["SN", "SNV"]
        assert confidence in ["High", "Medium", "Low"]

        total_validated += 1

    print("\nSource tag:", source_tag)
    print("Total:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))
    print("Exact keys validated: True")


assert total_validated == 32

print("\n" + "=" * 75)
print("ALL FIVE BATCHES VALIDATED SUCCESSFULLY")
print("Total mappings:", total_validated)
print("manual_review modified: NO")
print("=" * 75)

In [ ]:
# Cell 203: Apply five validated PPR mappings

total_applied = 0

for source_tag, mapping in new_ppr_mappings.items():

    print("\nProcessing:", source_tag)

    for key, (target_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        assert manual_review.loc[mask, "Target_Tag"].isna().all(), (
            f"Already mapped: {source_tag}, {key}"
        )

        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        total_applied += 1

    print("Mappings applied:", len(mapping))
    print("SN:", sum(v[0] == "SN" for v in mapping.values()))
    print("SNV:", sum(v[0] == "SNV" for v in mapping.values()))

assert total_applied == 32

print("\n" + "=" * 75)
print("ALL FIVE BATCHES APPLIED SUCCESSFULLY")
print("Total mappings applied:", total_applied)
print("=" * 75)

In [ ]:
# Cell 204: Exact-key audit of five PPR batches

audit_results = []

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ].copy()

    batch_rows = []

    for key, (expected_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == expected_tag, (
            f"Incorrect target tag: {source_tag}, {key}"
        )

        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        batch_rows.append(row)

    batch_df = pd.DataFrame(batch_rows)
    labels = batch_df["Target_Tag"]

    batch_keys = set(mapping.keys())

    group_keys = set(
        zip(group["sentence_id"], group["Token_Position"])
    )

    audit_results.append({
        "Source_Tag": source_tag,
        "Batch_Total": len(batch_df),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Unresolved": labels.isna().sum(),
        "Other": (
            labels.notna()
            & ~labels.isin(["SN", "SNV"])
        ).sum(),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"],
            keep=False
        ).sum(),
        "Outside_Batch": len(group_keys - batch_keys)
    })

audit_df = pd.DataFrame(audit_results)

print("=" * 90)
print("EXACT-KEY FIVE-BATCH AUDIT")
print("=" * 90)

display(audit_df)

print("\nTotal occurrences:", audit_df["Batch_Total"].sum())
print("Total SN:", audit_df["SN"].sum())
print("Total SNV:", audit_df["SNV"].sum())
print("Total unresolved:", audit_df["Unresolved"].sum())
print("Total other:", audit_df["Other"].sum())
print("Duplicate rows:", audit_df["Duplicates"].sum())
print("Previously outside batch:", audit_df["Outside_Batch"].sum())

all_passed = (
    audit_df["Batch_Total"].sum() == 32
    and audit_df["Unresolved"].sum() == 0
    and audit_df["Other"].sum() == 0
    and audit_df["Duplicates"].sum() == 0
    and audit_df["Outside_Batch"].sum() == 0
)

print("\nALL FIVE BATCH AUDITS PASSED:", all_passed)

assert all_passed, "Audit failed. Inspect the results."

In [ ]:
# Cell 205: Calculate remaining PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("=" * 75)
print("UPDATED PPR ANNOTATION PROGRESS")
print("=" * 75)

print("Unresolved PPR occurrences:", len(remaining_ppr))
print("Unresolved source tags:", len(remaining_summary))

print("\nNext five source tags:")

display(
    remaining_summary.head(5)
    .rename("Unresolved_Count")
    .to_frame()
)

print("\nLatest batch completed:", audit_df["Batch_Total"].sum())
print("Latest batch SN:", audit_df["SN"].sum())
print("Latest batch SNV:", audit_df["SNV"].sum())
print("Latest batch audit:", all_passed)

In [ ]:
# Cell 206: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.0.sg.1.dat.iv.n.n.prx",
    "PPR.0.pl.1.ins.iii.n.n.prx",
    "PPR.mas.pl.3.acc.ii.n.n.dst",
    "PPR.fem.pl.3.0.i.n.n.0",
    "PPR.fem.sg.3.acc.ii.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence",
    "Target_Tag"
]

for source_tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == source_tag)
        & (manual_review["Target_Tag"].isna())
    ].copy()

    group = group.sort_values(
        ["sentence_id", "Token_Position"]
    )

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("UNRESOLVED OCCURRENCES:", len(group))
    print("=" * 100)

    display(group[context_columns].reset_index(drop=True))

print("\nTotal occurrences inspected:",
      sum(
          (
              (manual_review["Source_Tag"] == tag)
              & manual_review["Target_Tag"].isna()
          ).sum()
          for tag in next_ppr_tags
      ))

In [ ]:
# Cell 207: Prepare exact occurrence-level PPR mappings

new_ppr_mappings = {

    "PPR.0.sg.1.dat.iv.n.n.prx": {
        (665, 15): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
        (1333, 6): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
        (1337, 0): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
        (1339, 1): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
        (1341, 4): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
        (1345, 7): ("SN", "Dative first-person pronoun functioning independently as recipient.", "High"),
    },

    "PPR.0.pl.1.ins.iii.n.n.prx": {
        (420, 6): ("SN", "Instrumental first-person plural pronoun functioning independently.", "High"),
        (547, 8): ("SN", "First-person plural pronoun referring independently to the speakers.", "High"),
        (1168, 3): ("SN", "Instrumental pronoun expressing the agent of रक्षिताः.", "High"),
        (1451, 11): ("SN", "Instrumental first-person plural pronoun functioning independently.", "Medium"),
        (1453, 12): ("SN", "Instrumental pronoun expressing the agent of निर्मीयन्ते.", "High"),
    },

    "PPR.mas.pl.3.acc.ii.n.n.dst": {
        (219, 4): ("SN", "Anaphoric pronoun referring independently to सगरपुत्रान्.", "High"),
        (317, 1): ("SN", "Accusative pronoun referring independently to the people concerned.", "Medium"),
        (499, 7): ("SN", "Anaphoric pronoun functioning independently; सर्वान् is an accompanying quantifier.", "Medium"),
        (727, 1): ("SN", "Accusative pronoun functioning independently as the object of आनयति.", "High"),
        (935, 3): ("SNV", "Demonstrative pronoun modifies the noun पशून् in तान् सर्वान् पशून्.", "High"),
    },

    "PPR.fem.pl.3.0.i.n.n.0": {
        (1178, 5): ("SN", "Collective pronoun used independently as a speaker label.", "High"),
        (1183, 0): ("SN", "Collective pronoun used independently as a speaker label.", "High"),
        (1189, 0): ("SN", "Collective pronoun used independently as a speaker label.", "High"),
        (1191, 5): ("SN", "Collective pronoun used independently as a speaker label.", "High"),
        (1204, 7): ("SN", "Collective pronoun used independently as a speaker label.", "High"),
    },

    "PPR.fem.sg.3.acc.ii.n.n.prx": {
        (6, 2): ("SN", "Demonstrative pronoun functioning independently as the object of कथयन्ति.", "High"),
        (8, 6): ("SN", "Demonstrative pronoun functioning independently as an object of पूजयन्ति.", "Medium"),
        (896, 2): ("SN", "Demonstrative pronoun functioning independently as the object of कथयन्ति.", "High"),
        (898, 6): ("SN", "Demonstrative pronoun functioning independently as an object of पूजयन्ति.", "Medium"),
        (965, 2): ("SN", "Demonstrative pronoun functioning independently as the object of अनुकुर्वन्तः.", "High"),
    }
}


# Validate exact occurrence keys before applying anything

expected_counts = {
    "PPR.0.sg.1.dat.iv.n.n.prx": 6,
    "PPR.0.pl.1.ins.iii.n.n.prx": 5,
    "PPR.mas.pl.3.acc.ii.n.n.dst": 5,
    "PPR.fem.pl.3.0.i.n.n.0": 5,
    "PPR.fem.sg.3.acc.ii.n.n.prx": 5,
}

validation_results = []

for source_tag, mapping in new_ppr_mappings.items():

    unresolved_group = manual_review[
        (manual_review["Source_Tag"] == source_tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(
        zip(
            unresolved_group["sentence_id"],
            unresolved_group["Token_Position"]
        )
    )

    proposed_keys = set(mapping.keys())

    missing_keys = actual_keys - proposed_keys
    extra_keys = proposed_keys - actual_keys

    assert len(mapping) == expected_counts[source_tag], (
        f"Unexpected mapping count for {source_tag}"
    )

    assert len(actual_keys) == expected_counts[source_tag], (
        f"Unexpected unresolved occurrence count for {source_tag}"
    )

    assert not missing_keys, (
        f"Missing occurrence keys for {source_tag}: {missing_keys}"
    )

    assert not extra_keys, (
        f"Invalid or already mapped keys for {source_tag}: {extra_keys}"
    )

    for key, (target_tag, reason, confidence) in mapping.items():
        assert target_tag in ["SN", "SNV"]
        assert isinstance(reason, str) and reason.strip()
        assert confidence in ["High", "Medium", "Low"]

    validation_results.append({
        "Source_Tag": source_tag,
        "Occurrences": len(mapping),
        "SN": sum(v[0] == "SN" for v in mapping.values()),
        "SNV": sum(v[0] == "SNV" for v in mapping.values()),
        "Validated": True
    })


validation_df = pd.DataFrame(validation_results)

print("=" * 85)
print("OCCURRENCE-LEVEL MAPPING VALIDATION")
print("=" * 85)

display(validation_df)

print("\nTotal mappings:", sum(len(m) for m in new_ppr_mappings.values()))
print(
    "Total SN:",
    sum(v[0] == "SN" for m in new_ppr_mappings.values() for v in m.values())
)
print(
    "Total SNV:",
    sum(v[0] == "SNV" for m in new_ppr_mappings.values() for v in m.values())
)

print("\nmanual_review has not been modified.")
print("ALL MAPPING VALIDATIONS PASSED.")

In [ ]:
# Cell 208: Apply validated PPR mappings

applied_count = 0

for source_tag, mapping in new_ppr_mappings.items():

    print(f"\nProcessing: {source_tag}")

    tag_applied = 0

    for (sid, pos), (target_tag, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
            & (manual_review["Target_Tag"].isna())
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch or already mapped: {source_tag}, {(sid, pos)}"
        )

        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        tag_applied += 1

    applied_count += tag_applied
    print("Mappings applied:", tag_applied)

assert applied_count == 26, (
    f"Expected 26 mappings, but applied {applied_count}"
)

print("\nALL FIVE BATCHES APPLIED SUCCESSFULLY")
print("Total mappings applied:", applied_count)

In [ ]:
# Cell 209: Exact-key audit of five PPR batches

audit_results = []

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ].copy()

    batch_rows = []

    for key, (expected_tag, reason, confidence) in mapping.items():

        sid, pos = key

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch: {source_tag}, {key}"
        )

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == expected_tag, (
            f"Incorrect target tag: {source_tag}, {key}"
        )

        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        batch_rows.append(row)

    batch_df = pd.DataFrame(batch_rows)
    labels = batch_df["Target_Tag"]

    batch_keys = set(mapping.keys())

    group_keys = set(
        zip(group["sentence_id"], group["Token_Position"])
    )

    audit_results.append({
        "Source_Tag": source_tag,
        "Batch_Total": len(batch_df),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Unresolved": labels.isna().sum(),
        "Other": (
            labels.notna()
            & ~labels.isin(["SN", "SNV"])
        ).sum(),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"],
            keep=False
        ).sum(),
        "Outside_Batch": len(group_keys - batch_keys)
    })

audit_df = pd.DataFrame(audit_results)

print("=" * 90)
print("EXACT-KEY FIVE-BATCH AUDIT")
print("=" * 90)

display(audit_df)

print("\nTotal occurrences:", audit_df["Batch_Total"].sum())
print("Total SN:", audit_df["SN"].sum())
print("Total SNV:", audit_df["SNV"].sum())
print("Total unresolved:", audit_df["Unresolved"].sum())
print("Total other:", audit_df["Other"].sum())
print("Duplicate rows:", audit_df["Duplicates"].sum())
print("Previously outside batch:", audit_df["Outside_Batch"].sum())

all_passed = (
    audit_df["Batch_Total"].sum() == 26
    and audit_df["Unresolved"].sum() == 0
    and audit_df["Other"].sum() == 0
    and audit_df["Duplicates"].sum() == 0
)

print("\nALL FIVE BATCH AUDITS PASSED:", all_passed)

assert all_passed, "Audit failed. Inspect the results."

In [ ]:
# Cell 210: Calculate remaining PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("=" * 75)
print("UPDATED PPR ANNOTATION PROGRESS")
print("=" * 75)

print("Unresolved PPR occurrences:", len(remaining_ppr))
print("Unresolved source tags:", len(remaining_summary))

print("\nNext five source tags:")

display(
    remaining_summary.head(5)
    .rename("Unresolved_Count")
    .to_frame()
)

In [ ]:
# Cell 211: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.neu.sg.3.acc.ii.n.n.dst",
    "PPR.0.sg.1.0.i.n.n.prx",
    "PPR.neu.sg.3.ins.iii.n.n.prx",
    "PPR.mas.pl.3.gen.vi.n.n.prx",
    "PPR.0.sg.2.dat.iv.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence",
    "Target_Tag"
]

total_inspected = 0

for source_tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == source_tag)
        & (manual_review["Target_Tag"].isna())
    ].copy()

    group = group.sort_values(
        ["sentence_id", "Token_Position"]
    )

    print("\n" + "=" * 100)
    print("SOURCE TAG:", source_tag)
    print("UNRESOLVED OCCURRENCES:", len(group))
    print("=" * 100)

    display(group[context_columns].reset_index(drop=True))

    total_inspected += len(group)

print("\nTOTAL OCCURRENCES INSPECTED:", total_inspected)

assert total_inspected == 21, (
    f"Expected 21 occurrences, found {total_inspected}"
)

In [ ]:
# Cell 212: Prepare exact occurrence-level PPR mappings

new_ppr_mappings = {

    "PPR.neu.sg.3.acc.ii.n.n.dst": {
        (252, 1): (
            "SN",
            "Demonstrative pronoun functioning independently as the object of आवरय.",
            "High"
        ),
        (463, 2): (
            "SNV",
            "Demonstrative pronoun agrees with and qualifies कारणम् in कारणं तद् एव.",
            "Medium"
        ),
        (570, 0): (
            "SNV",
            "अन्यं qualifies कञ्चित् in अन्यं कञ्चित् उद्दिश्य.",
            "Medium"
        ),
        (628, 0): (
            "SNV",
            "अन्यं qualifies कञ्चित् in अन्यं कञ्चित् उद्दिश्य.",
            "Medium"
        ),
        (682, 6): (
            "SN",
            "Demonstrative pronoun functions independently as the object of दृष्टवान्.",
            "High"
        ),
    },

    "PPR.0.sg.1.0.i.n.n.prx": {
        (97, 1): (
            "SN",
            "First-person pronoun functions independently as the subject.",
            "High"
        ),
        (109, 1): (
            "SN",
            "First-person pronoun functions independently as the subject.",
            "High"
        ),
        (312, 10): (
            "SN",
            "First-person pronoun functions independently as the subject in reported speech.",
            "High"
        ),
        (1303, 10): (
            "SN",
            "First-person pronoun functions independently in the quoted expression.",
            "Medium"
        ),
    },

    "PPR.neu.sg.3.ins.iii.n.n.prx": {
        (13, 0): (
            "SN",
            "Instrumental demonstrative pronoun functions independently, indicating means.",
            "High"
        ),
        (1269, 0): (
            "SN",
            "Instrumental demonstrative pronoun functions independently.",
            "High"
        ),
        (1454, 0): (
            "SN",
            "Instrumental demonstrative pronoun functions independently, indicating means.",
            "High"
        ),
        (1457, 0): (
            "SNV",
            "एतेन qualifies हेतुना in एतेन हेतुना.",
            "High"
        ),
    },

    "PPR.mas.pl.3.gen.vi.n.n.prx": {
        (95, 0): (
            "SNV",
            "Genitive pronoun qualifies छायायाम्, expressing possession.",
            "High"
        ),
        (101, 3): (
            "SNV",
            "Genitive pronoun सर्वेषाम् qualifies मुखम्.",
            "High"
        ),
        (324, 9): (
            "SNV",
            "Genitive pronoun qualifies सुखदुःखानि.",
            "High"
        ),
        (748, 6): (
            "SNV",
            "Genitive pronoun एतेषां qualifies स्थिरता.",
            "High"
        ),
    },

    "PPR.0.sg.2.dat.iv.n.n.prx": {
        (510, 8): (
            "SN",
            "Second-person dative pronoun functions independently as recipient.",
            "High"
        ),
        (1221, 5): (
            "SN",
            "Second-person dative pronoun functions independently as recipient.",
            "High"
        ),
        (1358, 2): (
            "SN",
            "Second-person dative pronoun functions independently as recipient.",
            "High"
        ),
        (1375, 1): (
            "SN",
            "Second-person dative pronoun functions independently as recipient.",
            "High"
        ),
    }
}


# Validate exact occurrence keys

expected_counts = {
    "PPR.neu.sg.3.acc.ii.n.n.dst": 5,
    "PPR.0.sg.1.0.i.n.n.prx": 4,
    "PPR.neu.sg.3.ins.iii.n.n.prx": 4,
    "PPR.mas.pl.3.gen.vi.n.n.prx": 4,
    "PPR.0.sg.2.dat.iv.n.n.prx": 4,
}

validation_results = []

for source_tag, mapping in new_ppr_mappings.items():

    unresolved_group = manual_review[
        (manual_review["Source_Tag"] == source_tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(
        unresolved_group["sentence_id"],
        unresolved_group["Token_Position"]
    ))

    proposed_keys = set(mapping.keys())

    assert len(mapping) == expected_counts[source_tag]
    assert len(actual_keys) == expected_counts[source_tag]

    assert actual_keys == proposed_keys, (
        f"Occurrence-key mismatch for {source_tag}. "
        f"Missing: {actual_keys - proposed_keys}; "
        f"Extra: {proposed_keys - actual_keys}"
    )

    for key, (target_tag, reason, confidence) in mapping.items():
        assert target_tag in ["SN", "SNV"]
        assert reason.strip()
        assert confidence in ["High", "Medium", "Low"]

    validation_results.append({
        "Source_Tag": source_tag,
        "Occurrences": len(mapping),
        "SN": sum(v[0] == "SN" for v in mapping.values()),
        "SNV": sum(v[0] == "SNV" for v in mapping.values()),
        "Validated": True
    })


validation_df = pd.DataFrame(validation_results)

print("=" * 85)
print("OCCURRENCE-LEVEL MAPPING VALIDATION")
print("=" * 85)

display(validation_df)

print("\nTotal mappings:",
      sum(len(m) for m in new_ppr_mappings.values()))

print("Total SN:",
      sum(v[0] == "SN" for m in new_ppr_mappings.values()
          for v in m.values()))

print("Total SNV:",
      sum(v[0] == "SNV" for m in new_ppr_mappings.values()
          for v in m.values()))

print("\nmanual_review has not been modified.")
print("ALL MAPPING VALIDATIONS PASSED.")

In [ ]:
# Cell 213: Apply validated PPR mappings

applied_count = 0

for source_tag, mapping in new_ppr_mappings.items():

    print(f"\nProcessing: {source_tag}")
    tag_applied = 0

    for (sid, pos), (target_tag, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
            & (manual_review["Target_Tag"].isna())
        )

        assert mask.sum() == 1, (
            f"Occurrence mismatch or already mapped: {source_tag}, {(sid, pos)}"
        )

        manual_review.loc[mask, "Target_Tag"] = target_tag
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        tag_applied += 1

    applied_count += tag_applied
    print("Mappings applied:", tag_applied)

assert applied_count == 21

print("\nALL FIVE BATCHES APPLIED SUCCESSFULLY")
print("Total mappings applied:", applied_count)

In [ ]:
# Fast audit: Current five PPR batches

results = []

for source_tag, mapping in new_ppr_mappings.items():

    for (sid, pos), (expected, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1, f"Key mismatch: {source_tag}, {(sid, pos)}"

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == expected
        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

    results.append({
        "Source_Tag": source_tag,
        "Total": len(mapping),
        "SN": sum(v[0] == "SN" for v in mapping.values()),
        "SNV": sum(v[0] == "SNV" for v in mapping.values())
    })

audit_df = pd.DataFrame(results)

assert audit_df["Total"].sum() == 21

print("AUDIT PASSED")
display(audit_df)

# Updated remaining workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

summary = remaining_ppr.groupby("Source_Tag").size().sort_values(
    ascending=False
)

print("\nRemaining unresolved PPR occurrences:", len(remaining_ppr))
print("Remaining source tags:", len(summary))
print("\nNext five source tags:")
display(summary.head(5).to_frame("Unresolved_Count"))

In [ ]:
# Cell 215: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.0.sg.1.gen.vi.n.n.prx",
    "PPR.mas.sg.3.acc.ii.n.n.prx",
    "PPR.fem.pl.3.nom.i.n.n.prx",
    "PPR.mas.sg.1.nom.i.n.n.prx",
    "PPR.neu.pl.3.nom.i.n.n.dst"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 17, f"Expected 17, found {total}"

In [ ]:
# Cell 216: Prepare exact occurrence-level mappings

new_ppr_mappings = {

    "PPR.0.sg.1.gen.vi.n.n.prx": {
        (545, 1): ("SNV", "मम् qualifies कारयानेन, expressing possession.", "High"),
        (545, 19): ("SNV", "मम् qualifies अपराधः, expressing possession.", "High"),
        (735, 8): ("SNV", "मे qualifies मार्जारक्रीडारङ्गम्, expressing possession.", "High"),
        (916, 3): ("SN", "मम् functions independently in the elliptical expression मम अपि अस्ति.", "Medium"),
    },

    "PPR.mas.sg.3.acc.ii.n.n.prx": {
        (355, 2): ("SN", "भवन्तं functions independently as the object of वदामि.", "High"),
        (526, 3): ("SN", "भवन्तं functions independently as the object of सूचयामि.", "High"),
        (1008, 3): ("SN", "भवन्तं functions independently as the object of नमस्करोमि.", "High"),
        (1179, 0): ("SNV", "एतं qualifies जलाशयम् in एतं जलाशयम्.", "High"),
    },

    "PPR.fem.pl.3.nom.i.n.n.prx": {
        (1182, 1): ("SN", "भवत्यः functions independently as the subject/addressee.", "High"),
        (1188, 1): ("SN", "भवत्यः functions independently as the subject/addressee.", "High"),
        (1200, 3): ("SN", "भवत्यः functions independently as the addressee.", "High"),
    },

    "PPR.mas.sg.1.nom.i.n.n.prx": {
        (237, 16): ("SN", "अहम् functions independently as the first-person subject.", "High"),
        (434, 3): ("SN", "अहं functions independently as the first-person subject.", "High"),
        (657, 1): ("SN", "अहम् functions independently as the first-person subject.", "High"),
    },

    "PPR.neu.pl.3.nom.i.n.n.dst": {
        (1261, 11): ("SN", "तानि functions independently as the subject of प्रतीयन्ते.", "High"),
        (1271, 2): ("SNV", "अन्यानि qualifies कलाकृत्यानि.", "High"),
    }
}

# The following occurrence is intentionally deferred:
deferred_key = (1316, 12)
deferred_tag = "PPR.neu.pl.3.nom.i.n.n.dst"

expected_counts = {
    "PPR.0.sg.1.gen.vi.n.n.prx": 4,
    "PPR.mas.sg.3.acc.ii.n.n.prx": 4,
    "PPR.fem.pl.3.nom.i.n.n.prx": 3,
    "PPR.mas.sg.1.nom.i.n.n.prx": 3,
    "PPR.neu.pl.3.nom.i.n.n.dst": 3,
}

results = []

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        (manual_review["Source_Tag"] == source_tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = set(mapping.keys())

    assert len(actual_keys) == expected_counts[source_tag]

    expected_deferred = (
        {deferred_key} if source_tag == deferred_tag else set()
    )

    assert actual_keys == proposed_keys | expected_deferred, (
        f"Key mismatch in {source_tag}"
    )

    for key, (target, reason, confidence) in mapping.items():
        assert target in ["SN", "SNV"]
        assert reason.strip()
        assert confidence in ["High", "Medium", "Low"]

    results.append({
        "Source_Tag": source_tag,
        "Total": len(actual_keys),
        "Mapped": len(mapping),
        "SN": sum(v[0] == "SN" for v in mapping.values()),
        "SNV": sum(v[0] == "SNV" for v in mapping.values()),
        "Deferred": len(expected_deferred),
        "Validated": True
    })

validation_df = pd.DataFrame(results)

display(validation_df)

print("\nTotal occurrences:", sum(expected_counts.values()))
print("Mappings prepared:", sum(len(m) for m in new_ppr_mappings.values()))
print("SN:", sum(v[0] == "SN" for m in new_ppr_mappings.values() for v in m.values()))
print("SNV:", sum(v[0] == "SNV" for m in new_ppr_mappings.values() for v in m.values()))
print("Deferred occurrence:", deferred_key)
print("\nmanual_review unchanged.")
print("ALL MAPPING VALIDATIONS PASSED.")

In [ ]:
# Cell 217: Combined apply + audit + progress

applied = 0
audit_results = []

# STEP 1: Revalidate all keys before modifying the DataFrame

for source_tag, mapping in new_ppr_mappings.items():

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
            & (manual_review["Target_Tag"].isna())
        )

        assert mask.sum() == 1, (
            f"Invalid or already mapped occurrence: {source_tag}, {(sid, pos)}"
        )

# Verify deferred occurrence is still unresolved
deferred_mask = (
    (manual_review["Source_Tag"] == deferred_tag)
    & (manual_review["sentence_id"] == deferred_key[0])
    & (manual_review["Token_Position"] == deferred_key[1])
)

assert deferred_mask.sum() == 1
assert manual_review.loc[deferred_mask, "Target_Tag"].isna().all()


# STEP 2: Apply mappings

for source_tag, mapping in new_ppr_mappings.items():

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        manual_review.loc[mask, "Target_Tag"] = target
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        applied += 1


# STEP 3: Exact-key audit

for source_tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        manual_review["Source_Tag"] == source_tag
    ]

    batch_rows = []

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == source_tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == target
        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        batch_rows.append(row)

    labels = pd.DataFrame(batch_rows)["Target_Tag"]

    audit_results.append({
        "Source_Tag": source_tag,
        "Total_Mapped": len(mapping),
        "SN": (labels == "SN").sum(),
        "SNV": (labels == "SNV").sum(),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"],
            keep=False
        ).sum()
    })

audit_df = pd.DataFrame(audit_results)

assert applied == 16
assert audit_df["Total_Mapped"].sum() == 16
assert audit_df["Duplicates"].sum() == 0

assert manual_review.loc[deferred_mask, "Target_Tag"].isna().all()

print("=" * 75)
print("BATCH AUDIT PASSED")
print("=" * 75)

display(audit_df)

print("Mappings applied:", applied)
print("SN:", audit_df["SN"].sum())
print("SNV:", audit_df["SNV"].sum())
print("Deferred occurrence remains unresolved:", deferred_key)


# STEP 4: Updated PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\n" + "=" * 75)
print("UPDATED PPR PROGRESS")
print("=" * 75)

print("Remaining unresolved occurrences:", len(remaining_ppr))
print("Remaining source tags:", len(remaining_summary))

print("\nNext five source tags:")
display(remaining_summary.head(5).to_frame("Unresolved_Count"))

In [ ]:
# Cell 218: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.0.sg.2.ins.iii.n.n.prx",
    "PPR.0.pl.2.nom.i.n.n.prx",
    "PPR.0.du.1.nom.i.n.n.prx",
    "PPR.0.sg.2.loc.vii.n.n.prx",
    "PPR.fem.du.3.gen.vi.n.n.dst"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 11, f"Expected 11, found {total}"

In [ ]:
# Cell 219: Combined PPR processing

new_ppr_mappings = {

    "PPR.0.sg.2.ins.iii.n.n.prx": {
        (814, 2): ("SN", "त्वया functions independently as the agent of न कृतम्.", "High"),
        (1167, 11): ("SN", "त्वया functions independently as the agent of निःसारिताः.", "High"),
    },

    "PPR.0.pl.2.nom.i.n.n.prx": {
        (864, 9): ("SN", "यूयम् functions independently as the second-person plural subject.", "High"),
        (932, 9): ("SN", "यूयम् functions independently as the second-person plural subject.", "High"),
    },

    "PPR.0.du.1.nom.i.n.n.prx": {
        (178, 0): ("SN", "आवां functions independently as the dual first-person subject.", "High"),
        (402, 5): ("SN", "आवां functions independently as the dual first-person subject.", "High"),
    },

    "PPR.0.sg.2.loc.vii.n.n.prx": {
        (423, 0): ("SN", "त्वयि functions independently in a locative absolute construction.", "High"),
        (812, 3): ("SN", "त्वयि functions independently as the complement of विश्र्वासः.", "High"),
    },

    "PPR.fem.du.3.gen.vi.n.n.dst": {
        (1066, 0): ("SNV", "तयोः qualifies पार्श्वे, expressing spatial relation.", "High"),
        (1067, 1): ("SNV", "तयोः qualifies विवादस्य, identifying the two parties involved.", "Medium"),
    }
}

deferred_tag = "PPR.0.sg.2.ins.iii.n.n.prx"
deferred_key = (408, 14)

expected_counts = {
    "PPR.0.sg.2.ins.iii.n.n.prx": 3,
    "PPR.0.pl.2.nom.i.n.n.prx": 2,
    "PPR.0.du.1.nom.i.n.n.prx": 2,
    "PPR.0.sg.2.loc.vii.n.n.prx": 2,
    "PPR.fem.du.3.gen.vi.n.n.dst": 2
}

# STEP 1: Validate all keys before applying

for tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = set(mapping.keys())

    deferred_keys = {deferred_key} if tag == deferred_tag else set()

    assert len(actual_keys) == expected_counts[tag]
    assert actual_keys == proposed_keys | deferred_keys, (
        f"Key mismatch: {tag}"
    )

    for target, reason, confidence in mapping.values():
        assert target in ["SN", "SNV"]
        assert reason.strip()
        assert confidence in ["High", "Medium", "Low"]

# STEP 2: Apply mappings

applied = 0

for tag, mapping in new_ppr_mappings.items():

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
            & (manual_review["Target_Tag"].isna())
        )

        assert mask.sum() == 1

        manual_review.loc[mask, "Target_Tag"] = target
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        applied += 1

# STEP 3: Audit exact keys

audit_results = []

for tag, mapping in new_ppr_mappings.items():

    labels = []

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == target
        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        labels.append(target)

    group = manual_review[manual_review["Source_Tag"] == tag]

    audit_results.append({
        "Source_Tag": tag,
        "Mapped": len(mapping),
        "SN": labels.count("SN"),
        "SNV": labels.count("SNV"),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"], keep=False
        ).sum()
    })

audit_df = pd.DataFrame(audit_results)

assert applied == 10
assert audit_df["Mapped"].sum() == 10
assert audit_df["Duplicates"].sum() == 0

# Deferred occurrence must remain unresolved
deferred_mask = (
    (manual_review["Source_Tag"] == deferred_tag)
    & (manual_review["sentence_id"] == deferred_key[0])
    & (manual_review["Token_Position"] == deferred_key[1])
)

assert deferred_mask.sum() == 1
assert manual_review.loc[deferred_mask, "Target_Tag"].isna().all()

print("BATCH AUDIT PASSED")
display(audit_df)

print("\nMappings applied:", applied)
print("SN:", audit_df["SN"].sum())
print("SNV:", audit_df["SNV"].sum())
print("Deferred:", deferred_key)

# STEP 4: Remaining PPR workload

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nRemaining unresolved PPR occurrences:", len(remaining_ppr))
print("Remaining source tags:", len(remaining_summary))

print("\nNext five source tags:")
display(remaining_summary.head(5).to_frame("Unresolved_Count"))

# STEP 5: Display full context for deferred occurrence

print("\nDEFERRED OCCURRENCE — COMPLETE SENTENCE")
print(
    repr(
        manual_review.loc[deferred_mask, "Full_Sentence"].iloc[0]
    )
)

In [ ]:
# Cell 220: Resolve deferred occurrence + inspect next batch

# STEP 1: Resolve previously deferred occurrence

sid, pos = 408, 14
source_tag = "PPR.0.sg.2.ins.iii.n.n.prx"

mask = (
    (manual_review["Source_Tag"] == source_tag)
    & (manual_review["sentence_id"] == sid)
    & (manual_review["Token_Position"] == pos)
    & (manual_review["Target_Tag"].isna())
)

assert mask.sum() == 1, "Deferred occurrence mismatch"

manual_review.loc[mask, "Target_Tag"] = "SN"
manual_review.loc[mask, "Mapping_Reason"] = (
    "त्वया functions independently as the instrumental agent "
    "in the elliptical construction त्वया अपि तथा."
)
manual_review.loc[mask, "Mapping_Confidence"] = "Medium"

assert manual_review.loc[mask, "Target_Tag"].iloc[0] == "SN"

print("Deferred occurrence resolved and verified:", (sid, pos))


# STEP 2: Inspect next five source tags

next_ppr_tags = [
    "PPR.fem.du.3.gen.vi.n.n.prx",
    "PPR.0.sg.2.acc.ii.n.n.prx",
    "PPR.0.pl.1.dat.iv.n.n.prx",
    "PPR.fem.sg.3.loc.vii.n.n.prx",
    "PPR.mas.du.3.nom.i.n.n.dst"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 10, f"Expected 10, found {total}"

In [ ]:
# Cell 221: Combined PPR processing

new_ppr_mappings = {

    "PPR.fem.du.3.gen.vi.n.n.prx": {
        (1078, 2): (
            "SNV",
            "एतयोः expresses the genitive relationship with विवादेन.",
            "High"
        ),
        (1428, 2): (
            "SN",
            "भवतीभ्यां functions independently as the agent in the passive construction.",
            "Medium"
        ),
    },

    "PPR.0.sg.2.acc.ii.n.n.prx": {
        (1209, 6): (
            "SN",
            "त्वां functions independently as the object of भस्मीकरोमि.",
            "High"
        ),
        (1370, 8): (
            "SN",
            "त्वां functions independently as the object of लङ्घयति.",
            "High"
        ),
    },

    "PPR.0.pl.1.dat.iv.n.n.prx": {
        (324, 14): (
            "SN",
            "अस्मभ्यम् functions independently as a dative pronoun.",
            "Medium"
        ),
        (1305, 5): (
            "SN",
            "अस्मभ्यं functions independently as a dative pronoun.",
            "High"
        ),
    },

    "PPR.fem.sg.3.loc.vii.n.n.prx": {
        (8, 3): (
            "SN",
            "अस्यां functions independently as a locative pronoun referring to the river.",
            "High"
        ),
        (898, 3): (
            "SN",
            "अस्यां functions independently as a locative pronoun referring to the river.",
            "High"
        ),
    },

    "PPR.mas.du.3.nom.i.n.n.dst": {
        (180, 1): (
            "SN",
            "तौ functions independently as the dual subject of खेलतः.",
            "High"
        ),
        (723, 1): (
            "SNV",
            "तौ qualifies उभौ in तौ उभौ अपि.",
            "High"
        ),
    }
}

expected_counts = {
    "PPR.fem.du.3.gen.vi.n.n.prx": 2,
    "PPR.0.sg.2.acc.ii.n.n.prx": 2,
    "PPR.0.pl.1.dat.iv.n.n.prx": 2,
    "PPR.fem.sg.3.loc.vii.n.n.prx": 2,
    "PPR.mas.du.3.nom.i.n.n.dst": 2
}

# STEP 1: Validate all occurrence keys before modification

for tag, mapping in new_ppr_mappings.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = set(mapping.keys())

    assert len(actual_keys) == expected_counts[tag]
    assert actual_keys == proposed_keys, f"Key mismatch: {tag}"

    for target, reason, confidence in mapping.values():
        assert target in ["SN", "SNV"]
        assert reason.strip()
        assert confidence in ["High", "Medium", "Low"]


# STEP 2: Apply mappings

applied = 0

for tag, mapping in new_ppr_mappings.items():

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
            & (manual_review["Target_Tag"].isna())
        )

        assert mask.sum() == 1

        manual_review.loc[mask, "Target_Tag"] = target
        manual_review.loc[mask, "Mapping_Reason"] = reason
        manual_review.loc[mask, "Mapping_Confidence"] = confidence

        applied += 1


# STEP 3: Exact-key audit

audit_results = []

for tag, mapping in new_ppr_mappings.items():

    labels = []

    for (sid, pos), (target, reason, confidence) in mapping.items():

        mask = (
            (manual_review["Source_Tag"] == tag)
            & (manual_review["sentence_id"] == sid)
            & (manual_review["Token_Position"] == pos)
        )

        assert mask.sum() == 1

        row = manual_review.loc[mask].iloc[0]

        assert row["Target_Tag"] == target
        assert row["Mapping_Reason"] == reason
        assert row["Mapping_Confidence"] == confidence

        labels.append(target)

    group = manual_review[manual_review["Source_Tag"] == tag]

    audit_results.append({
        "Source_Tag": tag,
        "Mapped": len(mapping),
        "SN": labels.count("SN"),
        "SNV": labels.count("SNV"),
        "Duplicates": group.duplicated(
            ["sentence_id", "Token_Position"],
            keep=False
        ).sum()
    })

audit_df = pd.DataFrame(audit_results)

assert applied == 10
assert audit_df["Mapped"].sum() == 10
assert audit_df["Duplicates"].sum() == 0

print("BATCH AUDIT PASSED")
display(audit_df)

print("\nMappings applied:", applied)
print("SN:", audit_df["SN"].sum())
print("SNV:", audit_df["SNV"].sum())


# STEP 4: Updated PPR progress

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_summary = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nRemaining unresolved PPR occurrences:", len(remaining_ppr))
print("Remaining source tags:", len(remaining_summary))

print("\nNext five source tags:")
display(remaining_summary.head(5).to_frame("Unresolved_Count"))

In [ ]:
# Cell 222: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.fem.sg.3.acc.ii.n.n.dst",
    "PPR.fem.sg.3.ins.iii.n.n.dst",
    "PPR.mas.pl.1.gen.vi.n.n.prx",
    "PPR.mas.sg.1.acc.ii.n.n.prx",
    "PPR.mas.pl.3.nom.i.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 10, f"Expected 10, found {total}"

In [ ]:
# Cell 223: Map and audit the next 10 PPR occurrences

new_ppr_mappings = {
    # तां modifies the noun सूचीं
    (660, 1): (
        "SNV",
        "तां directly modifies the feminine noun सूचीं; it functions as a demonstrative determiner.",
        "High"
    ),
    (660, 8): (
        "SNV",
        "तां directly modifies the feminine noun सूचीं in the same sentence.",
        "High"
    ),

    # तया refers independently to the preceding event/condition
    (1458, 3): (
        "SN",
        "तया independently refers to अतिवृष्टिः as the instrumental cause of जलाप्लावः.",
        "Medium"
    ),
    (1459, 3): (
        "SN",
        "तया independently refers to अनावृष्टिः as the instrumental cause of दुर्भिक्षम्.",
        "Medium"
    ),

    # अस्माकं functions as an independent genitive pronoun
    (568, 9): (
        "SN",
        "अस्माकं functions independently in the genitive construction; no directly modified noun is evident in the supplied context.",
        "Medium"
    ),
    (1149, 2): (
        "SN",
        "अस्माकं is an independent genitive pronoun in कः अभ्युपायः अस्माकं जीवनाय?",
        "High"
    ),

    # First-person singular pronouns
    (626, 2): (
        "SN",
        "मां is an independent first-person pronoun functioning as the object of भावयति.",
        "High"
    ),
    (1415, 1): (
        "SN",
        "मे is an independent first-person pronoun in the construction रोचते मे एषः भद्रमयूरः.",
        "Medium"
    ),

    # Third-person plural pronouns
    (114, 2): (
        "SN",
        "सर्वे functions substantivally as an independent pronoun; it does not modify a noun.",
        "High"
    ),
    (327, 0): (
        "SN",
        "भवन्तः is an independent honorific second-person pronoun functioning as the subject.",
        "High"
    )
}

expected_counts = {
    "PPR.fem.sg.3.acc.ii.n.n.dst": 2,
    "PPR.fem.sg.3.ins.iii.n.n.dst": 2,
    "PPR.mas.pl.1.gen.vi.n.n.prx": 2,
    "PPR.mas.sg.1.acc.ii.n.n.prx": 2,
    "PPR.mas.pl.3.nom.i.n.n.prx": 2
}

# STEP 1: Validate all keys before modifying the DataFrame
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))

    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Occurrence-key mismatch for {tag}. "
        f"Missing: {actual_keys - proposed_keys}; "
        f"Extra: {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 10
assert all(
    target in {"SN", "SNV"} and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 2: Apply mappings using exact occurrence keys
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected exactly one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 3: Audit every applied occurrence
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Check duplicate occurrence keys within the five source-tag groups
for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]
    assert not group.duplicated(["sentence_id", "Token_Position"]).any(), (
        f"Duplicate occurrence keys found in {tag}"
    )

print("BATCH AUDIT PASSED")

# STEP 4: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 5: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 224: Inspect next five PPR source-tag groups

next_ppr_tags = [
    "PPR.mas.sg.3.dat.iv.n.n.prx",
    "PPR.neu.pl.3.loc.vii.n.n.prx",
    "PPR.neu.pl.3.nom.i.n.n.0",
    "PPR.mas.sg.3.dat.iv.n.n.dst",
    "PPR.0.du.1.abl.v.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 9, f"Expected 9, found {total}"

In [ ]:
# Cell 225: Map and audit the next 9 PPR occurrences

new_ppr_mappings = {
    (1011, 3): (
        "SN",
        "भवते is an independent second-person pronoun in the dative case, functioning as the recipient of प्रदास्ये.",
        "High"
    ),
    (1386, 11): (
        "SN",
        "अस्मै is an independent demonstrative pronoun in the dative case; it does not modify a noun.",
        "Medium"
    ),
    (128, 1): (
        "SNV",
        "सर्वेषु directly modifies the locative plural noun कार्येषु.",
        "High"
    ),
    (135, 0): (
        "SNV",
        "सर्वेषु directly modifies the locative plural noun पत्रेषु.",
        "High"
    ),
    (1316, 4): (
        "SN",
        "अन्यानि is used substantivally in the enumeration भोजनं गृहं वस्त्रं धनम् अन्यानि च; no modified noun is expressed.",
        "Medium"
    ),
    (1451, 6): (
        "SNV",
        "अन्यानि directly modifies the neuter plural noun उपकरणानि.",
        "High"
    ),
    (659, 5): (
        "SN",
        "तस्मै is an independent demonstrative pronoun in the dative case, referring to the recipient of the list.",
        "High"
    ),
    (1116, 4): (
        "SN",
        "तस्मै is an independent demonstrative pronoun in the dative case, referring to the person receiving the cane blows.",
        "High"
    ),
    (403, 2): (
        "SN",
        "आवाभ्यां is an independent first-person dual pronoun referring to the two participants.",
        "Medium"
    )
}

expected_counts = {
    "PPR.mas.sg.3.dat.iv.n.n.prx": 2,
    "PPR.neu.pl.3.loc.vii.n.n.prx": 2,
    "PPR.neu.pl.3.nom.i.n.n.0": 2,
    "PPR.mas.sg.3.dat.iv.n.n.dst": 2,
    "PPR.0.du.1.abl.v.n.n.prx": 1
}

# STEP 1: Validate exact occurrence keys before modification
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 9

assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 2: Apply mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 3: Audit applied mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]
    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 4: Summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 5: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 226: Inspect next five PPR occurrences

next_ppr_tags = [
    "PPR.0.du.1.gen.vi.n.n.prx",
    "PPR.0.du.3.ins.iii.n.n.prx",
    "PPR.0.sg.1.0.iii.n.n.prx",
    "PPR.0.sg.1.nom.i.n.n.prx.n",
    "PPR.0.sg.2.gen.vi.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 5, f"Expected 5, found {total}"

In [ ]:
# Cell 227: Map and audit the next 5 PPR occurrences

new_ppr_mappings = {
    (186, 5): (
        "SNV",
        "आवयोः is a genitive dual pronoun directly associated with the noun गुणाः.",
        "High"
    ),
    (1452, 3): (
        "SN",
        "अस्माभिः is an independent first-person dual instrumental pronoun functioning as the agent.",
        "High"
    ),
    (306, 2): (
        "SN",
        "मया is an independent first-person singular instrumental pronoun used with सह.",
        "High"
    ),
    (410, 2): (
        "SN",
        "अहम् is an independent first-person singular nominative pronoun functioning as the subject.",
        "High"
    ),
    (1402, 2): (
        "SN",
        "ते is an independent second-person singular pronoun in the genitive/dative form; it does not directly modify a noun in the supplied context.",
        "Medium"
    )
}

expected_counts = {
    "PPR.0.du.1.gen.vi.n.n.prx": 1,
    "PPR.0.du.3.ins.iii.n.n.prx": 1,
    "PPR.0.sg.1.0.iii.n.n.prx": 1,
    "PPR.0.sg.1.nom.i.n.n.prx.n": 1,
    "PPR.0.sg.2.gen.vi.n.n.prx": 1
}

# STEP 1: Validate exact occurrence keys
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 5

# STEP 2: Validate mapping values
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 3: Apply mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 4: Audit each applied occurrence
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]
    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 5: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 6: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 228: Inspect next five PPR occurrences

next_ppr_tags = [
    "PPR.0.sg.3.gen.vi.n.n.prx",
    "PPR.fem.du.3.loc.vii.n.n.dst",
    "PPR.fem.du.3.nom.i.n.n.dst",
    "PPR.fem.pl.3.gen.vi.n.n.prx",
    "PPR.fem.pl.3.nom.i.n.n.dst"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 5, f"Expected 5, found {total}"

In [ ]:
# Cell 229: Map and audit the next 5 PPR occurrences

new_ppr_mappings = {
    (720, 6): (
        "SNV",
        "तव directly modifies the noun वचनम् in the possessive construction तव वचनम्.",
        "High"
    ),
    (1064, 0): (
        "SN",
        "तयोः is used independently in the partitive construction तयोः एका, meaning one of the two.",
        "High"
    ),
    (1065, 0): (
        "SNV",
        "ते directly determines the noun phrase द्वे भगिन्यौ.",
        "High"
    ),
    (900, 8): (
        "SNV",
        "अस्याः is a genitive pronoun associated with the noun पावनजलेन.",
        "Medium"
    ),
    (980, 0): (
        "SN",
        "ताः is an independent feminine plural pronoun functioning as the subject of भ्रमन्ति.",
        "High"
    )
}

expected_counts = {
    "PPR.0.sg.3.gen.vi.n.n.prx": 1,
    "PPR.fem.du.3.loc.vii.n.n.dst": 1,
    "PPR.fem.du.3.nom.i.n.n.dst": 1,
    "PPR.fem.pl.3.gen.vi.n.n.prx": 1,
    "PPR.fem.pl.3.nom.i.n.n.dst": 1
}

# STEP 1: Validate exact occurrence keys
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 5

# STEP 2: Validate mapping values
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 3: Apply exact occurrence mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 4: Audit applied mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Duplicate-key audit
for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]

    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 5: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 6: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 230: Inspect next five PPR occurrences

next_ppr_tags = [
    "PPR.fem.sg.1.nom.i.n.n.prx",
    "PPR.fem.sg.3.loc.vii.n.n.dst",
    "PPR.mas.du.2.gen.vi.n.n.dst",
    "PPR.mas.du.3.gen.vi.n.n.dst",
    "PPR.mas.pl.3.0.i.n.n.0"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 5, f"Expected 5, found {total}"

In [ ]:
# Cell 231: Map and audit the next 5 PPR occurrences

new_ppr_mappings = {
    (445, 2): (
        "SN",
        "एषा is an independent feminine demonstrative pronoun functioning as the subject.",
        "High"
    ),
    (25, 1): (
        "SNV",
        "तस्यां directly modifies the locative feminine noun रात्रौ.",
        "High"
    ),
    (703, 4): (
        "SNV",
        "तयोः is a genitive dual pronoun expressing possession or association with सम्भाषणम्.",
        "High"
    ),
    (438, 2): (
        "SNV",
        "तयोः is a genitive dual pronoun expressing possession of वर्षम् in the construction अन्तिमं वर्षं तयोः.",
        "High"
    ),
    (109, 7): (
        "SNV",
        "सर्वे directly qualifies the plural noun जनाः.",
        "High"
    )
}

expected_counts = {
    "PPR.fem.sg.1.nom.i.n.n.prx": 1,
    "PPR.fem.sg.3.loc.vii.n.n.dst": 1,
    "PPR.mas.du.2.gen.vi.n.n.dst": 1,
    "PPR.mas.du.3.gen.vi.n.n.dst": 1,
    "PPR.mas.pl.3.0.i.n.n.0": 1
}

# STEP 1: Validate exact occurrence keys
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 5

# STEP 2: Validate mapping values
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 3: Apply mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 4: Audit applied mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Duplicate-key audit
for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]

    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 5: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 6: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 232: Inspect next five PPR occurrences

next_ppr_tags = [
    "PPR.mas.pl.3.0.i.n.n.prx",
    "PPR.mas.pl.3.abl.v.n.n.dst",
    "PPR.mas.pl.3.acc.ii.n.n.prx",
    "PPR.mas.pl.3.dat.iv.n.n.dst",
    "PPR.mas.pl.3.dat.iv.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 5, f"Expected 5, found {total}"

In [ ]:
# Cell 233: Map and audit the next 5 PPR occurrences

new_ppr_mappings = {
    (1167, 7): (
        "SNV",
        "एते directly modifies the masculine plural noun रक्षकाः.",
        "High"
    ),
    (370, 4): (
        "SN",
        "तेभ्यः is an independent demonstrative pronoun referring to those capable of paying the bride price.",
        "High"
    ),
    (729, 7): (
        "SN",
        "तान् is an independent masculine plural demonstrative pronoun functioning as the object.",
        "High"
    ),
    (318, 6): (
        "SN",
        "तेभ्यः is an independent dative plural pronoun referring to the travellers mentioned in the sentence.",
        "Medium"
    ),
    (98, 3): (
        "SN",
        "सर्वेभ्यः is used substantivally to mean everyone; it does not directly modify an expressed noun.",
        "High"
    )
}

expected_counts = {
    "PPR.mas.pl.3.0.i.n.n.prx": 1,
    "PPR.mas.pl.3.abl.v.n.n.dst": 1,
    "PPR.mas.pl.3.acc.ii.n.n.prx": 1,
    "PPR.mas.pl.3.dat.iv.n.n.dst": 1,
    "PPR.mas.pl.3.dat.iv.n.n.prx": 1
}

# STEP 1: Validate exact occurrence keys
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 5

# STEP 2: Validate mapping values
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 3: Apply exact occurrence mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 4: Audit applied mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Duplicate-key audit
for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]

    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 5: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 6: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 234: Inspect next five PPR occurrences

next_ppr_tags = [
    "PPR.mas.pl.3.gen.vi.n.n.0",
    "PPR.mas.pl.3.ins.iii.n.n.prx",
    "PPR.mas.sg.0.gen.vi.n.n.dst",
    "PPR.mas.sg.0.nom.i.n.n.prx",
    "PPR.mas.sg.2.dat.iv.n.n.prx"
]

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

total = 0

for tag in next_ppr_tags:

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ].sort_values(["sentence_id", "Token_Position"])

    print(f"\n{'='*90}")
    print(f"SOURCE TAG: {tag} | OCCURRENCES: {len(group)}")
    print("="*90)

    display(group[context_columns].reset_index(drop=True))

    total += len(group)

print("\nTOTAL OCCURRENCES:", total)

assert total == 5, f"Expected 5, found {total}"

In [ ]:
# Cell 235: Map and audit the next 5 PPR occurrences

new_ppr_mappings = {
    (311, 0): (
        "SNV",
        "सर्वेषां is a genitive plural pronoun expressing possession or association with कथनम्.",
        "High"
    ),
    (1345, 6): (
        "SN",
        "भवद्भिः is an independent honorific second-person plural instrumental pronoun.",
        "High"
    ),
    (1274, 2): (
        "SNV",
        "तस्य directly modifies the masculine noun पिता in the possessive construction तस्य पिता.",
        "High"
    ),
    (1447, 7): (
        "SN",
        "एषः is an independent masculine demonstrative pronoun referring to आर्यपुत्रः.",
        "High"
    ),
    (1016, 2): (
        "SN",
        "ते is an independent second-person singular dative pronoun functioning as the recipient of ददामि.",
        "High"
    )
}

expected_counts = {
    "PPR.mas.pl.3.gen.vi.n.n.0": 1,
    "PPR.mas.pl.3.ins.iii.n.n.prx": 1,
    "PPR.mas.sg.0.gen.vi.n.n.dst": 1,
    "PPR.mas.sg.0.nom.i.n.n.prx": 1,
    "PPR.mas.sg.2.dat.iv.n.n.prx": 1
}

# STEP 1: Validate exact occurrence keys
for tag, expected_count in expected_counts.items():

    group = manual_review[
        (manual_review["Source_Tag"] == tag)
        & (manual_review["Target_Tag"].isna())
    ]

    actual_keys = set(zip(group["sentence_id"], group["Token_Position"]))
    proposed_keys = {
        key for key in new_ppr_mappings
        if key in actual_keys
    }

    assert len(group) == expected_count, (
        f"{tag}: expected {expected_count}, found {len(group)}"
    )

    assert actual_keys == proposed_keys, (
        f"Key mismatch in {tag}: "
        f"Missing {actual_keys - proposed_keys}; "
        f"Extra {proposed_keys - actual_keys}"
    )

assert len(new_ppr_mappings) == 5

# STEP 2: Validate mapping values
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

# STEP 3: Apply exact occurrence mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1, (
        f"Expected one unresolved row for {(sentence_id, token_position)}"
    )

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# STEP 4: Audit applied mappings
for (sentence_id, token_position), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sentence_id)
        & (manual_review["Token_Position"] == token_position)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Duplicate-key audit
for tag in expected_counts:
    group = manual_review[manual_review["Source_Tag"] == tag]

    assert not group.duplicated(
        ["sentence_id", "Token_Position"]
    ).any(), f"Duplicate keys in {tag}"

print("BATCH AUDIT PASSED")

# STEP 5: Batch summary
applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"]) in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE:")
display(applied_rows["Mapping_Confidence"].value_counts())

# STEP 6: Remaining PPR workload
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

remaining_by_tag = (
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
)

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", len(remaining_by_tag))

print("\nNEXT FIVE SOURCE-TAG GROUPS:")
display(remaining_by_tag.head(5).rename("Unresolved_Count").to_frame())

In [ ]:
# Cell 236: Collect ALL unresolved PPR occurrences

import pandas as pd

# Collect all unresolved PPR records
remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
].copy()

# Sort by source tag and occurrence
remaining_ppr = remaining_ppr.sort_values(
    ["Source_Tag", "sentence_id", "Token_Position"]
).reset_index(drop=True)

# Required review columns
review_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Source_Tag",
    "Previous_Words",
    "Next_Words",
    "Full_Sentence"
]

print("=" * 100)
print("COMPLETE UNRESOLVED PPR REVIEW")
print("=" * 100)

print("\nTotal unresolved occurrences:", len(remaining_ppr))
print("Unique source tags:", remaining_ppr["Source_Tag"].nunique())

# Check duplicate occurrence keys
duplicate_keys = remaining_ppr[
    remaining_ppr.duplicated(
        ["sentence_id", "Token_Position"],
        keep=False
    )
]

print("Duplicate occurrence keys:", len(duplicate_keys))

# Display all occurrences
display(remaining_ppr[review_columns])

# Source-tag summary
print("\nSOURCE TAG SUMMARY:")
display(
    remaining_ppr.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
    .rename("Unresolved_Count")
    .to_frame()
)

# Export review data for backup
review_path = "" + str(RESULTS_DIR) + "/PPR_remaining_review.csv"

remaining_ppr[review_columns].to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nReview file saved:", review_path)
print("\nINSPECTION COMPLETED")

In [ ]:
# Cell 237: Complete PPR mapping for all 18 unresolved occurrences

# Format:
# (sentence_id, Token_Position): (Target_Tag, Reason, Confidence)

new_ppr_mappings = {

    (1208, 0): (
        "SNV",
        "ते expresses possession and is associated with the noun शक्तिः.",
        "Medium"
    ),

    (811, 9): (
        "SN",
        "त्वयि is an independent second-person pronoun in the locative construction भक्तिः त्वयि.",
        "High"
    ),

    (1415, 2): (
        "SNV",
        "एषः directly identifies the noun भद्रमयूरः.",
        "High"
    ),

    (1042, 0): (
        "SN",
        "तस्मात् is an independent demonstrative pronoun in the ablative case.",
        "High"
    ),

    (1291, 0): (
        "SNV",
        "एतस्मात् directly modifies अनुभवात् in the construction एतस्मात् अनुभवात् अनन्तरम्.",
        "High"
    ),

    (476, 0): (
        "SN",
        "तस्य functions as an independent genitive pronoun with अन्तः, meaning inside him.",
        "Medium"
    ),

    (348, 8): (
        "SN",
        "तस्मिन् is an independent demonstrative pronoun in the locative case.",
        "High"
    ),

    (375, 22): (
        "SN",
        "एताभ्याम् is an independent demonstrative dual pronoun in the instrumental/dative case.",
        "Medium"
    ),

    (1313, 30): (
        "SNV",
        "अन्यानि qualifies the neuter plural noun कर्माणि.",
        "High"
    ),

    (399, 5): (
        "SN",
        "अहम् is an independent first-person singular pronoun functioning as the subject.",
        "High"
    ),

    (656, 12): (
        "SNV",
        "अन्या directly modifies the feminine noun सेवा in the construction अन्या सेवा.",
        "High"
    ),

    (309, 0): (
        "SN",
        "एतस्मात् is an independent demonstrative pronoun in the ablative case.",
        "High"
    ),

    (852, 1): (
        "SNV",
        "तस्य expresses possession or association with रक्षणस्य.",
        "High"
    ),

    (401, 5): (
        "SN",
        "तेन independently refers to the previously mentioned rail vehicle.",
        "High"
    ),

    (665, 2): (
        "SNV",
        "तस्मिन् directly modifies the locative noun क्षणे.",
        "High"
    ),

    (279, 0): (
        "SNV",
        "एतस्मिन् directly modifies the locative noun जगति.",
        "High"
    ),

    (690, 2): (
        "SNV",
        "एकस्मिन् directly modifies the locative noun घटे.",
        "High"
    )
}

# One historical occurrence is intentionally deferred.
deferred_key = (1316, 12)
deferred_word = "तानि"
deferred_tag = "PPR.neu.pl.3.nom.i.n.n.dst"

expected_counts = {
    "PPR.mas.sg.2.gen.vi.n.n.prx": 1,
    "PPR.mas.sg.2.loc.vii.n.n.prx": 1,
    "PPR.mas.sg.3.0.i.n.n.prx": 1,
    "PPR.mas.sg.3.abl.v.n.n.dst": 1,
    "PPR.mas.sg.3.abl.v.n.n.prx": 1,
    "PPR.mas.sg.3.gen.vi.n.n.dst": 1,
    "PPR.mas.sg.3.loc.vii.n.n.dst": 1,
    "PPR.neu.du.3.dat.iv.n.n.prx": 1,
    "PPR.neu.pl.3.0.i.n.n.0": 1,
    "PPR.neu.pl.3.nom.i.n.n.dst": 1,
    "PPR.neu.sg.1.nom.i.n.n.prx": 1,
    "PPR.neu.sg.3.0.i.n.n.dst": 1,
    "PPR.neu.sg.3.abl.v.n.n.prx": 1,
    "PPR.neu.sg.3.gen.vi.n.n.dst": 1,
    "PPR.neu.sg.3.ins.iii.n.n.dst": 1,
    "PPR.neu.sg.3.loc.vii.n.n.dst": 1,
    "PPR.neu.sg.3.loc.vii.n.n.prx": 1,
    "PPR.neu.sg.3.loc.viii.n.n.prx": 1
}

# Expected word and source-tag checks
expected_words = {
    (1208, 0): "ते",
    (811, 9): "त्वयि",
    (1415, 2): "एषः",
    (1042, 0): "तस्मात्",
    (1291, 0): "एतस्मात्",
    (476, 0): "तस्य",
    (348, 8): "तस्मिन्",
    (375, 22): "एताभ्याम्",
    (1313, 30): "अन्यानि",
    (399, 5): "अहम्",
    (656, 12): "अन्या",
    (309, 0): "एतस्मात्",
    (852, 1): "तस्य",
    (401, 5): "तेन",
    (665, 2): "तस्मिन्",
    (279, 0): "एतस्मिन्",
    (690, 2): "एकस्मिन्",
    deferred_key: deferred_word
}

expected_tag_by_key = {
    (1208, 0): "PPR.mas.sg.2.gen.vi.n.n.prx",
    (811, 9): "PPR.mas.sg.2.loc.vii.n.n.prx",
    (1415, 2): "PPR.mas.sg.3.0.i.n.n.prx",
    (1042, 0): "PPR.mas.sg.3.abl.v.n.n.dst",
    (1291, 0): "PPR.mas.sg.3.abl.v.n.n.prx",
    (476, 0): "PPR.mas.sg.3.gen.vi.n.n.dst",
    (348, 8): "PPR.mas.sg.3.loc.vii.n.n.dst",
    (375, 22): "PPR.neu.du.3.dat.iv.n.n.prx",
    (1313, 30): "PPR.neu.pl.3.0.i.n.n.0",
    deferred_key: deferred_tag,
    (399, 5): "PPR.neu.sg.1.nom.i.n.n.prx",
    (656, 12): "PPR.neu.sg.3.0.i.n.n.dst",
    (309, 0): "PPR.neu.sg.3.abl.v.n.n.prx",
    (852, 1): "PPR.neu.sg.3.gen.vi.n.n.dst",
    (401, 5): "PPR.neu.sg.3.ins.iii.n.n.dst",
    (665, 2): "PPR.neu.sg.3.loc.vii.n.n.dst",
    (279, 0): "PPR.neu.sg.3.loc.vii.n.n.prx",
    (690, 2): "PPR.neu.sg.3.loc.viii.n.n.prx"
}

# ============================================================
# STEP 1: PRE-VALIDATION — DO NOT MODIFY DATA YET
# ============================================================

assert len(new_ppr_mappings) == 17
assert len(expected_counts) == 18

all_expected_keys = set(new_ppr_mappings) | {deferred_key}

assert len(all_expected_keys) == 18

for tag, expected_count in expected_counts.items():

    group = remaining_ppr[
        remaining_ppr["Source_Tag"] == tag
    ]

    assert len(group) == expected_count, (
        f"Count mismatch for {tag}: {len(group)}"
    )

actual_keys = set(
    zip(remaining_ppr["sentence_id"], remaining_ppr["Token_Position"])
)

assert actual_keys == all_expected_keys, (
    f"Occurrence mismatch. Missing: {all_expected_keys - actual_keys}; "
    f"Unexpected: {actual_keys - all_expected_keys}"
)

# Validate every word and source tag against the inspected output
for key in all_expected_keys:

    sid, pos = key

    row = manual_review[
        (manual_review["sentence_id"] == sid)
        & (manual_review["Token_Position"] == pos)
        & (manual_review["Target_Tag"].isna())
    ]

    assert len(row) == 1, f"Invalid occurrence key: {key}"

    assert row.iloc[0]["Word"] == expected_words[key], (
        f"Word mismatch at {key}: "
        f"{row.iloc[0]['Word']} != {expected_words[key]}"
    )

    assert row.iloc[0]["Source_Tag"] == expected_tag_by_key[key], (
        f"Source-tag mismatch at {key}"
    )

# Validate all proposed labels
assert all(
    target in {"SN", "SNV"}
    and confidence in {"High", "Medium", "Low"}
    and reason.strip()
    for target, reason, confidence in new_ppr_mappings.values()
)

print("PRE-VALIDATION PASSED")
print("17 mappings validated.")
print("1 occurrence retained for investigation.")

# ============================================================
# STEP 2: APPLY ONLY THE 17 VERIFIED MAPPINGS
# ============================================================

for (sid, pos), (target, reason, confidence) in new_ppr_mappings.items():

    mask = (
        (manual_review["sentence_id"] == sid)
        & (manual_review["Token_Position"] == pos)
        & (manual_review["Target_Tag"].isna())
    )

    assert mask.sum() == 1

    manual_review.loc[mask, "Target_Tag"] = target
    manual_review.loc[mask, "Mapping_Reason"] = reason
    manual_review.loc[mask, "Mapping_Confidence"] = confidence

# ============================================================
# STEP 3: POST-MAPPING AUDIT
# ============================================================

for (sid, pos), (target, reason, confidence) in new_ppr_mappings.items():

    row = manual_review[
        (manual_review["sentence_id"] == sid)
        & (manual_review["Token_Position"] == pos)
    ]

    assert len(row) == 1
    assert row.iloc[0]["Target_Tag"] == target
    assert row.iloc[0]["Mapping_Reason"] == reason
    assert row.iloc[0]["Mapping_Confidence"] == confidence

# Confirm deferred occurrence remains untouched
deferred_row = manual_review[
    (manual_review["sentence_id"] == deferred_key[0])
    & (manual_review["Token_Position"] == deferred_key[1])
]

assert len(deferred_row) == 1
assert pd.isna(deferred_row.iloc[0]["Target_Tag"])
assert deferred_row.iloc[0]["Word"] == deferred_word

print("\nPOST-MAPPING AUDIT PASSED")

# ============================================================
# STEP 4: SUMMARY
# ============================================================

applied_rows = manual_review[
    manual_review.apply(
        lambda r: (r["sentence_id"], r["Token_Position"])
        in new_ppr_mappings,
        axis=1
    )
]

print("\nMAPPINGS APPLIED:", len(applied_rows))
print("SN:", (applied_rows["Target_Tag"] == "SN").sum())
print("SNV:", (applied_rows["Target_Tag"] == "SNV").sum())

print("\nCONFIDENCE DISTRIBUTION:")
display(applied_rows["Mapping_Confidence"].value_counts())

print("\nDEFERRED OCCURRENCE:")
display(
    deferred_row[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence"
        ]
    ]
)

# ============================================================
# STEP 5: REMAINING PPR WORKLOAD
# ============================================================

remaining_ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
    & manual_review["Target_Tag"].isna()
]

print("\nREMAINING UNRESOLVED PPR OCCURRENCES:", len(remaining_ppr))
print("REMAINING SOURCE TAGS:", remaining_ppr["Source_Tag"].nunique())

if len(remaining_ppr) > 0:
    display(
        remaining_ppr[
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Source_Tag",
                "Full_Sentence"
            ]
        ]
    )

print("\nCOMPLETE BATCH PROCESSING FINISHED")

In [ ]:
# Cell 238: Investigate final unresolved PPR occurrence

sid = 1316
pos = 12

row = manual_review[
    (manual_review["sentence_id"] == sid)
    & (manual_review["Token_Position"] == pos)
]

assert len(row) == 1

r = row.iloc[0]

print("=" * 100)
print("FINAL UNRESOLVED OCCURRENCE")
print("=" * 100)

print("Sentence ID:", sid)
print("Token Position:", pos)
print("Word:", repr(r["Word"]))
print("Source Tag:", r["Source_Tag"])

print("\nCOMPLETE SENTENCE:")
print(r["Full_Sentence"])

print("\nPREVIOUS WORDS:")
print(r["Previous_Words"])

print("\nNEXT WORDS:")
print(r["Next_Words"])

print("\nCURRENT MAPPING:")
print("Target Tag:", r["Target_Tag"])
print("Reason:", r.get("Mapping_Reason"))
print("Confidence:", r.get("Mapping_Confidence"))

print("\n" + "=" * 100)

In [ ]:
# Cell 239: Resolve final PPR occurrence and audit entire PPR dataset

sid, pos = 1316, 12

final_mapping = (
    "SN",
    "तानि refers independently to the previously mentioned items and functions as the subject of अन्तर्भवन्ति. It does not directly modify an expressed noun.",
    "High"
)

# STEP 1: Validate exact occurrence
mask = (
    (manual_review["sentence_id"] == sid)
    & (manual_review["Token_Position"] == pos)
)

row = manual_review[mask]

assert len(row) == 1, "Occurrence key must identify exactly one row"
assert row.iloc[0]["Word"] == "तानि"
assert row.iloc[0]["Source_Tag"] == "PPR.neu.pl.3.nom.i.n.n.dst"
assert pd.isna(row.iloc[0]["Target_Tag"])

# STEP 2: Apply final mapping
manual_review.loc[mask, "Target_Tag"] = final_mapping[0]
manual_review.loc[mask, "Mapping_Reason"] = final_mapping[1]
manual_review.loc[mask, "Mapping_Confidence"] = final_mapping[2]

# STEP 3: Verify final occurrence
row = manual_review[mask]

assert row.iloc[0]["Target_Tag"] == "SN"
assert row.iloc[0]["Mapping_Reason"] == final_mapping[1]
assert row.iloc[0]["Mapping_Confidence"] == "High"

print("FINAL OCCURRENCE MAPPED SUCCESSFULLY")

# STEP 4: Complete PPR audit
ppr = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
].copy()

unresolved = ppr[ppr["Target_Tag"].isna()]

invalid_targets = ppr[
    ~ppr["Target_Tag"].isin(["SN", "SNV"])
]

missing_reasons = ppr[
    ppr["Mapping_Reason"].isna()
    | (ppr["Mapping_Reason"].astype(str).str.strip() == "")
]

missing_confidence = ppr[
    ~ppr["Mapping_Confidence"].isin(["High", "Medium", "Low"])
]

duplicate_keys = ppr[
    ppr.duplicated(
        ["sentence_id", "Token_Position"],
        keep=False
    )
]

# STEP 5: Final statistics
print("\n" + "=" * 80)
print("COMPLETE PPR AUDIT REPORT")
print("=" * 80)

print("Total PPR occurrences:", len(ppr))
print("Unique source tags:", ppr["Source_Tag"].nunique())

print("\nTARGET TAG DISTRIBUTION:")
display(ppr["Target_Tag"].value_counts(dropna=False))

print("\nCONFIDENCE DISTRIBUTION:")
display(ppr["Mapping_Confidence"].value_counts(dropna=False))

print("\nUNRESOLVED OCCURRENCES:", len(unresolved))
print("INVALID TARGET TAGS:", len(invalid_targets))
print("MISSING MAPPING REASONS:", len(missing_reasons))
print("MISSING/INVALID CONFIDENCE:", len(missing_confidence))
print("DUPLICATE OCCURRENCE ROWS:", len(duplicate_keys))

# STEP 6: Assertions
assert len(unresolved) == 0, "Unresolved PPR occurrences remain"
assert len(invalid_targets) == 0, "Invalid target tags found"
assert len(missing_reasons) == 0, "Missing mapping reasons found"
assert len(missing_confidence) == 0, "Missing/invalid confidence found"
assert len(duplicate_keys) == 0, "Duplicate occurrence keys found"

print("\n" + "=" * 80)
print("ALL PPR AUDIT CHECKS PASSED")
print("=" * 80)

In [ ]:
# Save completed PPR mapping results

from pathlib import Path

OUT_DIR = Path(
    str(RESULTS_DIR)
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save complete occurrence-level review
manual_review.to_csv(
    OUT_DIR / "manual_review_PPR_completed.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save PPR-specific audit data
ppr_final = manual_review[
    manual_review["Source_Tag"].str.startswith("PPR.", na=False)
].copy()

ppr_final.to_csv(
    OUT_DIR / "PPR_final_mappings.csv",
    index=False,
    encoding="utf-8-sig"
)

print("PPR backup completed.")
print("Total PPR occurrences:", len(ppr_final))
print("SN:", (ppr_final["Target_Tag"] == "SN").sum())
print("SNV:", (ppr_final["Target_Tag"] == "SNV").sum())
print("Unresolved:", ppr_final["Target_Tag"].isna().sum())

In [ ]:
# Cell 240: Identify remaining unresolved source-tag families

unresolved_all = manual_review[
    manual_review["Target_Tag"].isna()
].copy()

print("TOTAL UNRESOLVED OCCURRENCES:", len(unresolved_all))

print("\nSOURCE-TAG FAMILY DISTRIBUTION:")

family_summary = (
    unresolved_all.groupby("Source_Family")
    .size()
    .sort_values(ascending=False)
    .rename("Unresolved_Count")
    .to_frame()
)

display(family_summary)

print("\nTOP 15 UNRESOLVED SOURCE TAGS:")

display(
    unresolved_all.groupby("Source_Tag")
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("Unresolved_Count")
    .to_frame()
)

In [ ]:
# Cell 241: Inspect unresolved JQ occurrences

jq_unresolved = manual_review[
    (manual_review["Source_Family"] == "JQ") &
    (manual_review["Target_Tag"].isna())
].copy()

print("TOTAL UNRESOLVED JQ OCCURRENCES:", len(jq_unresolved))

tag_counts = (
    jq_unresolved["Source_Tag"]
    .value_counts()
    .rename_axis("Source_Tag")
    .reset_index(name="Count")
)

display(tag_counts.head(20))

print("\nCONTEXT SAMPLES — TOP 10 JQ TAGS")

for tag in tag_counts.head(10)["Source_Tag"]:
    samples = jq_unresolved[jq_unresolved["Source_Tag"] == tag].head(2)

    print("\n" + "=" * 100)
    print("SOURCE TAG:", tag)
    print("TOTAL OCCURRENCES:", int(tag_counts.loc[
        tag_counts["Source_Tag"] == tag, "Count"
    ].iloc[0]))

    for _, row in samples.iterrows():
        print("-" * 80)
        print("Sentence ID:", row["sentence_id"])
        print("Token Position:", row["Token_Position"])
        print("Word:", row["Word"])
        print("Previous:", row["Previous_Words"])
        print("Next:", row["Next_Words"])
        print("Full sentence:", row["Full_Sentence"])

In [ ]:
# Cell 242: Full JQ pattern review

jq_unresolved = manual_review[
    (manual_review["Source_Family"] == "JQ") &
    (manual_review["Target_Tag"].isna())
].copy()

for tag, group in jq_unresolved.groupby("Source_Tag", sort=False):

    print("\n" + "=" * 110)
    print(f"SOURCE TAG: {tag} | TOTAL: {len(group)}")

    for _, row in group.head(3).iterrows():
        print("-" * 90)
        print(
            f"ID: ({row['sentence_id']}, {row['Token_Position']})"
            f" | Word: {row['Word']}"
        )
        print("Sentence:", row["Full_Sentence"])

In [ ]:
# Cell 243: Prepare occurrence-level JQ mapping review

jq_review = manual_review[
    (manual_review["Source_Family"] == "JQ") &
    (manual_review["Target_Tag"].isna())
].copy()

for _column in ("Proposed_Target_Tag", "Mapping_Reason", "Mapping_Confidence"):
    jq_review[_column] = pd.Series(pd.NA, index=jq_review.index, dtype="object")

jq_review = jq_review.sort_values(
    ["Source_Tag", "sentence_id", "Token_Position"]
).reset_index(drop=True)

output_path = (
    "" + str(RESULTS_DIR) + "/"
    "JQ_occurrence_review.csv"
)

jq_review.to_csv(output_path, index=False, encoding="utf-8-sig")

print("JQ REVIEW FILE CREATED")
print("Total occurrences:", len(jq_review))
print("Unique source tags:", jq_review["Source_Tag"].nunique())
print("Unresolved target labels:", jq_review["Proposed_Target_Tag"].isna().sum())
print("Saved to:", output_path)

display(
    jq_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].head(10)
)

In [ ]:
# Cell 244: Preliminary lexical grouping of JQ occurrences

import pandas as pd

jq_review = pd.read_csv(
    "" + str(RESULTS_DIR) + "/"
    "JQ_occurrence_review.csv"
)

def classify_jq_word(word):
    word = str(word).strip()

    degree_adverbs = {
        "अतीव", "बहुधा", "सर्वथा", "कोटिशः"
    }

    quantifier_words = {
        "सर्वे", "सर्वेषां", "सर्वेषु", "सर्वासु",
        "सर्वान्", "सर्वाणि", "सर्वं", "सर्वा",
        "सकलाः", "सकला", "सकलाम्",
        "अनेके", "अनेकानि", "अनेकेषु",
        "बहवः", "बहूनां", "बहु", "लक्षाणि"
    }

    ordinal_words = {
        "प्रथमः", "प्रथमा", "प्रथमे", "प्रथमं",
        "अष्टमे", "चतुर्दशे", "द्वादशे",
        "पञ्चम्यां", "अन्तिमं", "आद्यं",
        "द्विगुणितः", "त्रिगुणितः",
        "१८६०तमे", "१८५७तमे", "१८८३तमे",
        "विंशतिततम्यां", "नवविंशतितम्यां",
        "द्वाविंशतमः"
    }

    if word in degree_adverbs:
        return "DEGREE_ADVERB"

    if word in quantifier_words:
        return "QUANTIFIER"

    if word in ordinal_words:
        return "ORDINAL"

    if word in {"एकः", "एका", "एकम्", "एकं", "एकां",
                "एकाम्", "एकेन", "एकस्मिन्", "एकस्यां"}:
        return "CARDINAL_ONE"

    if word in {"द्वौ", "द्वे", "उभौ", "उभे", "दौ"}:
        return "DUAL_QUANTIFIER"

    return "OTHER_REVIEW"

jq_review["Review_Group"] = jq_review["Word"].apply(classify_jq_word)

print("JQ PRELIMINARY GROUP DISTRIBUTION")
display(
    jq_review["Review_Group"]
    .value_counts()
    .rename_axis("Review_Group")
    .to_frame("Occurrences")
)

print("\nSOURCE TAG DISTRIBUTION WITHIN GROUPS")
display(
    pd.crosstab(
        jq_review["Review_Group"],
        jq_review["Source_Family"]
    )
)

print("\nUNCLASSIFIED WORDS — REVIEW THESE")
display(
    jq_review.loc[
        jq_review["Review_Group"] == "OTHER_REVIEW",
        ["Word", "Source_Tag", "sentence_id", "Full_Sentence"]
    ].head(30)
)

jq_review.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "JQ_occurrence_review_grouped.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nGrouped review file saved.")

In [ ]:
# Cell 245: Complete OTHER_REVIEW audit

import pandas as pd

jq_review = pd.read_csv(
    "" + str(RESULTS_DIR) + "/"
    "JQ_occurrence_review_grouped.csv"
)

other = jq_review[
    jq_review["Review_Group"] == "OTHER_REVIEW"
].copy()

other = other.sort_values(
    ["Word", "sentence_id", "Token_Position"]
).reset_index(drop=True)

print("TOTAL OTHER_REVIEW OCCURRENCES:", len(other))
print("UNIQUE WORD FORMS:", other["Word"].nunique())

print("\nWORD FREQUENCY DISTRIBUTION")
display(
    other["Word"]
    .value_counts()
    .rename_axis("Word")
    .to_frame("Occurrences")
)

print("\nCOMPLETE OCCURRENCE-LEVEL CONTEXT")

for _, row in other.iterrows():
    print(
        f"({row['sentence_id']}, {row['Token_Position']}) "
        f"| {row['Word']} "
        f"| {row['Source_Tag']}"
    )
    print("Sentence:", row["Full_Sentence"])
    print("-" * 90)

assert len(other) == 135, "Unexpected OTHER_REVIEW count"

assert not other.duplicated(
    ["sentence_id", "Token_Position"]
).any(), "Duplicate occurrence keys found"

print("\nOTHER_REVIEW AUDIT COMPLETED")

In [ ]:

import pandas as pd
from pathlib import Path

JQ_FILE = Path(
    "" + str(RESULTS_DIR) + "/"
    "JQ_occurrence_review_grouped.csv"
)

jq = pd.read_csv(JQ_FILE)
for _column in ("Proposed_Target_Tag", "Mapping_Reason", "Mapping_Confidence"):
    if _column in jq.columns:
        jq[_column] = jq[_column].astype(object)

print("Total JQ occurrences:", len(jq))
print("\nAvailable columns:")
print(jq.columns.tolist())

# Identify the group column
group_cols = [c for c in jq.columns if "group" in c.lower()]

if group_cols:
    GROUP_COL = group_cols[0]

    print("\nJQ group distribution:")
    print(jq[GROUP_COL].value_counts(dropna=False).to_string())

    print("\nOccurrences by group:")
    for group in jq[GROUP_COL].dropna().unique():
        subset = jq[jq[GROUP_COL] == group]

        print(f"\n{'='*80}")
        print(f"GROUP: {group} | Occurrences: {len(subset)}")
        print("="*80)

        cols = [
            c for c in [
                "sentence_id", "Token_Position", "Word",
                "Source_Tag", "Full_Sentence"
            ] if c in subset.columns
        ]

        display(subset[cols].head(15))
else:
    print("\nNo group column detected. Please share the column names.")

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

# Display every JQ occurrence in batches of 25
for start in range(0, len(jq), 25):
    end = min(start + 25, len(jq))

    print(f"\n{'='*100}")
    print(f"JQ OCCURRENCES {start + 1}–{end} of {len(jq)}")
    print("="*100)

    cols = [
        "sentence_id",
        "Token_Position",
        "Word",
        "Source_Tag",
        "Review_Group",
        "Previous_Words",
        "Next_Words",
        "Full_Sentence"
    ]

    display(jq.iloc[start:end][cols].reset_index(drop=True))

In [ ]:
# JQ Batch 1: Degree, manner and quantitative expressions

jq_batch1 = {
    (863, 4): ("AVP", "अतीव modifies the adjective प्रसन्नः as a degree adverb.", "High"),
    (879, 3): ("AVP", "अतीव modifies the adjective प्रसन्नाः as a degree adverb.", "High"),
    (415, 8): ("AVP", "अतीव modifies the adjective मितभाषिणी.", "High"),
    (446, 0): ("AVP", "अतीव modifies the adjective मेधाविनी.", "High"),
    (480, 3): ("AVP", "बहुधा modifies कृतम् by expressing manner or frequency.", "Medium"),
    (982, 3): ("AVP", "अतीव modifies the adjective रमणीया.", "High"),
    (1264, 5): ("AVP", "अतीव modifies the adjective मनोहारि.", "High"),
    (852, 5): ("AVP", "सर्वथा modifies the predicate न शोभते.", "High"),
    (510, 9): ("AVC", "कोटिशः expresses a quantitative count of salutations.", "Medium")
}

for (sid, pos), (tag, reason, confidence) in jq_batch1.items():
    mask = (
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one occurrence for {(sid, pos)}, "
        f"found {mask.sum()}"
    )

    jq.loc[mask, "Proposed_Target_Tag"] = tag
    jq.loc[mask, "Mapping_Reason"] = reason
    jq.loc[mask, "Mapping_Confidence"] = confidence

print("Batch 1 proposals recorded:", len(jq_batch1))

display(
    jq[
        jq.apply(
            lambda r: (r["sentence_id"], r["Token_Position"]) in jq_batch1,
            axis=1
        )
    ][[
        "sentence_id",
        "Token_Position",
        "Word",
        "Proposed_Target_Tag",
        "Mapping_Reason",
        "Mapping_Confidence"
    ]]
)

In [ ]:
# Fix proposal columns to use string/object-compatible dtypes
for col in [
    "Proposed_Target_Tag",
    "Mapping_Reason",
    "Mapping_Confidence"
]:
    jq[col] = jq[col].astype("object")

print("Column dtypes corrected:")
print(jq[
    ["Proposed_Target_Tag", "Mapping_Reason", "Mapping_Confidence"]
].dtypes)

In [ ]:
# Inspect dual-quantifier occurrences in JQ

dual_forms = ["उभौ", "उभे", "द्वे", "द्वौ"]

dual_review = jq[
    jq["Word"].astype(str).str.contains(
        "|".join(dual_forms),
        regex=True,
        na=False
    )
].copy()

print("Dual-quantifier occurrences:", len(dual_review))
print("Already proposed:", dual_review["Proposed_Target_Tag"].notna().sum())

display(
    dual_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["Word", "sentence_id"])
)

In [ ]:
# JQ Batch 2A: Independent dual pronouns

jq_batch2a = {
    (1410, 0): (
        "SN",
        "उभे functions independently as a dual pronoun referring to both speakers in the dialogue label.",
        "Medium"
    ),
    (1420, 5): (
        "SN",
        "उभे functions independently as a dual pronoun in the dialogue-speaker label.",
        "Medium"
    ),
    (1429, 0): (
        "SN",
        "उभे functions independently as a dual pronoun in the dialogue-speaker label.",
        "Medium"
    ),
    (337, 0): (
        "SN",
        "उभौ independently refers to two persons; no expressed noun is being modified.",
        "High"
    ),
    (723, 2): (
        "SN",
        "उभौ independently refers back to तौ and functions as a pronoun.",
        "High"
    ),
    (1398, 3): (
        "SN",
        "उभौ functions independently as the object-like pronoun in उभौ निर्वर्ण्य.",
        "High"
    )
}

# Validate exact occurrence keys
for key, value in jq_batch2a.items():
    mask = (
        (jq["sentence_id"] == key[0]) &
        (jq["Token_Position"] == key[1])
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one occurrence for {key}, found {mask.sum()}"
        )

    if jq.loc[mask, "Proposed_Target_Tag"].notna().any():
        raise ValueError(f"Occurrence already proposed: {key}")

# Record proposals
for (sentence_id, position), (tag, reason, confidence) in jq_batch2a.items():
    mask = (
        (jq["sentence_id"] == sentence_id) &
        (jq["Token_Position"] == position)
    )

    jq.loc[mask, "Proposed_Target_Tag"] = tag
    jq.loc[mask, "Mapping_Reason"] = reason
    jq.loc[mask, "Mapping_Confidence"] = confidence

print("Batch 2A proposals recorded:", len(jq_batch2a))

display(
    jq[
        jq.apply(
            lambda r: (r["sentence_id"], r["Token_Position"])
            in jq_batch2a,
            axis=1
        )
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)

In [ ]:
# Inspect existing proposals for all six Batch 2A occurrences

batch2a_keys = [
    (1410, 0),
    (1420, 5),
    (1429, 0),
    (337, 0),
    (723, 2),
    (1398, 3)
]

for sentence_id, position in batch2a_keys:
    mask = (
        (jq["sentence_id"] == sentence_id) &
        (jq["Token_Position"] == position)
    )

    row = jq.loc[
        mask,
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]

    print(f"\nOccurrence: {(sentence_id, position)}")
    display(row)

In [ ]:
# Inspect remaining dual cardinal numerals

batch2a_keys = {
    (1410, 0), (1420, 5), (1429, 0),
    (337, 0), (723, 2), (1398, 3)
}

remaining_duals = jq[
    jq.apply(
        lambda r: (
            r["Word"] in ["द्वे", "द्वौ"]
            and (r["sentence_id"], r["Token_Position"]) not in batch2a_keys
        ),
        axis=1
    )
].copy()

print("Remaining dual numeral occurrences:", len(remaining_duals))

display(
    remaining_duals[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values("sentence_id")
)

In [ ]:
# JQ Batch 2B: Declined cardinal numerals

jq_batch2b = {
    (152, 0): (
        "A",
        "द्वौ qualifies the noun बकौ by expressing its cardinal quantity; it is an inflected numeral.",
        "Medium"
    ),
    (488, 12): (
        "A",
        "द्वौ expresses cardinal quantity in apposition to the demonstrative pronoun एतौ.",
        "Medium"
    ),
    (960, 4): (
        "A",
        "द्वौ qualifies स्वर्णपदकौ by expressing cardinal quantity; it functions adjectivally.",
        "Medium"
    ),
    (1056, 6): (
        "A",
        "द्वे qualifies अस्त्रे and agrees with the neuter dual nominal expression.",
        "Medium"
    ),
    (1063, 6): (
        "A",
        "द्वे qualifies स्त्रियौ and expresses their cardinal quantity.",
        "Medium"
    ),
    (1065, 1): (
        "A",
        "द्वे qualifies भगिन्यौ and expresses their cardinal quantity.",
        "Medium"
    ),
    (1079, 2): (
        "A",
        "द्वे qualifies मातरौ and expresses their cardinal quantity.",
        "Medium"
    ),
    (1270, 1): (
        "A",
        "द्वे qualifies चलचित्रे and expresses their cardinal quantity.",
        "Medium"
    )
}

# Validate all occurrence keys before writing
for key in jq_batch2b:
    mask = (
        (jq["sentence_id"] == key[0]) &
        (jq["Token_Position"] == key[1])
    )

    if mask.sum() != 1:
        raise ValueError(f"Occurrence validation failed: {key}")

    if jq.loc[mask, "Proposed_Target_Tag"].notna().any():
        raise ValueError(f"Existing proposal found; not overwriting: {key}")

# Record proposals
for (sentence_id, position), (tag, reason, confidence) in jq_batch2b.items():
    mask = (
        (jq["sentence_id"] == sentence_id) &
        (jq["Token_Position"] == position)
    )

    jq.loc[mask, "Proposed_Target_Tag"] = tag
    jq.loc[mask, "Mapping_Reason"] = reason
    jq.loc[mask, "Mapping_Confidence"] = confidence

print("Batch 2B proposals recorded:", len(jq_batch2b))

display(
    jq[
        jq.apply(
            lambda r: (r["sentence_id"], r["Token_Position"]) in jq_batch2b,
            axis=1
        )
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ]
)

# Part B2 — Safe mapping workbench and current JQ review

The workbench saves proposal and unresolved queues separately from final labels. The optional explicit-approval commit cell is not a routine execution cell: only use it after reviewing and authorizing the exact occurrence keys. The JQ context/recommendation/exception/approval-queue cells are retained as the latest review workflow.

# 12. Optimized Mapping Workbench — Safe Continuation

This section is added to the original research notebook. The historical dataset-preparation, modeling, and occurrence-level mapping cells are retained. The duplicated failed JQ Batch 2A retry cell has been removed; its six successful proposals remain in the preceding successful cell.

## Purpose
- Continue mapping without repeating old manual cells.
- Apply proposals only to exact `(sentence_id, Token_Position)` occurrence keys.
- Keep proposals separate from final gold labels until explicitly approved.
- Prevent accidental overwrites, duplicate keys, invalid tags, and silent partial application.
- Export resumable CSV checkpoints and a complete unresolved review queue.

**Important:** Run the original notebook setup and mapping preparation cells first, through the cells that create `manual_review` and `jq`. This workbench expects those DataFrames to exist. It does not infer or auto-assign labels from source-tag prefixes. Context-sensitive decisions still require linguistic review.


## 12.1 Mapping rules and target inventory

The approved target inventory is the project's 24-tag list. The workbench validates against it. A proposal is not automatically a gold annotation: it remains in `Proposed_Target_Tag` until it is reviewed and deliberately committed.

The current JQ decisions recorded in the notebook are:
- Batch 1: 9 proposals.
- Batch 2A: 6 independent dual-pronoun proposals (`SN`).
- Batch 2B: 8 declined cardinal numeral proposals (`A`, provisional convention; Medium confidence).

The notebook's previous `manual_review["Target_Tag"]` mappings remain intact.


In [ ]:
# 12.2 Initialize safe mapping utilities

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
from datetime import datetime

TARGET_TAGS = [
    "N", "NS", "NV", "KNV", "SN", "SNV", "SNS",
    "SAM", "SAMY", "SAMP", "KP", "UP", "AVN",
    "AVP", "AVUP", "SAMB", "AVU", "K", "AKV",
    "AB", "AVC", "DVI", "A", "PU"
]

PROPOSAL_COLUMNS = [
    "Proposed_Target_Tag",
    "Mapping_Reason",
    "Mapping_Confidence"
]
VALID_CONFIDENCE = {"High", "Medium", "Low"}

required_review_columns = {
    "sentence_id", "Token_Position", "Word", "Source_Tag",
    "Source_Family", "Full_Sentence", "Target_Tag"
}
missing = required_review_columns - set(manual_review.columns)
if missing:
    raise ValueError(f"manual_review is missing required columns: {sorted(missing)}")

for col in PROPOSAL_COLUMNS:
    if col not in manual_review.columns:
        manual_review[col] = pd.Series(index=manual_review.index, dtype="object")
    else:
        manual_review[col] = manual_review[col].astype("object")

if "jq" in globals():
    for col in PROPOSAL_COLUMNS:
        if col not in jq.columns:
            jq[col] = pd.Series(index=jq.index, dtype="object")
        else:
            jq[col] = jq[col].astype("object")

print("Target tag inventory:", len(TARGET_TAGS))
print("manual_review rows:", len(manual_review))
print("JQ review DataFrame available:", "jq" in globals())
print("Proposal columns ready.")


In [ ]:
# 12.3 Exact-key proposal function (safe by default)

def apply_occurrence_proposals(
    df,
    proposals,
    family=None,
    overwrite=False,
    target_column="Proposed_Target_Tag"
):
    """Apply { (sentence_id, Token_Position): (tag, reason, confidence) }.

    Safety properties:
    - each occurrence key must match exactly one row;
    - target tags and confidence values are validated;
    - existing proposals are not overwritten unless overwrite=True;
    - changes are validated before any write occurs.
    """
    required = {"sentence_id", "Token_Position", target_column,
                "Mapping_Reason", "Mapping_Confidence"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing columns: {sorted(missing_cols)}")

    for _column in (target_column, "Mapping_Reason", "Mapping_Confidence"):
        df[_column] = df[_column].astype(object)

    validated = []
    errors = []

    for key, value in proposals.items():
        if len(key) != 2:
            errors.append(f"Invalid occurrence key: {key}")
            continue
        sid, pos = key
        if len(value) != 3:
            errors.append(f"Expected (tag, reason, confidence) for {key}")
            continue

        tag, reason, confidence = value
        if tag not in TARGET_TAGS:
            errors.append(f"Invalid target tag {tag!r} for {key}")
        if not isinstance(reason, str) or not reason.strip():
            errors.append(f"Missing mapping reason for {key}")
        if confidence not in VALID_CONFIDENCE:
            errors.append(f"Invalid confidence {confidence!r} for {key}")

        mask = (
            (df["sentence_id"] == sid) &
            (df["Token_Position"] == pos)
        )
        if family is not None and "Source_Family" in df.columns:
            mask &= df["Source_Family"].eq(family)

        count = int(mask.sum())
        if count != 1:
            errors.append(f"{key} matched {count} rows (expected exactly 1)")
            continue

        if pd.notna(df.loc[mask, target_column]).any() and not overwrite:
            existing = df.loc[mask, target_column].iloc[0]
            errors.append(f"{key} already has proposal {existing!r}; not overwritten")
            continue

        validated.append((mask, tag, reason.strip(), confidence, key))

    if errors:
        raise ValueError(
            "No proposals were written. Resolve these validation issues first:\n- "
            + "\n- ".join(errors)
        )

    for mask, tag, reason, confidence, key in validated:
        df.loc[mask, target_column] = tag
        df.loc[mask, "Mapping_Reason"] = reason
        df.loc[mask, "Mapping_Confidence"] = confidence

    return len(validated)


def audit_proposals(df, family=None, target_column="Proposed_Target_Tag"):
    view = df.copy()
    if family is not None and "Source_Family" in view.columns:
        view = view[view["Source_Family"].eq(family)].copy()

    mapped = view[target_column].notna()
    invalid = view[mapped & ~view[target_column].isin(TARGET_TAGS)]
    missing_reason = view[mapped & view["Mapping_Reason"].isna()]
    missing_conf = view[mapped & ~view["Mapping_Confidence"].isin(VALID_CONFIDENCE)]
    duplicate_keys = view.duplicated(["sentence_id", "Token_Position"], keep=False)

    return {
        "rows": len(view),
        "proposed": int(mapped.sum()),
        "unproposed": int((~mapped).sum()),
        "invalid_tags": len(invalid),
        "missing_reasons": len(missing_reason),
        "invalid_confidence": len(missing_conf),
        "duplicate_occurrence_rows": int(duplicate_keys.sum())
    }

print("Safe mapping utilities defined.")


## 12.4 Rebuild the current JQ proposal view

This cell synchronizes the proposal columns already recorded in `jq` back to the corresponding JQ rows in `manual_review` **without changing `Target_Tag`**. It refuses ambiguous or duplicate occurrence matches.


In [ ]:
# Synchronize existing JQ proposals to the review ledger (proposal columns only)

if "jq" not in globals():
    raise NameError("jq is not available. Run the JQ review preparation cells first.")

jq_proposals = jq[jq["Proposed_Target_Tag"].notna()].copy()
sync_count = 0
sync_errors = []

for row in jq_proposals.itertuples(index=False):
    sid = getattr(row, "sentence_id")
    pos = getattr(row, "Token_Position")
    mask = (
        manual_review["Source_Family"].eq("JQ") &
        manual_review["sentence_id"].eq(sid) &
        manual_review["Token_Position"].eq(pos)
    )
    if int(mask.sum()) != 1:
        sync_errors.append((sid, pos, int(mask.sum())))
        continue

    # Do not overwrite a conflicting proposal already in the master ledger.
    existing = manual_review.loc[mask, "Proposed_Target_Tag"].iloc[0]
    if pd.notna(existing) and existing != row.Proposed_Target_Tag:
        sync_errors.append((sid, pos, "conflicting proposal"))
        continue

    for col in PROPOSAL_COLUMNS:
        manual_review.loc[mask, col] = getattr(row, col)
    sync_count += 1

if sync_errors:
    print("Sync exceptions (no conflicting rows overwritten):")
    display(pd.DataFrame(sync_errors, columns=["sentence_id", "Token_Position", "issue"]))
else:
    print("JQ proposal rows synchronized:", sync_count)

print("JQ proposal audit:", audit_proposals(jq))


## 12.5 Create one consolidated unresolved work queue

This produces a CSV for all remaining source families, with full sentence context and stable occurrence identifiers. It is a review queue, not an automatic mapping operation.


In [ ]:
# Build a consolidated queue of still-unmapped occurrences

unresolved_queue = manual_review[
    manual_review["Target_Tag"].isna()
].copy()

if "Proposed_Target_Tag" in unresolved_queue.columns:
    unresolved_queue = unresolved_queue[
        unresolved_queue["Proposed_Target_Tag"].isna()
    ].copy()

queue_columns = [
    c for c in [
        "Source_Family", "sentence_id", "Token_Position", "Word",
        "Source_Tag", "Previous_Words", "Next_Words", "Full_Sentence",
        "Proposed_Target_Tag", "Mapping_Reason", "Mapping_Confidence"
    ] if c in unresolved_queue.columns
]

unresolved_queue = unresolved_queue[queue_columns].sort_values(
    ["Source_Family", "Source_Tag", "sentence_id", "Token_Position"],
    kind="stable"
).reset_index(drop=True)

family_queue_summary = (
    unresolved_queue.groupby("Source_Family", dropna=False)
    .agg(
        unresolved_occurrences=("Word", "size"),
        unique_words=("Word", "nunique"),
        unique_source_tags=("Source_Tag", "nunique")
    )
    .sort_values("unresolved_occurrences", ascending=False)
)

print("Unresolved occurrences awaiting a proposal:", len(unresolved_queue))
display(family_queue_summary)
display(unresolved_queue.head(30))


## 12.6 Export resumable checkpoints

The checkpoint includes the complete master review ledger, the JQ review table (if present), the consolidated unresolved queue, and a JSON summary. The output directory uses the existing `RESULTS_DIR` when available; otherwise it uses a local folder.


In [ ]:
# Save mapping checkpoints

if "RESULTS_DIR" in globals():
    MAPPING_OUT_DIR = Path(RESULTS_DIR)
else:
    MAPPING_OUT_DIR = Path("./Sanskrit_POS_Mapping_Checkpoints")

MAPPING_OUT_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

manual_checkpoint = MAPPING_OUT_DIR / "occurrence_level_mapping_review_checkpoint.csv"
queue_checkpoint = MAPPING_OUT_DIR / "unresolved_mapping_queue.csv"
jq_checkpoint = MAPPING_OUT_DIR / "JQ_occurrence_review_checkpoint.csv"
summary_checkpoint = MAPPING_OUT_DIR / "mapping_checkpoint_summary.json"

manual_review.to_csv(manual_checkpoint, index=False, encoding="utf-8-sig")
unresolved_queue.to_csv(queue_checkpoint, index=False, encoding="utf-8-sig")

if "jq" in globals():
    jq.to_csv(jq_checkpoint, index=False, encoding="utf-8-sig")

summary = {
    "saved_at": stamp,
    "manual_review_rows": int(len(manual_review)),
    "unresolved_queue_rows": int(len(unresolved_queue)),
    "family_summary": {
        str(k): int(v)
        for k, v in unresolved_queue["Source_Family"].value_counts().items()
    },
    "jq_audit": audit_proposals(jq) if "jq" in globals() else None,
    "target_tags": TARGET_TAGS,
    "note": "Proposals are separate from final Target_Tag gold annotations."
}
with open(summary_checkpoint, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved mapping checkpoints:")
for p in [manual_checkpoint, queue_checkpoint, jq_checkpoint, summary_checkpoint]:
    if p.exists():
        print(f" - {p} ({p.stat().st_size:,} bytes)")


## 12.7 Commit reviewed proposals to final Target_Tag — explicit approval only

Do not run this cell until the listed occurrence keys have been linguistically reviewed and approved. It commits only the explicitly listed keys, checks that each row is unique, and preserves all other rows. Use `APPROVED_KEYS` as an explicit allowlist.


In [ ]:
# OPTIONAL: explicitly commit reviewed proposals to Target_Tag
# Leave APPROVED_KEYS empty until you have reviewed the corresponding proposal rows.

APPROVED_KEYS = set()  # Example: {(863, 4), (879, 3)}

def commit_approved_proposals(master_df, approved_keys, family=None):
    if not approved_keys:
        print("No approved keys supplied; nothing committed.")
        return 0

    planned = []
    errors = []

    for sid, pos in approved_keys:
        mask = (
            master_df["sentence_id"].eq(sid) &
            master_df["Token_Position"].eq(pos)
        )
        if family is not None:
            mask &= master_df["Source_Family"].eq(family)

        if int(mask.sum()) != 1:
            errors.append(f"{(sid, pos)} matched {int(mask.sum())} rows")
            continue

        row = master_df.loc[mask].iloc[0]
        tag = row.get("Proposed_Target_Tag")
        reason = row.get("Mapping_Reason")
        confidence = row.get("Mapping_Confidence")

        if pd.isna(tag) or tag not in TARGET_TAGS:
            errors.append(f"{(sid, pos)} has no valid proposed tag")
        if pd.isna(reason) or not str(reason).strip():
            errors.append(f"{(sid, pos)} has no mapping reason")
        if confidence not in VALID_CONFIDENCE:
            errors.append(f"{(sid, pos)} has invalid confidence")
        if pd.notna(row.get("Target_Tag")):
            errors.append(f"{(sid, pos)} already has Target_Tag={row.get('Target_Tag')!r}")

        planned.append((mask, tag))

    if errors:
        raise ValueError("Nothing committed. Resolve:\n- " + "\n- ".join(errors))

    for mask, tag in planned:
        master_df.loc[mask, "Target_Tag"] = tag

    print("Explicitly approved mappings committed:", len(planned))
    return len(planned)

# Uncomment only after reviewing the exact keys:
# committed = commit_approved_proposals(manual_review, APPROVED_KEYS, family="JQ")


## 12.8 Final mapping audit before model preparation

This audit reports completed and unresolved counts by family, invalid labels, missing mapping explanations, and duplicate occurrence keys. A successful audit checks data consistency; it does not independently prove linguistic correctness. Expert adjudication is still required for uncertain decisions.


In [ ]:
# Global occurrence-level mapping audit

mapped_mask = manual_review["Target_Tag"].notna()
invalid_gold = manual_review[
    mapped_mask & ~manual_review["Target_Tag"].isin(TARGET_TAGS)
].copy()

missing_reason_gold = manual_review[
    mapped_mask & (
        manual_review["Mapping_Reason"].isna() |
        manual_review["Mapping_Reason"].astype(str).str.strip().eq("")
    )
].copy()

duplicate_occurrences = manual_review[
    manual_review.duplicated(["Source_Family", "sentence_id", "Token_Position"], keep=False)
].copy()

global_summary = (
    manual_review.assign(is_mapped=mapped_mask)
    .groupby("Source_Family", dropna=False)
    .agg(
        total_occurrences=("Word", "size"),
        mapped=("is_mapped", "sum"),
        unresolved=("is_mapped", lambda s: int((~s).sum()))
    )
    .sort_values("unresolved", ascending=False)
)

print("GLOBAL MAPPING AUDIT")
display(global_summary)
print("Invalid Target_Tag values:", len(invalid_gold))
print("Mapped rows missing Mapping_Reason:", len(missing_reason_gold))
print("Duplicate family/sentence/token keys:", len(duplicate_occurrences))

if len(invalid_gold):
    display(invalid_gold[["Source_Family", "sentence_id", "Token_Position", "Word", "Target_Tag"]])
if len(missing_reason_gold):
    display(missing_reason_gold[["Source_Family", "sentence_id", "Token_Position", "Word"]])
if len(duplicate_occurrences):
    display(duplicate_occurrences[["Source_Family", "sentence_id", "Token_Position", "Word"]])

print("\nAudit complete. Resolve all data-integrity issues before final model training.")


In [ ]:

# Check JQ proposal status

print("JQ DataFrame shape:", jq.shape)

print("\nAvailable columns:")
print(jq.columns.tolist())

print("\nProposal counts:")
print(
    jq["Proposed_Target_Tag"]
    .value_counts(dropna=False)
)

print("\nConfidence counts:")
print(
    jq["Mapping_Confidence"]
    .value_counts(dropna=False)
)

print("\nSample proposals:")
display(
    jq[
        jq["Proposed_Target_Tag"].notna()
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].head(15)
)


In [ ]:

# ============================================================
# JQ PROPOSAL STATUS CHECK
# Read-only verification — does not modify any mappings
# ============================================================

if "jq" not in globals():
    print("ERROR: jq DataFrame not found.")
    print("Please rerun the JQ preparation and proposal cells.")
else:
    print("JQ DataFrame shape:", jq.shape)

    required_cols = [
        "sentence_id",
        "Token_Position",
        "Word",
        "Proposed_Target_Tag",
        "Mapping_Reason",
        "Mapping_Confidence"
    ]

    missing_cols = [
        col for col in required_cols if col not in jq.columns
    ]

    if missing_cols:
        print("\nMissing columns:", missing_cols)
    else:
        proposals = jq[jq["Proposed_Target_Tag"].notna()].copy()

        print("\nTotal JQ occurrences:", len(jq))
        print("Proposed mappings:", len(proposals))
        print("Without proposals:", len(jq) - len(proposals))

        print("\nProposed target tag distribution:")
        display(proposals["Proposed_Target_Tag"].value_counts())

        print("\nConfidence distribution:")
        display(proposals["Mapping_Confidence"].value_counts())

        print("\nProposal details:")
        display(proposals[required_cols].sort_values(
            ["sentence_id", "Token_Position"]
        ))


In [ ]:

# ============================================================
# JQ BATCH REVIEW — FULL SENTENCE CONTEXT
# Read-only: no mapping changes
# ============================================================

proposals = jq[jq["Proposed_Target_Tag"].notna()].copy()

context_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Previous_Words",
    "Full_Sentence",
    "Next_Words",
    "Proposed_Target_Tag",
    "Mapping_Reason",
    "Mapping_Confidence"
]

available_columns = [
    col for col in context_columns if col in proposals.columns
]

missing_columns = [
    col for col in context_columns if col not in proposals.columns
]

print("Total proposals:", len(proposals))
print("Available context columns:", available_columns)

if missing_columns:
    print("Columns not present in jq:", missing_columns)

display(
    proposals[available_columns]
    .sort_values(["sentence_id", "Token_Position"])
    .reset_index(drop=True)
)


In [ ]:

# ============================================================
# JQ REVIEW — GUIDELINE-BASED RECOMMENDATIONS
# SAFE MODE: Does not commit gold labels
# ============================================================

import pandas as pd

# Ensure required columns exist
for col in ["Review_Recommendation", "Review_Status", "Review_Notes"]:
    if col not in jq.columns:
        jq[col] = None

# Review recommendations based on the Consortium POS guidelines
for idx, row in jq[jq["Proposed_Target_Tag"].notna()].iterrows():

    proposed = row["Proposed_Target_Tag"]
    word = row["Word"]

    if proposed == "AVP":
        jq.at[idx, "Review_Recommendation"] = "AKV"
        jq.at[idx, "Review_Status"] = "REVIEW_REQUIRED"
        jq.at[idx, "Review_Notes"] = (
            "AVP is interrogative indeclinable; "
            "AKV is the candidate for an adverbial indeclinable."
        )

    elif proposed == "A":
        jq.at[idx, "Review_Recommendation"] = "SAMY"
        jq.at[idx, "Review_Status"] = "REVIEW_REQUIRED"
        jq.at[idx, "Review_Notes"] = (
            "Check SAMY classification: numeral agreeing "
            "with the following nominal expression."
        )

    elif proposed == "SN":
        jq.at[idx, "Review_Recommendation"] = "SN"
        jq.at[idx, "Review_Status"] = "REVIEW_REQUIRED"
        jq.at[idx, "Review_Notes"] = (
            "Pronoun candidate. Verify independent pronominal "
            "use, especially dialogue-label contexts."
        )

    elif proposed == "AVC":
        jq.at[idx, "Review_Recommendation"] = None
        jq.at[idx, "Review_Status"] = "LINGUISTIC_REVIEW_REQUIRED"
        jq.at[idx, "Review_Notes"] = (
            "कोटिशः requires separate classification; "
            "do not assume it is a conjunction."
        )

# Display recommendation summary
print("JQ recommendation summary:")
display(
    jq[jq["Proposed_Target_Tag"].notna()]
    .groupby(
        ["Proposed_Target_Tag", "Review_Recommendation", "Review_Status"],
        dropna=False
    )
    .size()
    .reset_index(name="Occurrences")
)

print("\nDetailed review:")
display(
    jq[jq["Proposed_Target_Tag"].notna()][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Proposed_Target_Tag",
            "Review_Recommendation",
            "Review_Status",
            "Review_Notes"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)


In [ ]:

# ============================================================
# JQ FINAL LINGUISTIC REVIEW SHEET
# Safe mode: no changes to Target_Tag
# ============================================================

jq_review = jq[jq["Proposed_Target_Tag"].notna()].copy()

# Provisional recommendations based on Consortium guidelines
def get_jq_review(row):

    word = str(row["Word"])
    sid = row["sentence_id"]
    proposed = row["Proposed_Target_Tag"]

    if proposed == "AVP":
        return pd.Series({
            "Candidate_Gold_Tag": "AKV",
            "Decision": "PROVISIONAL",
            "Review_Note": (
                "Adverbial indeclinable. Confirm its function "
                "in the complete sentence."
            )
        })

    if proposed == "A":
        return pd.Series({
            "Candidate_Gold_Tag": "SAMY",
            "Decision": "PROVISIONAL",
            "Review_Note": (
                "Inflected numeral. Verify agreement with the "
                "associated nominal expression and its role."
            )
        })

    if proposed == "SN":
        if sid in [1410, 1420, 1429]:
            note = (
                "Dialogue-label context. Verify whether this "
                "is a genuine pronoun token or a speaker marker."
            )
        else:
            note = (
                "Independent dual pronoun candidate. "
                "Verify its syntactic use."
            )

        return pd.Series({
            "Candidate_Gold_Tag": "SN",
            "Decision": "PROVISIONAL",
            "Review_Note": note
        })

    if word == "कोटिशः":
        return pd.Series({
            "Candidate_Gold_Tag": "AKV",
            "Decision": "MANUAL_REVIEW",
            "Review_Note": (
                "Quantitative adverbial expression. Verify "
                "whether it modifies नमः and fits AKV."
            )
        })

    return pd.Series({
        "Candidate_Gold_Tag": None,
        "Decision": "MANUAL_REVIEW",
        "Review_Note": "Requires individual linguistic review."
    })


jq_review[
    ["Candidate_Gold_Tag", "Decision", "Review_Note"]
] = jq_review.apply(get_jq_review, axis=1)

review_columns = [
    "sentence_id",
    "Token_Position",
    "Word",
    "Full_Sentence",
    "Proposed_Target_Tag",
    "Candidate_Gold_Tag",
    "Decision",
    "Review_Note",
    "Mapping_Confidence"
]

display(
    jq_review[review_columns]
    .sort_values(["sentence_id", "Token_Position"])
    .reset_index(drop=True)
)

print("\nCandidate tag distribution:")
display(jq_review["Candidate_Gold_Tag"].value_counts(dropna=False))

print("\nDecision distribution:")
display(jq_review["Decision"].value_counts())


In [ ]:

# ============================================================
# JQ EXCEPTION AUDIT — BEFORE FINAL APPROVAL
# Read-only with respect to gold Target_Tag
# ============================================================

# Sentence IDs requiring special attention
special_review_ids = [960, 1056, 1270, 1410, 1420, 1429]

special_cases = jq_review[
    jq_review["sentence_id"].isin(special_review_ids)
].copy()

# Remaining candidate rows
other_cases = jq_review[
    ~jq_review["sentence_id"].isin(special_review_ids)
].copy()

print("TOTAL JQ PROPOSALS:", len(jq_review))
print("SPECIAL REVIEW CASES:", len(special_cases))
print("OTHER CANDIDATES:", len(other_cases))

print("\n========== SPECIAL REVIEW CASES ==========")

display(
    special_cases[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence",
            "Proposed_Target_Tag",
            "Candidate_Gold_Tag",
            "Review_Note"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\n========== OTHER CANDIDATES ==========")

display(
    other_cases[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence",
            "Proposed_Target_Tag",
            "Candidate_Gold_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

# Save review sheet without changing final mappings
output_path = RESULTS_DIR / "JQ_linguistic_review_sheet.csv"

jq_review.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nReview sheet saved to:", output_path)


In [ ]:

# ============================================================
# JQ — PREPARE EXPERT APPROVAL QUEUE
# No automatic changes to Target_Tag
# ============================================================

# Special cases requiring additional review
hold_keys = {
    (960, 4),    # Gender disagreement
    (1410, 0),   # Dialogue-label context
    (1420, 5),   # Dialogue-label context
    (1429, 0),   # Dialogue-label context
    (510, 9),    # कोटिशः: individual review
}

# Create approval status column
jq_review["Expert_Approval_Status"] = "READY_FOR_REVIEW"

for idx, row in jq_review.iterrows():
    key = (row["sentence_id"], row["Token_Position"])

    if key in hold_keys:
        jq_review.at[idx, "Expert_Approval_Status"] = "HOLD"

# Add linguistic rationale for the recommended candidates
def approval_reason(row):
    tag = row["Candidate_Gold_Tag"]

    if tag == "AKV":
        return (
            "Adverbial indeclinable candidate; "
            "verify its adverbial function."
        )

    if tag == "SAMY":
        return (
            "Cardinal numeral agreeing with the associated "
            "nominal expression; verify gender and context."
        )

    if tag == "SN":
        return (
            "Independent pronoun candidate; verify that it "
            "does not modify an expressed noun."
        )

    return "Individual linguistic review required."


jq_review["Expert_Review_Reason"] = jq_review.apply(
    approval_reason, axis=1
)

print("Approval queue summary:")
display(
    jq_review.groupby(
        ["Candidate_Gold_Tag", "Expert_Approval_Status"],
        dropna=False
    ).size().reset_index(name="Occurrences")
)

print("\nCandidates ready for expert review:")
display(
    jq_review[
        jq_review["Expert_Approval_Status"] == "READY_FOR_REVIEW"
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence",
            "Candidate_Gold_Tag",
            "Mapping_Confidence",
            "Expert_Review_Reason"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

print("\nCases on hold:")
display(
    jq_review[
        jq_review["Expert_Approval_Status"] == "HOLD"
    ][
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence",
            "Candidate_Gold_Tag",
            "Review_Note"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

# Save the review queue
jq_review.to_csv(
    RESULTS_DIR / "JQ_expert_approval_queue.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: JQ_expert_approval_queue.csv")


In [ ]:
# Check current mapping status in the live runtime

mapping_status = (
    manual_review
    .assign(
        Is_Mapped=manual_review["Target_Tag"].notna()
    )
    .groupby("Source_Family")
    .agg(
        Total=("Target_Tag", "size"),
        Mapped=("Is_Mapped", "sum"),
        Unresolved=("Is_Mapped", lambda x: (~x).sum())
    )
    .sort_values("Unresolved", ascending=False)
)

display(mapping_status)

print("\nTOTAL OCCURRENCES:", len(manual_review))
print("TOTAL MAPPED:", manual_review["Target_Tag"].notna().sum())
print("TOTAL UNRESOLVED:", manual_review["Target_Tag"].isna().sum())

In [ ]:

# ============================================================
# FAMILY-WISE POS MAPPING WORKFLOW
# First test: JQ (295 occurrences)
# Reusable for all remaining source families
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

def prepare_family_review(family_name, save=True):

    required = [
        "Source_Family",
        "sentence_id",
        "Token_Position",
        "Word",
        "Source_Tag",
        "Full_Sentence",
        "Target_Tag"
    ]

    missing = [c for c in required if c not in manual_review.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Select unresolved gold occurrences from this family
    family_df = manual_review[
        (manual_review["Source_Family"] == family_name) &
        (manual_review["Target_Tag"].isna())
    ].copy()

    if family_df.empty:
        print(f"No unresolved occurrences found for {family_name}.")
        return family_df

    # Preserve existing proposal information
    proposal_cols = [
        "Proposed_Target_Tag",
        "Mapping_Reason",
        "Mapping_Confidence"
    ]

    for col in proposal_cols:
        if col not in family_df.columns:
            family_df[col] = None

    # Add review status without changing gold labels
    family_df["Review_Status"] = np.where(
        family_df["Proposed_Target_Tag"].notna(),
        "PROPOSAL_EXISTS_REVIEW_REQUIRED",
        "NEEDS_MAPPING"
    )

    # Sort for easier linguistic review
    family_df = family_df.sort_values(
        ["Source_Tag", "sentence_id", "Token_Position"]
    ).reset_index(drop=True)

    # Summary by source tag
    summary = (
        family_df.groupby("Source_Tag")
        .agg(
            Occurrences=("Word", "size"),
            Unique_Words=("Word", "nunique"),
            Existing_Proposals=("Proposed_Target_Tag",
                                lambda x: x.notna().sum())
        )
        .sort_values("Occurrences", ascending=False)
    )

    print("=" * 65)
    print(f"FAMILY-WISE REVIEW: {family_name}")
    print("=" * 65)
    print("Unresolved gold occurrences:", len(family_df))
    print("Existing proposals:",
          family_df["Proposed_Target_Tag"].notna().sum())
    print("Still needing proposals:",
          family_df["Proposed_Target_Tag"].isna().sum())
    print("Unique source tags:", family_df["Source_Tag"].nunique())
    print("\nSOURCE TAG SUMMARY")
    display(summary)

    # Save complete family review sheet
    if save:
        output_dir = Path(
            str(RESULTS_DIR) if "RESULTS_DIR" in globals()
            else "/content"
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        output_file = output_dir / f"{family_name}_family_review.csv"
        family_df.to_csv(output_file, index=False, encoding="utf-8-sig")

        print("\nReview file saved:", output_file)

    print("\nFULL OCCURRENCE REVIEW")
    display(family_df)

    return family_df


# ------------------------------------------------------------
# CHANGE THIS VALUE TO PROCESS ANOTHER FAMILY
# ------------------------------------------------------------

CURRENT_FAMILY = "JQ"

family_review = prepare_family_review(CURRENT_FAMILY)


In [ ]:
from pathlib import Path

print("Grouped review saved locally at:", Path(RESULTS_DIR) / "JQ_grouped_word_contexts.csv")


In [ ]:
from pathlib import Path

print("Grouped review saved locally at:", Path(RESULTS_DIR) / "JQ_grouped_word_contexts.csv")


In [ ]:
# ============================================================
# VERIFY EXISTING सर्वे PROPOSALS
# ============================================================

# Define the expected JQ 'सर्वे' batch mappings to resolve the NameError
jq_sarve_batch = {
    (7, 0): ("SN", "Independent plural quantifier pronoun used substantively.", "High"),
    (39, 7): ("SNV", "Quantifier pronoun qualifying an accompanying relative pronoun.", "High"),
    (51, 0): ("SN", "Independent quantifier pronoun meaning 'everyone' functioning as subject.", "High"),
    (51, 3): ("SN", "Independent quantifier pronoun meaning 'everyone' functioning as subject.", "High"),
    (52, 0): ("SN", "Independent quantifier pronoun meaning 'everyone' functioning as subject.", "High"),
    (235, 3): ("SNV", "Quantifier pronoun directly modifying first-person plural pronoun वयं.", "High"),
    (292, 0): ("SN", "Independent quantifier pronoun functioning as the subject of भवन्तु.", "High"),
    (294, 0): ("SN", "Independent quantifier pronoun functioning as the subject of भवन्तु.", "High"),
    (316, 2): ("SN", "Independent quantifier pronoun referring back to the workers.", "High"),
    (327, 1): ("SNV", "Quantifier pronoun directly modifying the honorific pronoun भवन्तः.", "High"),
    (350, 8): ("SN", "Independent quantifier pronoun functioning substantively.", "High"),
    (432, 2): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (542, 1): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (592, 2): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (599, 6): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (880, 1): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (883, 0): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (897, 0): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (1136, 1): ("SN", "Independent third-person plural pronoun used substantively.", "High"),
    (1169, 0): ("SN", "Independent third-person plural pronoun used substantively.", "High")
}

sarve_keys = list(jq_sarve_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (row["sentence_id"], row["Token_Position"])
        in sarve_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarve_keys))
print("Rows found:", len(verified))
print("\nProposal distribution:")

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

print("\nDetailed verification:")
display(
    verified[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# Compare intended proposals with existing stored proposals

comparison = []

for key, (expected_tag, reason, confidence) in jq_sarve_batch.items():

    sentence_id, token_position = key

    row = jq[
        (jq["sentence_id"] == sentence_id) &
        (jq["Token_Position"] == token_position)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sentence_id,
            "Token_Position": token_position,
            "Word": row.iloc[0]["Word"],
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

display(comparison_df[comparison_df["Match"] == False])

print("Total occurrences:", len(comparison_df))
print("Matching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

In [ ]:
# ============================================================
# JQ NEXT GROUP — प्रथमा
# Inspect contexts; no modifications.
# ============================================================

prathama_review = jq[
    jq["Word"] == "प्रथमा"
].copy()

print("Total प्रथमा occurrences:", len(prathama_review))

display(
    prathama_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 4 — प्रथमा (8 occurrences)
# Occurrence-specific proposals; gold labels remain unchanged.
# ============================================================

jq_prathama_batch = {
    (384, 1): (
        "A",
        "प्रथमा is an ordinal adjective modifying रात्रिः in नवोढयोः प्रथमा रात्रिः.",
        "High"
    ),

    (964, 8): (
        "A",
        "प्रथमा is an ordinal adjective describing भारतीयमहिला as the first Indian woman to achieve the stated result.",
        "High"
    ),

    (1364, 0): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    ),

    (1373, 5): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    ),

    (1384, 0): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    ),

    (1416, 10): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    ),

    (1423, 0): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    ),

    (1427, 0): (
        "NS",
        "प्रथमा is a character's proper name used as the speaker label in dialogue.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_prathama_batch)

# Audit
audit_proposals(jq)

print("\nप्रथमा proposals submitted:", len(jq_prathama_batch))
print("Expected distribution: A = 2, NS = 6")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY प्रथमा PROPOSALS
# Read-only verification; no changes made.
# ============================================================

prathama_keys = list(jq_prathama_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in prathama_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(prathama_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

print("\nDetailed verification:")
display(
    verified[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

# Compare stored labels against intended labels
comparison = []

for key, (expected_tag, _, _) in jq_prathama_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]
        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

In [ ]:
# ============================================================
# JQ NEXT GROUP — एकः
# Read-only inspection
# ============================================================

ekah_review = jq[
    jq["Word"] == "एकः"
].copy()

print("Total एकः occurrences:", len(ekah_review))

display(
    ekah_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 5 — एकः (14 occurrences)
# Context-based proposals; no gold labels changed.
# ============================================================

jq_ekah_batch = {
    (151, 6): (
        "A",
        "एकः modifies the masculine noun पक्षिराजः in एकः पक्षिराजः मयूरः.",
        "High"
    ),
    (182, 2): (
        "A",
        "एकः modifies the masculine noun गुणः.",
        "High"
    ),
    (185, 2): (
        "A",
        "एकः modifies the masculine noun गुणः.",
        "High"
    ),
    (716, 12): (
        "A",
        "एकः modifies the masculine noun प्राणिविशेषः.",
        "High"
    ),
    (743, 6): (
        "A",
        "एकः modifies the masculine noun मूषकः.",
        "High"
    ),
    (856, 2): (
        "A",
        "एकः modifies the masculine noun वटवृक्षः.",
        "High"
    ),
    (857, 2): (
        "A",
        "एकः modifies the masculine noun व्याधः.",
        "High"
    ),
    (869, 8): (
        "A",
        "एकः modifies the masculine noun मूषकः.",
        "High"
    ),
    (876, 2): (
        "A",
        "एकः modifies the masculine noun उत्सवः.",
        "High"
    ),
    (1059, 0): (
        "A",
        "एकः modifies the masculine noun नृपः.",
        "High"
    ),
    (1066, 2): (
        "A",
        "एकः modifies the masculine noun शिशुः.",
        "High"
    ),
    (1079, 0): (
        "A",
        "एकः modifies the masculine noun बालः.",
        "High"
    ),
    (1208, 6): (
        "A",
        "एकः modifies the masculine noun भुजः.",
        "High"
    ),
    (1406, 6): (
        "A",
        "एकः modifies the masculine noun अन्वयः in the question.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_ekah_batch)

# Audit proposals
audit_proposals(jq)

print("\nएकः proposals submitted:", len(jq_ekah_batch))
print("Expected distribution: A = 14")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY एकः PROPOSALS
# Read-only verification
# ============================================================

ekah_keys = list(jq_ekah_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in ekah_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(ekah_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_ekah_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ NEXT GROUP — एकं
# Read-only inspection
# ============================================================

ekam_review = jq[
    jq["Word"] == "एकं"
].copy()

print("Total एकं occurrences:", len(ekam_review))

display(
    ekam_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 6 — एकं (8 occurrences)
# Add proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_ekam_batch = {
    (247, 1): (
        "A",
        "एकं modifies the neuter noun फलं.",
        "High"
    ),
    (340, 0): (
        "A",
        "एकं modifies the neuter noun गृहं.",
        "High"
    ),
    (392, 7): (
        "A",
        "एकं modifies मधुरखण्डम् in the accusative construction.",
        "High"
    ),
    (770, 1): (
        "A",
        "एकं modifies the neuter noun जीपयानम्.",
        "High"
    ),
    (858, 3): (
        "A",
        "एकं modifies the neuter noun जालं.",
        "High"
    ),
    (960, 6): (
        "A",
        "एकं quantifies रजतपदकं, contrasting with द्वौ स्वर्णपदकौ.",
        "High"
    ),
    (1087, 0): (
        "A",
        "एकं modifies the masculine accusative noun खण्डं.",
        "High"
    ),
    (1333, 7): (
        "A",
        "एकं modifies the neuter noun कर्गदं.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_ekam_batch)

# Run proposal audit
audit_proposals(jq)

print("\nएकं proposals submitted:", len(jq_ekam_batch))
print("Expected distribution: A = 8")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY एकं PROPOSALS
# Read-only verification; no changes made.
# ============================================================

ekam_keys = list(jq_ekam_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in ekam_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(ekam_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_ekam_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ NEXT GROUP — एका
# Read-only inspection
# ============================================================

eka_review = jq[
    jq["Word"] == "एका"
].copy()

print("Total एका occurrences:", len(eka_review))

display(
    eka_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 7 — एका (9 occurrences)
# Occurrence-specific proposals; gold labels unchanged.
# ============================================================

jq_eka_batch = {
    (4, 2): (
        "A",
        "एका modifies the feminine noun कथा.",
        "High"
    ),
    (127, 5): (
        "A",
        "एका modifies the feminine noun महिला.",
        "High"
    ),
    (669, 8): (
        "A",
        "एका modifies the feminine noun सूची.",
        "High"
    ),
    (780, 1): (
        "A",
        "एका modifies the feminine noun नौका.",
        "High"
    ),
    (894, 2): (
        "A",
        "एका modifies the feminine noun कथा.",
        "High"
    ),
    (1064, 1): (
        "SN",
        "एका is used independently to refer to one of two women; अपरा च लता provides the contrast.",
        "High"
    ),
    (1130, 4): (
        "A",
        "एका modifies the feminine noun गुलिका, reinforced by एव.",
        "High"
    ),
    (1269, 1): (
        "A",
        "एका modifies the feminine noun घटना.",
        "High"
    ),
    (1359, 1): (
        "A",
        "एका modifies the feminine noun परिहासिका.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_eka_batch)

# Run audit
audit_proposals(jq)

print("\nएका proposals submitted:", len(jq_eka_batch))
print("Expected distribution: A = 8, SN = 1")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY एका PROPOSALS
# Read-only verification
# ============================================================

eka_keys = list(jq_eka_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in eka_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(eka_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_eka_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ NEXT GROUP — एकस्मिन्
# Read-only inspection
# ============================================================

ekasmin_review = jq[
    jq["Word"] == "एकस्मिन्"
].copy()

print("Total एकस्मिन् occurrences:", len(ekasmin_review))

display(
    ekasmin_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 8 — एकस्मिन् (5 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_ekasmin_batch = {
    (492, 3): (
        "A",
        "एकस्मिन् modifies the locative noun विद्यालये.",
        "High"
    ),
    (927, 2): (
        "A",
        "एकस्मिन् modifies the locative noun महापात्रे.",
        "High"
    ),
    (968, 3): (
        "A",
        "एकस्मिन् modifies the locative noun वर्षे.",
        "High"
    ),
    (1063, 0): (
        "A",
        "एकस्मिन् modifies the locative noun दिवसे.",
        "High"
    ),
    (1458, 0): (
        "A",
        "एकस्मिन् modifies the locative noun समये.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_ekasmin_batch)

# Audit proposals
audit_proposals(jq)

print("\nएकस्मिन् proposals submitted:", len(jq_ekasmin_batch))
print("Expected distribution: A = 5")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY एकस्मिन् PROPOSALS
# Read-only verification; no changes made.
# ============================================================

ekasmin_keys = list(jq_ekasmin_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in ekasmin_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(ekasmin_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_ekasmin_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ NEXT GROUP — सर्वं
# Read-only inspection
# ============================================================

sarvam_review = jq[
    jq["Word"] == "सर्वं"
].copy()

print("Total सर्वं occurrences:", len(sarvam_review))

display(
    sarvam_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 9 — सर्वं (9 occurrences)
# Occurrence-specific proposals; gold labels unchanged.
# ============================================================

jq_sarvam_batch = {
    (544, 4): (
        "SN",
        "सर्वं is used substantively as the object of वदन्, referring to everything being said.",
        "Medium"
    ),
    (688, 28): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (694, 12): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (702, 3): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (722, 1): (
        "SN",
        "सर्वं is used independently in सर्वं साधयितुं शक्यम्, meaning everything can be accomplished.",
        "High"
    ),
    (1293, 6): (
        "SN",
        "सर्वं is used substantively as the object of त्यक्तवान्, meaning everything was renounced.",
        "High"
    ),
    (1321, 9): (
        "SN",
        "सर्वं refers to the totality expressed by तत् तत् and functions substantively.",
        "Medium"
    ),
    (1450, 10): (
        "SN",
        "सर्वं refers collectively to the previously mentioned भूमिः, जलं and वायुः.",
        "High"
    ),
    (1455, 9): (
        "SN",
        "सर्वं collectively refers to भूमिः, जलं and वायुः rather than modifying one individual noun.",
        "Medium"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_sarvam_batch)

# Audit proposals
audit_proposals(jq)

print("\nसर्वं proposals submitted:", len(jq_sarvam_batch))
print("Expected distribution: A = 3, SN = 6")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# JQ BATCH 9 — सर्वं (9 occurrences)
# Occurrence-specific proposals; gold labels unchanged.
# ============================================================

# Redefining apply_occurrence_proposals for safety
def apply_occurrence_proposals(
    df,
    proposals,
    family=None,
    overwrite=False,
    target_column="Proposed_Target_Tag"
):
    required = {"sentence_id", "Token_Position", target_column,
                "Mapping_Reason", "Mapping_Confidence"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing columns: {sorted(missing_cols)}")

    for _column in (target_column, "Mapping_Reason", "Mapping_Confidence"):
        df[_column] = df[_column].astype(object)

    validated = []
    errors = []

    for key, value in proposals.items():
        sid, pos = key
        tag, reason, confidence = value
        if tag not in TARGET_TAGS:
            errors.append(f"Invalid target tag {tag!r} for {key}")
        if confidence not in VALID_CONFIDENCE:
            errors.append(f"Invalid confidence {confidence!r} for {key}")

        mask = (
            (df["sentence_id"] == sid) &
            (df["Token_Position"] == pos)
        )
        count = int(mask.sum())
        if count != 1:
            errors.append(f"{key} matched {count} rows")
            continue

        validated.append((mask, tag, reason.strip(), confidence, key))

    if errors:
        raise ValueError("Validation issues:\n- " + "\n- ".join(errors))

    for mask, tag, reason, confidence, key in validated:
        df.loc[mask, target_column] = tag
        df.loc[mask, "Mapping_Reason"] = reason
        df.loc[mask, "Mapping_Confidence"] = confidence

    return len(validated)

# Define the JQ Batch 9 proposals
jq_sarvam_batch = {
    (544, 4): (
        "SN",
        "सर्वं is used substantively as the object of वदन्, referring to everything being said.",
        "Medium"
    ),
    (688, 28): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (694, 12): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (702, 3): (
        "A",
        "सर्वं modifies the expressed noun धनं.",
        "High"
    ),
    (722, 1): (
        "SN",
        "सर्वं is used independently in सर्वं साधयितुं शक्यम्, meaning everything can be accomplished.",
        "High"
    ),
    (1293, 6): (
        "SN",
        "सर्वं is used substantively as the object of त्यक्तवान्, meaning everything was renounced.",
        "High"
    ),
    (1321, 9): (
        "SN",
        "सर्वं refers to the totality expressed by तत् तत् and functions substantively.",
        "Medium"
    ),
    (1450, 10): (
        "SN",
        "सर्वं refers collectively to the previously mentioned भूमिः, जलं and वायुः.",
        "High"
    ),
    (1455, 9): (
        "SN",
        "सर्वं collectively refers to भूमिः, जलं and वायुः rather than modifying one individual noun.",
        "Medium"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_sarvam_batch)

print("\nसर्वं proposals submitted successfully:", len(jq_sarvam_batch))

In [ ]:
# ============================================================
# VERIFY JQ BATCH 9 — सर्वं
# Read-only verification
# ============================================================

sarvam_keys = list(jq_sarvam_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarvam_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarvam_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

# Compare expected and stored proposals
comparison = []

for key, (expected_tag, _, _) in jq_sarvam_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

print("\nDetailed verification:")
display(comparison_df)

# Confirm gold labels remain unchanged
print(
    "\nGold Target_Tag values still missing:",
    verified["Target_Tag"].isna().sum(),
    "out of",
    len(verified)
)

In [ ]:
# ============================================================
# JQ NEXT GROUP — सर्वान्
# Read-only inspection
# ============================================================

sarvan_review = jq[
    jq["Word"] == "सर्वान्"
].copy()

print("Total सर्वान् occurrences:", len(sarvan_review))

display(
    sarvan_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 10 — सर्वान् (3 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_sarvan_batch = {
    (499, 8): (
        "SN",
        "सर्वान् is used substantively with तान्, referring to the previously mentioned people without an expressed noun.",
        "Medium"
    ),
    (935, 4): (
        "A",
        "सर्वान् modifies the expressed masculine plural accusative noun पशून्.",
        "High"
    ),
    (992, 3): (
        "A",
        "सर्वान् modifies the expressed masculine plural accusative noun जनान्.",
        "High"
    )
}

# Apply proposals using the currently available workbench function
apply_occurrence_proposals(jq, jq_sarvan_batch)

print("\nसर्वान् proposals submitted:", len(jq_sarvan_batch))
print("Expected distribution: A = 2, SN = 1")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 10 — सर्वान्
# Read-only verification
# ============================================================

sarvan_keys = list(jq_sarvan_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarvan_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarvan_keys))
print("Rows found:", len(verified))

print("\nProposal distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarvan_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    if len(row) == 1:
        actual_tag = row.iloc[0]["Proposed_Target_Tag"]

        comparison.append({
            "sentence_id": sid,
            "Token_Position": pos,
            "Expected_Tag": expected_tag,
            "Existing_Tag": actual_tag,
            "Match": expected_tag == actual_tag
        })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ NEXT GROUP — एकम्
# Read-only inspection
# ============================================================

ekam2_review = jq[
    jq["Word"] == "एकम्"
].copy()

print("Total एकम् occurrences:", len(ekam2_review))

display(
    ekam2_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Full_Sentence",
            "Target_Tag",
            "Proposed_Target_Tag",
            "Mapping_Confidence"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 11 — एकम् (5 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_ekam_word_batch = {
    (77, 0): (
        "A",
        "एकम् modifies the neuter noun गीतम्.",
        "High"
    ),
    (324, 15): (
        "A",
        "एकम् modifies the neuter noun अवसरं in एकम् उत्तमम् अवसरं.",
        "High"
    ),
    (637, 2): (
        "A",
        "एकम् modifies the neuter noun पत्रम्.",
        "High"
    ),
    (1083, 2): (
        "A",
        "एकम् modifies the masculine accusative noun आरक्षकम्.",
        "High"
    ),
    (1239, 4): (
        "A",
        "एकम् modifies the neuter noun उत्तरम्.",
        "High"
    )
}

# Apply proposals
apply_occurrence_proposals(jq, jq_ekam_word_batch)

print("\nएकम् proposals submitted:", len(jq_ekam_word_batch))
print("Expected distribution: A = 5")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# JQ BATCH 12 — INSPECT सर्वासु
# ============================================================

sarvasu_review = jq[jq["Word"] == "सर्वासु"].copy()

print("Total occurrences:", len(sarvasu_review))

display(
    sarvasu_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 12 — सर्वासु (3 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_sarvasu_batch = {
    (3, 1): (
        "A",
        "सर्वासु qualifies the feminine plural locative noun नदीषु.",
        "High"
    ),
    (893, 1): (
        "A",
        "सर्वासु qualifies the feminine plural locative noun नदीषु.",
        "High"
    ),
    (1323, 3): (
        "A",
        "सर्वासु qualifies the feminine plural locative noun क्रियासु.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_sarvasu_batch)

print("सर्वासु proposals submitted:", len(jq_sarvasu_batch))
print("Expected distribution: A = 3")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 12 — सर्वासु
# ============================================================

sarvasu_keys = list(jq_sarvasu_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarvasu_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarvasu_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarvasu_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ BATCH 13 — INSPECT द्वितीया
# ============================================================

dvitiiya_review = jq[jq["Word"] == "द्वितीया"].copy()

print("Total occurrences:", len(dvitiiya_review))

display(
    dvitiiya_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 13 — द्वितीया (4 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_dvitiya_batch = {
    (1370, 0): (
        "NS",
        "द्वितीया is used as a speaker's proper name before the stage direction तपस्विनी.",
        "High"
    ),
    (1379, 0): (
        "NS",
        "द्वितीया is used as a speaker's proper name in the dialogue label.",
        "High"
    ),
    (1411, 0): (
        "NS",
        "द्वितीया is used as a speaker's proper name in the dialogue label.",
        "High"
    ),
    (1432, 5): (
        "NS",
        "द्वितीया is used as a speaker's proper name in the dialogue label.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_dvitiya_batch)

print("द्वितीया proposals submitted:", len(jq_dvitiya_batch))
print("Expected distribution: NS = 4")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 13 — द्वितीया
# ============================================================

dvitiya_keys = list(jq_dvitiya_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in dvitiya_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(dvitiya_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_dvitiya_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)


In [ ]:
# ============================================================
# JQ BATCH 14 — INSPECT सर्वेभ्यः
# ============================================================

sarvebhyah_review = jq[jq["Word"] == "सर्वेभ्यः"].copy()

print("Total occurrences:", len(sarvebhyah_review))

display(
    sarvebhyah_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 14 — सर्वेभ्यः (2 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_sarvebhyah_batch = {
    (992, 6): (
        "SN",
        "सर्वेभ्यः is used substantively, referring to all people who are pleased by ऋतुराजः.",
        "High"
    ),
    (1305, 6): (
        "SN",
        "सर्वेभ्यः is used substantively, referring to all of us who receive peace.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_sarvebhyah_batch)

print("सर्वेभ्यः proposals submitted:", len(jq_sarvebhyah_batch))
print("Expected distribution: SN = 2")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 14 — सर्वेभ्यः
# ============================================================

sarvebhyah_keys = list(jq_sarvebhyah_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarvebhyah_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarvebhyah_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarvebhyah_batch.items():
    sid, pos = key
    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ BATCH 15 — INSPECT सर्वैः
# ============================================================

sarvaih_review = jq[jq["Word"] == "सर्वैः"].copy()

print("Total occurrences:", len(sarvaih_review))

display(
    sarvaih_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 15 — सर्वैः (2 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_sarvaih_batch = {
    (1270, 3): (
        "SN",
        "सर्वैः is used substantively to refer to all people in the instrumental plural.",
        "Medium"
    ),
    (1451, 14): (
        "A",
        "सर्वैः qualifies the pronoun तैः in तैः सर्वैः.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_sarvaih_batch)

print("सर्वैः proposals submitted:", len(jq_sarvaih_batch))
print("Expected distribution: SN = 1, A = 1")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 15 — सर्वैः
# ============================================================

sarvaih_keys = list(jq_sarvaih_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarvaih_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarvaih_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarvaih_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ BATCH 16 — INSPECT सर्वेषु
# ============================================================

sarveshu_review = jq[jq["Word"] == "सर्वेषु"].copy()

print("Total occurrences:", len(sarveshu_review))

display(
    sarveshu_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 16 — सर्वेषु (2 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_sarveshu_batch = {
    (885, 2): (
        "A",
        "सर्वेषु qualifies the locative plural noun विद्यालयेषु and extends to कार्यालयेषु.",
        "High"
    ),
    (972, 4): (
        "A",
        "सर्वेषु qualifies the locative plural noun ऋतुषु.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_sarveshu_batch)

print("सर्वेषु proposals submitted:", len(jq_sarveshu_batch))
print("Expected distribution: A = 2")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# VERIFY JQ BATCH 16 — सर्वेषु
# ============================================================

sarveshu_keys = list(jq_sarveshu_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarveshu_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarveshu_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarveshu_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ BATCH 17 — INSPECT अधिकः
# ============================================================

adhikah_review = jq[jq["Word"] == "अधिकः"].copy()

print("Total occurrences:", len(adhikah_review))

display(
    adhikah_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 17 — अधिकः (2 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_adhikah_batch = {
    (309, 3): (
        "A",
        "अधिकः qualifies the masculine singular noun आयः.",
        "High"
    ),
    (658, 7): (
        "A",
        "अधिकः qualifies the masculine singular noun धनमदः.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_adhikah_batch)

print("अधिकः proposals submitted:", len(jq_adhikah_batch))
print("Expected distribution: A = 2")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# COUNT REMAINING JQ WORD BATCHES
# ============================================================

pending = jq[jq["Proposed_Target_Tag"].isna()].copy()

pending_words = sorted(pending["Word"].dropna().unique())

print("Total unresolved occurrences:", len(pending))
print("Unique words with pending occurrences:", len(pending_words))

print("\nRemaining words:")
for i, word in enumerate(pending_words, 1):
    count = (pending["Word"] == word).sum()
    print(f"{i}. {word} — {count} occurrences")

In [ ]:
# Check previously processed words
previous_words = [
    "सर्वे", "प्रथमा", "एकः", "एकं", "एका",
    "एकस्मिन्", "सर्वं", "सर्वान्", "एकम्",
    "सर्वासु", "द्वितीया", "सर्वेभ्यः",
    "सर्वैः", "सर्वेषु", "अधिकः"
]

for word in previous_words:
    subset = jq[jq["Word"] == word]

    print(
        f"{word}: "
        f"Total={len(subset)}, "
        f"Proposed={subset['Proposed_Target_Tag'].notna().sum()}, "
        f"Unresolved={subset['Proposed_Target_Tag'].isna().sum()}"
    )

In [ ]:
# ============================================================
# RESTORE CHECK — सर्वे
# ============================================================

sarve_review = jq[jq["Word"] == "सर्वे"].copy()

print("Total occurrences:", len(sarve_review))

display(
    sarve_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# सर्वे — REVIEW OCCURRENCES BEFORE RESTORING PROPOSALS
# ============================================================

sarve_review = jq[jq["Word"] == "सर्वे"].copy()

for _, row in sarve_review.sort_values(
    ["sentence_id", "Token_Position"]
).iterrows():

    print(
        f"\nSentence ID: {row['sentence_id']}"
        f"\nPosition: {row['Token_Position']}"
        f"\nSentence: {row['Full_Sentence']}"
    )

In [ ]:
# ============================================================
# RESTORE JQ BATCH 3 — सर्वे (20 occurrences)
# ============================================================

jq_sarve_batch = {
    (292, 0): ("SN", "Independent subject meaning everyone.", "High"),
    (294, 0): ("SN", "Independent subject meaning everyone.", "High"),
    (334, 13): ("SN", "Independent subject of प्राप्तवन्तः.", "High"),
    (440, 3): ("SN", "Independent subject of आगमिष्यन्ति.", "High"),
    (499, 2): ("SN", "Independent subject referring to all people.", "High"),
    (771, 1): ("A", "Qualifies ग्रामजनाः.", "High"),
    (862, 0): ("A", "Qualifies कपोताः.", "High"),
    (864, 10): ("SN", "Independent substantive use referring to यूयम्.", "High"),
    (865, 1): ("A", "Qualifies कपोताः.", "High"),
    (871, 3): ("A", "Qualifies कपोताः.", "High"),
    (873, 1): ("A", "Qualifies कपोताः.", "High"),
    (882, 0): ("A", "Qualifies जनाः.", "High"),
    (886, 1): ("A", "Qualifies जनाः.", "High"),
    (931, 5): ("A", "Qualifies पशवः.", "High"),
    (937, 3): ("A", "Qualifies पशवः.", "High"),
    (939, 10): ("A", "Qualifies शृगालाः.", "High"),
    (1107, 0): ("SN", "Independent subject of प्रशंसितवन्तः.", "High"),
    (1146, 0): ("A", "Qualifies जनाः.", "High"),
    (1254, 3): ("SN", "Independent substantive use referring to वयं.", "High"),
    (1303, 7): ("SN", "Independent substantive use referring to वयं.", "High")
}

apply_occurrence_proposals(jq, jq_sarve_batch)

print("सर्वे proposals submitted:", len(jq_sarve_batch))
print("Expected distribution: A = 11, SN = 9")
print("Gold Target_Tag values remain unchanged.")

In [ ]:
# ============================================================
# VERIFY RESTORED BATCH 3 — सर्वे
# ============================================================

sarve_keys = list(jq_sarve_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in sarve_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(sarve_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_sarve_batch.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# VERIFY JQ BATCH 17 — अधिकः
# ============================================================

adhikah_keys = list(jq_adhikah_batch.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in adhikah_keys,
        axis=1
    )
].copy()

print("Expected proposals:", len(adhikah_keys))
print("Rows found:", len(verified))

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

comparison = []

for key, (expected_tag, _, _) in jq_adhikah_batch.items():
    sid, pos = key
    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

In [ ]:
# ============================================================
# JQ BATCH 18 — INSPECT अग्रिमे
# ============================================================

agrime_review = jq[jq["Word"] == "अग्रिमे"].copy()

print("Total occurrences:", len(agrime_review))

display(
    agrime_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence",
            "Proposed_Target_Tag"
        ]
    ].sort_values(["sentence_id", "Token_Position"])
)

In [ ]:
# ============================================================
# JQ BATCH 18 — अग्रिमे (4 occurrences)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_agrime_batch = {
    (310, 0): (
        "A",
        "अग्रिमे qualifies the locative singular noun वर्षे.",
        "High"
    ),
    (659, 10): (
        "A",
        "अग्रिमे qualifies the locative singular noun जन्मनि.",
        "High"
    ),
    (661, 7): (
        "A",
        "अग्रिमे qualifies the locative singular noun जन्मनि.",
        "High"
    ),
    (665, 23): (
        "A",
        "अग्रिमे qualifies the locative singular noun जन्मनि.",
        "High"
    )
}

apply_occurrence_proposals(jq, jq_agrime_batch)

print("अग्रिमे proposals submitted:", len(jq_agrime_batch))
print("Expected distribution: A = 4")
print("Gold Target_Tag values have not been committed.")

In [ ]:
# ============================================================
# GROUP-BASED ANNOTATION — BATCH 1
# Extract next 50 unresolved occurrences
# ============================================================

pending = jq[jq["Proposed_Target_Tag"].isna()].copy()

pending = pending.sort_values(
    ["sentence_id", "Token_Position"]
)

batch_1 = pending.head(50).copy()

print("Batch size:", len(batch_1))
print("Unique words:", batch_1["Word"].nunique())

display(
    batch_1[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].reset_index(drop=True)
)

print("\nWord frequency in this batch:")

display(
    batch_1.groupby("Word")
    .size()
    .reset_index(name="Occurrences")
    .sort_values("Occurrences", ascending=False)
)

In [ ]:
# ============================================================
# GROUP ANNOTATION — BATCH 1 (50 OCCURRENCES)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_group_batch_1 = {

    (8, 0): ("A", "Qualifies श्रद्धालुजनाः.", "High"),
    (9, 3): ("A", "Qualifies तीर्थस्थानानि.", "High"),
    (10, 3): ("SN", "Quantity expression used substantively.", "Medium"),
    (13, 4): ("A", "Qualifies अन्नम्.", "High"),
    (14, 4): ("A", "Qualifies उपकारम्.", "High"),

    (21, 0): ("A", "Qualifies वर्षे.", "High"),
    (23, 0): ("A", "Qualifies वर्षे.", "High"),
    (35, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (36, 4): ("A", "Ordinal qualifies वर्षे.", "High"),
    (41, 4): ("A", "Ordinal qualifies वर्षे.", "High"),

    (61, 3): ("AVU", "Used adverbially to mean not more.", "Medium"),
    (81, 0): ("A", "Qualifies शृगालः.", "High"),
    (102, 2): ("A", "Qualifies नाम.", "High"),
    (134, 5): ("A", "Qualifies पत्राणि.", "High"),

    (140, 7): ("A", "Qualifies सन्देशे.", "High"),
    (140, 13): ("A", "Qualifies सेवकः.", "High"),

    (153, 0): ("A", "Numeral qualifies हंसाः.", "High"),
    (154, 0): ("A", "Numeral qualifies काकाः.", "High"),
    (155, 0): ("A", "Numeral qualifies कोकिलाः.", "High"),
    (156, 0): ("A", "Numeral qualifies कुक्कुटाः.", "High"),
    (157, 0): ("A", "Numeral qualifies शुकाः.", "High"),
    (158, 0): ("A", "Numeral qualifies कपोताः.", "High"),
    (159, 0): ("A", "Numeral qualifies सारिकाः.", "High"),
    (160, 0): ("A", "Numeral qualifies उल्लूकाः.", "High"),

    (162, 2): ("A", "Numeral qualifies छात्रौ.", "High"),
    (217, 1): ("A", "Quantifies पुत्राः.", "Medium"),

    (266, 2): ("A", "Numeral qualifies नेत्राणि.", "High"),
    (266, 8): ("A", "Ordinal/numeral qualifies नेत्राणि.", "High"),

    (297, 3): ("A", "Qualifies घोषणाम्.", "High"),
    (307, 8): ("A", "Predicative adjective describing वेतनम्.", "High"),

    (308, 1): ("A", "Qualifies करशुल्कम्.", "High"),
    (308, 4): ("A", "Qualifies धनम्.", "High"),
    (310, 2): ("A", "Qualifies आयम्.", "High"),

    (310, 23): ("A", "Qualifies आयः.", "High"),
    (310, 27): ("A", "Predicative adjective describing the increase.", "High"),

    (313, 3): ("AVU", "Indeclinable relative expression.", "Medium"),
    (313, 6): ("SN", "Substantive meaning half.", "Medium"),

    (320, 0): ("A", "Qualifies कूपानाम्.", "High"),
    (328, 17): ("A", "Quantifies सुवर्णनाणकानि.", "Medium"),
    (338, 0): ("AVU", "Adverbial use meaning further/ahead.", "High"),

    (343, 4): ("A", "Qualifies अनादरः.", "High"),
    (347, 0): ("A", "Qualifies कन्याः.", "High"),
    (365, 5): ("A", "Predicative adjective meaning great.", "High"),

    (369, 1): ("A", "Qualifies आर्थिकसामर्थ्यम्.", "High"),
    (372, 1): ("SN", "Substantive quantity expression.", "Medium"),

    (378, 6): ("SN", "Substantive plural expression.", "Medium"),
    (380, 1): ("A", "Expresses sufficient quantity of water.", "Medium"),
    (386, 14): ("SN", "Substantive genitive plural expression.", "Medium"),

    (389, 0): ("A", "Qualifies निद्रा.", "High"),
    (391, 4): ("SN", "Substantive genitive plural expression.", "Medium")
}

print("Prepared proposals:", len(jq_group_batch_1))
print("Expected occurrences: 50")

In [ ]:
apply_occurrence_proposals(jq, jq_group_batch_1)

print("Batch 1 proposals submitted:", len(jq_group_batch_1))
print("Gold Target_Tag remains unchanged.")

In [ ]:
# ============================================================
# VERIFY GROUP ANNOTATION — BATCH 1
# ============================================================

batch_keys = list(jq_group_batch_1.keys())

verified = jq[
    jq.apply(
        lambda row: (
            row["sentence_id"],
            row["Token_Position"]
        ) in batch_keys,
        axis=1
    )
].copy()

print("Expected occurrences:", len(batch_keys))
print("Rows found:", len(verified))

# Check tag distribution
print("\nProposed tag distribution:")
display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

# Check every occurrence individually
comparison = []

for key, (expected_tag, _, _) in jq_group_batch_1.items():
    sid, pos = key

    row = jq[
        (jq["sentence_id"] == sid) &
        (jq["Token_Position"] == pos)
    ]

    actual_tag = (
        row.iloc[0]["Proposed_Target_Tag"]
        if len(row) == 1 else None
    )

    comparison.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Expected_Tag": expected_tag,
        "Existing_Tag": actual_tag,
        "Match": expected_tag == actual_tag
    })

comparison_df = pd.DataFrame(comparison)

print("\nMatching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(comparison_df)

# Check gold labels were not changed
gold_unchanged = verified["Target_Tag"].isna().all()

print("\nGold Target_Tag remains unchanged:", gold_unchanged)

In [ ]:
# ============================================================
# GROUP ANNOTATION — BATCH 2
# Extract next 50 unresolved occurrences
# ============================================================

pending = jq[jq["Proposed_Target_Tag"].isna()].copy()

pending = pending.sort_values(
    ["sentence_id", "Token_Position"]
)

batch_2 = pending.head(50).copy()

print("Batch size:", len(batch_2))
print("Unique words:", batch_2["Word"].nunique())

display(
    batch_2[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].reset_index(drop=True)
)

print("\nWord frequency:")

display(
    batch_2.groupby("Word")
    .size()
    .reset_index(name="Occurrences")
    .sort_values("Occurrences", ascending=False)
)

In [ ]:
# ============================================================
# GROUP ANNOTATION — BATCH 2 (50 OCCURRENCES)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_group_batch_2 = {

    (415, 0): ("A", "Qualifies स्थानम्.", "High"),
    (422, 7): ("A", "Qualifies स्थानम्.", "High"),
    (437, 1): ("A", "Qualifies वर्गे.", "High"),
    (438, 0): ("A", "Qualifies वर्षम्.", "High"),

    (442, 4): ("A", "Qualifies पुत्रः.", "High"),
    (443, 6): ("A", "Qualifies शब्देन.", "High"),
    (451, 0): ("A", "Qualifies रात्रौ.", "High"),

    (462, 2): ("AVU", "Quantity expression modifying वक्तुम्.", "Medium"),
    (468, 5): ("AVU", "Modifies भाषते adverbially.", "High"),
    (469, 1): ("SN", "Substantive use meaning all this.", "Medium"),
    (470, 3): ("A", "Qualifies मौनम्.", "High"),

    (475, 0): ("A", "Qualifies मनःस्थितिः.", "High"),
    (477, 0): ("A", "Qualifies रात्रौ.", "High"),
    (498, 0): ("A", "Qualifies कष्टजातम्.", "High"),
    (513, 1): ("AVU", "Adverbial use meaning forward.", "High"),

    (637, 9): ("A", "Quantifies रुप्यकाणि.", "High"),
    (641, 2): ("A", "Qualifies पत्रेण.", "High"),
    (651, 3): ("A", "Qualifies गर्वः.", "High"),
    (652, 2): ("A", "Qualifies सम्पत्तिः.", "High"),
    (653, 6): ("A", "Quantifies पताकाः.", "High"),

    (659, 2): ("A", "Qualifies सूचीम्.", "High"),
    (665, 16): ("A", "Qualifies सूचीम्.", "High"),
    (669, 17): ("A", "Qualifies सम्पत्तिम्.", "High"),

    (680, 1): ("AVU", "Modifies चतुरः adverbially.", "High"),
    (683, 10): ("A", "Qualifies धनम्.", "High"),
    (697, 2): ("A", "Qualifies चोरभीतिः.", "High"),
    (703, 6): ("AVU", "Adverbial use meaning completely.", "Medium"),

    (725, 2): ("A", "Numeral quantifies मार्जाराः.", "High"),
    (748, 4): ("AVU", "Correlative quantity expression in यावत्...तावत्.", "Medium"),

    (833, 2): ("A", "Qualifies वर्षे.", "High"),
    (833, 7): ("A", "Quantifies छात्राः.", "High"),
    (836, 2): ("A", "Quantifies संस्कृतानुरागिणः.", "Medium"),
    (838, 5): ("A", "Qualifies देशे.", "High"),
    (840, 4): ("A", "Ordinal qualifies वर्षे.", "High"),
    (841, 1): ("AVU", "Temporal adverb.", "High"),

    (845, 5): ("A", "Qualifies विषयाणां.", "High"),
    (849, 0): ("A", "Qualifies देशः.", "High"),
    (861, 2): ("A", "Quantifies कपोताः.", "High"),

    (897, 3): ("AVU", "Modifies पावनम् adverbially.", "High"),
    (898, 0): ("A", "Qualifies श्रद्धालुजनाः.", "High"),
    (899, 3): ("A", "Qualifies तीर्थस्थानानि.", "High"),
    (900, 3): ("A", "Quantifies जनाः.", "Medium"),
    (903, 4): ("A", "Qualifies अन्नम्.", "High"),

    (907, 5): ("SN", "Substantive use meaning both.", "Medium"),

    (945, 2): ("A", "Ordinal qualifies वर्षे.", "High"),
    (945, 6): ("A", "Ordinal qualifies तारिकायाम्.", "High"),
    (948, 1): ("A", "Ordinal qualifies वर्षे.", "High"),

    (954, 2): ("A", "Qualifies प्रशिक्षकः.", "High"),
    (954, 10): ("AVU", "Modifies प्राशंसत् adverbially.", "High"),
    (956, 3): ("A", "Qualifies भारोत्तोलकेन.", "High")
}

print("Prepared proposals:", len(jq_group_batch_2))
print("Expected occurrences: 50")

In [ ]:
# Inspect the two unmatched occurrences
for sid, pos in [(415, 0), (948, 1)]:
    print(f"\nLooking for sentence_id={sid}, Token_Position={pos}")

    rows = jq[jq["sentence_id"] == sid]

    if rows.empty:
        print("Sentence ID not found in jq.")
    else:
        display(
            rows[
                [
                    "sentence_id",
                    "Token_Position",
                    "Word",
                    "Full_Sentence",
                    "Proposed_Target_Tag"
                ]
            ]
        )

In [ ]:
# Remove incorrect keys
jq_group_batch_2.pop((415, 0))
jq_group_batch_2.pop((948, 1))

# Add corrected keys with the same proposals
jq_group_batch_2[(415, 2)] = (
    "A", "Qualifies स्थानम्.", "High"
)

jq_group_batch_2[(948, 3)] = (
    "A", "Ordinal qualifies वर्षे.", "High"
)

print("Total proposals:", len(jq_group_batch_2))
print("Corrected keys:", (415, 2), (948, 3))

In [ ]:
apply_occurrence_proposals(jq, jq_group_batch_2)

print("Batch 2 proposals submitted:", len(jq_group_batch_2))

In [ ]:
batch_keys = list(jq_group_batch_2.keys())

verified = jq[
    jq.apply(
        lambda row: (row["sentence_id"], row["Token_Position"]) in batch_keys,
        axis=1
    )
].copy()

print("Expected occurrences:", len(batch_keys))
print("Rows found:", len(verified))

comparison = []

for key, (expected_tag, reason, confidence) in jq_group_batch_2.items():
    rows = verified[
        (verified["sentence_id"] == key[0]) &
        (verified["Token_Position"] == key[1])
    ]

    actual_tag = (
        rows.iloc[0]["Proposed_Target_Tag"]
        if len(rows) == 1 else "MISSING/DUPLICATE"
    )

    comparison.append({
        "Key": key,
        "Expected": expected_tag,
        "Actual": actual_tag,
        "Match": actual_tag == expected_tag
    })

comparison_df = pd.DataFrame(comparison)

print("Matching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(verified.groupby("Proposed_Target_Tag").size().reset_index(name="Count"))
display(comparison_df[~comparison_df["Match"]])

print("Gold Target_Tag unchanged:", verified["Target_Tag"].isna().all())

In [ ]:
pending = jq[jq["Proposed_Target_Tag"].isna()].copy()

pending = pending.sort_values(
    ["sentence_id", "Token_Position"]
)

batch_3 = pending.head(50).copy()

print("Batch 3 occurrences:", len(batch_3))
print("Unique words:", batch_3["Word"].nunique())

display(
    batch_3[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].reset_index(drop=True)
)

In [ ]:
# ============================================================
# GROUP ANNOTATION — BATCH 3 (50 OCCURRENCES)
# Proposals only; gold Target_Tag remains unchanged.
# ============================================================

jq_group_batch_3 = {

    # Ordinal years and numerals
    (959, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (960, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (961, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (961, 4): ("A", "Numeral quantifies स्वर्णपदकान्.", "High"),
    (962, 6): ("A", "Numeral quantifies स्वर्णपदकान्.", "High"),
    (964, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (968, 17): ("A", "Numeral quantifies ऋतवः.", "High"),

    # Quantifiers and adjectives
    (972, 8): ("SN", "Substantive genitive meaning of all.", "Medium"),
    (973, 6): ("A", "Qualifies नामानि.", "High"),
    (975, 0): ("A", "Qualifies धरा.", "High"),
    (984, 0): ("A", "Qualifies संगीतकार्यक्रमाः.", "High"),
    (986, 2): ("A", "Ordinal qualifies तिथौ.", "High"),
    (989, 2): ("A", "Qualifies स्थानेषु.", "High"),
    (991, 4): ("A", "Qualifies धराम्.", "High"),
    (992, 8): ("AVU", "Degree adverb modifying रोचते.", "High"),
    (994, 3): ("A", "Qualifies शरीरम्.", "High"),

    # Mahattara and quantity expressions
    (1006, 0): ("A", "Qualifies भिक्षाम्.", "High"),
    (1010, 1): ("A", "Qualifies भिक्षाम्.", "High"),
    (1011, 1): ("A", "Qualifies भिक्षाम्.", "High"),
    (1012, 0): ("SN", "Substantive compound meaning a thousand cows.", "Medium"),
    (1016, 1): ("SN", "Substantive quantity expression meaning many thousands.", "Medium"),
    (1023, 0): ("A", "Qualifies कनकम्.", "High"),
    (1057, 7): ("A", "Qualifies विश्वस्य.", "High"),
    (1064, 3): ("SN", "Substantive use meaning the other one.", "Medium"),
    (1085, 9): ("SN", "Substantive expression meaning two pieces.", "Medium"),
    (1086, 7): ("SN", "Substantive expression meaning two pieces.", "Medium"),
    (1087, 3): ("SN", "Substantive use meaning the other part.", "Medium"),
    (1097, 11): ("A", "Predicate adjective qualifying जीवनम्.", "High"),
    (1105, 7): ("SN", "Substantive use meaning that is sufficient.", "Medium"),

    # Adverbial expressions
    (1107, 3): ("AVU", "Adverb meaning more or further.", "High"),
    (1116, 5): ("AVU", "Frequency expression meaning fifteen times.", "High"),

    # Numerals and adjectives
    (1125, 3): ("A", "Ordinal qualifies दिवसः.", "High"),
    (1128, 5): ("A", "Qualifies सैनिकान्.", "High"),
    (1129, 6): ("A", "Qualifies सैनिकाः.", "High"),
    (1206, 1): ("A", "Numeral quantifies फणान्.", "High"),
    (1257, 1): ("A", "Ordinal qualifies वर्षे.", "High"),
    (1260, 8): ("A", "Distributive adjective qualifying भावस्य and कर्मणः.", "Medium"),
    (1260, 12): ("A", "Qualifies अंशानाम्.", "High"),
    (1263, 2): ("AVU", "Degree adverb modifying विकसितम्.", "High"),
    (1269, 4): ("A", "Qualifies कौशलेन.", "High"),

    # Dates and ordinal expressions
    (1273, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (1273, 3): ("A", "Ordinal qualifies तारिकायाम्.", "High"),
    (1283, 4): ("A", "Ordinal qualifies तले.", "High"),
    (1299, 0): ("A", "Ordinal qualifies वर्षे.", "High"),
    (1299, 3): ("A", "Ordinal qualifies दिवसे.", "High"),
    (1302, 2): ("A", "Ordinal qualifies वर्षे.", "High"),

    # Four purusharthas
    (1308, 0): ("A", "Numeral quantifies पुरुषार्थाः.", "High"),
    (1309, 6): ("A", "Numeral quantifies पुरुषार्थाः.", "High"),
    (1314, 1): ("A", "Ordinal qualifies पुरुषार्थः.", "High"),
    (1315, 1): ("A", "Ordinal qualifies पुरुषार्थः.", "High")
}

print("Prepared proposals:", len(jq_group_batch_3))
print("Expected occurrences: 50")

In [ ]:
apply_occurrence_proposals(jq, jq_group_batch_3)

print("Batch 3 proposals submitted:", len(jq_group_batch_3))
print("Gold Target_Tag remains unchanged.")

In [ ]:
batch_keys = list(jq_group_batch_3.keys())

verified = jq[
    jq.apply(
        lambda row: (row["sentence_id"], row["Token_Position"]) in batch_keys,
        axis=1
    )
].copy()

comparison = []

for key, (expected_tag, reason, confidence) in jq_group_batch_3.items():
    rows = verified[
        (verified["sentence_id"] == key[0]) &
        (verified["Token_Position"] == key[1])
    ]

    actual_tag = (
        rows.iloc[0]["Proposed_Target_Tag"]
        if len(rows) == 1 else "MISSING/DUPLICATE"
    )

    comparison.append({
        "Key": key,
        "Expected": expected_tag,
        "Actual": actual_tag,
        "Match": actual_tag == expected_tag
    })

comparison_df = pd.DataFrame(comparison)

print("Expected occurrences:", len(batch_keys))
print("Rows found:", len(verified))
print("Matching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

display(comparison_df[~comparison_df["Match"]])

print("Gold Target_Tag unchanged:", verified["Target_Tag"].isna().all())

In [ ]:
pending = jq[jq["Proposed_Target_Tag"].isna()].copy()

pending = pending.sort_values(
    ["sentence_id", "Token_Position"]
)

batch_4 = pending.head(50).copy()

print("Batch 4 occurrences:", len(batch_4))
print("Unique words:", batch_4["Word"].nunique())

display(
    batch_4[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Source_Tag",
            "Previous_Words",
            "Next_Words",
            "Full_Sentence"
        ]
    ].reset_index(drop=True)
)

In [ ]:
for sid in [1324, 1328]:
    print(f"\nSentence ID: {sid}")

    rows = jq[jq["sentence_id"] == sid]

    display(
        rows[
            [
                "sentence_id",
                "Token_Position",
                "Word",
                "Full_Sentence",
                "Proposed_Target_Tag"
            ]
        ]
    )

In [ ]:
jq_group_batch_4 = globals().get("jq_group_batch_4", {})

# Remove incorrect occurrence keys
# Remove incorrect occurrence keys only if they exist
jq_group_batch_4.pop((1324, 2), None)
jq_group_batch_4.pop((1328, 2), None)

print("Correction keys checked successfully.")

# Insert corrected keys, preserving the original proposals
jq_group_batch_4[(1324, 5)] = (
    "SN",
    "Substantive pronoun meaning all of it.",
    "Medium"
)

jq_group_batch_4[(1328, 3)] = (
    "A",
    "Qualifies पर्यायः.",
    "High"
)

print("Prepared proposals:", len(jq_group_batch_4))
print("Corrected keys:", (1324, 5), (1328, 3))

In [ ]:
print("Dictionary size:", len(jq_group_batch_4))
print("Key (1324, 2) exists:", (1324, 2) in jq_group_batch_4)
print("Key (1328, 2) exists:", (1328, 2) in jq_group_batch_4)

In [ ]:
# Inspect the problematic proposal
key = (1420, 9)

print("Proposal:")
print(jq_group_batch_4.get(key))

print("\nMatching sentence IDs in jq:")
print(jq[
    jq["sentence_id"].astype(str).str.strip() == "1420"
].to_string(index=False))

In [ ]:
row = src[
    src["sentence_id"].astype(str).str.strip() == "1420"
]

if len(row):
    print("Original source sentence:")
    print(row.to_string(index=False))
else:
    print("Sentence ID 1420 not found in src.")

In [ ]:
# Remove the incorrect key if it exists
jq_group_batch_4.pop((1420, 9), None)

# Add the proposal using the actual Token_Position
jq_group_batch_4[(1420, 5)] = (
    "SN",
    "उभे is used independently as a dual pronoun; it does not modify an adjacent noun.",
    "Medium"
)

print("Corrected proposal:")
print((1420, 5), "->", jq_group_batch_4[(1420, 5)])

In [ ]:
apply_occurrence_proposals(jq, jq_group_batch_4)

print("Batch 4 proposals submitted:", len(jq_group_batch_4))
print("Gold Target_Tag remains unchanged.")

In [ ]:
batch_keys = list(jq_group_batch_4.keys())

verified = jq[
    jq.apply(
        lambda row: (row["sentence_id"], row["Token_Position"]) in batch_keys,
        axis=1
    )
].copy()

comparison = []

for key, (expected_tag, reason, confidence) in jq_group_batch_4.items():
    rows = verified[
        (verified["sentence_id"] == key[0]) &
        (verified["Token_Position"] == key[1])
    ]

    actual_tag = (
        rows.iloc[0]["Proposed_Target_Tag"]
        if len(rows) == 1 else "MISSING/DUPLICATE"
    )

    comparison.append({
        "Key": key,
        "Expected": expected_tag,
        "Actual": actual_tag,
        "Match": actual_tag == expected_tag
    })

comparison_df = pd.DataFrame(comparison)

print("Expected occurrences:", len(batch_keys))
print("Rows found:", len(verified))
print("Matching:", comparison_df["Match"].sum())
print("Different:", (~comparison_df["Match"]).sum())

display(
    verified.groupby("Proposed_Target_Tag")
    .size()
    .reset_index(name="Count")
)

display(comparison_df[~comparison_df["Match"]])

print("Gold Target_Tag unchanged:", verified["Target_Tag"].isna().all())

In [ ]:
# ============================================================
# FINAL AUDIT — OVERALL PROPOSAL STATUS
# ============================================================

print("Total occurrences:", len(jq))

print(
    "Proposed occurrences:",
    jq["Proposed_Target_Tag"].notna().sum()
)

print(
    "Unresolved occurrences:",
    jq["Proposed_Target_Tag"].isna().sum()
)

print(
    "Unique words:",
    jq["Word"].nunique()
)

print("\nProposal distribution:")
display(
    jq["Proposed_Target_Tag"]
    .value_counts(dropna=False)
    .rename_axis("Proposed_Target_Tag")
    .reset_index(name="Count")
)

print("\nConfidence distribution:")
display(
    jq["Mapping_Confidence"]
    .value_counts(dropna=False)
    .rename_axis("Mapping_Confidence")
    .reset_index(name="Count")
)

In [ ]:
duplicate_keys = jq[
    jq.duplicated(
        subset=["sentence_id", "Token_Position"],
        keep=False
    )
].sort_values(["sentence_id", "Token_Position"])

print("Duplicate occurrence rows:", len(duplicate_keys))

display(duplicate_keys)

In [ ]:
invalid_tags = sorted(
    set(jq["Proposed_Target_Tag"].dropna())
    - set(TARGET_TAGS)
)

print("Invalid proposed tags:", invalid_tags)

In [ ]:
from pathlib import Path
from datetime import datetime

AUDIT_DIR = Path(
    "" + str(RESULTS_DIR) + "/Mapping_Audit"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

backup_path = AUDIT_DIR / "jq_proposals_295_backup.csv"

jq.to_csv(backup_path, index=False, encoding="utf-8-sig")

print("Backup saved:", backup_path)
print("Rows saved:", len(jq))
print("Proposals saved:", jq["Proposed_Target_Tag"].notna().sum())

In [ ]:

import pandas as pd
from pathlib import Path

REVIEW_DIR = Path(
    "" + str(RESULTS_DIR) + "/Gold_Review"
)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Work on a separate copy; never overwrite jq or Target_Tag
fast_review = jq.copy().reset_index(drop=True)

fast_review["Approved_Target_Tag"] = pd.NA
fast_review["Review_Status"] = "Pending"
fast_review["Reviewer_Notes"] = ""

# Provisionally accept High-confidence proposals
high_mask = fast_review["Mapping_Confidence"].eq("High")

fast_review.loc[high_mask, "Approved_Target_Tag"] = (
    fast_review.loc[high_mask, "Proposed_Target_Tag"]
)
fast_review.loc[high_mask, "Review_Status"] = "Provisional_Accepted"

# Keep Medium-confidence proposals for targeted review
medium_mask = fast_review["Mapping_Confidence"].eq("Medium")
fast_review.loc[medium_mask, "Review_Status"] = "Needs_Review"

# Export complete workbench and medium-confidence queue
fast_review.to_csv(
    REVIEW_DIR / "Sanskrit_POS_Fast_Review_All.csv",
    index=False,
    encoding="utf-8-sig"
)

medium_review = fast_review[medium_mask].copy()

medium_review.to_csv(
    REVIEW_DIR / "Sanskrit_POS_Medium_Review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("FAST-TRACK REVIEW CREATED")
print("Total occurrences:", len(fast_review))
print("High-confidence provisional:", high_mask.sum())
print("Medium-confidence pending:", medium_mask.sum())
print("Approved labels:", fast_review["Approved_Target_Tag"].notna().sum())
print("Original Target_Tag unchanged:",
      fast_review["Target_Tag"].equals(jq["Target_Tag"]))

display(
    medium_review[
        [
            "sentence_id", "Token_Position", "Word",
            "Full_Sentence", "Proposed_Target_Tag",
            "Mapping_Reason", "Mapping_Confidence",
            "Review_Status"
        ]
    ].head(15)
)


In [ ]:
print("Total medium-confidence cases:", len(medium_review))

display(
    medium_review[
        [
            "sentence_id",
            "Token_Position",
            "Word",
            "Full_Sentence",
            "Proposed_Target_Tag",
            "Mapping_Reason",
            "Mapping_Confidence"
        ]
    ].reset_index(drop=True)
)

In [ ]:

# ============================================================
# FAST REVIEW — ALL 49 MEDIUM-CONFIDENCE OCCURRENCES
# ============================================================

medium_decisions = {
    (480, 3): "AVP",
    (1079, 2): "A",
    (1410, 0): "SN",
    (1429, 0): "SN",
    (1063, 6): "A",
    (1065, 1): "A",
    (1064, 3): "SN",
    (152, 0): "A",
    (1420, 5): "SN",
    (488, 12): "A",
    (378, 6): "SN",
    (499, 8): "SN",
    (386, 14): "SN",
    (972, 8): "SN",
    (1270, 3): "SN",
    (836, 2): "A",
    (510, 9): "AVC",
    (1429, 2): "AVU",
    (1425, 6): "SN",
    (1085, 9): "SN",
    (1086, 7): "SN",
    (1087, 3): "SN",
    (1056, 6): "A",
    (1270, 1): "A",
    (960, 4): "A",
    (391, 4): "SN",
    (900, 3): "A",
    (10, 3): "A",       # Corrected: was SN
    (217, 1): "A",
    (1012, 0): "SN",
    (313, 3): "AVU",
    (313, 6): "SN",
    (372, 1): "SN",
    (462, 2): "AVU",
    (544, 4): "SN",
    (748, 4): "AVU",
    (1105, 7): "SN",
    (1260, 8): "A",
    (1455, 9): "SN",
    (1318, 2): "A",
    (328, 17): "A",
    (1016, 1): "SN",
    (61, 3): "AVU",
    (703, 6): "A",     # Corrected: was AVU
    (1324, 5): "SN",
    (907, 5): "SN",
    (380, 1): "A",
    (469, 1): "SN",
    (1321, 9): "SN"
}

assert len(medium_decisions) == 49, "Decision count must be 49"

# Apply decisions to the full review dataset
for (sid, pos), tag in medium_decisions.items():
    mask = (
        (fast_review["sentence_id"] == sid) &
        (fast_review["Token_Position"] == pos)
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {(sid, pos)}, "
            f"found {mask.sum()}"
        )

    fast_review.loc[mask, "Approved_Target_Tag"] = tag

    original_proposal = fast_review.loc[
        mask, "Proposed_Target_Tag"
    ].iloc[0]

    fast_review.loc[mask, "Review_Status"] = (
        "Corrected" if tag != original_proposal else "Approved"
    )

    fast_review.loc[mask, "Reviewer_Notes"] = (
        "Contextual review completed; provisional label decision."
        if tag == original_proposal
        else f"Corrected from {original_proposal} to {tag} after contextual review."
    )

# Save completed review dataset
review_path = REVIEW_DIR / "Sanskrit_POS_Fast_Review_All.csv"

fast_review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("MEDIUM-CONFIDENCE REVIEW COMPLETED")
print("Decisions:", len(medium_decisions))
print("Total reviewed labels:", fast_review["Approved_Target_Tag"].notna().sum())
print("Pending:", fast_review["Approved_Target_Tag"].isna().sum())

print("\nReview status:")
display(fast_review["Review_Status"].value_counts())

print("\nFinal approved-label distribution:")
display(fast_review["Approved_Target_Tag"].value_counts())

print("\nOriginal gold Target_Tag unchanged:",
      fast_review["Target_Tag"].equals(jq["Target_Tag"]))

print("\nSaved:", review_path)


In [ ]:

# ============================================================
# TRAINING DATA STRUCTURE INSPECTION
# No data is modified in this cell.
# ============================================================

candidate_names = [
    "source_prepared",
    "source_train",
    "source_validation",
    "source_test",
    "target_prepared",
    "news_train_pool",
    "news_validation",
    "news_test_locked",
    "train_sequences",
    "source_sequences"
]

for name in candidate_names:
    if name not in globals():
        continue

    obj = globals()[name]

    print("\n" + "=" * 65)
    print("VARIABLE:", name)
    print("TYPE:", type(obj))

    if isinstance(obj, dict):
        print("KEYS:", list(obj.keys()))

        for key, value in obj.items():
            if isinstance(value, (list, tuple)):
                print(f"{key}: {len(value)} items")
                if value and isinstance(value[0], (list, tuple)):
                    print(" First sequence length:", len(value[0]))
                    print(" First sequence sample:", value[0][:5])
            else:
                print(f"{key}: {type(value)}")

    elif isinstance(obj, (list, tuple)):
        print("Number of items:", len(obj))
        if obj:
            print("First item type:", type(obj[0]))
            print("First item sample:", str(obj[0])[:400])

    elif isinstance(obj, pd.DataFrame):
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))
        display(obj.head(3))

print("\n" + "=" * 65)
print("REVIEW MAPPING SOURCE TAG COUNTS")

display(
    fast_review.groupby(
        ["Source_Tag", "Approved_Target_Tag"]
    ).size().reset_index(name="Occurrences")
)


In [ ]:
# Add a stable occurrence identifier
fast_review["Occurrence_ID"] = (
    fast_review["sentence_id"].astype(str)
    + "_"
    + fast_review["Token_Position"].astype(str)
)

print("Occurrence_ID created:", "Occurrence_ID" in fast_review.columns)
print("Unique occurrence IDs:", fast_review["Occurrence_ID"].nunique())
print("Total rows:", len(fast_review))

In [ ]:
# ============================================================
# DIAGNOSE FINAL REVIEW AUDIT
# ============================================================

print("Total occurrences:", len(fast_review))

print(
    "Missing Approved_Target_Tag:",
    fast_review["Approved_Target_Tag"].isna().sum()
)

print(
    "Invalid approved labels:",
    (~fast_review["Approved_Target_Tag"].isin(TARGET_TAGS)
     & fast_review["Approved_Target_Tag"].notna()).sum()
)

print(
    "Duplicate Occurrence_ID:",
    fast_review["Occurrence_ID"].duplicated().sum()
)

# Display unresolved occurrences
unresolved = fast_review[
    fast_review["Approved_Target_Tag"].isna()
]

print("\nUNRESOLVED OCCURRENCES:")
display(unresolved[
    [
        "Occurrence_ID",
        "sentence_id",
        "Token_Position",
        "Word",
        "Source_Tag",
        "Target_Tag",
        "Proposed_Target_Tag",
        "Review_Status",
        "Review_Notes"
    ]
])

In [ ]:
print("Review status distribution:")
display(fast_review["Review_Status"].value_counts(dropna=False))

print("\nProposed target tag distribution:")
display(fast_review["Proposed_Target_Tag"].value_counts(dropna=False))

In [ ]:
fast_review["Target_Tag"].equals(jq["Target_Tag"])

In [ ]:

# Inspect available training data structures
for name in [
    "source_prepared",
    "source_train",
    "source_validation",
    "source_test",
    "target_prepared",
    "news_train_pool",
    "news_validation",
    "news_test_locked"
]:
    if name not in globals():
        continue

    obj = globals()[name]

    print("\n" + "=" * 55)
    print("VARIABLE:", name)
    print("TYPE:", type(obj))

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"\n{key}: {type(value)}")
            if isinstance(value, (list, tuple)):
                print("Length:", len(value))
                if value:
                    print("Sample:", str(value[0])[:300])

    elif isinstance(obj, pd.DataFrame):
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))
        display(obj.head(2))

    elif isinstance(obj, (list, tuple)):
        print("Length:", len(obj))
        if obj:
            print("Sample:", str(obj[0])[:300])


In [ ]:

# ============================================================
# DRY-RUN: MATCH REVIEWED OCCURRENCES TO SOURCE TRAIN
# No training labels are modified.
# ============================================================

import pandas as pd

source_check = source_train.copy(deep=True)
review_check = fast_review.copy()

# Check whether sentence IDs are unique in source_train
duplicate_sentence_ids = source_check[
    source_check.duplicated("sentence_id", keep=False)
]

print("Source training sentences:", len(source_check))
print("Duplicate sentence-ID rows:", len(duplicate_sentence_ids))

# Build sentence lookup
source_lookup = {}

for idx, row in source_check.iterrows():
    sid = row["sentence_id"]
    source_lookup.setdefault(sid, []).append(idx)

match_results = []

for _, review_row in review_check.iterrows():

    sid = review_row["sentence_id"]
    pos = int(review_row["Token_Position"])
    word = str(review_row["Word"]).strip()

    candidate_indices = source_lookup.get(sid, [])

    result = "NOT_FOUND"
    source_word = None
    source_doc_id = None

    for idx in candidate_indices:
        tokens = source_check.at[idx, "tokens"]

        if pos < len(tokens):
            source_word = str(tokens[pos]).strip()
            source_doc_id = source_check.at[idx, "doc_id"]

            if source_word == word:
                result = "EXACT_MATCH"
                break
            else:
                result = "WORD_MISMATCH"

    match_results.append({
        "sentence_id": sid,
        "Token_Position": pos,
        "Word": word,
        "Source_Word": source_word,
        "doc_id": source_doc_id,
        "Approved_Tag": review_row["Approved_Target_Tag"],
        "Match_Status": result
    })

match_audit = pd.DataFrame(match_results)

print("\nMATCHING AUDIT")
print("Total reviewed occurrences:", len(match_audit))
print("Exact matches:", (match_audit["Match_Status"] == "EXACT_MATCH").sum())
print("Word mismatches:", (match_audit["Match_Status"] == "WORD_MISMATCH").sum())
print("Not found:", (match_audit["Match_Status"] == "NOT_FOUND").sum())

display(
    match_audit[
        match_audit["Match_Status"] != "EXACT_MATCH"
    ]
)


In [ ]:
# ============================================================
# IDENTIFY CORRECT DATASET FOR THE 295 REVIEWED OCCURRENCES
# No modifications.
# ============================================================

print("FAST REVIEW COLUMNS:")
print(fast_review.columns.tolist())

print("\nSOURCE TRAIN:")
print("Rows:", len(source_train))
print("Sentence ID range:",
      source_train["sentence_id"].min(),
      source_train["sentence_id"].max())

print("\nNEWS TRAIN POOL:")
print("Rows:", len(news_train_pool))
print("Sentence ID range:",
      news_train_pool["sentence_id"].min(),
      news_train_pool["sentence_id"].max())

print("Document ID range:",
      news_train_pool["doc_id"].min(),
      news_train_pool["doc_id"].max())

print("\nREVIEW SENTENCE ID RANGE:")
print(fast_review["sentence_id"].min(),
      fast_review["sentence_id"].max())

print("\nREVIEW SAMPLE:")
display(
    fast_review[
        ["sentence_id", "Token_Position", "Word",
         "Approved_Target_Tag"]
    ].head(10)
)

print("\nNEWS TRAIN SAMPLE WITH MATCHING SENTENCE IDS:")
sample_ids = fast_review["sentence_id"].head(10).tolist()

display(
    news_train_pool[
        news_train_pool["sentence_id"].isin(sample_ids)
    ][["sentence_id", "doc_id", "tokens", "tags", "__sentence_key"]]
)

In [ ]:

# ============================================================
# DIAGNOSTIC: MATCH REVIEWED OCCURRENCES BY SENTENCE CONTENT
# ============================================================

import re
import pandas as pd

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

# Build a normalized sentence lookup from news_train_pool
news_lookup = {}

for idx, row in news_train_pool.iterrows():
    sentence = normalize_text(" ".join(row["tokens"]))
    news_lookup.setdefault(sentence, []).append(idx)

# Match reviewed full sentences
sentence_matches = []

for _, r in fast_review.iterrows():
    normalized = normalize_text(r["Full_Sentence"])
    candidates = news_lookup.get(normalized, [])

    sentence_matches.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Review_Sentence_ID": r["sentence_id"],
        "Word": r["Word"],
        "Token_Position": r["Token_Position"],
        "Matched_Rows": len(candidates),
        "Candidate_Indices": candidates,
        "Candidate_Keys": [
            news_train_pool.loc[i, "__sentence_key"]
            for i in candidates
        ]
    })

sentence_match_audit = pd.DataFrame(sentence_matches)

print("Reviewed occurrences:", len(sentence_match_audit))
print(
    "Sentence matches:",
    (sentence_match_audit["Matched_Rows"] == 1).sum()
)
print(
    "Ambiguous matches:",
    (sentence_match_audit["Matched_Rows"] > 1).sum()
)
print(
    "Unmatched sentences:",
    (sentence_match_audit["Matched_Rows"] == 0).sum()
)

display(sentence_match_audit.head(15))


In [ ]:
print("source_train sentence_id dtype:", source_train["sentence_id"].dtype)
print("fast_review sentence_id dtype:", fast_review["sentence_id"].dtype)

print("\nSource sentence ID 10:")
display(
    source_train[
        source_train["sentence_id"].astype(str).str.strip() == "10"
    ][["sentence_id", "doc_id", "tokens", "tags"]]
)

print("\nReviewed occurrence 10_3:")
display(
    fast_review[
        (fast_review["sentence_id"].astype(str).str.strip() == "10") &
        (fast_review["Token_Position"] == 3)
    ][["Occurrence_ID", "Word", "Token_Position", "Approved_Target_Tag"]]
)

In [ ]:
import pandas as pd

# Build source sentence lookup using normalized sentence IDs
source_lookup = {}

for idx, row in source_train.iterrows():
    sid = str(row["sentence_id"]).strip()
    source_lookup.setdefault(sid, []).append(idx)

audit_rows = []

for _, r in fast_review.iterrows():
    sid = str(int(r["sentence_id"]))
    pos = int(r["Token_Position"])
    word = str(r["Word"]).strip()

    candidates = source_lookup.get(sid, [])

    matched = []
    for idx in candidates:
        tokens = source_train.at[idx, "tokens"]

        if pos < len(tokens) and str(tokens[pos]).strip() == word:
            matched.append(idx)

    audit_rows.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Sentence_ID": sid,
        "Word": word,
        "Token_Position": pos,
        "Approved_Target_Tag": r["Approved_Target_Tag"],
        "Candidate_Rows": len(candidates),
        "Exact_Matches": len(matched),
        "Matched_Indices": matched
    })

source_match_audit = pd.DataFrame(audit_rows)

print("Total reviewed occurrences:", len(source_match_audit))
print("Exact matches:", (source_match_audit["Exact_Matches"] == 1).sum())
print("Ambiguous matches:", (source_match_audit["Exact_Matches"] > 1).sum())
print("Unmatched:", (source_match_audit["Exact_Matches"] == 0).sum())

display(source_match_audit.head(20))

In [ ]:
unmatched = source_match_audit[
    source_match_audit["Exact_Matches"] == 0
].copy()

diagnostics = []

for _, r in unmatched.iterrows():
    sid = str(r["Sentence_ID"])
    word = r["Word"]
    pos = int(r["Token_Position"])

    candidates = source_train[
        source_train["sentence_id"].astype(str).str.strip() == sid
    ]

    if candidates.empty:
        reason = "Sentence_ID_not_in_source_train"
        actual_tokens = None
    else:
        tokens = candidates.iloc[0]["tokens"]
        actual_tokens = " ".join(map(str, tokens))

        if word in tokens:
            reason = "Word_exists_at_different_position"
        else:
            reason = "Word_not_found_in_sentence"

    diagnostics.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Sentence_ID": sid,
        "Word": word,
        "Expected_Position": pos,
        "Reason": reason,
        "Source_Tokens": actual_tokens
    })

unmatched_diagnostics = pd.DataFrame(diagnostics)

print(unmatched_diagnostics["Reason"].value_counts())
display(unmatched_diagnostics.head(25))

In [ ]:
# Identify the single occurrence whose word exists at another position
different_position = unmatched_diagnostics[
    unmatched_diagnostics["Reason"] == "Word_exists_at_different_position"
]

for _, r in different_position.iterrows():
    sid = r["Sentence_ID"]
    word = r["Word"]

    row = source_train[
        source_train["sentence_id"].astype(str).str.strip() == sid
    ].iloc[0]

    tokens = row["tokens"]

    print("Occurrence:", r["Occurrence_ID"])
    print("Expected position:", r["Expected_Position"])
    print("Word:", word)
    print("Actual positions:", [i for i, token in enumerate(tokens) if str(token).strip() == word])
    print("Tokens:", tokens)
    print("Tags:", row["tags"])

In [ ]:
exact_matches = source_match_audit[
    source_match_audit["Exact_Matches"] == 1
].copy()

check_rows = []

for _, r in exact_matches.iterrows():
    idx = r["Matched_Indices"][0]
    pos = int(r["Token_Position"])

    current_tag = source_train.at[idx, "tags"][pos]

    check_rows.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Source_Index": idx,
        "Sentence_ID": r["Sentence_ID"],
        "Word": r["Word"],
        "Position": pos,
        "Current_Tag": current_tag,
        "Approved_Tag": r["Approved_Target_Tag"],
        "Needs_Fill": pd.isna(current_tag),
        "Tag_Conflict": (
            pd.notna(current_tag) and
            current_tag != r["Approved_Target_Tag"]
        )
    })

update_audit = pd.DataFrame(check_rows)

print("Exact matches:", len(update_audit))
print("Missing tags:", update_audit["Needs_Fill"].sum())
print("Existing-tag conflicts:", update_audit["Tag_Conflict"].sum())

display(update_audit["Current_Tag"].value_counts(dropna=False))
display(update_audit[update_audit["Tag_Conflict"]])

In [ ]:
import copy

# Keep a backup before modifying source_train
source_train_before_mapping = copy.deepcopy(source_train)

updated_count = 0

for _, r in update_audit.iterrows():
    idx = int(r["Source_Index"])
    pos = int(r["Position"])
    approved_tag = r["Approved_Tag"]

    tags = list(source_train.at[idx, "tags"])

    # Safety check: update only missing positions
    if pd.isna(tags[pos]):
        tags[pos] = approved_tag
        source_train.at[idx, "tags"] = tags
        updated_count += 1

print("Labels successfully filled:", updated_count)

In [ ]:
from pathlib import Path
import pickle

OUT_DIR = Path(
    "" + str(RESULTS_DIR) + "/Gold_Review"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUT_DIR / "source_train_mapped_229.pkl"

with open(output_path, "wb") as f:
    pickle.dump(source_train, f)

print("Saved successfully:", output_path)
print("Dataset shape:", source_train.shape)
print("Mapped occurrences:", updated_count)

In [ ]:
import pandas as pd

print("Available source-related DataFrames:\n")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if {"sentence_id", "tokens", "tags"}.issubset(obj.columns):
            print(
                f"{name}: shape={obj.shape}, "
                f"sentence_id dtype={obj['sentence_id'].dtype}, "
                f"unique sentences={obj['sentence_id'].nunique()}"
            )

In [ ]:
# Match the 66 unmatched occurrences against source_val

val_lookup = {}

for idx, row in source_val.iterrows():
    sid = str(row["sentence_id"]).strip()
    val_lookup.setdefault(sid, []).append(idx)

val_audit_rows = []

for _, r in unmatched.iterrows():
    sid = str(r["Sentence_ID"]).strip()
    pos = int(r["Token_Position"])
    word = str(r["Word"]).strip()

    candidates = val_lookup.get(sid, [])
    matched = []

    for idx in candidates:
        tokens = source_val.at[idx, "tokens"]

        if pos < len(tokens) and str(tokens[pos]).strip() == word:
            matched.append(idx)

    val_audit_rows.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Sentence_ID": sid,
        "Word": word,
        "Token_Position": pos,
        "Approved_Tag": r["Approved_Target_Tag"],
        "Candidate_Rows": len(candidates),
        "Exact_Matches": len(matched),
        "Matched_Indices": matched
    })

val_match_audit = pd.DataFrame(val_audit_rows)

print("Unmatched occurrences checked:", len(val_match_audit))
print("Exact matches in source_val:",
      (val_match_audit["Exact_Matches"] == 1).sum())
print("Ambiguous matches:",
      (val_match_audit["Exact_Matches"] > 1).sum())
print("Still unmatched:",
      (val_match_audit["Exact_Matches"] == 0).sum())

display(val_match_audit.head(25))

In [ ]:
# Inspect the one occurrence not matched in either split

still_unmatched = val_match_audit[
    val_match_audit["Exact_Matches"] == 0
]

for _, r in still_unmatched.iterrows():
    sid = str(r["Sentence_ID"])
    word = str(r["Word"]).strip()
    pos = int(r["Token_Position"])

    print("Occurrence:", r["Occurrence_ID"])
    print("Sentence ID:", sid)
    print("Word:", word)
    print("Expected position:", pos)

    rows = source_train_all[
        source_train_all["sentence_id"].astype(str).str.strip() == sid
    ]

    print("Rows in source_train_all:", len(rows))

    for _, row in rows.iterrows():
        tokens = row["tokens"]
        print("Tokens:", tokens)
        print("Actual positions:",
              [i for i, t in enumerate(tokens) if str(t).strip() == word])

In [ ]:
sid = "443"

for name, df in [
    ("source_train", source_train),
    ("source_val", source_val),
    ("source_train_all", source_train_all)
]:
    rows = df[df["sentence_id"].astype(str).str.strip() == sid]

    print(f"\n{name}: {len(rows)} matching rows")

    if not rows.empty:
        row = rows.iloc[0]
        print("Tokens:", list(enumerate(row["tokens"])))
        print("Tags:", list(enumerate(row["tags"])))

In [ ]:
# Correct the final occurrence using the verified position
sid = "443"
correct_position = 5
approved_tag = "A"

row_idx = source_train.index[
    source_train["sentence_id"].astype(str).str.strip() == sid
][0]

tokens = list(source_train.at[row_idx, "tokens"])
tags = list(source_train.at[row_idx, "tags"])

assert tokens[correct_position] == "एकेन"
assert pd.isna(tags[correct_position])

tags[correct_position] = approved_tag
source_train.at[row_idx, "tags"] = tags

print("Occurrence 443_5 updated successfully.")
print("Token:", source_train.at[row_idx, "tokens"][correct_position])
print("Updated tag:", source_train.at[row_idx, "tags"][correct_position])

In [ ]:
import pickle

output_path = RESULTS_DIR / "Gold_Review" / "source_train_mapped_230.pkl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "wb") as f:
    pickle.dump(source_train, f)

print("Saved:", output_path)

In [ ]:
validation_audit_rows = []

for _, r in val_match_audit[
    val_match_audit["Exact_Matches"] == 1
].iterrows():

    idx = int(r["Matched_Indices"][0])
    pos = int(r["Token_Position"])

    tokens = list(source_val.at[idx, "tokens"])
    tags = list(source_val.at[idx, "tags"])

    validation_audit_rows.append({
        "Occurrence_ID": r["Occurrence_ID"],
        "Source_Index": idx,
        "Sentence_ID": r["Sentence_ID"],
        "Word": r["Word"],
        "Position": pos,
        "Current_Tag": tags[pos],
        "Approved_Tag": r["Approved_Tag"],
        "Token_Matches": str(tokens[pos]).strip() == str(r["Word"]).strip(),
        "Needs_Fill": pd.isna(tags[pos]),
        "Tag_Conflict": (
            pd.notna(tags[pos]) and tags[pos] != r["Approved_Tag"]
        )
    })

validation_audit = pd.DataFrame(validation_audit_rows)

print("Validation occurrences:", len(validation_audit))
print("Token mismatches:", (~validation_audit["Token_Matches"]).sum())
print("Missing tags:", validation_audit["Needs_Fill"].sum())
print("Existing-tag conflicts:", validation_audit["Tag_Conflict"].sum())

display(validation_audit["Current_Tag"].value_counts(dropna=False))
display(validation_audit[validation_audit["Tag_Conflict"]])

In [ ]:
import copy
import pandas as pd

# Preserve original validation data
source_val_reviewed = copy.deepcopy(source_val)

updated_count = 0

for _, r in validation_audit.iterrows():
    idx = r["Source_Index"]
    pos = int(r["Position"])
    approved_tag = r["Approved_Tag"]

    tags = list(source_val_reviewed.at[idx, "tags"])

    if pd.isna(tags[pos]):
        tags[pos] = approved_tag
        source_val_reviewed.at[idx, "tags"] = tags
        updated_count += 1

print("Validation labels applied:", updated_count)

In [ ]:
from collections import Counter
import pandas as pd

def missing_tag_report(df, name):
    missing_by_token = Counter()
    total_tokens = 0
    total_missing = 0

    for _, row in df.iterrows():
        for token, tag in zip(row["tokens"], row["tags"]):
            total_tokens += 1

            if pd.isna(tag):
                total_missing += 1
                missing_by_token[str(token)] += 1

    print(f"\n{name}")
    print("Total tokens:", total_tokens)
    print("Missing tags:", total_missing)
    print("Tagged tokens:", total_tokens - total_missing)
    print("Missing percentage:", round(100 * total_missing / total_tokens, 2))
    print("\nTop 20 tokens with missing tags:")
    display(pd.DataFrame(
        missing_by_token.most_common(20),
        columns=["Token", "Missing_Count"]
    ))

missing_tag_report(source_train, "SOURCE TRAIN")
missing_tag_report(source_val_reviewed, "SOURCE VALIDATION")

In [ ]:
print("SRC columns:")
print(src.columns.tolist())

print("\nSRC data types:")
print(src.dtypes)

print("\nSRC first 10 rows:")
display(src.head(10))

print("\nSOURCE TRAIN sample with missing labels:")

for _, row in source_train.iterrows():
    if any(pd.isna(tag) for tag in row["tags"]):
        print("Sentence ID:", row["sentence_id"])
        print("Tokens:", row["tokens"])
        print("Tags:", row["tags"])
        break

In [ ]:
from collections import Counter, defaultdict
import pandas as pd

unmapped_prefix_counts = Counter()
unmapped_token_examples = defaultdict(list)
unmapped_original_tags = defaultdict(Counter)

train_ids = set(source_train["sentence_id"].astype(str))
val_ids = set(source_val_reviewed["sentence_id"].astype(str))

for _, row in src.iterrows():
    sid = str(row["sentence_id"])

    if sid in train_ids:
        split = "Train"
    elif sid in val_ids:
        split = "Validation"
    else:
        continue

    for token, original_tag, prefix, mapped_tag in zip(
        row["tokens"],
        row["tags"],
        row["source_prefixes"],
        row["mapped_tags"]
    ):
        if pd.isna(mapped_tag):
            unmapped_prefix_counts[(split, str(prefix))] += 1
            unmapped_original_tags[(split, str(prefix))][str(original_tag)] += 1

            if len(unmapped_token_examples[(split, str(prefix))]) < 5:
                unmapped_token_examples[(split, str(prefix))].append(str(token))

report = []

for (split, prefix), count in unmapped_prefix_counts.items():
    report.append({
        "Split": split,
        "Source_Prefix": prefix,
        "Count": count,
        "Original_Tag_Examples": list(
            unmapped_original_tags[(split, prefix)].most_common(3)
        ),
        "Token_Examples": unmapped_token_examples[(split, prefix)]
    })

unmapped_report = pd.DataFrame(report)

if not unmapped_report.empty:
    unmapped_report = unmapped_report.sort_values(
        ["Split", "Count"],
        ascending=[True, False]
    )

display(unmapped_report)

print("\nTotal unmapped occurrences:")
print("Train:", sum(v for (s, _), v in unmapped_prefix_counts.items() if s == "Train"))
print("Validation:", sum(v for (s, _), v in unmapped_prefix_counts.items() if s == "Validation"))

In [ ]:
print("Existing SOURCE_PREFIX_TO_TARGET mapping:\n")

for prefix, target in sorted(SOURCE_PREFIX_TO_TARGET.items()):
    print(f"{prefix:10s} -> {target}")

print("\nTarget tagset:")
print(TARGET_TAGS)

print("\nExisting mapping coverage:")
print("Mapped source occurrences:", src["mapped_count"].sum())
print("Unmapped source occurrences:", src["unmapped_count"].sum())

In [ ]:
from collections import defaultdict, Counter
import pandas as pd

prefix_stats = defaultdict(Counter)
prefix_examples = defaultdict(list)

for _, row in src.iterrows():
    for token, prefix, mapped_tag in zip(
        row["tokens"],
        row["source_prefixes"],
        row["mapped_tags"]
    ):
        prefix = str(prefix).strip() if pd.notna(prefix) else "EMPTY"

        prefix_stats[prefix]["Total"] += 1

        if pd.notna(mapped_tag):
            prefix_stats[prefix]["Mapped"] += 1
            prefix_stats[prefix][f"Target:{mapped_tag}"] += 1
        else:
            prefix_stats[prefix]["Unmapped"] += 1

            if len(prefix_examples[prefix]) < 10:
                prefix_examples[prefix].append(str(token))

rows = []

for prefix, stats in prefix_stats.items():
    total = stats["Total"]
    mapped = stats["Mapped"]

    rows.append({
        "Source_Prefix": prefix,
        "Total": total,
        "Mapped": mapped,
        "Unmapped": stats["Unmapped"],
        "Coverage_%": round(100 * mapped / total, 2),
        "Target_Distribution": {
            k.replace("Target:", ""): v
            for k, v in stats.items()
            if k.startswith("Target:")
        },
        "Unmapped_Examples": prefix_examples[prefix]
    })

prefix_inventory = pd.DataFrame(rows).sort_values(
    "Unmapped", ascending=False
)

display(prefix_inventory)

print("\nTotal source tokens:", prefix_inventory["Total"].sum())
print("Total mapped:", prefix_inventory["Mapped"].sum())
print("Total unmapped:", prefix_inventory["Unmapped"].sum())

In [ ]:
import pandas as pd

PREFIXES_TO_INSPECT = [
    "CX", "VBP", "ALC", "CEM", "AMN", "DAB"
]

context_rows = []

for _, row in src.iterrows():

    tokens = row["tokens"]
    original_tags = row["tags"]
    prefixes = row["source_prefixes"]
    mapped_tags = row["mapped_tags"]

    sentence = " ".join(map(str, tokens))

    for pos, (token, original, prefix, mapped) in enumerate(
        zip(tokens, original_tags, prefixes, mapped_tags)
    ):
        prefix = str(prefix).strip()

        if prefix not in PREFIXES_TO_INSPECT:
            continue

        if pd.notna(mapped):
            continue

        context_rows.append({
            "Sentence_ID": row["sentence_id"],
            "Token_Position": pos,
            "Token": token,
            "Source_Tag": original,
            "Source_Prefix": prefix,
            "Previous_Token": tokens[pos-1] if pos > 0 else "",
            "Next_Token": tokens[pos+1] if pos < len(tokens)-1 else "",
            "Full_Sentence": sentence
        })

context_df = pd.DataFrame(context_rows)

# Show up to 5 examples per prefix
for prefix in PREFIXES_TO_INSPECT:
    print(f"\n{'='*70}")
    print("PREFIX:", prefix)

    subset = context_df[
        context_df["Source_Prefix"] == prefix
    ].drop_duplicates("Sentence_ID").head(5)

    display(subset)

In [ ]:

# STEP 1: Inspect source-prefix occurrences before applying new mappings

import pandas as pd
from collections import Counter

# Prefixes requiring immediate mapping or review
PREFIXES_TO_REVIEW = [
    "ALC", "AMN", "VBP", "VBG",
    "CNG", "CEM", "JQ",
    "DAB", "DRL", "DWH",
    "CX", "CCL", "RDF", "RDS", "RDX"
]

rows = []

for _, row in src.iterrows():
    tokens = row["tokens"]
    prefixes = row["source_prefixes"]
    mapped = row["mapped_tags"]

    for i, (token, prefix, current_tag) in enumerate(
        zip(tokens, prefixes, mapped)
    ):
        if prefix in PREFIXES_TO_REVIEW:
            rows.append({
                "sentence_id": row["sentence_id"],
                "token_position": i,
                "token": token,
                "source_prefix": prefix,
                "current_target_tag": current_tag
            })

review_inventory = pd.DataFrame(rows)

print("Total occurrences:", len(review_inventory))
print("\nOccurrences by source prefix:")
print(review_inventory["source_prefix"].value_counts())

print("\nSample occurrences (first 60):")
display(review_inventory.head(60))

# Save an inspection copy
review_inventory.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "source_prefix_mapping_inventory.csv",
    index=False
)


In [ ]:

# STEP 2: Inspect ambiguous source categories using actual sentence context

from collections import Counter, defaultdict
import pandas as pd

AMBIGUOUS_PREFIXES = [
    "CX", "JQ", "DAB", "DRL", "DWH",
    "RDX", "RDF", "RDS", "CCL"
]

word_counts = defaultdict(Counter)
examples = defaultdict(list)

for _, row in src.iterrows():
    tokens = row["tokens"]
    prefixes = row["source_prefixes"]

    for i, (token, prefix) in enumerate(zip(tokens, prefixes)):
        if prefix not in AMBIGUOUS_PREFIXES:
            continue

        word_counts[prefix][str(token)] += 1

        if len(examples[prefix]) < 12:
            examples[prefix].append({
                "sentence_id": row["sentence_id"],
                "position": i,
                "word": token,
                "sentence": " ".join(map(str, tokens))
            })

# Show the 20 most frequent words in each category
for prefix in AMBIGUOUS_PREFIXES:
    print("\n" + "=" * 80)
    print(f"SOURCE PREFIX: {prefix}")
    print("MOST FREQUENT WORDS:")
    print(word_counts[prefix].most_common(20))

    print("\nSENTENCE EXAMPLES:")
    display(pd.DataFrame(examples[prefix]))

# Save complete word-frequency inventory
frequency_rows = []

for prefix, counter in word_counts.items():
    for word, count in counter.items():
        frequency_rows.append({
            "source_prefix": prefix,
            "word": word,
            "frequency": count
        })

word_frequency_df = pd.DataFrame(frequency_rows)
word_frequency_df.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "ambiguous_prefix_word_frequencies.csv",
    index=False
)

print("\nUnique words by prefix:")
print(word_frequency_df.groupby("source_prefix")["word"].nunique())


In [ ]:

# STEP 3: Apply high-confidence source-to-target mappings
import pandas as pd
import re
import copy

AUTO_PREFIX_MAP = {
    "ALC": "AVP",
    "AMN": "AVP",
    "VBP": "NV",
    "VBG": "NV",
    "CNG": "AVN",
    "CEM": "AVN",
    "DAB": "SNV",
    "DRL": "SNV"
}

def is_missing_tag(value):
    return value is None or (
        isinstance(value, float) and pd.isna(value)
    )

def is_numeric_token(token):
    return bool(re.fullmatch(r"[०-९0-9]+", str(token).strip()))

# Work on copies first
src = src.copy(deep=True)
source_train = source_train.copy(deep=True)
source_val = source_val.copy(deep=True)

# Build sentence lookup from source inventory
src_lookup = {
    str(row["sentence_id"]): row
    for _, row in src.iterrows()
}

# Update grouped source inventory
mapping_counts = {}

for idx, row in src.iterrows():
    prefixes = list(row["source_prefixes"])
    tokens = list(row["tokens"])
    tags = list(row["mapped_tags"])

    for i, (prefix, token) in enumerate(zip(prefixes, tokens)):
        proposed = AUTO_PREFIX_MAP.get(prefix)

        if prefix == "RDX" and is_numeric_token(token):
            proposed = "AVC"

        if proposed is not None and is_missing_tag(tags[i]):
            tags[i] = proposed
            mapping_counts[proposed] = mapping_counts.get(proposed, 0) + 1

    src.at[idx, "mapped_tags"] = tags

# Apply the same mappings to training and validation sequences
def update_split(df, split_name):
    df = df.copy(deep=True)
    added = 0
    skipped = 0

    for idx, row in df.iterrows():
        sid = str(row["sentence_id"])

        if sid not in src_lookup:
            skipped += 1
            continue

        source_row = src_lookup[sid]
        prefixes = list(source_row["source_prefixes"])
        tokens = list(source_row["tokens"])
        tags = list(row["tags"])

        if not (len(prefixes) == len(tokens) == len(tags)):
            skipped += 1
            continue

        for i, (prefix, token) in enumerate(zip(prefixes, tokens)):
            proposed = AUTO_PREFIX_MAP.get(prefix)

            if prefix == "RDX" and is_numeric_token(token):
                proposed = "AVC"

            if proposed is not None and is_missing_tag(tags[i]):
                tags[i] = proposed
                added += 1

        df.at[idx, "tags"] = tags

    print(f"{split_name}: labels added = {added}, rows skipped = {skipped}")
    return df

source_train = update_split(source_train, "Training")
source_val = update_split(source_val, "Validation")

print("\nMappings added to src:")
print(mapping_counts)

print("\nTarget tag distribution in source_train:")
print(pd.Series([t for seq in source_train["tags"] for t in seq]).value_counts())

print("\nRemaining missing training tags:",
      sum(is_missing_tag(t) for seq in source_train["tags"] for t in seq))

print("Remaining missing validation tags:",
      sum(is_missing_tag(t) for seq in source_val["tags"] for t in seq))


In [ ]:

# STEP 5: Create focused review inventory for CX and JQ

review_prefixes = ["CX", "JQ"]

cx_jq_rows = []

for _, row in src.iterrows():
    tokens = list(row["tokens"])
    prefixes = list(row["source_prefixes"])

    for i, (token, prefix) in enumerate(zip(tokens, prefixes)):
        if prefix in review_prefixes:
            cx_jq_rows.append({
                "sentence_id": row["sentence_id"],
                "position": i,
                "token": token,
                "source_prefix": prefix,
                "previous_word": tokens[i-1] if i > 0 else "",
                "next_word": tokens[i+1] if i+1 < len(tokens) else "",
                "sentence": " ".join(map(str, tokens))
            })

cx_jq_review = pd.DataFrame(cx_jq_rows)

for prefix in review_prefixes:
    subset = cx_jq_review[
        cx_jq_review["source_prefix"] == prefix
    ]

    print(f"\n{prefix}: {len(subset)} occurrences")
    print("Unique words:", subset["token"].nunique())
    print("\nMost frequent words:")
    print(subset["token"].value_counts().head(50))

    print("\nContext examples:")
    display(subset.head(30))

cx_jq_review.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "CX_JQ_Context_Review.csv",
    index=False
)


In [ ]:

# STEP 6: Context-informed CX and JQ mapping
# Only fills missing labels. Original source prefixes are preserved.

import pandas as pd

# CX: clear verbal forms and indeclinables
CX_WORD_MAP = {
    # Gerunds, participles and infinitives -> NV
    "श्रुत्वा": "NV", "कृत्वा": "NV", "गत्वा": "NV",
    "उक्त्वा": "NV", "आगत्य": "NV", "दृष्ट्वा": "NV",
    "आदाय": "NV", "अवलोक्य": "NV", "भूत्वा": "NV",
    "स्थित्वा": "NV", "लब्ध्वा": "NV", "मत्वा": "NV",
    "उद्दिश्य": "NV", "परित्यज्य": "NV", "विलोक्य": "NV",
    "ज्ञात्वा": "NV", "नीत्वा": "NV", "गृहीत्वा": "NV",
    "जित्वा": "NV", "समाप्य": "NV", "निवृत्य": "NV",
    "पठितुम्": "NV", "वारयितुम्": "NV", "दातुं": "NV",
    "कर्तुं": "NV", "वक्तुम्": "NV", "प्राप्तुं": "NV",
    "नेतुं": "NV", "दत्वा": "NV", "परित्यज्य": "NV",
    "विरुध्य": "NV", "सन्": "NV",

    # General indeclinables / particles -> AVN
    "इति": "AVN", "तु": "AVN", "स्म": "AVN",
    "इव": "AVN", "वा": "AVN", "अपि": "AVN",
    "एव": "AVN", "न": "AVN", "खलु": "AVN",
    "केवलम्": "AVN", "नाम": "AVN", "अलं": "AVN",

    # Clear adverbial uses -> AVP
    "ततः": "AVP", "अतः": "AVP", "पुरतः": "AVP",
    "परितः": "AVP", "पुनः": "AVP", "तथा": "AVP",

    # Case-governing particles -> K
    "सह": "K", "प्रति": "K"
}

# JQ: map only selected, clear examples
JQ_WORD_MAP = {
    # Qualifying adjectives
    "महत्तरां": "A", "महत्तरं": "A", "महत्तरः": "A",
    "अपर्याप्तं": "A", "सम्पूर्णस्य": "A",
    "महती": "A", "महान्": "A", "महत्तरम्": "A",
    "पर्याप्तम्": "A", "अत्यधिकं": "A",

    # Pronominal / quantifying determiners modifying nouns
    "सर्वे": "SNV", "सर्वं": "SNV", "सर्वासु": "SNV",
    "सर्वान्": "SNV", "सर्वेषु": "SNV", "सर्वैः": "SNV",
    "एकः": "SNV", "एका": "SNV", "एकं": "SNV",
    "एकम्": "SNV", "एकस्मिन्": "SNV",
    "द्वे": "SNV", "द्वौ": "SNV", "उभे": "SNV",
    "उभौ": "SNV", "अनेकानि": "SNV",
    "चत्वारः": "SNV", "त्रीन्": "SNV", "त्रीणि": "SNV",

    # Nominal quantity expressions
    "लक्षाणि": "N", "गोसहस्रं": "N",
    "खण्डद्वयं": "N",

    # Indeclinable numerical/frequency expression
    "पञ्चदशवारं": "AVC"
}

# Source sentence lookup
source_lookup = {
    str(row["sentence_id"]): row
    for _, row in src.iterrows()
}

def missing_tag(x):
    return x is None or (
        isinstance(x, float) and pd.isna(x)
    )

def propose_tag(prefix, token, next_word=""):
    token = str(token)

    if prefix == "CX":
        return CX_WORD_MAP.get(token)

    if prefix == "JQ":
        return JQ_WORD_MAP.get(token)

    return None

# Apply by exact sentence ID + token position + token match
def apply_cx_jq_rules(df, split_name):
    df = df.copy(deep=True)
    counts = {}
    mismatches = 0
    unmatched_sentences = 0

    for idx, row in df.iterrows():
        sid = str(row["sentence_id"])
        source_row = source_lookup.get(sid)

        if source_row is None:
            unmatched_sentences += 1
            continue

        tokens = list(row["tokens"])
        tags = list(row["tags"])
        source_tokens = list(source_row["tokens"])
        prefixes = list(source_row["source_prefixes"])

        if len(tokens) != len(tags) or len(tokens) != len(source_tokens):
            mismatches += 1
            continue

        for i, token in enumerate(tokens):
            if str(token) != str(source_tokens[i]):
                mismatches += 1
                continue

            prefix = prefixes[i]
            if prefix not in ["CX", "JQ"]:
                continue

            next_word = tokens[i+1] if i+1 < len(tokens) else ""
            proposed = propose_tag(prefix, token, next_word)

            if proposed and missing_tag(tags[i]):
                tags[i] = proposed
                counts[(prefix, proposed)] = (
                    counts.get((prefix, proposed), 0) + 1
                )

        df.at[idx, "tags"] = tags

    print(f"\n{split_name}")
    print("Added labels:", sum(counts.values()))
    print("Mapping breakdown:", counts)
    print("Token mismatches:", mismatches)
    print("Unmatched sentences:", unmatched_sentences)

    return df

source_train = apply_cx_jq_rules(source_train, "TRAIN")
source_val = apply_cx_jq_rules(source_val, "VALIDATION")

print("\nRemaining missing labels:")
for name, df in [
    ("Train", source_train),
    ("Validation", source_val)
]:
    missing = sum(
        missing_tag(tag)
        for sequence in df["tags"]
        for tag in sequence
    )
    print(name, ":", missing)


In [ ]:

# STEP 7: Inspect only the still-unmapped tokens
from collections import Counter, defaultdict
import pandas as pd

def is_missing(x):
    return x is None or (
        isinstance(x, float) and pd.isna(x)
    )

def collect_unmapped(df, split_name):
    records = []

    for _, row in df.iterrows():
        sid = str(row["sentence_id"])
        source_row = source_lookup.get(sid)

        if source_row is None:
            continue

        tokens = list(row["tokens"])
        tags = list(row["tags"])
        source_tokens = list(source_row["tokens"])
        prefixes = list(source_row["source_prefixes"])

        if not (len(tokens) == len(tags) == len(prefixes)):
            print("Length mismatch:", sid)
            continue

        for i, (token, tag) in enumerate(zip(tokens, tags)):
            if not is_missing(tag):
                continue

            if str(token) != str(source_tokens[i]):
                print("Token mismatch:", sid, i, token)
                continue

            records.append({
                "sentence_id": sid,
                "position": i,
                "token": str(token),
                "source_prefix": prefixes[i],
                "previous_word": str(tokens[i-1]) if i > 0 else "",
                "next_word": str(tokens[i+1]) if i+1 < len(tokens) else "",
                "sentence": " ".join(map(str, tokens))
            })

    result = pd.DataFrame(records)

    print(f"\n{'='*65}\n{split_name}")
    print("Remaining missing occurrences:", len(result))

    print("\nCounts by source prefix:")
    print(result["source_prefix"].value_counts().to_string())

    print("\nTop unresolved words by prefix:")
    for prefix in result["source_prefix"].drop_duplicates():
        subset = result[result["source_prefix"] == prefix]
        print(f"\n--- {prefix}: {len(subset)} occurrences ---")
        print(subset["token"].value_counts().head(25).to_string())
        print("\nContext examples:")
        display(subset.head(8))

    return result

unmapped_train = collect_unmapped(source_train, "TRAIN")
unmapped_val = collect_unmapped(source_val, "VALIDATION")

unmapped_train.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "unmapped_literature_train.csv", index=False
)

unmapped_val.to_csv(
    "" + str(RESULTS_DIR) + "/"
    "unmapped_literature_validation.csv", index=False
)

print("\nCombined unresolved occurrences:",
      len(unmapped_train) + len(unmapped_val))


In [ ]:

# STEP 8: Targeted mapping of remaining literature labels

import pandas as pd

# Clear CX mappings supported by the observed examples
CX_FINAL_MAP = {
    # Verbal derivatives: gerunds, infinitives and participles
    "अधिकृत्य": "NV", "प्रणम्य": "NV", "आरूह्य": "NV",
    "प्रवेष्टुम्": "NV", "लेखितुम्": "NV", "स्मृत्वा": "NV",
    "उत्थाप्य": "NV", "द्रष्टुं": "NV", "निमित्तीकृत्य": "NV",
    "उपविश्य": "NV", "गन्तुं": "NV", "निषिध्य": "NV",
    "भवितुम्": "NV", "पीत्वा": "NV", "निश्र्चित्य": "NV",
    "प्रविश्य": "NV", "अन्वेष्टुं": "NV", "कथयितुं": "NV",
    "विरमयितुम्": "NV", "उद्घाटयितुम्ं": "NV",
    "अवशोषयितुं": "NV", "विहृत्य": "NV",
    "सम्पादयितुं": "NV", "विहाय": "NV",
    "पुनरागत्य": "NV", "आदाय": "NV",

    # General particles
    "च": "AVN", "वै": "AVN", "मा": "AVN",
    "हि": "AVN", "नूनम्": "AVN", "कृपया": "AVN",

    # Adverbial forms
    "इतः": "AVP", "कुत्र": "AVP", "एकदा": "AVP",
    "पृथक्": "AVP", "सम्यक्": "AVP", "कदाचित्": "AVP",

    # Governing particles
    "कृते": "K", "विना": "K"
}

# JQ: only map words with a defensible grammatical function
JQ_FINAL_MAP = {
    "द्वाविंशतमः": "A",
    "प्रथमे": "A",
    "प्रथमः": "A",
    "तृतीयं": "A",
    "अग्रिमे": "A",
    "अधिकः": "A",
    "अधिकम्": "A",
    "द्विगुणितः": "A",
    "त्रिगुणितः": "A",
    "सकलाः": "A",
    "पर्याप्तमात्रेण": "A",
    "अन्तिमं": "A",
    "प्रथमा": "A",
    "द्वितीया": "A",
    "अष्टमे": "A",
    "प्रथमं": "A",
    "वह्वयः": "A",

    # Quantifiers modifying nouns
    "सर्वाणि": "SNV",
    "एकाम्": "SNV",
    "पञ्च": "SNV",
    "दश": "SNV",
    "सहस्त्रं": "N",
    "१८६०तमे": "A"
}

# DWH: interrogative word functioning as a noun modifier
DWH_MODIFIERS = {"कः", "का", "कानि", "कियत्", "कीदृशम्"}

# CCL: use context for pronominal quantifiers
CCL_PRONOUNS = {
    "किञ्चित्", "किङ्चित", "किमपि", "कोऽपि",
    "केचन", "काञ्चित्", "कञ्चित्", "कश्र्चन",
    "कस्याञ्चित्", "कस्मिंश्रित्", "कांश्र्चित्",
    "किञ्चित्त्", "केनापि", "किञ्चिद्"
}

# Explicitly grounded proper names from the examples
RDF_PROPER_NAMES = {
    "आजाद", "इलाहाबाद", "जुरासिकपार्क",
    "टाइटैनिक", "लद्दाख", "रियाबोंको",
    "जलियांवालाबाग"
}

def missing_value(x):
    return x is None or (
        isinstance(x, float) and pd.isna(x)
    )

def final_proposal(prefix, token, next_word, sentence):
    token = str(token).strip()
    next_word = str(next_word).strip()

    if prefix == "CX":
        return CX_FINAL_MAP.get(token)

    if prefix == "JQ":
        return JQ_FINAL_MAP.get(token)

    if prefix == "DWH":
        # Interrogative determiner before a noun
        if token in DWH_MODIFIERS and next_word:
            return "SNV"

    if prefix == "CCL":
        if token == "पुरा":
            return "AVP"
        if token in CCL_PRONOUNS and next_word:
            return "SNV"

    if prefix == "RDF":
        if token in RDF_PROPER_NAMES:
            return "NS"

    if prefix == "RDS":
        if token.endswith("."):
            return "AB"
        if token.isdigit() or "/" in token and any(
            ch.isdigit() for ch in token
        ):
            return "AVC"

    if prefix == "RDX":
        if token == "जलियांवालाबाग":
            return "NS"

    # Missing source prefix: use only obvious cases
    if prefix in ("", None):
        if token in {"।", "॥", ",", ".", ";", ":", "?", "!", "'", '"', "[", "]"}:
            return "PU"
        if token == "अलिखत्":
            return "KP"
        if token == "सन्देशः":
            return "N"
        if token == "तेन":
            return "SN"

    return None


def apply_final_rules(df, split_name):
    df = df.copy(deep=True)
    counts = {}
    unresolved = 0
    mismatches = 0

    for idx, row in df.iterrows():
        sid = str(row["sentence_id"])
        source_row = source_lookup.get(sid)

        if source_row is None:
            continue

        tokens = list(row["tokens"])
        tags = list(row["tags"])
        source_tokens = list(source_row["tokens"])
        prefixes = list(source_row["source_prefixes"])

        if not (len(tokens) == len(tags) == len(source_tokens) == len(prefixes)):
            mismatches += 1
            continue

        for i, token in enumerate(tokens):
            if str(token) != str(source_tokens[i]):
                mismatches += 1
                continue

            if not missing_value(tags[i]):
                continue

            next_word = tokens[i+1] if i+1 < len(tokens) else ""
            prefix = prefixes[i]

            proposed = final_proposal(
                prefix, token, next_word, tokens
            )

            if proposed in TARGET_TAGS:
                tags[i] = proposed
                counts[proposed] = counts.get(proposed, 0) + 1
            else:
                unresolved += 1

        df.at[idx, "tags"] = tags

    print(f"\n{split_name}")
    print("New labels:", sum(counts.values()))
    print("Distribution:", counts)
    print("Still unresolved:", unresolved)
    print("Token mismatches:", mismatches)

    return df


source_train = apply_final_rules(source_train, "TRAIN")
source_val = apply_final_rules(source_val, "VALIDATION")

print("\nFINAL MISSING LABEL COUNTS")
for name, df in [
    ("Train", source_train),
    ("Validation", source_val)
]:
    missing = sum(
        missing_value(tag)
        for sequence in df["tags"]
        for tag in sequence
    )
    total = sum(len(sequence) for sequence in df["tags"])
    print(f"{name}: {missing}/{total} missing")


In [ ]:
import pandas as pd
import numpy as np

# Build source lookup using the original source inventory
src_lookup = {
    str(row["sentence_id"]): row
    for _, row in src.iterrows()
}

def get_unresolved(df, split_name):
    rows = []

    for _, row in df.iterrows():
        sid = str(row["sentence_id"])
        tokens = row["tokens"]
        tags = row["tags"]

        source_row = src_lookup.get(sid)
        source_prefixes = (
            source_row["source_prefixes"]
            if source_row is not None else []
        )

        for i, (token, tag) in enumerate(zip(tokens, tags)):
            if pd.isna(tag) or str(tag).strip() == "":
                prefix = (
                    source_prefixes[i]
                    if i < len(source_prefixes) else ""
                )

                rows.append({
                    "split": split_name,
                    "sentence_id": sid,
                    "position": i,
                    "token": token,
                    "source_prefix": prefix,
                    "previous_word": tokens[i-1] if i > 0 else "",
                    "next_word": tokens[i+1] if i+1 < len(tokens) else "",
                    "sentence": " ".join(tokens)
                })

    return pd.DataFrame(rows)


unmapped_train = get_unresolved(source_train, "TRAIN")
unmapped_val = get_unresolved(source_val, "VALIDATION")

unmapped_all = pd.concat(
    [unmapped_train, unmapped_val],
    ignore_index=True
)

print("TRAIN:", len(unmapped_train))
print("VALIDATION:", len(unmapped_val))
print("\nCombined source-prefix distribution:")
print(unmapped_all["source_prefix"].value_counts(dropna=False))

print("\nUnresolved tokens by split and prefix:")
print(pd.crosstab(
    unmapped_all["source_prefix"],
    unmapped_all["split"]
))

display(unmapped_all)

In [ ]:
# Display every unresolved occurrence without truncation
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

for prefix, group in unmapped_all.groupby(
    "source_prefix", dropna=False, sort=True
):
    print("\n" + "=" * 100)
    print(f"SOURCE PREFIX: {prefix!r} | OCCURRENCES: {len(group)}")
    print("=" * 100)

    for _, r in group.iterrows():
        print(
            f'{r["split"]} | ID={r["sentence_id"]} | '
            f'POS={r["position"]} | TOKEN={r["token"]} | '
            f'PREV={r["previous_word"]} | NEXT={r["next_word"]}'
        )
        print(f'  SENTENCE: {r["sentence"]}')

In [ ]:

import pandas as pd
import re

# High-confidence lexical mappings for the remaining CX tokens
CX_NV = {
    "प्रसाद्य", "दर्शयित्वा", "परिगृह्य", "विक्षोभ्य",
    "वर्धयितुं", "वर्णयितुं", "अभिनीय", "निष्क्रम्य",
    "वस्तुम्", "दृष्टुम्", "श्रोतुं", "दृष्टुं", "आघ्रातुं",
    "शयितुम्", "उपगम्य", "निर्वर्ण्य", "आदातुम्",
    "अवलम्व्य", "वर्जयित्वा", "पूरयितुं", "विचिन्त्य",
    "निर्गत्य", "उध्दत्य", "आलिङ्गितुं", "धृत्वा",
    "आहुय", "स्वीकर्तुं", "आरोप्य", "नमस्कृत्य",
    "वक्तुं", "कारयित्वा", "आलिङ्गय", "अवगन्तुं",
    "अनुसृत्य", "उपायनीकृत्य", "कर्तुम्", "स्वीकृत्य",
    "लिखित्वा", "निक्षिप्य", "आकृष्य", "उपवेश्य",
    "आप्रष्टुं", "विहस्य", "अनुक्त्वा", "सति",
    "ज्ञापयितुं", "उद्दिश्य", "आनेतुम्", "प्रसार्य",
    "संयम्य", "मार्जयितुं", "अर्पयितुं", "आह्वातुं",
    "द्योतयितुं", "संस्थाप्य", "दत्त्वा", "प्रत्यागत्य",
    "स्थापयित्वा", "पिधाय", "आरुह्य", "पतित्वा",
    "साधयितुं", "निष्कास्य", "अथगयित्वा", "आनीय",
    "रक्षितुम्", "प्रक्षिप्य", "भक्षयितुं", "हत्वा",
    "अवरुद्ध्य", "बद्ध्वा", "पलायितुं", "संवर्धयितुं",
    "उन्नीय", "उच्चार्य", "नेतुम्", "शिक्षियितुं",
    "दॄष्ट्वा", "प्रहर्तुं", "प्रत्यक्षीकर्तुं",
    "प्रत्यक्षिकर्तुं", "बहिष्कर्तुं", "आदातुम्",
    "प्रवेष्टुम्", "लेखितुम्", "स्मृत्वा", "उत्थाप्य",
    "द्रष्टुं", "निमित्तीकृत्य", "उपविश्य", "गन्तुं",
    "निषिध्य", "भवितुम्", "पीत्वा", "निश्र्चित्य",
    "प्रविश्य", "अन्वेष्टुं", "कथयितुं", "विरमयितुम्",
    "उद्घाटयितुम्ं", "अवशोषयितुं", "विहृत्य",
    "सम्पादयितुं", "विहाय", "पुनरागत्य", "आदाय",
    "प्रशंस्य", "संस्तुत्य", "प्रतिनिवर्तितुम्",
    "उत्प्लुत्य", "अवलम्ब्य", "उत्थाय", "आप्त्वा",
    "गन्तुम्", "मलिनीकर्तुं", "निर्मलीकर्तुं",
    "आनाय्य", "आहूय", "गोपयित्वा", "उपवेशयितुं",
    "दर्शयित्वा", "आवश्यक", "आवश्यकताः"
}

CX_AVP = {
    "कुतः", "इतः", "अतएव", "अन्यत्र", "यथाकालं",
    "यथेष्टं", "तदर्थ", "यथेच्छम्", "प्रतिपक्षतः",
    "एकतः", "अन्यतः", "तत्र", "सहसा", "बहिः",
    "प्रायेण", "सदैव", "सर्वतः", "दूरतः", "सर्वत्र",
    "कदाचित्", "पृथक्", "सम्यक्", "मनाक्", "अतीव",
    "बहुधा", "भृशं", "कोटिशः", "एकदा", "अति",
    "यथाकालम्"
}

CX_AVN = {
    "ओम्", "नमः", "ननु", "नहि", "नु", "किल",
    "आम्", "आदि", "केवलं", "तर्हि", "नहि",
    "नित्यं", "हि", "नूनम्", "इत्यादि", "एव",
    "परम्", "न", "च", "वै", "मा", "इव", "अपि",
    "वा", "तु", "स्म", "खलु", "इति", "अथ",
    "नमोनमः"
}

CX_K = {
    "बिना", "अन्तरा", "उपरि", "अलम्"
}

CX_SAMB = {
    "यथा", "अपितु", "उत", "नोचेत्"
}

# Prefix RDF: map only clear Sanskrit-compatible categories
RDF_MAP = {
    "चौसा": "NS",       # named mango variety; retain this convention consistently
    "लँगडा": "NS",      # named mango variety
    "लद्दाख": "NS",
    "ताजिया": "N",
    "अक्टुबर": "N",
    "अक्तोबर्": "N",
    "अक्टोबर्": "N",
    "अक्टोबर": "N",
    "मे": "N",
    "कृपया": "AVN"
}

# JQ: classify by grammatical function in the displayed context
JQ_MAP = {
    "अतीव": "AVP",
    "बहुधा": "AVP",
    "कोटिशः": "AVP",
    "भृशं": "AVP",
    "एतावत्": "SNV",
    "एकां": "SNV",
    "बहवः": "SNV",
    "अपरः": "SNV",
    "सहस्रं": "N",
    "सकला": "A",
    "आद्यं": "A"
}

def clean_token(x):
    return re.sub(r'^[\'"“”‘’]+|[\'"“”‘’,।!?]+$', '', str(x)).strip()

def classify_remaining(row):
    prefix = str(row["source_prefix"]).strip()
    token = clean_token(row["token"])
    next_word = clean_token(row["next_word"])

    if prefix == "CX":
        if token in CX_NV:
            return "NV"
        if token in CX_K:
            return "K"
        if token in CX_SAMB:
            return "SAMB"
        if token in CX_AVP:
            return "AVP"
        if token in CX_AVN:
            return "AVN"

        # Pronouns used independently or as modifiers
        if token == "अन्यत्":
            return "SN"

        if token in {"कश्चित्", "कश्र्चित्", "कश्र्चन",
                     "कानिचन", "कस्याञ्चित्", "कस्मिंश्चित्",
                     "कस्मिंश्रित्", "काञ्चित्", "कांश्र्चित्",
                     "कञ्चित्", "किञ्चित्", "किमपि",
                     "कापि", "केचन", "कश्चन"}:

            if token == "किञ्चित्":
                if next_word in {"हसन्ती", "अन्तर्मुखी", "पृष्टे"}:
                    return "AVP"
                if next_word in {"फलम्", "कालान्तरम्"}:
                    return "SNV"
                if next_word in {"विहस्य", "अनुक्त्वा"}:
                    return "SN"

            if token == "किमपि":
                if next_word in {"फलम्", "करणीयम्", "नित्यः"}:
                    return "SNV"
                if next_word in {"वक्तुम्", "अनुक्त्वा"}:
                    return "SN"

            if token in {"कापि", "कश्चित्", "कश्र्चित्",
                         "कश्र्चन", "कानिचन", "कस्याञ्चित्",
                         "कस्मिंश्चित्", "कस्मिंश्रित्",
                         "काञ्चित्", "कांश्र्चित्", "केचन"}:
                if next_word in {
                    "उपायः", "आर्थिकसमस्या", "न्यूनता",
                    "मनश्र्चिकित्सकः", "कम्पः",
                    "आनुकूल्यानि", "स्वायत्तसंस्थायां",
                    "वने", "वृश्र्चिकान्"
                }:
                    return "SNV"

            if token == "केचन" and next_word in {"एतत्", "उक्तवन्तः", "चिन्तितवान्"}:
                return "SN"

            if token == "कञ्चित्" and next_word == "उद्दिश्य":
                return "SN"

        if token == "उभयम्":
            return "SN"

        if token == "साधु":
            return "AVU"

        if token == "तमे":
            return "A"

        if token == "तावत्":
            return "AVN"

        if token in {"यथा-"}:
            return "AVP"

    elif prefix == "JQ":
        return JQ_MAP.get(token)

    elif prefix == "RDF":
        return RDF_MAP.get(token)

    elif prefix == "RDX":
        if token == "बिभ्रन्न":
            return "NV"

    elif prefix == "RDS":
        # "क" is a house-number identifier, not a Sanskrit word.
        return None

    elif prefix == "":
        if token in {"..."}:
            return "PU"

    return None


def apply_remaining_mapping(df, split_name):
    added = 0
    distribution = {}

    for _, r in unmapped_all[unmapped_all["split"] == split_name].iterrows():
        tag = classify_remaining(r)

        if tag is None:
            continue

        sid = str(r["sentence_id"])
        pos = int(r["position"])

        mask = df["sentence_id"].astype(str) == sid
        if not mask.any():
            continue

        idx = df.index[mask][0]
        tags = list(df.at[idx, "tags"])

        if pos >= len(tags):
            continue

        old = tags[pos]
        if pd.notna(old) and str(old).strip():
            continue

        tags[pos] = tag
        df.at[idx, "tags"] = tags

        added += 1
        distribution[tag] = distribution.get(tag, 0) + 1

    print(f"\n{split_name}")
    print("New labels:", added)
    print("Distribution:", distribution)


apply_remaining_mapping(source_train, "TRAIN")
apply_remaining_mapping(source_val, "VALIDATION")


In [ ]:
def audit_split(df, name):
    total = 0
    missing = 0
    invalid = 0

    for _, row in df.iterrows():
        tokens = row["tokens"]
        tags = row["tags"]

        if len(tokens) != len(tags):
            invalid += 1
            continue

        total += len(tokens)
        missing += sum(
            pd.isna(t) or str(t).strip() == ""
            for t in tags
        )

    print(
        f"{name}: total={total}, missing={missing}, "
        f"coverage={(total-missing)/total*100:.2f}%, "
        f"length_errors={invalid}"
    )

audit_split(source_train, "TRAIN")
audit_split(source_val, "VALIDATION")

print(
    "Sentence ID overlap:",
    len(
        set(source_train["sentence_id"].astype(str))
        & set(source_val["sentence_id"].astype(str))
    )
)

In [ ]:
import pandas as pd
from pathlib import Path

def keep_complete_sentences(df):
    complete_indices = []
    incomplete_indices = []

    for idx, row in df.iterrows():
        tokens = row["tokens"]
        tags = row["tags"]

        valid = (
            len(tokens) == len(tags)
            and all(
                pd.notna(tag) and str(tag).strip() != ""
                for tag in tags
            )
        )

        if valid:
            complete_indices.append(idx)
        else:
            incomplete_indices.append(idx)

    return (
        df.loc[complete_indices].copy().reset_index(drop=True),
        df.loc[incomplete_indices].copy().reset_index(drop=True)
    )


source_train_final, source_train_incomplete = keep_complete_sentences(source_train)
source_val_final, source_val_incomplete = keep_complete_sentences(source_val)

print("TRAIN")
print("Original sentences:", len(source_train))
print("Complete sentences:", len(source_train_final))
print("Excluded sentences:", len(source_train_incomplete))

print("\nVALIDATION")
print("Original sentences:", len(source_val))
print("Complete sentences:", len(source_val_final))
print("Excluded sentences:", len(source_val_incomplete))

# Verify the final datasets
assert all(
    len(r["tokens"]) == len(r["tags"])
    and all(pd.notna(t) and str(t).strip() for t in r["tags"])
    for _, r in source_train_final.iterrows()
)

assert all(
    len(r["tokens"]) == len(r["tags"])
    and all(pd.notna(t) and str(t).strip() for t in r["tags"])
    for _, r in source_val_final.iterrows()
)

# Save without overwriting previous intermediate files
OUT_DIR = Path(
    "" + str(RESULTS_DIR) + "/Final_Literature_Data"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

source_train_final.to_pickle(OUT_DIR / "literature_train_final.pkl")
source_val_final.to_pickle(OUT_DIR / "literature_validation_final.pkl")

source_train_incomplete.to_csv(
    OUT_DIR / "excluded_train_incomplete.csv", index=False
)
source_val_incomplete.to_csv(
    OUT_DIR / "excluded_validation_incomplete.csv", index=False
)

print("\nFinal datasets saved successfully.")

In [ ]:
from collections import Counter
import pandas as pd

TARGET_TAGS = [
    "N", "NS", "NV", "KNV", "SN", "SNV", "SNS",
    "SAM", "SAMY", "SAMP", "KP", "UP", "AVN",
    "AVP", "AVUP", "SAMB", "AVU", "K", "AKV",
    "AB", "AVC", "DVI", "A", "PU"
]

def get_tag_counts(df):
    counts = Counter(
        tag
        for tags in df["tags"]
        for tag in tags
    )
    return counts

datasets = {
    "Literature Train": source_train_final,
    "Literature Validation": source_val_final,
    "News Train Pool": news_train_pool,
    "News Validation": news_val,
    "News Test": news_test
}

distribution = pd.DataFrame({
    name: get_tag_counts(df)
    for name, df in datasets.items()
}).fillna(0).astype(int)

distribution.index.name = "POS Tag"

print("TAG DISTRIBUTION")
display(distribution)

print("\nTAGS ABSENT FROM LITERATURE TRAIN:")
train_tags = set(get_tag_counts(source_train_final))
print(sorted(set(TARGET_TAGS) - train_tags))

print("\nTAGS NOT IN TARGET TAGSET:")
for name, df in datasets.items():
    observed = set(get_tag_counts(df))
    unexpected = observed - set(TARGET_TAGS)
    print(name, sorted(unexpected))

print("\nDATASET COLUMN STRUCTURE:")
for name, df in datasets.items():
    print(name, list(df.columns), "Shape:", df.shape)

In [ ]:
from collections import Counter
import pandas as pd

# Original source tag distribution
source_tag_counts = Counter(
    tag
    for tags in src["tags"]
    for tag in tags
)

print("ORIGINAL SOURCE TAG DISTRIBUTION")
display(
    pd.DataFrame(
        source_tag_counts.items(),
        columns=["Source_Tag", "Count"]
    ).sort_values("Count", ascending=False)
)

print("\nCURRENT SOURCE-TO-TARGET MAPPING")
for source_tag, target_tag in SOURCE_PREFIX_TO_TARGET.items():
    print(f"{source_tag:10} -> {target_tag}")

print("\nUNMAPPED SOURCE PREFIXES")
unmapped_prefix_counts = Counter(
    prefix
    for prefixes in src["source_prefixes"]
    for prefix in prefixes
    if pd.notna(prefix)
    and str(prefix).strip()
    and str(prefix).strip() not in SOURCE_PREFIX_TO_TARGET
)

display(
    pd.DataFrame(
        unmapped_prefix_counts.items(),
        columns=["Source_Prefix", "Count"]
    ).sort_values("Count", ascending=False)
)

In [ ]:
from collections import Counter
import pandas as pd

def prefix_distribution(df):
    counts = Counter(
        str(prefix).strip()
        for prefixes in df["source_prefixes"]
        for prefix in prefixes
        if pd.notna(prefix) and str(prefix).strip()
    )
    return counts

train_prefix_counts = prefix_distribution(src)

mapping_audit = []

for prefix, count in train_prefix_counts.items():
    mapped = prefix in SOURCE_PREFIX_TO_TARGET

    mapping_audit.append({
        "Source_Prefix": prefix,
        "Count": count,
        "In_Current_Dictionary": mapped,
        "Current_Target": SOURCE_PREFIX_TO_TARGET.get(prefix, "NOT IN DICTIONARY")
    })

mapping_audit_df = pd.DataFrame(mapping_audit)

display(
    mapping_audit_df.sort_values(
        ["In_Current_Dictionary", "Count"],
        ascending=[True, False]
    )
)

print("Total source prefixes:", len(mapping_audit_df))
print(
    "Prefixes in current dictionary:",
    mapping_audit_df["In_Current_Dictionary"].sum()
)
print(
    "Prefixes missing from dictionary:",
    (~mapping_audit_df["In_Current_Dictionary"]).sum()
)

In [ ]:
SOURCE_PREFIX_TO_TARGET = {
    # Nouns, adjectives and verbs
    "NC": "N",
    "NP": "NS",
    "JJ": "A",
    "V": "KP",
    "VBP": "NV",
    "VBG": "NV",

    # Punctuation
    "PU": "PU",

    # Adverbs and particles
    "ALC": "AVP",
    "AMN": "AVP",
    "CEM": "AVN",
    "CNG": "AVN",

    # Conjunctions and interjections
    "CIN": "AVU",
    "CCD": "SAMB",
    "CSB": "SAMB",

    # Pronoun families: provisional default
    "PPR": "SN",
    "PRC": "SN",
    "PRF": "SN",
    "PRL": "SN",
    "PWH": "SN"
}

CONTEXTUAL_PREFIXES = {
    "CCL", "CX", "DAB", "DRL", "DWH",
    "JQ", "RDF", "RDS", "RDX"
}

all_prefixes = set(train_prefix_counts.keys())
classified = set(SOURCE_PREFIX_TO_TARGET) | CONTEXTUAL_PREFIXES

print("Total prefixes:", len(all_prefixes))
print("Direct mappings:", len(all_prefixes & set(SOURCE_PREFIX_TO_TARGET)))
print("Contextual prefixes:", len(all_prefixes & CONTEXTUAL_PREFIXES))
print("Unclassified prefixes:", sorted(all_prefixes - classified))

assert all_prefixes <= classified, "Some source prefixes remain unclassified!"

print("\nSUCCESS: All source prefixes are classified.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np

# Configure seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch configuration initiated for experiments.")

# Part C — Complete Experimental Pipeline
## Sanskrit POS Cross-Domain Generalization & Few-Shot Adaptation

This section implements and executes the complete research methodology for your M.Tech dissertation:

- **Target Tagset**: 24 standardized Sanskrit POS labels.
- **Models Evaluated**:
  1. **M1**: BiLSTM-CRF (word embeddings + 2-layer BiLSTM + Linear-Chain CRF).
  2. **M2**: XLM-RoBERTa (`FacebookAI/xlm-roberta-base` with word-to-subword alignment).
  3. **M3**: IndicBERT v2 (`ai4bharat/IndicBERTv2-MLM-only` with word-to-subword alignment and Sanskrit `<sa>` token).
- **Four Research Experiments**:
  - **Experiment 1**: Literature In-Domain (Literature Train $\rightarrow$ Validation $\rightarrow$ Held-Out Test).
  - **Experiment 2**: News In-Domain Reference (News Adaptation Pool $\rightarrow$ Validation $\rightarrow$ Locked Test).
  - **Experiment 3**: Zero-Shot Cross-Domain Generalization (Literature $\rightarrow$ News Locked Test).
  - **Experiment 4**: Few-Shot Target Adaptation (Fine-tuning literature checkpoints on 50, 100, 200, 500, 1000 sentences across seeds 42, 123, 2026).
- **Downstream Application**: Inference on 3,000 unannotated Sanskrit news sentences.
- **Reporting & Visualizations**: Automatic compilation of Tables 1, 2, 3, 4, confusion matrix heatmaps, and learning curve plots.

In [ ]:
# ============================================================
# C.1: LOAD LEAK-FREE DATASET SPLITS AND VERIFY LEAKAGE
# ============================================================

import json
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Standard target 24 POS tags
TARGET_TAGS = [
    "N", "NS", "NV", "KNV", "SN", "SNV", "SNS", "SAM", "SAMY", "SAMP",
    "KP", "UP", "AVN", "AVP", "AVUP", "SAMB", "AVU", "K", "AKV", "AB",
    "AVC", "DVI", "A", "PU"
]
TAG2ID = {tag: idx for idx, tag in enumerate(TARGET_TAGS)}
ID2TAG = {idx: tag for idx, tag in enumerate(TARGET_TAGS)}

# Ensure PROJECT_DIR is defined
if "PROJECT_DIR" not in globals():
    if Path("/content").is_dir():
        PROJECT_DIR = Path("/content/Cross_Domain")
    elif Path("/kaggle/working").is_dir():
        PROJECT_DIR = Path("/kaggle/working/Cross_Domain")
    else:
        PROJECT_DIR = Path.cwd().resolve()
        if PROJECT_DIR.name != "Cross_Domain" and (PROJECT_DIR / "Cross_Domain").is_dir():
            PROJECT_DIR = PROJECT_DIR / "Cross_Domain"

lit_dir = PROJECT_DIR / "outputs" / "datasets" / "Final_Literature_Data"
datasets_dir = PROJECT_DIR / "outputs" / "datasets"
lit_dir.mkdir(parents=True, exist_ok=True)
datasets_dir.mkdir(parents=True, exist_ok=True)

def sentence_key(row):
    doc_id = getattr(row, "doc_id", None)
    sentence_id = getattr(row, "sentence_id", None)
    if doc_id is None and isinstance(row, (dict, pd.Series)):
        doc_id = row.get("doc_id", row.get("sentence_id"))
        sentence_id = row.get("sentence_id")
    return f"{doc_id}::{sentence_id}"

def validate_split_leakage(splits_dict):
    # Literature domain check
    lit_seen = {}
    for sname in ["literature_train", "literature_validation", "literature_test"]:
        if sname in splits_dict:
            for row in splits_dict[sname].itertuples():
                key = sentence_key(row)
                if key in lit_seen:
                    raise AssertionError(f"Literature leakage: {key} in {lit_seen[key]} and {sname}")
                lit_seen[key] = sname

    # News domain document check
    news_docs = {}
    for sname in ["news_adaptation_pool", "news_validation", "news_test"]:
        if sname in splits_dict:
            for doc in splits_dict[sname]["doc_id"].unique():
                if doc in news_docs:
                    raise AssertionError(f"News document leakage: {doc} in {news_docs[doc]} and {sname}")
                news_docs[doc] = sname

# ------------------------------------------------------------
# STEP 1: If user clicked 'Run All', use verified in-memory datasets directly
# ------------------------------------------------------------
splits = {}

if "source_train_final" in globals() and "news_train_pool" in globals():
    print("Using verified dataset splits directly from notebook session...")
    groups = source_train_final["doc_id"].astype(str).drop_duplicates().tolist()
    train_groups, test_groups = train_test_split(groups, test_size=0.2, random_state=42)

    splits["literature_train"] = source_train_final[
        source_train_final["doc_id"].astype(str).isin(set(train_groups))
    ].copy().reset_index(drop=True)
    splits["literature_test"] = source_train_final[
        source_train_final["doc_id"].astype(str).isin(set(test_groups))
    ].copy().reset_index(drop=True)
    splits["literature_validation"] = source_val_final.copy().reset_index(drop=True)

    splits["news_adaptation_pool"] = news_train_pool.copy().reset_index(drop=True)
    splits["news_validation"] = news_val.copy().reset_index(drop=True)
    splits["news_test"] = news_test.copy().reset_index(drop=True)

    # Persist to disk for subsequent cells
    for name, df in splits.items():
        folder = lit_dir if "literature" in name else datasets_dir
        folder.mkdir(parents=True, exist_ok=True)
        try:
            df.to_json(folder / f"{name}.json", orient="records", force_ascii=False, indent=2)
            df.to_pickle(folder / f"{name}.pkl")
        except Exception:
            pass

# ------------------------------------------------------------
# STEP 2: If standalone run, load from disk or auto-fetch from Kaggle
# ------------------------------------------------------------
else:
    def ensure_splits_available():
        """Checks if split files exist; if not, searches local machine or auto-downloads from Kaggle."""
        check_files = [
            lit_dir / "literature_train.json",
            lit_dir / "literature_train.pkl",
            lit_dir / "literature_train.csv",
            datasets_dir / "literature_train.csv",
        ]
        for cf in check_files:
            if cf.is_file():
                if cf.suffix == ".json":
                    return
                if cf.suffix == ".pkl":
                    try:
                        pd.read_pickle(cf)
                        return
                    except Exception:
                        pass

        # Search locally across search roots
        search_roots = [
            PROJECT_DIR,
            PROJECT_DIR.parent,
            Path("/content"),
            Path("/kaggle"),
            Path.cwd(),
            Path.home() / "Desktop" / "Cross_Domain",
        ]
        for root in search_roots:
            if root.is_dir():
                found = list(root.rglob("literature_train.*"))
                for f in found:
                    if f.suffix in [".json", ".csv"]:
                        found_dir = f.parent
                        print(f"Found existing dataset splits in {found_dir}. Synchronizing...")
                        shutil.copytree(found_dir, lit_dir, dirs_exist_ok=True)
                        if found_dir.parent.is_dir():
                            shutil.copytree(found_dir.parent, datasets_dir, dirs_exist_ok=True)
                        return

        # Auto-fetch from Kaggle directly
        print("Split files not found on remote VM. Automatically downloading from Kaggle...")
        os.environ["KAGGLE_API_TOKEN"] = "KGAT_c595dea7dbc740b795edcc95f32ed945"
        k_dir = Path.home() / ".kaggle"
        k_dir.mkdir(parents=True, exist_ok=True)
        (k_dir / "access_token").write_text("KGAT_c595dea7dbc740b795edcc95f32ed945", encoding="utf-8")
        try:
            (k_dir / "access_token").chmod(0o600)
        except Exception:
            pass
        os.system(f"pip install -q kaggle && kaggle datasets download -d badshahkumar/sanskrit-pos-crossdomain --unzip -p {PROJECT_DIR}")
        for b_src, b_dst in [
            (PROJECT_DIR / "kaggle_bundle/outputs/datasets", datasets_dir),
            (PROJECT_DIR / "kaggle_bundle/outputs/datasets/Final_Literature_Data", lit_dir),
            (PROJECT_DIR / "kaggle_bundle/configs", PROJECT_DIR / "configs"),
            (PROJECT_DIR / "outputs/datasets", datasets_dir),
            (PROJECT_DIR / "outputs/datasets/Final_Literature_Data", lit_dir),
        ]:
            if b_src.is_dir() and b_src != b_dst:
                shutil.copytree(b_src, b_dst, dirs_exist_ok=True)

    ensure_splits_available()

    def load_split_dataset(name, folder, basename, csv_fallback=None):
        folder = Path(folder)
        candidates = [
            folder / f"{basename}.json",
            datasets_dir / f"{basename}.json",
            lit_dir / f"{basename}.json",
            folder / f"{basename}.pkl",
            datasets_dir / f"{basename}.pkl",
            lit_dir / f"{basename}.pkl",
        ]
        if csv_fallback:
            candidates.extend([
                folder / csv_fallback,
                datasets_dir / csv_fallback,
                lit_dir / csv_fallback,
            ])

        for p in candidates:
            if p.is_file():
                if p.suffix == ".json":
                    try:
                        with open(p, "r", encoding="utf-8") as f:
                            return pd.DataFrame(json.load(f))
                    except Exception:
                        pass
                elif p.suffix == ".pkl":
                    try:
                        return pd.read_pickle(p)
                    except Exception:
                        pass
                elif p.suffix == ".csv":
                    try:
                        df_csv = pd.read_csv(p)
                        tag_col = "tag" if "tag" in df_csv.columns else "gold_sanskrit_tag"
                        return df_csv.groupby(["doc_id", "sentence_id"], sort=False).agg({
                            "token": list,
                            tag_col: list
                        }).reset_index().rename(columns={"token": "tokens", tag_col: "tags"})
                    except Exception:
                        pass

        for pattern in [f"{basename}.json", f"{basename}.pkl", csv_fallback]:
            if not pattern:
                continue
            for search_root in [PROJECT_DIR, Path("/content"), Path("/kaggle")]:
                if search_root.is_dir():
                    matches = list(search_root.rglob(pattern))
                    for m in matches:
                        if m.is_file():
                            if m.suffix == ".json":
                                try:
                                    with open(m, "r", encoding="utf-8") as f:
                                        return pd.DataFrame(json.load(f))
                                except Exception:
                                    pass
                            elif m.suffix == ".pkl":
                                try:
                                    return pd.read_pickle(m)
                                except Exception:
                                    pass
                            elif m.suffix == ".csv":
                                try:
                                    df_csv = pd.read_csv(m)
                                    tag_col = "tag" if "tag" in df_csv.columns else "gold_sanskrit_tag"
                                    return df_csv.groupby(["doc_id", "sentence_id"], sort=False).agg({
                                        "token": list,
                                        tag_col: list
                                    }).reset_index().rename(columns={"token": "tokens", tag_col: "tags"})
                                except Exception:
                                    pass

        raise FileNotFoundError(f"Could not load split '{name}' from JSON, PKL, or CSV.")

    splits = {
        "literature_train": load_split_dataset("literature_train", lit_dir, "literature_train", "literature_train.csv"),
        "literature_validation": load_split_dataset("literature_validation", lit_dir, "literature_validation", "literature_validation.csv"),
        "literature_test": load_split_dataset("literature_test", lit_dir, "literature_test_locked", "literature_test_gold.csv"),
        "news_adaptation_pool": load_split_dataset("news_adaptation_pool", datasets_dir, "news_train_pool", "news_adaptation_pool.csv"),
        "news_validation": load_split_dataset("news_validation", datasets_dir, "news_validation", "news_validation.csv"),
        "news_test": load_split_dataset("news_test", datasets_dir, "news_test_locked", "news_test_gold.csv"),
    }

validate_split_leakage(splits)
print("SUCCESS: Zero sentence and zero document leakage verified across all splits!")
print()

for name, df in splits.items():
    total_tokens = sum(len(seq) for seq in df["tokens"])
    print(f"{name:25s}: {len(df):4d} sentences, {total_tokens:5d} tokens")

unann_path = PROJECT_DIR / "data" / "raw" / "News_Unannotated.csv" if "PROJECT_DIR" in globals() else Path("data/raw/News_Unannotated.csv")
if not unann_path.is_file():
    cand_unann = list(PROJECT_DIR.rglob("News_Unannotated.csv"))
    if cand_unann:
        unann_path = cand_unann[0]
news_unannotated = pd.read_csv(unann_path)
print(f"news_unannotated         : {len(news_unannotated):4d} sentences")



In [ ]:
# ============================================================
# C.2: TARGET POS TAGSET (24 TAGS)
# ============================================================

print("Target 24-Tag POS Schema:")
for idx, tag in enumerate(TARGET_TAGS):
    print(f"{idx:02d}: {tag:4s}", end="  " if (idx + 1) % 6 != 0 else "\n")

assert len(TARGET_TAGS) == 24
assert len(TAG2ID) == 24
assert len(ID2TAG) == 24
print("\nTagset definition verified.")

In [ ]:
# ============================================================
# C.3: M1 ARCHITECTURE — MORPHOLOGICALLY-AWARE BiLSTM-CRF
#       (Word + Character-CNN + Suffix/Prefix Affix Features)
# ============================================================

import torch
import torch.nn as nn
from pathlib import Path
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Execution device:", DEVICE)

def build_vocabulary(sequences, min_frequency=1):
    """
    Builds multi-tier lexical, affix, and character vocabularies.
    This directly addresses Sanskrit morphological inflection (vibhakti/tiṅanta)
    and dramatically reduces cross-domain Out-Of-Vocabulary (OOV) rates.
    """
    w_counts, s_counts, p_counts, c_counts = {}, {}, {}, {}
    for seq in sequences:
        for w in seq:
            w_counts[w] = w_counts.get(w, 0) + 1
            s = w[-3:] if len(w) >= 3 else w
            p = w[:3] if len(w) >= 3 else w
            s_counts[s] = s_counts.get(s, 0) + 1
            p_counts[p] = p_counts.get(p, 0) + 1
            for c in w:
                c_counts[c] = c_counts.get(c, 0) + 1

    def make_vocab(counts, min_f=1):
        v = {"<pad>": 0, "<unk>": 1}
        for item, count in sorted(counts.items()):
            if count >= min_f and item not in ["<pad>", "<unk>"]:
                v[item] = len(v)
        return v

    return {
        "word": make_vocab(w_counts, min_frequency),
        "suffix": make_vocab(s_counts, 1),
        "prefix": make_vocab(p_counts, 1),
        "char": make_vocab(c_counts, 1),
    }

class SequenceTaggingDataset(Dataset):
    """
    Morphologically-aware sequence dataset extracting word IDs, suffix IDs,
    prefix IDs, and character matrices.
    """
    def __init__(self, frame, vocabulary, label_to_id=None, max_word_len=15):
        self.examples = []
        label_to_id = label_to_id or TAG2ID

        if isinstance(vocabulary, dict) and "word" in vocabulary:
            self.w_voc = vocabulary["word"]
            self.s_voc = vocabulary["suffix"]
            self.p_voc = vocabulary["prefix"]
            self.c_voc = vocabulary["char"]
            self.has_morph = True
        else:
            self.w_voc = vocabulary
            self.has_morph = False

        self.max_word_len = max_word_len

        for row in frame.itertuples():
            w_ids = [self.w_voc.get(tok, 1) for tok in row.tokens]
            labels = [label_to_id.get(tag, 0) for tag in row.tags]

            item = {
                "input_ids": torch.tensor(w_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "mask": torch.ones(len(w_ids), dtype=torch.bool),
            }

            if self.has_morph:
                s_ids = [self.s_voc.get(tok[-3:] if len(tok) >= 3 else tok, 1) for tok in row.tokens]
                p_ids = [self.p_voc.get(tok[:3] if len(tok) >= 3 else tok, 1) for tok in row.tokens]

                c_matrix = []
                for tok in row.tokens:
                    c_row = [self.c_voc.get(c, 1) for c in tok[:max_word_len]]
                    if len(c_row) < max_word_len:
                        c_row += [0] * (max_word_len - len(c_row))
                    c_matrix.append(c_row)

                item["suffix_ids"] = torch.tensor(s_ids, dtype=torch.long)
                item["prefix_ids"] = torch.tensor(p_ids, dtype=torch.long)
                item["char_ids"] = torch.tensor(c_matrix, dtype=torch.long)

            self.examples.append(item)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_sequences(batch, pad_id=0):
    collated = {
        "input_ids": pad_sequence([item["input_ids"] for item in batch], batch_first=True, padding_value=pad_id),
        "labels": pad_sequence([item["labels"] for item in batch], batch_first=True, padding_value=0),
        "mask": pad_sequence([item["mask"] for item in batch], batch_first=True, padding_value=False),
    }
    if "suffix_ids" in batch[0]:
        collated["suffix_ids"] = pad_sequence([item["suffix_ids"] for item in batch], batch_first=True, padding_value=pad_id)
        collated["prefix_ids"] = pad_sequence([item["prefix_ids"] for item in batch], batch_first=True, padding_value=pad_id)
        collated["char_ids"] = pad_sequence([item["char_ids"] for item in batch], batch_first=True, padding_value=pad_id)
    return collated

class BiLSTMCRF(nn.Module):
    """
    Enhanced BiLSTM-CRF sequence tagger with integrated Char-CNN,
    Suffix/Prefix affix representations, and 2-layer LayerNorm BiLSTM.
    """
    def __init__(
        self,
        vocab_size,
        num_labels=24,
        embedding_dim=128,
        hidden_size=128,
        dropout=0.3,
        pad_idx=0,
        use_morphology=True,
        char_vocab_size=120,
        suffix_vocab_size=1500,
        prefix_vocab_size=1500,
        char_dim=32,
        char_filters=64,
        affix_dim=32,
        num_layers=2,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_labels = num_labels
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.dropout_prob = dropout
        self.pad_idx = pad_idx
        self.use_morphology = use_morphology
        self.num_layers = num_layers
        self.char_vocab_size = char_vocab_size
        self.suffix_vocab_size = suffix_vocab_size
        self.prefix_vocab_size = prefix_vocab_size
        self.char_dim = char_dim
        self.char_filters = char_filters
        self.affix_dim = affix_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        if use_morphology:
            self.suffix_embedding = nn.Embedding(suffix_vocab_size, affix_dim, padding_idx=0)
            self.prefix_embedding = nn.Embedding(prefix_vocab_size, affix_dim, padding_idx=0)
            self.char_embedding = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
            self.char_cnn = nn.Conv1d(char_dim, char_filters, kernel_size=3, padding=1)
            lstm_in_dim = embedding_dim + (affix_dim * 2) + char_filters
        else:
            lstm_in_dim = embedding_dim

        self.encoder = nn.LSTM(
            lstm_in_dim,
            hidden_size,
            num_layers=num_layers if use_morphology else 1,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if (use_morphology and num_layers > 1) else 0,
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2) if use_morphology else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def emissions(self, token_ids, suffix_ids=None, prefix_ids=None, char_ids=None):
        embedded = self.embedding(token_ids)
        if self.use_morphology and suffix_ids is not None and char_ids is not None:
            s_emb = self.suffix_embedding(suffix_ids)
            p_emb = self.prefix_embedding(prefix_ids)
            B, S, C = char_ids.shape
            chars_flat = char_ids.view(B * S, C)
            c_emb = self.char_embedding(chars_flat).transpose(1, 2)
            c_conv = torch.relu(self.char_cnn(c_emb))
            c_pooled, _ = torch.max(c_conv, dim=2)
            c_repr = c_pooled.view(B, S, -1)
            token_repr = torch.cat([embedded, s_emb, p_emb, c_repr], dim=-1)
        else:
            token_repr = embedded

        encoded, _ = self.encoder(token_repr)
        encoded = self.layer_norm(encoded)
        dropped = self.dropout(encoded)
        return self.classifier(dropped)

    def forward(self, token_ids, mask, tags=None, suffix_ids=None, prefix_ids=None, char_ids=None):
        emissions = self.emissions(token_ids, suffix_ids, prefix_ids, char_ids)
        crf_mask = mask.to(dtype=torch.bool)
        if tags is not None:
            return -self.crf(emissions, tags, mask=crf_mask, reduction="mean")
        return self.crf.decode(emissions, mask=crf_mask)

    def save(self, checkpoint_path, vocabulary, config):
        checkpoint_path = Path(checkpoint_path)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "state_dict": self.state_dict(),
            "vocabulary": vocabulary,
            "config": config,
            "vocab_size": self.vocab_size,
            "num_labels": self.num_labels,
            "embedding_dim": self.embedding_dim,
            "hidden_size": self.hidden_size,
            "dropout": self.dropout_prob,
            "pad_idx": self.pad_idx,
            "use_morphology": self.use_morphology,
            "num_layers": self.num_layers,
            "char_vocab_size": self.char_vocab_size,
            "suffix_vocab_size": self.suffix_vocab_size,
            "prefix_vocab_size": self.prefix_vocab_size,
            "char_dim": self.char_dim,
            "char_filters": self.char_filters,
            "affix_dim": self.affix_dim,
        }
        torch.save(payload, checkpoint_path)

    @classmethod
    def load(cls, checkpoint_path, device=torch.device("cpu")):
        try:
            payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            payload = torch.load(checkpoint_path, map_location=device)
        model = cls(
            vocab_size=payload["vocab_size"],
            num_labels=payload["num_labels"],
            embedding_dim=payload["embedding_dim"],
            hidden_size=payload["hidden_size"],
            dropout=payload["dropout"],
            pad_idx=payload["pad_idx"],
            use_morphology=payload.get("use_morphology", True),
            num_layers=payload.get("num_layers", 2),
            char_vocab_size=payload.get("char_vocab_size", 120),
            suffix_vocab_size=payload.get("suffix_vocab_size", 1500),
            prefix_vocab_size=payload.get("prefix_vocab_size", 1500),
            char_dim=payload.get("char_dim", 32),
            char_filters=payload.get("char_filters", 64),
            affix_dim=payload.get("affix_dim", 32),
        )
        model.load_state_dict(payload["state_dict"])
        model.to(device)
        return model, payload["vocabulary"], payload.get("config", {})

lit_vocab = build_vocabulary(splits["literature_train"]["tokens"])
print(f"Literature multi-tier vocabulary built: {len(lit_vocab['word'])} words, {len(lit_vocab['suffix'])} suffixes, {len(lit_vocab['char'])} characters.")

In [ ]:
# ============================================================
# C.4: M2, M3, M4, M5 TRANSFORMER ARCHITECTURES & ALIGNMENT
#      (XLM-RoBERTa, IndicBERT v2, MuRIL, and Sanskrit-BERT)
#      Includes Automated Devanagari -> IAST Transliteration (Option A)
# ============================================================

from transformers import AutoTokenizer, AutoModelForTokenClassification

def devanagari_to_iast(text: str) -> str:
    """Accurately transliterates Devanagari Sanskrit text to IAST Latin script.
    Preserves 1:1 token boundary alignment so word-level supervision
    remains strictly identical to Devanagari models.
    Used for Latin/IAST-pretrained models like Matej/bert-base-buddhist-sanskrit-v2 (M5)."""
    vowels = {
        'अ': 'a', 'आ': 'ā', 'इ': 'i', 'ई': 'ī', 'उ': 'u', 'ऊ': 'ū',
        'ऋ': 'ṛ', 'ॠ': 'ṝ', 'ऌ': 'ḷ', 'ॡ': 'ḹ', 'ए': 'e', 'ऐ': 'ai',
        'ओ': 'o', 'औ': 'au'
    }
    consonants = {
        'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ṅ',
        'च': 'c', 'छ': 'ch', 'ज': 'j', 'झ': 'jh', 'ञ': 'ñ',
        'ट': 'ṭ', 'ठ': 'ṭh', 'ड': 'ḍ', 'ढ': 'ḍh', 'ण': 'ṇ',
        'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
        'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
        'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v',
        'श': 'ś', 'ष': 'ṣ', 'स': 's', 'ह': 'h'
    }
    matras = {
        'ा': 'ā', 'ि': 'i', 'ी': 'ī', 'ु': 'u', 'ू': 'ū',
        'ृ': 'ṛ', 'ॄ': 'ṝ', 'ॢ': 'ḷ', 'ॣ': 'ḹ', 'े': 'e', 'ै': 'ai',
        'ो': 'o', 'ौ': 'au'
    }
    specials = {
        'ं': 'ṁ', 'ः': 'ḥ', 'ँ': 'm̐', 'ऽ': "'", '।': '.', '॥': '..',
        '०': '0', '१': '1', '२': '2', '३': '3', '४': '4',
        '५': '5', '६': '6', '७': '7', '८': '8', '९': '9'
    }

    res = []
    i = 0
    n = len(text)
    while i < n:
        c = text[i]
        if c in consonants:
            base = consonants[c]
            if i + 1 < n:
                nxt = text[i + 1]
                if nxt == '्':
                    res.append(base)
                    i += 2
                    continue
                elif nxt in matras:
                    res.append(base + matras[nxt])
                    i += 2
                    continue
                else:
                    res.append(base + 'a')
                    i += 1
                    continue
            else:
                res.append(base + 'a')
                i += 1
                continue
        elif c in vowels:
            res.append(vowels[c])
            i += 1
        elif c in matras:
            res.append(matras[c])
            i += 1
        elif c in specials:
            res.append(specials[c])
            i += 1
        elif c == '्':
            i += 1
        else:
            res.append(c)
            i += 1
    return ''.join(res)

class WordTokenClassificationDataset(Dataset):
    """Dataset with strict word-to-subword alignment for Sanskrit tokens.
    First subword of each word gets label ID; subsequent subwords & special tokens get -100.
    When is_iast=True, words are transliterated before subword tokenization,
    preserving exact 1:1 word index alignment."""
    def __init__(self, frame, tokenizer, label_to_id=None, max_length=256, is_labeled=True, is_iast=False):
        self.examples = []
        self.raw_tokens = []
        self.is_iast = is_iast
        label_to_id = label_to_id or TAG2ID
        
        for row in frame.itertuples():
            tokens = list(row.tokens)
            self.raw_tokens.append(tokens)
            tokens_to_tokenize = [devanagari_to_iast(t) for t in tokens] if is_iast else tokens
            encoded = tokenizer(
                tokens_to_tokenize,
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors=None,
            )
            word_ids = encoded.word_ids()
            
            if is_labeled and hasattr(row, "tags") and row.tags is not None:
                labels = []
                for index, word_id in enumerate(word_ids):
                    if word_id is None:
                        labels.append(-100)
                    elif index > 0 and word_id == word_ids[index - 1]:
                        labels.append(-100)
                    else:
                        labels.append(label_to_id[row.tags[word_id]])
            else:
                labels = [-100] * len(encoded["input_ids"])

            self.examples.append({
                "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "word_ids": word_ids,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

class TransformerTagger:
    """Wrapper around HuggingFace AutoModelForTokenClassification."""
    def __init__(self, model_id, tokenizer=None, model=None, cache_dir=None, is_iast=None):
        self.model_id = model_id
        if is_iast is not None:
            self.is_iast = is_iast
        else:
            self.is_iast = "buddhist-sanskrit" in model_id.lower() or "sanskrit_bert" in model_id.lower() or "sanskrit-bert" in model_id.lower()
        self.tokenizer = tokenizer or AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
        self.model = model or AutoModelForTokenClassification.from_pretrained(
            model_id,
            num_labels=len(TARGET_TAGS),
            id2label=ID2TAG,
            label2id=TAG2ID,
            cache_dir=cache_dir,
            ignore_mismatched_sizes=True,
        )

    def to(self, device):
        self.model.to(device)
        return self

    def predict_words(self, sentences, device, batch_size=16, max_length=256):
        self.model.eval()
        predictions = []
        with torch.no_grad():
            for i in range(0, len(sentences), batch_size):
                batch_words = sentences[i : i + batch_size]
                if self.is_iast:
                    batch_to_encode = [[devanagari_to_iast(w) for w in words] for words in batch_words]
                else:
                    batch_to_encode = list(batch_words)

                encoded = self.tokenizer(
                    batch_to_encode,
                    is_split_into_words=True,
                    truncation=True,
                    max_length=max_length,
                    padding=True,
                    return_tensors="pt",
                ).to(device)

                outputs = self.model(**encoded)
                preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

                for j, words in enumerate(batch_words):
                    word_ids = encoded.word_ids(batch_index=j)
                    sent_preds = []
                    seen_word_ids = set()

                    for token_idx, word_id in enumerate(word_ids):
                        if word_id is not None and word_id not in seen_word_ids:
                            pred_tag_id = int(preds[j, token_idx])
                            sent_preds.append(ID2TAG[pred_tag_id])
                            seen_word_ids.add(word_id)

                    while len(sent_preds) < len(words):
                        sent_preds.append("N")

                    predictions.append(sent_preds)
        return predictions

    def save(self, output_dir, config=None):
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        meta = {"is_iast": self.is_iast, "model_id": self.model_id}
        (output_dir / "tagger_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
        if config:
            (output_dir / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, checkpoint_dir, device=torch.device("cpu"), is_iast=None):
        checkpoint_dir = Path(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
        model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
        model.to(device)
        meta_file = checkpoint_dir / "tagger_meta.json"
        if is_iast is None and meta_file.exists():
            try:
                meta = json.loads(meta_file.read_text(encoding="utf-8"))
                is_iast = meta.get("is_iast", False)
            except Exception:
                pass
        return cls(model_id=str(checkpoint_dir), tokenizer=tokenizer, model=model, is_iast=is_iast)

# ============================================================
# MASTER SPECIFICATION FOR THE FIVE EVALUATED MODELS
# ============================================================
MODEL_SPECS = {
    "bilstm_crf": {
        "id": "M1",
        "display_name": "BiLSTM_CRF",
        "type": "bilstm",
        "config_path": "configs/bilstm_crf.json",
    },
    "xlm_roberta": {
        "id": "M2",
        "display_name": "XLM_RoBERTa",
        "type": "transformer",
        "hf_model_id": "FacebookAI/xlm-roberta-base",
        "config_path": "configs/xlm_roberta.json",
        "is_iast": False,
    },
    "indicbert_v2": {
        "id": "M3",
        "display_name": "IndicBERT_v2",
        "type": "transformer",
        "hf_model_id": "ai4bharat/IndicBERTv2-MLM-only",
        "config_path": "configs/indicbert_v2.json",
        "is_iast": False,
    },
    "muril": {
        "id": "M4",
        "display_name": "MuRIL",
        "type": "transformer",
        "hf_model_id": "google/muril-base-cased",
        "config_path": "configs/muril.json",
        "is_iast": False,
    },
    "sanskrit_bert": {
        "id": "M5",
        "display_name": "Sanskrit_BERT",
        "type": "transformer",
        "hf_model_id": "Matej/bert-base-buddhist-sanskrit-v2",
        "config_path": "configs/sanskrit_bert.json",
        "is_iast": True,
    },
}

print(f"Loaded definitions for {len(MODEL_SPECS)} models: {[s['display_name'] for s in MODEL_SPECS.values()]}")


def load_model_config(spec):
    """Resiliently load model hyperparameter configuration, falling back to verified defaults."""
    candidates = [
        PROJECT_DIR / spec.get("config_path", ""),
        PROJECT_DIR / "kaggle_bundle" / spec.get("config_path", ""),
        Path("configs") / Path(spec.get("config_path", "")).name,
    ]
    for c in candidates:
        if c and c.is_file():
            try:
                with open(c, encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                pass
    if spec.get("type") == "bilstm":
        return {"epochs": 20, "batch_size": 32, "learning_rate": 0.001, "early_stopping_patience": 3, "seed": 42}
    return {"epochs": 10, "batch_size": 16, "learning_rate": 3e-5, "weight_decay": 0.01, "warmup_ratio": 0.1, "early_stopping_patience": 3, "seed": 42}



In [ ]:
# ============================================================
# C.5: UNIFIED EVALUATION ENGINE (24 TARGET TAG METRICS)
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

def evaluate_predictions(frame, predictions):
    """Compute comprehensive token-level POS evaluation metrics."""
    if len(frame) != len(predictions):
        raise ValueError(f"Sentence count mismatch: frame={len(frame)}, pred={len(predictions)}")

    gold_tokens = [tag for tags in frame["tags"] for tag in tags]
    pred_tokens = [tag for seq in predictions for tag in seq]

    if len(gold_tokens) != len(pred_tokens):
        raise ValueError(f"Token count mismatch: gold={len(gold_tokens)}, pred={len(pred_tokens)}")

    summary_metrics = {
        "accuracy": float(accuracy_score(gold_tokens, pred_tokens)),
        "macro_precision": float(precision_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="macro", zero_division=0)),
        "weighted_precision": float(precision_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(gold_tokens, pred_tokens, labels=TARGET_TAGS, average="weighted", zero_division=0)),
        "total_tokens": len(gold_tokens),
        "total_sentences": len(frame),
    }

    report_dict = classification_report(gold_tokens, pred_tokens, labels=TARGET_TAGS, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).T.reset_index(names="tag")

    per_tag_rows = []
    for tag in TARGET_TAGS:
        tdata = report_dict.get(tag, {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})
        per_tag_rows.append({
            "tag": tag,
            "precision": float(tdata["precision"]),
            "recall": float(tdata["recall"]),
            "f1": float(tdata["f1-score"]),
            "support": int(tdata["support"]),
        })
    per_tag_df = pd.DataFrame(per_tag_rows)
    conf_mat = confusion_matrix(gold_tokens, pred_tokens, labels=TARGET_TAGS)
    confusion_df = pd.DataFrame(conf_mat, index=TARGET_TAGS, columns=TARGET_TAGS)

    return summary_metrics, per_tag_df, report_df, confusion_df

def save_evaluation_results(output_dir, summary_metrics, per_tag_df, report_df, confusion_df, predictions, frame):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    (output_dir / "metrics.json").write_text(json.dumps(summary_metrics, indent=2), encoding="utf-8")
    pd.DataFrame([summary_metrics]).to_csv(output_dir / "metrics.csv", index=False)
    per_tag_df.to_csv(output_dir / "per_tag_metrics.csv", index=False)
    report_df.to_csv(output_dir / "classification_report.csv", index=False)
    confusion_df.to_csv(output_dir / "confusion_matrix.csv")

    pred_rows = []
    for row, pred_seq in zip(frame.itertuples(), predictions):
        doc_id = getattr(row, "doc_id", row.sentence_id)
        sentence_id = row.sentence_id
        for tid, (token, gold_tag, pred_tag) in enumerate(zip(row.tokens, row.tags, pred_seq)):
            pred_rows.append({
                "doc_id": doc_id,
                "sentence_id": sentence_id,
                "token_id": tid,
                "token": token,
                "gold": gold_tag,
                "predicted": pred_tag,
            })
    pd.DataFrame(pred_rows).to_csv(output_dir / "predictions.csv", index=False)

print("Evaluation functions defined: accuracy, macro/weighted F1, confusion matrix, and prediction export.")

In [ ]:
# ============================================================
# C.6: UNIFIED TRAINING ENGINE WITH VALIDATION EARLY STOPPING
#      (Supports BiLSTM-CRF and all Transformer Taggers)
# ============================================================

import time
import logging
from transformers import get_linear_schedule_with_warmup

def decode_bilstm_predictions(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["mask"].to(device)
            best_paths = model(input_ids, mask=mask)
            for path in best_paths:
                predictions.append([ID2TAG[idx] for idx in path])
    return predictions

def train_bilstm_crf(model, train_df, val_df, vocabulary, device, run_dir, config):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    epochs = config.get("epochs", 20)
    batch_size = config.get("batch_size", 32)
    lr = config.get("learning_rate", 0.001)
    weight_decay = config.get("weight_decay", 1e-4)
    patience = config.get("early_stopping_patience", 3)

    train_ds = SequenceTaggingDataset(train_df, vocabulary)
    val_ds = SequenceTaggingDataset(val_df, vocabulary)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_sequences)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_sequences)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = -1.0
    best_epoch = -1
    best_metrics = {}
    patience_counter = 0
    history_records = []
    checkpoint_dir = run_dir / "checkpoint"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        total_train_loss = 0.0
        num_batches = 0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            loss = model(input_ids, mask=mask, tags=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_train_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_train_loss / max(num_batches, 1)

        val_preds = decode_bilstm_predictions(model, val_loader, device)
        val_metrics, _, _, _ = evaluate_predictions(val_df, val_preds)
        val_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]
        val_wf1 = val_metrics["weighted_f1"]
        elapsed = time.time() - t0

        print(f"Epoch {epoch:02d}/{epochs:02d} [{elapsed:.1f}s] - Train Loss: {avg_train_loss:.4f} - Val Acc: {val_acc:.4f} - Val Macro F1: {val_f1:.4f} - Val Weighted F1: {val_wf1:.4f}")

        history_records.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1,
            "val_weighted_f1": val_wf1,
            "duration_seconds": elapsed,
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_metrics = val_metrics
            patience_counter = 0
            model.save(checkpoint_dir / "model.pt", vocabulary, config)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement.")
                break

    history_df = pd.DataFrame(history_records)
    history_df.to_csv(run_dir / "training_history.csv", index=False)

    best_model, _, _ = BiLSTMCRF.load(checkpoint_dir / "model.pt", device=device)
    print(f"Training completed. Best epoch: {best_epoch} with Val Macro F1: {best_val_f1:.4f}")
    return best_model, best_metrics, history_df


def train_transformer(tagger, train_df, val_df, device, run_dir, config):
    """Train transformer tagger with word-level validation Macro F1 early stopping."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    epochs = config.get("epochs", 10)
    batch_size = config.get("batch_size", 8)
    lr = config.get("learning_rate", 3e-5)
    weight_decay = config.get("weight_decay", 0.01)
    max_length = config.get("max_length", 256)
    warmup_ratio = config.get("warmup_ratio", 0.1)
    patience = config.get("early_stopping_patience", 2)

    train_ds = WordTokenClassificationDataset(
        train_df, tagger.tokenizer, max_length=max_length, is_iast=getattr(tagger, "is_iast", False)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)

    optimizer = torch.optim.AdamW(tagger.model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_val_f1 = -1.0
    best_epoch = -1
    best_metrics = {}
    patience_counter = 0
    history_records = []
    checkpoint_dir = run_dir / "checkpoint"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tagger.model.train()
        total_train_loss = 0.0
        num_batches = 0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = tagger.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tagger.model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_train_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_train_loss / max(num_batches, 1)

        # Word-level validation evaluation
        val_preds = tagger.predict_words(list(val_df["tokens"]), device=device, batch_size=batch_size, max_length=max_length)
        val_metrics, _, _, _ = evaluate_predictions(val_df, val_preds)
        val_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]
        val_wf1 = val_metrics["weighted_f1"]
        elapsed = time.time() - t0

        print(f"Epoch {epoch:02d}/{epochs:02d} [{elapsed:.1f}s] - Train Loss: {avg_train_loss:.4f} - Val Acc: {val_acc:.4f} - Val Macro F1: {val_f1:.4f} - Val Weighted F1: {val_wf1:.4f}")

        history_records.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1,
            "val_weighted_f1": val_wf1,
            "duration_seconds": elapsed,
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_metrics = val_metrics
            patience_counter = 0
            tagger.save(checkpoint_dir, config)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement.")
                break

    history_df = pd.DataFrame(history_records)
    history_df.to_csv(run_dir / "training_history.csv", index=False)

    best_tagger = TransformerTagger.load(checkpoint_dir, device=device)
    print(f"Training completed. Best epoch: {best_epoch} with Val Macro F1: {best_val_f1:.4f}")
    return best_tagger, best_metrics, history_df

print("Training engines defined for BiLSTM-CRF and all Transformers.")


In [ ]:
# ============================================================
# C.7: EXPERIMENT TRACKING & NESTED FEW-SHOT SAMPLER
# ============================================================

import csv
import uuid
from datetime import datetime, timezone

EXP_ROOT = PROJECT_DIR / "outputs" / "experiments"
REGISTRY_PATH = EXP_ROOT / "experiment_registry.csv"

REGISTRY_COLUMNS = [
    "experiment_id", "model", "setting", "annotation_budget", "seed", "status",
    "run_dir", "validation_macro_f1", "test_macro_f1", "test_accuracy",
    "test_weighted_f1", "error", "started_at", "completed_at",
]

def get_nested_few_shot_subsets(pool_df, budgets=(0, 50, 100, 200, 500, 1000), seed=42):
    """Sample strictly nested subsets: budget 50 ⊂ 100 ⊂ 200 ⊂ 500 ⊂ 1000."""
    sorted_budgets = sorted(set(budgets))
    rng = np.random.RandomState(seed)
    indices = rng.permutation(len(pool_df))

    subsets = {}
    for b in sorted_budgets:
        if b == 0:
            subsets[0] = pool_df.iloc[:0].copy().reset_index(drop=True)
        else:
            subset_indices = indices[:b]
            subsets[b] = pool_df.iloc[subset_indices].copy().reset_index(drop=True)
    return subsets

def create_hierarchical_run_directory(experiments_root, model, setting, annotation_budget=0, seed=42):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_token = uuid.uuid4().hex[:6]
    experiment_id = f"{model}_{setting}_{annotation_budget}sent_s{seed}_{timestamp}_{run_token}"

    if setting == "few_shot":
        run_dir = experiments_root / model / setting / str(annotation_budget) / f"seed_{seed}" / f"run_{timestamp}_{run_token}"
    else:
        run_dir = experiments_root / model / setting / f"run_{timestamp}_{run_token}"

    run_dir.mkdir(parents=True, exist_ok=False)
    return experiment_id, run_dir

def append_registry(registry_path, experiment_id, model, setting, annotation_budget, seed, status,
                    run_dir, validation_macro_f1=None, test_macro_f1=None, test_accuracy=None,
                    test_weighted_f1=None, error=None, started_at=None, completed_at=None):
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    row = {
        "experiment_id": experiment_id, "model": model, "setting": setting,
        "annotation_budget": annotation_budget, "seed": seed, "status": status,
        "run_dir": str(run_dir), "validation_macro_f1": validation_macro_f1,
        "test_macro_f1": test_macro_f1, "test_accuracy": test_accuracy,
        "test_weighted_f1": test_weighted_f1, "error": error,
        "started_at": started_at or datetime.now(timezone.utc).isoformat(),
        "completed_at": completed_at or datetime.now(timezone.utc).isoformat(),
    }
    exists = registry_path.exists()
    with registry_path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=REGISTRY_COLUMNS)
        if not exists:
            writer.writeheader()
        writer.writerow(row)
    update_best_model_index(registry_path)

def update_best_model_index(registry_path):
    if not registry_path.exists():
        return
    df = pd.read_csv(registry_path)
    succ = df[df["status"] == "SUCCESS"].dropna(subset=["validation_macro_f1"])
    if succ.empty:
        return
    best_indices = succ.groupby(["model", "setting", "annotation_budget"])["validation_macro_f1"].idxmax()
    best_df = succ.loc[best_indices].reset_index(drop=True)
    best_df.to_csv(registry_path.parent / "best_model_index.csv", index=False)

print("Tracking utilities and nested subset sampler defined.")

In [ ]:
# ============================================================
# C.8: EXPERIMENT 1 — LITERATURE IN-DOMAIN EVALUATION
#      (Executes All Selected Models on Sanskrit Literature)
# ============================================================

import copy

# Select which models to run: ["bilstm_crf", "xlm_roberta", "indicbert_v2", "muril", "sanskrit_bert"]
MODELS_TO_RUN = ["bilstm_crf", "xlm_roberta", "indicbert_v2", "muril", "sanskrit_bert"]

lit_train = splits["literature_train"]
lit_val = splits["literature_validation"]
lit_test = splits["literature_test"]

lit_checkpoints = {}

for m_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    print(f"\n{'='*70}\n[EXPERIMENT 1] Literature In-Domain: {display_name} ({spec['id']})\n{'='*70}")
    
    config = load_model_config(spec)
        
    seed = config.get("seed", 42)
    torch.manual_seed(seed)
    
    exp_id, run_dir = create_hierarchical_run_directory(
        EXP_ROOT, model=display_name, setting="literature_in_domain", seed=seed
    )
    started_at = datetime.now(timezone.utc).isoformat()
    
    try:
        if spec["type"] == "bilstm":
            lit_vocab = build_vocabulary(lit_train["tokens"])
            model = BiLSTMCRF(
                vocab_size=len(lit_vocab),
                num_labels=len(TARGET_TAGS),
                embedding_dim=config.get("embedding_dim", 128),
                hidden_size=config.get("hidden_size", 128),
                dropout=config.get("dropout", 0.2),
            ).to(DEVICE)
            
            best_model, val_metrics, _ = train_bilstm_crf(
                model=model,
                train_df=lit_train,
                val_df=lit_val,
                vocabulary=lit_vocab,
                device=DEVICE,
                run_dir=run_dir,
                config=config,
            )
            
            test_ds = SequenceTaggingDataset(lit_test, lit_vocab)
            test_loader = DataLoader(test_ds, batch_size=config.get("batch_size", 32), shuffle=False, collate_fn=collate_sequences)
            test_preds = decode_bilstm_predictions(best_model, test_loader, DEVICE)
            lit_checkpoints[m_key] = {"run_dir": run_dir, "vocab": lit_vocab, "type": "bilstm"}
            
        else:
            tagger = TransformerTagger(model_id=spec["hf_model_id"], is_iast=spec.get("is_iast", False)).to(DEVICE)
            best_tagger, val_metrics, _ = train_transformer(
                tagger=tagger,
                train_df=lit_train,
                val_df=lit_val,
                device=DEVICE,
                run_dir=run_dir,
                config=config,
            )
            test_preds = best_tagger.predict_words(
                list(lit_test["tokens"]),
                device=DEVICE,
                batch_size=config.get("batch_size", 16),
                max_length=config.get("max_length", 256),
            )
            lit_checkpoints[m_key] = {"run_dir": run_dir, "type": "transformer"}
            
        summary, per_tag, report, conf_mat = evaluate_predictions(lit_test, test_preds)
        save_evaluation_results(run_dir, summary, per_tag, report, conf_mat, test_preds, lit_test)
        
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="literature_in_domain",
            annotation_budget=0,
            seed=seed,
            status="SUCCESS",
            run_dir=run_dir,
            validation_macro_f1=val_metrics.get("macro_f1"),
            test_macro_f1=summary["macro_f1"],
            test_accuracy=summary["accuracy"],
            test_weighted_f1=summary["weighted_f1"],
            started_at=started_at,
        )
        print(f"==> {display_name} Exp 1 Result: Accuracy={summary['accuracy']*100:.2f}%, Test Macro F1={summary['macro_f1']*100:.2f}%")
        
    except Exception as e:
        print(f"Error evaluating {display_name} on Literature: {e}")
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="literature_in_domain",
            annotation_budget=0,
            seed=seed,
            status="FAILED",
            run_dir=run_dir,
            error=str(e),
            started_at=started_at,
        )

print("\nLiterature In-Domain Evaluation Suite Completed!")



In [ ]:
# ============================================================
# C.9: EXPERIMENT 2 — NEWS IN-DOMAIN REFERENCE BENCHMARK
#      (Upper-Bound Reference Trained and Evaluated on News)
# ============================================================

news_pool = splits["news_adaptation_pool"]
news_val = splits["news_validation"]
news_test = splits["news_test"]

for m_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    print(f"\n{'='*70}\n[EXPERIMENT 2] News In-Domain Reference: {display_name} ({spec['id']})\n{'='*70}")
    
    config = load_model_config(spec)
        
    seed = config.get("seed", 42)
    torch.manual_seed(seed)
    
    exp_id, run_dir = create_hierarchical_run_directory(
        EXP_ROOT, model=display_name, setting="news_in_domain", seed=seed
    )
    started_at = datetime.now(timezone.utc).isoformat()
    
    try:
        if spec["type"] == "bilstm":
            news_vocab = build_vocabulary(news_pool["tokens"])
            model = BiLSTMCRF(
                vocab_size=len(news_vocab),
                num_labels=len(TARGET_TAGS),
                embedding_dim=config.get("embedding_dim", 128),
                hidden_size=config.get("hidden_size", 128),
                dropout=config.get("dropout", 0.2),
            ).to(DEVICE)
            
            best_model, val_metrics, _ = train_bilstm_crf(
                model=model,
                train_df=news_pool,
                val_df=news_val,
                vocabulary=news_vocab,
                device=DEVICE,
                run_dir=run_dir,
                config=config,
            )
            
            test_ds = SequenceTaggingDataset(news_test, news_vocab)
            test_loader = DataLoader(test_ds, batch_size=config.get("batch_size", 32), shuffle=False, collate_fn=collate_sequences)
            test_preds = decode_bilstm_predictions(best_model, test_loader, DEVICE)
            
        else:
            tagger = TransformerTagger(model_id=spec["hf_model_id"], is_iast=spec.get("is_iast", False)).to(DEVICE)
            best_tagger, val_metrics, _ = train_transformer(
                tagger=tagger,
                train_df=news_pool,
                val_df=news_val,
                device=DEVICE,
                run_dir=run_dir,
                config=config,
            )
            test_preds = best_tagger.predict_words(
                list(news_test["tokens"]),
                device=DEVICE,
                batch_size=config.get("batch_size", 16),
                max_length=config.get("max_length", 256),
            )
            
        summary, per_tag, report, conf_mat = evaluate_predictions(news_test, test_preds)
        save_evaluation_results(run_dir, summary, per_tag, report, conf_mat, test_preds, news_test)
        
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="news_in_domain",
            annotation_budget=0,
            seed=seed,
            status="SUCCESS",
            run_dir=run_dir,
            validation_macro_f1=val_metrics.get("macro_f1"),
            test_macro_f1=summary["macro_f1"],
            test_accuracy=summary["accuracy"],
            test_weighted_f1=summary["weighted_f1"],
            started_at=started_at,
        )
        print(f"==> {display_name} Exp 2 Result: Accuracy={summary['accuracy']*100:.2f}%, News Test Macro F1={summary['macro_f1']*100:.2f}%")
        
    except Exception as e:
        print(f"Error evaluating {display_name} on News: {e}")
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="news_in_domain",
            annotation_budget=0,
            seed=seed,
            status="FAILED",
            run_dir=run_dir,
            error=str(e),
            started_at=started_at,
        )

print("\nNews In-Domain Reference Benchmark Suite Completed!")



In [ ]:
# ============================================================
# C.10: EXPERIMENT 3 — ZERO-SHOT CROSS-DOMAIN GENERALIZATION
#       (Literature Trained -> Evaluated Directly on News Locked Test)
# ============================================================

for m_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    print(f"\n{'='*70}\n[EXPERIMENT 3] Zero-Shot Transfer: {display_name} ({spec['id']})\n{'='*70}")
    
    config = load_model_config(spec)
        
    seed = config.get("seed", 42)
    exp_id, run_dir = create_hierarchical_run_directory(
        EXP_ROOT, model=display_name, setting="zero_shot", seed=seed
    )
    started_at = datetime.now(timezone.utc).isoformat()
    
    try:
        lit_info = lit_checkpoints[m_key]
        chk_dir = lit_info["run_dir"] / "checkpoint"
        
        if spec["type"] == "bilstm":
            lit_model, lit_vocab, _ = BiLSTMCRF.load(chk_dir / "model.pt", device=DEVICE)
            test_ds = SequenceTaggingDataset(news_test, lit_vocab)
            test_loader = DataLoader(test_ds, batch_size=config.get("batch_size", 32), shuffle=False, collate_fn=collate_sequences)
            test_preds = decode_bilstm_predictions(lit_model, test_loader, DEVICE)
            
        else:
            lit_tagger = TransformerTagger.load(chk_dir, device=DEVICE)
            test_preds = lit_tagger.predict_words(
                list(news_test["tokens"]),
                device=DEVICE,
                batch_size=config.get("batch_size", 16),
                max_length=config.get("max_length", 256),
            )
            
        summary, per_tag, report, conf_mat = evaluate_predictions(news_test, test_preds)
        save_evaluation_results(run_dir, summary, per_tag, report, conf_mat, test_preds, news_test)
        
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="zero_shot",
            annotation_budget=0,
            seed=seed,
            status="SUCCESS",
            run_dir=run_dir,
            validation_macro_f1=None,
            test_macro_f1=summary["macro_f1"],
            test_accuracy=summary["accuracy"],
            test_weighted_f1=summary["weighted_f1"],
            started_at=started_at,
        )
        print(f"==> {display_name} Exp 3 Result: Zero-Shot Accuracy={summary['accuracy']*100:.2f}%, News Test Macro F1={summary['macro_f1']*100:.2f}%")
        
    except Exception as e:
        print(f"Error in Zero-Shot evaluation for {display_name}: {e}")
        append_registry(
            REGISTRY_PATH,
            experiment_id=exp_id,
            model=display_name,
            setting="zero_shot",
            annotation_budget=0,
            seed=seed,
            status="FAILED",
            run_dir=run_dir,
            error=str(e),
            started_at=started_at,
        )

print("\nZero-Shot Cross-Domain Generalization Suite Completed!")



In [ ]:
# ============================================================
# C.11: EXPERIMENT 4 — FEW-SHOT TARGET DOMAIN ADAPTATION
#       (Annotation Budgets: 50, 100, 200, 500, 1000 sentences)
# ============================================================

budgets = [50, 100, 200, 500, 1000]
# Multi-seed dissertation protocol: [42, 123, 2026] (set seeds=[42] for single fast run)
seeds = [42]

print(f"Running Few-Shot Target Domain Adaptation across budgets {budgets} and seeds {seeds}...")

for m_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    print(f"\n{'='*70}\n[EXPERIMENT 4] Few-Shot Adaptation: {display_name} ({spec['id']})\n{'='*70}")
    
    base_config = load_model_config(spec)
        
    lit_info = lit_checkpoints[m_key]
    chk_dir = lit_info["run_dir"] / "checkpoint"
    
    for s in seeds:
        print(f"\n--- Seed {s} ---")
        nested_subsets = get_nested_few_shot_subsets(news_pool, budgets=budgets, seed=s)
        
        for b in budgets:
            sub_df = nested_subsets[b]
            token_count = len([t for ts in sub_df['tokens'] for t in ts])
            print(f"Fine-tuning budget = {b:4d} sentences ({token_count:5d} tokens)...")
            
            exp_id, run_dir = create_hierarchical_run_directory(
                EXP_ROOT, model=display_name, setting="few_shot", annotation_budget=b, seed=s
            )
            started_at = datetime.now(timezone.utc).isoformat()
            
            try:
                torch.manual_seed(s)
                if spec["type"] == "bilstm":
                    ft_model, ft_vocab, _ = BiLSTMCRF.load(chk_dir / "model.pt", device=DEVICE)
                    ft_config = copy.deepcopy(base_config)
                    ft_config["learning_rate"] = base_config.get("learning_rate", 0.001) * 0.5
                    ft_config["epochs"] = 15
                    
                    best_ft, ft_val, _ = train_bilstm_crf(
                        model=ft_model,
                        train_df=sub_df,
                        val_df=news_val,
                        vocabulary=ft_vocab,
                        device=DEVICE,
                        run_dir=run_dir,
                        config=ft_config,
                    )
                    test_ds = SequenceTaggingDataset(news_test, ft_vocab)
                    test_loader = DataLoader(test_ds, batch_size=base_config.get("batch_size", 32), shuffle=False, collate_fn=collate_sequences)
                    test_preds = decode_bilstm_predictions(best_ft, test_loader, DEVICE)
                    
                else:
                    ft_tagger = TransformerTagger.load(chk_dir, device=DEVICE)
                    ft_config = copy.deepcopy(base_config)
                    ft_config["learning_rate"] = base_config.get("learning_rate", 3e-5) * 0.7
                    ft_config["epochs"] = 6
                    ft_config["early_stopping_patience"] = 2
                    
                    best_ft, ft_val, _ = train_transformer(
                        tagger=ft_tagger,
                        train_df=sub_df,
                        val_df=news_val,
                        device=DEVICE,
                        run_dir=run_dir,
                        config=ft_config,
                    )
                    test_preds = best_ft.predict_words(
                        list(news_test["tokens"]),
                        device=DEVICE,
                        batch_size=base_config.get("batch_size", 16),
                        max_length=base_config.get("max_length", 256),
                    )
                    
                summary, per_tag, report, conf_mat = evaluate_predictions(news_test, test_preds)
                save_evaluation_results(run_dir, summary, per_tag, report, conf_mat, test_preds, news_test)
                
                append_registry(
                    REGISTRY_PATH,
                    experiment_id=exp_id,
                    model=display_name,
                    setting="few_shot",
                    annotation_budget=b,
                    seed=s,
                    status="SUCCESS",
                    run_dir=run_dir,
                    validation_macro_f1=ft_val.get("macro_f1"),
                    test_macro_f1=summary["macro_f1"],
                    test_accuracy=summary["accuracy"],
                    test_weighted_f1=summary["weighted_f1"],
                    started_at=started_at,
                )
                print(f"--> Budget {b:4d} sent: Val F1={ft_val.get('macro_f1', 0)*100:.2f}%, Test Macro F1={summary['macro_f1']*100:.2f}%, Acc={summary['accuracy']*100:.2f}%")
                
            except Exception as e:
                print(f"Error in few-shot budget {b}: {e}")
                append_registry(
                    REGISTRY_PATH,
                    experiment_id=exp_id,
                    model=display_name,
                    setting="few_shot",
                    annotation_budget=b,
                    seed=seed,
                    status="FAILED",
                    run_dir=run_dir,
                    error=str(e),
                    started_at=started_at,
                )

print("\nFew-Shot Target Domain Adaptation Suite Completed!")



In [ ]:
# ============================================================
# C.12: DOWNSTREAM INFERENCE ON UNANNOTATED SANSKRIT NEWS
#       (3,000 Unseen Target Sentences)
# ============================================================

import re

def simple_sanskrit_tokenize(text):
    tokens = []
    raw_tokens = str(text).strip().split()
    for tok in raw_tokens:
        sub_tokens = re.findall(r"\w+|[।॥.,;:!?]", tok)
        tokens.extend(sub_tokens if sub_tokens else [tok])
    return tokens

print("Running batch inference on 3,000 unannotated Sanskrit news sentences...")
sentence_tokens = [simple_sanskrit_tokenize(row.sentence) for row in news_unannotated.itertuples()]

unann_out_dir = PROJECT_DIR / "outputs" / "predictions" / "unannotated_news"
unann_out_dir.mkdir(parents=True, exist_ok=True)

for m_key, lit_info in lit_checkpoints.items():
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    chk_dir = lit_info["run_dir"] / "checkpoint"
    print(f"Predicting 3,000 unannotated sentences with {display_name}...")
    
    if spec["type"] == "bilstm":
        lit_model, lit_vocab, _ = BiLSTMCRF.load(chk_dir / "model.pt", device=DEVICE)
        lit_model.eval()
        eval_frame = pd.DataFrame({
            "doc_id": news_unannotated["doc_id"],
            "sentence_id": news_unannotated["sentence_id"],
            "tokens": sentence_tokens,
            "tags": [["N"] * len(t) for t in sentence_tokens],
        })
        unann_ds = SequenceTaggingDataset(eval_frame, lit_vocab)
        unann_loader = DataLoader(unann_ds, batch_size=32, shuffle=False, collate_fn=collate_sequences)
        all_unann_preds = []
        with torch.no_grad():
            for batch in unann_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                mask = batch["mask"].to(DEVICE)
                paths = lit_model(input_ids, mask=mask)
                for path in paths:
                    all_unann_preds.append([ID2TAG[idx] for idx in path])
    else:
        lit_tagger = TransformerTagger.load(chk_dir, device=DEVICE)
        all_unann_preds = lit_tagger.predict_words(sentence_tokens, device=DEVICE, batch_size=16)

    token_rows = []
    sentence_rows = []
    for row, tokens, preds in zip(news_unannotated.itertuples(), sentence_tokens, all_unann_preds):
        for tid, (tok, ptag) in enumerate(zip(tokens, preds)):
            token_rows.append({
                "doc_id": row.doc_id,
                "sentence_id": row.sentence_id,
                "token_id": tid,
                "token": tok,
                "predicted_pos": ptag,
                "model_name": display_name,
                "confidence": 1.0,
            })
        sentence_rows.append({
            "doc_id": row.doc_id,
            "sentence_id": row.sentence_id,
            "sentence": row.sentence,
            "tokens": " ".join(tokens),
            "predicted_tags": " ".join(preds),
            "model_name": display_name,
        })

    token_df = pd.DataFrame(token_rows)
    sent_df = pd.DataFrame(sentence_rows)
    token_df.to_csv(unann_out_dir / f"{m_key}_unannotated_token_predictions.csv", index=False)
    sent_df.to_csv(unann_out_dir / f"{m_key}_unannotated_sentence_predictions.csv", index=False)
    print(f"Saved {len(token_df)} token predictions to {unann_out_dir / f'{m_key}_unannotated_token_predictions.csv'}")

print(f"\nAll unannotated news predictions saved to {unann_out_dir}")


In [ ]:
# ============================================================
# C.13: DISSERTATION COMPARISON TABLES (TABLES 1, 2, 3, 4)
#       AND MASTER EXCEL WORKBOOK GENERATOR
# ============================================================

import openpyxl

BASELINE_OUT_DIR = PROJECT_DIR / "outputs" / "baseline_experiments"
BASELINE_OUT_DIR.mkdir(parents=True, exist_ok=True)
EVALS_DIR = PROJECT_DIR / "outputs" / "evaluations"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

reg_df = pd.read_csv(REGISTRY_PATH)

# ------------------------------------------------------------
# 1. In-Domain Table (Literature vs News)
# ------------------------------------------------------------
in_domain_df = reg_df[reg_df["setting"].isin(["literature_in_domain", "news_in_domain"]) & (reg_df["status"] == "SUCCESS")].copy()
in_domain_summary = []
for m_name in in_domain_df["model"].unique():
    m_runs = in_domain_df[in_domain_df["model"] == m_name]
    lit_run = m_runs[m_runs["setting"] == "literature_in_domain"]
    news_run = m_runs[m_runs["setting"] == "news_in_domain"]
    
    in_domain_summary.append({
        "Model": m_name,
        "Literature_Val_Macro_F1": lit_run["validation_macro_f1"].values[0] if not lit_run.empty else None,
        "Literature_Test_Macro_F1": lit_run["test_macro_f1"].values[0] if not lit_run.empty else None,
        "Literature_Test_Accuracy": lit_run["test_accuracy"].values[0] if not lit_run.empty else None,
        "News_Val_Macro_F1": news_run["validation_macro_f1"].values[0] if not news_run.empty else None,
        "News_Test_Macro_F1": news_run["test_macro_f1"].values[0] if not news_run.empty else None,
        "News_Test_Accuracy": news_run["test_accuracy"].values[0] if not news_run.empty else None,
    })
t1_df = pd.DataFrame(in_domain_summary)
t1_df.to_csv(BASELINE_OUT_DIR / "literature_in_domain_results.csv", index=False)
t1_df.to_csv(EVALS_DIR / "table1_in_domain.csv", index=False)

# ------------------------------------------------------------
# 2. Zero-Shot Cross-Domain Table & Domain Degradation
# ------------------------------------------------------------
zs_df = reg_df[(reg_df["setting"] == "zero_shot") & (reg_df["status"] == "SUCCESS")].copy()
zs_summary = []
for _, row in zs_df.iterrows():
    m_name = row["model"]
    lit_match = in_domain_df[(in_domain_df["model"] == m_name) & (in_domain_df["setting"] == "literature_in_domain")]
    lit_f1 = lit_match["test_macro_f1"].values[0] if not lit_match.empty else None
    zs_f1 = row["test_macro_f1"]
    abs_drop = (lit_f1 - zs_f1) if (lit_f1 is not None and zs_f1 is not None) else None
    rel_drop = (abs_drop / lit_f1 * 100) if (lit_f1 and abs_drop is not None) else None
    
    zs_summary.append({
        "Model": m_name,
        "Literature_InDomain_Macro_F1": lit_f1,
        "News_ZeroShot_Macro_F1": zs_f1,
        "Absolute_F1_Drop": abs_drop,
        "Relative_F1_Drop_Pct": rel_drop,
        "ZeroShot_Accuracy": row["test_accuracy"],
        "ZeroShot_Weighted_F1": row["test_weighted_f1"],
    })
t2_df = pd.DataFrame(zs_summary)
t2_df.to_csv(BASELINE_OUT_DIR / "cross_domain_zero_shot_results.csv", index=False)
t2_df.to_csv(EVALS_DIR / "table2_zero_shot.csv", index=False)

# ------------------------------------------------------------
# 3. Few-Shot Adaptation Progression Table
# ------------------------------------------------------------
fs_df = reg_df[reg_df["setting"].isin(["few_shot", "zero_shot"]) & (reg_df["status"] == "SUCCESS")].copy()
fs_df.loc[fs_df["setting"] == "zero_shot", "annotation_budget"] = 0

t3_pivot = fs_df.pivot_table(
    index="model",
    columns="annotation_budget",
    values="test_macro_f1",
    aggfunc="mean"
).reset_index()
t3_pivot.to_csv(BASELINE_OUT_DIR / "few_shot_results.csv", index=False)
t3_pivot.to_csv(EVALS_DIR / "table3_few_shot.csv", index=False)

# ------------------------------------------------------------
# 4. Master Excel Workbook
# ------------------------------------------------------------
excel_path = BASELINE_OUT_DIR / "baseline_experiment_results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    t1_df.to_excel(writer, sheet_name="In_Domain_Benchmarks", index=False)
    t2_df.to_excel(writer, sheet_name="Zero_Shot_Degradation", index=False)
    t3_pivot.to_excel(writer, sheet_name="Few_Shot_Progression", index=False)
    reg_df.to_excel(writer, sheet_name="Experiment_Registry", index=False)

print(f"Generated comparison tables and Excel workbook at: {excel_path}")
display(t1_df)
display(t2_df)
display(t3_pivot)


In [ ]:
# ============================================================
# C.14: PUBLICATION-READY FIGURES & VISUALIZATIONS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

reg_df = pd.read_csv(REGISTRY_PATH)

# 1. Few-Shot Learning Curves Plot
fs_runs = reg_df[reg_df["setting"].isin(["few_shot", "zero_shot"]) & (reg_df["status"] == "SUCCESS")].copy()
fs_runs.loc[fs_runs["setting"] == "zero_shot", "annotation_budget"] = 0

if not fs_runs.empty:
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=fs_runs, x="annotation_budget", y="test_macro_f1", hue="model", marker="o", linewidth=2.5)
    plt.title("Few-Shot Adaptation: Target News Macro F1 vs Annotation Budget", fontsize=13, fontweight="bold")
    plt.xlabel("Annotation Budget (News Sentences)", fontsize=11)
    plt.ylabel("Locked News Test Macro F1", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    curve_path = FIGURE_DIR / "few_shot_macro_f1_curve.png"
    plt.savefig(curve_path, dpi=300)
    plt.close()
    print(f"Generated Few-Shot Learning Curve: {curve_path}")
    display(Image(filename=str(curve_path)))

# 2. Zero-Shot Cross-Domain Degradation Bar Chart
zs_csv = PROJECT_DIR / "outputs" / "baseline_experiments" / "cross_domain_zero_shot_results.csv"
if zs_csv.exists():
    zdf = pd.read_csv(zs_csv)
    if not zdf.empty:
        plt.figure(figsize=(10, 6))
        x = np.arange(len(zdf))
        w = 0.35
        plt.bar(x - w/2, zdf["Literature_InDomain_Macro_F1"] * 100, width=w, label="Literature In-Domain F1", color="#2b5c8f")
        plt.bar(x + w/2, zdf["News_ZeroShot_Macro_F1"] * 100, width=w, label="News Zero-Shot F1", color="#d95f02")
        plt.xticks(x, zdf["Model"], rotation=15, ha="right", fontsize=10)
        plt.ylabel("Macro F1 (%)", fontsize=11)
        plt.title("Domain Shift Impact: In-Domain Literature vs Zero-Shot News", fontsize=13, fontweight="bold")
        plt.legend(frameon=True)
        plt.grid(axis="y", linestyle="--", alpha=0.6)
        plt.tight_layout()
        deg_path = FIGURE_DIR / "zero_shot_degradation.png"
        plt.savefig(deg_path, dpi=300)
        plt.close()
        print(f"Generated Zero-Shot Degradation Chart: {deg_path}")
        display(Image(filename=str(deg_path)))


In [ ]:
# ============================================================
# C.15: KAGGLE GPU ACCELERATION & BUNDLE PACKAGING
# ============================================================

import zipfile
import shutil

def create_kaggle_bundle(project_root, output_bundle_path=None):
    root = Path(project_root).resolve()
    output_bundle_path = output_bundle_path or root / "outputs" / "kaggle_bundle.zip"
    output_bundle_path.parent.mkdir(parents=True, exist_ok=True)

    items_to_bundle = [
        ("data/raw/Literature_Corpus.csv", "data/raw/Literature_Corpus.csv"),
        ("data/raw/News_Annotated.csv", "data/raw/News_Annotated.csv"),
        ("data/raw/News_Unannotated.csv", "data/raw/News_Unannotated.csv"),
        ("outputs/datasets", "outputs/datasets"),
        ("configs", "configs"),
        ("src", "src"),
        ("outputs/baseline_experiments", "outputs/baseline_experiments"),
    ]

    with zipfile.ZipFile(output_bundle_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for src_rel, arc_rel in items_to_bundle:
            src_path = root / src_rel
            if src_path.is_file():
                zipf.write(src_path, arc_rel)
            elif src_path.is_dir():
                for f in src_path.rglob("*"):
                    if f.is_file() and not any(p in f.parts for p in [".git", "__pycache__", ".ipynb_checkpoints"]):
                        rel_in_dir = f.relative_to(src_path)
                        zipf.write(f, str(Path(arc_rel) / rel_in_dir))

    bundle_mb = output_bundle_path.stat().st_size / (1024 * 1024)
    print(f"SUCCESS: Kaggle/Colab bundle created at {output_bundle_path} ({bundle_mb:.2f} MB)")
    return output_bundle_path

bundle = create_kaggle_bundle(PROJECT_DIR)


In [ ]:
# ============================================================
# C.16: MASTER EXPERIMENT RUNNER FOR M2 -> M5 (ONE BY ONE)
#       Executes all 4 experimental stages sequentially:
#         Stage 1: Literature In-Domain (Source Training)
#         Stage 2: News In-Domain Reference Benchmark
#         Stage 3: Zero-Shot Cross-Domain Generalization
#         Stage 4: Few-Shot Target Domain Adaptation (50-1000 sents)
#       And saves checkpoints, predictions, tables, and Excel workbook.
# ============================================================

import copy
import gc
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Select models to execute:
# To run one model at a time, uncomment one:
#   MODELS_TO_RUN = ["xlm_roberta"]    # M2: XLM-RoBERTa
#   MODELS_TO_RUN = ["indicbert_v2"]   # M3: IndicBERT v2
#   MODELS_TO_RUN = ["muril"]          # M4: MuRIL
#   MODELS_TO_RUN = ["sanskrit_bert"]  # M5: Sanskrit-BERT (Option A Transliteration)
# Or run all remaining models sequentially one by one:
MODELS_TO_RUN = ["xlm_roberta", "indicbert_v2", "muril", "sanskrit_bert"]

lit_train = splits["literature_train"]
lit_val = splits["literature_validation"]
lit_test = splits["literature_test"]
news_pool = splits["news_adaptation_pool"]
news_val = splits["news_validation"]
news_test = splits["news_test"]

def run_full_pipeline_for_model(m_key, seeds=[42], budgets=[50, 100, 200, 500, 1000]):
    spec = MODEL_SPECS[m_key]
    display_name = spec["display_name"]
    model_id_tag = spec["id"]
    
    print()
    print("=" * 75)
    print(f"STARTING FULL EXPERIMENTAL SUITE: {display_name} ({model_id_tag})")
    print(f"   Model: {spec.get('hf_model_id', m_key)}")
    print(f"   IAST Transliteration: {spec.get('is_iast', False)}")
    print("=" * 75)

    config = load_model_config(spec)
    seed = config.get("seed", 42)
    torch.manual_seed(seed)

    # ----------------------------------------------------
    # STAGE 1: Literature In-Domain Training & Evaluation
    # ----------------------------------------------------
    print(f"\n>>> [STAGE 1/4] Literature In-Domain: {display_name}...")
    exp_id, run_dir = create_hierarchical_run_directory(EXP_ROOT, model=display_name, setting="literature_in_domain", seed=seed)
    started_at = datetime.now(timezone.utc).isoformat()

    tagger = TransformerTagger(model_id=spec["hf_model_id"], is_iast=spec.get("is_iast", False)).to(DEVICE)
    best_tagger, val_metrics, _ = train_transformer(
        tagger=tagger, train_df=lit_train, val_df=lit_val, device=DEVICE, run_dir=run_dir, config=config
    )
    test_preds = best_tagger.predict_words(list(lit_test["tokens"]), device=DEVICE, batch_size=config.get("batch_size", 16))
    summary_lit, per_tag_lit, report_lit, conf_mat_lit = evaluate_predictions(lit_test, test_preds)
    save_evaluation_results(run_dir, summary_lit, per_tag_lit, report_lit, conf_mat_lit, test_preds, lit_test)
    append_registry(
        REGISTRY_PATH, experiment_id=exp_id, model=display_name, setting="literature_in_domain",
        annotation_budget=0, seed=seed, status="SUCCESS", run_dir=run_dir,
        validation_macro_f1=val_metrics.get("macro_f1"), test_macro_f1=summary_lit["macro_f1"],
        test_accuracy=summary_lit["accuracy"], test_weighted_f1=summary_lit["weighted_f1"], started_at=started_at
    )
    lit_chk_dir = run_dir / "checkpoint"
    print(f"Stage 1 Done: Lit Val F1={val_metrics.get('macro_f1', 0)*100:.2f}%, Lit Test Acc={summary_lit['accuracy']*100:.2f}%, Test Macro F1={summary_lit['macro_f1']*100:.2f}%")

    # Free GPU memory
    del tagger, best_tagger
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ----------------------------------------------------
    # STAGE 2: News In-Domain Reference Benchmark
    # ----------------------------------------------------
    print(f"\n>>> [STAGE 2/4] News In-Domain Reference: {display_name}...")
    exp_id, run_dir = create_hierarchical_run_directory(EXP_ROOT, model=display_name, setting="news_in_domain", seed=seed)
    started_at = datetime.now(timezone.utc).isoformat()

    news_tagger = TransformerTagger(model_id=spec["hf_model_id"], is_iast=spec.get("is_iast", False)).to(DEVICE)
    best_news_tagger, news_val_metrics, _ = train_transformer(
        tagger=news_tagger, train_df=news_pool, val_df=news_val, device=DEVICE, run_dir=run_dir, config=config
    )
    test_preds = best_news_tagger.predict_words(list(news_test["tokens"]), device=DEVICE, batch_size=config.get("batch_size", 16))
    summary_news, per_tag_news, report_news, conf_mat_news = evaluate_predictions(news_test, test_preds)
    save_evaluation_results(run_dir, summary_news, per_tag_news, report_news, conf_mat_news, test_preds, news_test)
    append_registry(
        REGISTRY_PATH, experiment_id=exp_id, model=display_name, setting="news_in_domain",
        annotation_budget=0, seed=seed, status="SUCCESS", run_dir=run_dir,
        validation_macro_f1=news_val_metrics.get("macro_f1"), test_macro_f1=summary_news["macro_f1"],
        test_accuracy=summary_news["accuracy"], test_weighted_f1=summary_news["weighted_f1"], started_at=started_at
    )
    print(f"Stage 2 Done: News Val F1={news_val_metrics.get('macro_f1', 0)*100:.2f}%, News Test Acc={summary_news['accuracy']*100:.2f}%, Test Macro F1={summary_news['macro_f1']*100:.2f}%")

    # Free GPU memory
    del news_tagger, best_news_tagger
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ----------------------------------------------------
    # STAGE 3: Zero-Shot Cross-Domain Generalization
    # ----------------------------------------------------
    print(f"\n>>> [STAGE 3/4] Zero-Shot Cross-Domain (Literature -> News Locked Test): {display_name}...")
    exp_id, run_dir = create_hierarchical_run_directory(EXP_ROOT, model=display_name, setting="zero_shot", seed=seed)
    started_at = datetime.now(timezone.utc).isoformat()

    lit_loaded = TransformerTagger.load(lit_chk_dir, device=DEVICE)
    test_preds = lit_loaded.predict_words(list(news_test["tokens"]), device=DEVICE, batch_size=config.get("batch_size", 16))
    summary_zs, per_tag_zs, report_zs, conf_mat_zs = evaluate_predictions(news_test, test_preds)
    save_evaluation_results(run_dir, summary_zs, per_tag_zs, report_zs, conf_mat_zs, test_preds, news_test)
    append_registry(
        REGISTRY_PATH, experiment_id=exp_id, model=display_name, setting="zero_shot",
        annotation_budget=0, seed=seed, status="SUCCESS", run_dir=run_dir,
        validation_macro_f1=None, test_macro_f1=summary_zs["macro_f1"],
        test_accuracy=summary_zs["accuracy"], test_weighted_f1=summary_zs["weighted_f1"], started_at=started_at
    )
    abs_drop = summary_lit["macro_f1"] - summary_zs["macro_f1"]
    rel_drop = (abs_drop / summary_lit["macro_f1"]) * 100 if summary_lit["macro_f1"] else 0
    print(f"Stage 3 Done: Zero-Shot News Acc={summary_zs['accuracy']*100:.2f}%, Test Macro F1={summary_zs['macro_f1']*100:.2f}%")
    print(f"   Domain Degradation: Absolute Drop = -{abs_drop*100:.2f} pp, Relative Drop = -{rel_drop:.2f}%")

    # Free GPU memory
    del lit_loaded
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ----------------------------------------------------
    # STAGE 4: Few-Shot Target Domain Adaptation
    # ----------------------------------------------------
    print(f"\n>>> [STAGE 4/4] Few-Shot Adaptation (Budgets: {budgets}): {display_name}...")
    for s in seeds:
        nested_subsets = get_nested_few_shot_subsets(news_pool, budgets=budgets, seed=s)
        for b in budgets:
            sub_df = nested_subsets[b]
            token_count = sum(len(t) for t in sub_df["tokens"])
            exp_id, run_dir = create_hierarchical_run_directory(EXP_ROOT, model=display_name, setting="few_shot", annotation_budget=b, seed=s)
            started_at = datetime.now(timezone.utc).isoformat()

            torch.manual_seed(s)
            ft_tagger = TransformerTagger.load(lit_chk_dir, device=DEVICE)
            ft_config = copy.deepcopy(config)
            ft_config["learning_rate"] = config.get("learning_rate", 3e-5) * 0.7
            ft_config["epochs"] = 6
            ft_config["early_stopping_patience"] = 2

            best_ft, ft_val, _ = train_transformer(tagger=ft_tagger, train_df=sub_df, val_df=news_val, device=DEVICE, run_dir=run_dir, config=ft_config)
            test_preds = best_ft.predict_words(list(news_test["tokens"]), device=DEVICE, batch_size=config.get("batch_size", 16))
            summary_fs, per_tag_fs, report_fs, conf_mat_fs = evaluate_predictions(news_test, test_preds)
            save_evaluation_results(run_dir, summary_fs, per_tag_fs, report_fs, conf_mat_fs, test_preds, news_test)
            append_registry(
                REGISTRY_PATH, experiment_id=exp_id, model=display_name, setting="few_shot",
                annotation_budget=b, seed=s, status="SUCCESS", run_dir=run_dir,
                validation_macro_f1=ft_val.get("macro_f1"), test_macro_f1=summary_fs["macro_f1"],
                test_accuracy=summary_fs["accuracy"], test_weighted_f1=summary_fs["weighted_f1"], started_at=started_at
            )
            gain = (summary_fs["macro_f1"] - summary_zs["macro_f1"]) * 100
            print(f"   --> Budget {b:4d} sent ({token_count:5d} tokens): Val F1={ft_val.get('macro_f1', 0)*100:.2f}%, Test Macro F1={summary_fs['macro_f1']*100:.2f}%, Acc={summary_fs['accuracy']*100:.2f}% (Gain: +{gain:.2f} pp)")

            # Clean GPU memory per budget run
            del ft_tagger, best_ft
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\nALL 4 EXPERIMENTAL STAGES COMPLETED FOR {display_name}!")

# Execute models sequentially
for model_key in MODELS_TO_RUN:
    run_full_pipeline_for_model(model_key)

print()
print("=" * 75)
print("ALL REQUESTED MODELS FINISHED! GENERATING CONSOLIDATED BENCHMARK TABLES")
print("=" * 75)

BASELINE_OUT_DIR = PROJECT_DIR / "outputs" / "baseline_experiments"
BASELINE_OUT_DIR.mkdir(parents=True, exist_ok=True)
EVALS_DIR = PROJECT_DIR / "outputs" / "evaluations"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

if REGISTRY_PATH.is_file():
    reg_df = pd.read_csv(REGISTRY_PATH)

    # 1. In-Domain Table
    in_domain_df = reg_df[reg_df["setting"].isin(["literature_in_domain", "news_in_domain"]) & (reg_df["status"] == "SUCCESS")].copy()
    in_domain_summary = []
    for m_name in in_domain_df["model"].unique():
        m_runs = in_domain_df[in_domain_df["model"] == m_name]
        lit_run = m_runs[m_runs["setting"] == "literature_in_domain"]
        news_run = m_runs[m_runs["setting"] == "news_in_domain"]
        in_domain_summary.append({
            "Model": m_name,
            "Literature_Val_Macro_F1": lit_run["validation_macro_f1"].values[0] if not lit_run.empty else None,
            "Literature_Test_Macro_F1": lit_run["test_macro_f1"].values[0] if not lit_run.empty else None,
            "Literature_Test_Accuracy": lit_run["test_accuracy"].values[0] if not lit_run.empty else None,
            "News_Val_Macro_F1": news_run["validation_macro_f1"].values[0] if not news_run.empty else None,
            "News_Test_Macro_F1": news_run["test_macro_f1"].values[0] if not news_run.empty else None,
            "News_Test_Accuracy": news_run["test_accuracy"].values[0] if not news_run.empty else None,
        })
    t1_df = pd.DataFrame(in_domain_summary)
    t1_df.to_csv(BASELINE_OUT_DIR / "literature_in_domain_results.csv", index=False)
    t1_df.to_csv(EVALS_DIR / "table1_in_domain.csv", index=False)

    # 2. Zero-Shot Cross-Domain Table
    zs_df = reg_df[(reg_df["setting"] == "zero_shot") & (reg_df["status"] == "SUCCESS")].copy()
    zs_summary = []
    for _, row in zs_df.iterrows():
        m_name = row["model"]
        lit_match = in_domain_df[(in_domain_df["model"] == m_name) & (in_domain_df["setting"] == "literature_in_domain")]
        lit_f1 = lit_match["test_macro_f1"].values[0] if not lit_match.empty else None
        zs_f1 = row["test_macro_f1"]
        abs_drop = (lit_f1 - zs_f1) if (lit_f1 is not None and zs_f1 is not None) else None
        rel_drop = (abs_drop / lit_f1 * 100) if (lit_f1 and abs_drop is not None) else None
        zs_summary.append({
            "Model": m_name,
            "Literature_InDomain_Macro_F1": lit_f1,
            "News_ZeroShot_Macro_F1": zs_f1,
            "Absolute_F1_Drop": abs_drop,
            "Relative_F1_Drop_Pct": rel_drop,
            "ZeroShot_Accuracy": row["test_accuracy"],
            "ZeroShot_Weighted_F1": row["test_weighted_f1"],
        })
    t2_df = pd.DataFrame(zs_summary)
    t2_df.to_csv(BASELINE_OUT_DIR / "cross_domain_zero_shot_results.csv", index=False)
    t2_df.to_csv(EVALS_DIR / "table2_zero_shot.csv", index=False)

    # 3. Few-Shot Adaptation Progression Table
    fs_df = reg_df[reg_df["setting"].isin(["few_shot", "zero_shot"]) & (reg_df["status"] == "SUCCESS")].copy()
    fs_df.loc[fs_df["setting"] == "zero_shot", "annotation_budget"] = 0
    t3_pivot = fs_df.pivot_table(
        index="model",
        columns="annotation_budget",
        values="test_macro_f1",
        aggfunc="mean"
    ).reset_index()
    t3_pivot.to_csv(BASELINE_OUT_DIR / "few_shot_results.csv", index=False)
    t3_pivot.to_csv(EVALS_DIR / "table3_few_shot.csv", index=False)

    # 4. Master Excel Workbook
    try:
        import openpyxl
        excel_path = BASELINE_OUT_DIR / "baseline_experiment_results.xlsx"
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            t1_df.to_excel(writer, sheet_name="In_Domain_Benchmarks", index=False)
            t2_df.to_excel(writer, sheet_name="Zero_Shot_Degradation", index=False)
            t3_pivot.to_excel(writer, sheet_name="Few_Shot_Progression", index=False)
            reg_df.to_excel(writer, sheet_name="Experiment_Registry", index=False)
        print(f"Generated comparison tables and Excel workbook at: {excel_path}\n")
    except Exception as ex:
        print("Note: Excel export skipped (openpyxl):", ex)

    print("=== Table 1: In-Domain Performance ===")
    display(t1_df)
    print("\n=== Table 2: Zero-Shot Cross-Domain Generalization ===")
    display(t2_df)
    print("\n=== Table 3: Few-Shot Adaptation Progression ===")
    display(t3_pivot)

